In [1]:
import os
import sys
import warnings
import numpy as np
import pandas as pd
from scipy import linalg
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import accuracy_score
import statsmodels.formula.api as smf

warnings.filterwarnings("ignore")

# ==============================================================================
# CONFIGURATION & SUBJECT LIST
# ==============================================================================
BASE_DIR = "datasets/DB2"
SUBJECTS = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12]  # Subject 11 excluded
WINDOW_SIZES_MS = [30, 40, 50]                  # Analysis windows to evaluate
FIXED_THRESHOLD_DM = 3.0                        # Critical Mahalanobis threshold

OUTPUT_DIR = os.path.join(BASE_DIR, "analysis_results_angle_multi_window")
os.makedirs(OUTPUT_DIR, exist_ok=True)
LOG_FILE = os.path.join(OUTPUT_DIR, "angle_multi_window_analysis_log.txt")


class Logger:
    """Tees stdout to both terminal and a persistent log file."""
    def __init__(self, filepath):
        self.terminal = sys.stdout
        self.logfile = open(filepath, "w", encoding="utf-8")

    def write(self, message):
        self.terminal.write(message)
        self.logfile.write(message)

    def flush(self):
        self.terminal.flush()
        self.logfile.flush()


sys.stdout = Logger(LOG_FILE)


# ==============================================================================
# 1. DATA INGESTION & CONSOLIDATION
# ==============================================================================
print("=" * 90)
print(f"INGESTING DATA ({len(SUBJECTS)} SUBJECTS | ANGLE-DERIVED ONSETS | WINDOWS: {WINDOW_SIZES_MS} ms)")
print("=" * 90)

dfs = []
for s in SUBJECTS:
    candidates = [
        os.path.join(BASE_DIR, f"DB2_s{s}", f"S{s}_E1_A1_pruned_emg_features.csv"),
        os.path.join(BASE_DIR, f"s{s}", f"S{s}_E1_A1_pruned_emg_features.csv"),
    ]
    csv_path = next((p for p in candidates if os.path.isfile(p)), None)

    if csv_path is None:
        print(f"[WARN] No feature CSV found for Subject {s}. Skipping.")
        continue

    df_sub = pd.read_csv(csv_path)
    df_sub["subject"] = s
    dfs.append(df_sub)

if not dfs:
    raise FileNotFoundError(f"No feature files found in {BASE_DIR}. Check directory structure.")

raw_df = pd.concat(dfs, ignore_index=True)

# Sift out only angle epochs and quiet baseline
target_epochs = ["rest_gray", "pre_tkin_angle", "post_tkin_angle"]
raw_df = raw_df[raw_df["epoch"].isin(target_epochs)].copy()

meta_cols = ["repetition", "movement", "epoch", "window_ms", "idx_start", "idx_end", "subject"]
feature_cols = [c for c in raw_df.columns if c not in meta_cols and pd.api.types.is_numeric_dtype(raw_df[c])]

print(f"Total rows ingested   : {len(raw_df)}")
print(f"Usable feature metrics: {len(feature_cols)}")
print(f"Active subjects       : {sorted(raw_df['subject'].unique().tolist())}\n")


# ==============================================================================
# PIPELINE RUNNER FOR A SPECIFIC WINDOW DURATION
# ==============================================================================
def run_angle_window_pipeline(win_ms, df_all):
    label = f"EVALUATION WINDOW: {win_ms} ms (t_kin ANGLE)"
    print("=" * 90)
    print(f"{label:^90}")
    print("=" * 90)

    df_win = df_all[df_all["window_ms"] == win_ms].copy()
    pre_epoch = "pre_tkin_angle"
    post_epoch = "post_tkin_angle"

    # --------------------------------------------------------------------------
    # PHASE 1: Population Feature Screening (LME + Cohen's d + Directional Ratio)
    # --------------------------------------------------------------------------
    screening_records = []
    df_rest = df_win[df_win["epoch"] == "rest_gray"].dropna(subset=["edc_RMS"])
    df_pre = df_win[df_win["epoch"] == pre_epoch].dropna(subset=["edc_RMS"])
    df_post = df_win[df_win["epoch"] == post_epoch].dropna(subset=["edc_RMS"])

    for feat in feature_cols:
        # A. Subject-Normalized Cohen's d
        cohens_pre, cohens_post = [], []
        for s in SUBJECTS:
            s_rest = df_rest[df_rest["subject"] == s][feat].dropna()
            s_pre = df_pre[df_pre["subject"] == s][feat].dropna()
            s_post = df_post[df_post["subject"] == s][feat].dropna()

            if len(s_rest) > 1 and len(s_pre) > 1:
                pooled_std = np.sqrt((s_rest.var() + s_pre.var()) / 2.0)
                if pooled_std > 1e-8:
                    cohens_pre.append((s_pre.mean() - s_rest.mean()) / pooled_std)

            if len(s_rest) > 1 and len(s_post) > 1:
                pooled_std = np.sqrt((s_rest.var() + s_post.var()) / 2.0)
                if pooled_std > 1e-8:
                    cohens_post.append((s_post.mean() - s_rest.mean()) / pooled_std)

        med_d_pre = float(np.median(cohens_pre)) if cohens_pre else 0.0
        med_d_post = float(np.median(cohens_post)) if cohens_post else 0.0

        # B. Linear Mixed-Effects Model: feat ~ epoch (fixed) + (1 | subject)
        sub_lme = df_win[df_win["epoch"].isin(["rest_gray", pre_epoch])].dropna(subset=[feat, "subject", "epoch"]).copy()
        t_stat_lme, p_val_lme = 0.0, 1.0

        if len(sub_lme["subject"].unique()) > 1:
            try:
                sub_lme["feat_scaled"] = sub_lme.groupby("subject")[feat].transform(
                    lambda x: (x - x.mean()) / (x.std() + 1e-8)
                )
                model = smf.mixedlm(
                    "feat_scaled ~ C(epoch, Treatment('rest_gray'))",
                    data=sub_lme,
                    groups="subject"
                )
                res = model.fit(disp=False)
                t_cols = [c for c in res.tvalues.index if pre_epoch in c]
                if t_cols and not np.isnan(res.tvalues[t_cols[0]]):
                    t_stat_lme = float(res.tvalues[t_cols[0]])
                    p_val_lme = float(res.pvalues[t_cols[0]])
            except Exception:
                # Clustered OLS fallback if mixedlm Hessian is non-invertible
                try:
                    ols_mod = smf.ols("feat_scaled ~ C(epoch, Treatment('rest_gray'))", data=sub_lme).fit(
                        cov_type='cluster', cov_kwds={'groups': sub_lme['subject']}
                    )
                    t_cols = [c for c in ols_mod.tvalues.index if pre_epoch in c]
                    if t_cols:
                        t_stat_lme = float(ols_mod.tvalues[t_cols[0]])
                        p_val_lme = float(ols_mod.pvalues[t_cols[0]])
                except Exception:
                    t_stat_lme, p_val_lme = 0.0, 1.0

        # C. Pre-Kinematic Directional Fisher Separability (M5 vs M6)
        m5_vals = df_pre[df_pre["movement"] == 5][feat].dropna()
        m6_vals = df_pre[df_pre["movement"] == 6][feat].dropna()
        fisher_dir = 0.0
        if len(m5_vals) > 1 and len(m6_vals) > 1:
            var_sum = m5_vals.var() + m6_vals.var()
            if var_sum > 1e-8:
                fisher_dir = (m5_vals.mean() - m6_vals.mean()) ** 2 / var_sum

        screening_records.append({
            "feature": feat,
            "cohen_d_pre": med_d_pre,
            "cohen_d_post": med_d_post,
            "lme_t_stat": t_stat_lme,
            "lme_p_val": p_val_lme,
            "dir_fisher_ratio": fisher_dir
        })

    scr_df = pd.DataFrame(screening_records)
    scr_df["composite_score"] = (
        scr_df["cohen_d_pre"].abs() * 0.4 +
        scr_df["lme_t_stat"].abs() * 0.3 +
        scr_df["dir_fisher_ratio"] * 0.3
    )
    scr_df = scr_df.sort_values("composite_score", ascending=False).reset_index(drop=True)
    scr_df.to_csv(os.path.join(OUTPUT_DIR, f"feature_screening_{win_ms}ms.csv"), index=False)

    print(f"\n[PHASE 1: TOP 5 CANDIDATE METRICS - {win_ms} ms]")
    print(f"{'Feature':<28} {'d(Pre)':<10} {'d(Post)':<10} {'LME t-stat':<12} {'Fisher Dir':<10}")
    print("-" * 74)
    for _, r in scr_df.head(5).iterrows():
        print(f"{r['feature']:<28} {r['cohen_d_pre']:<10.2f} {r['cohen_d_post']:<10.2f} "
              f"{r['lme_t_stat']:<12.2f} {r['dir_fisher_ratio']:<10.2f}")
    print()

    # Select orthogonal subspace triplet: Relational, Extensor, Flexor
    chosen_features = ["antagonist_diff_ratio"]
    edc_cand = scr_df[scr_df["feature"].str.startswith("edc_")]["feature"].tolist()
    fds_cand = scr_df[scr_df["feature"].str.startswith("fds_")]["feature"].tolist()
    if edc_cand: chosen_features.append(edc_cand[0])
    if fds_cand: chosen_features.append(fds_cand[0])

    print(f"Selected Subspace Triplet ({win_ms} ms): {chosen_features}\n")

    # --------------------------------------------------------------------------
    # PHASE 2: State Detection via Whitened Mahalanobis Distance (D_M)
    # --------------------------------------------------------------------------
    dm_records = []
    for s in SUBJECTS:
        sub_df = df_win[df_win["subject"] == s]
        rest_mat = sub_df[sub_df["epoch"] == "rest_gray"][chosen_features].dropna().values
        pre_mat = sub_df[sub_df["epoch"] == pre_epoch][chosen_features].dropna().values
        post_mat = sub_df[sub_df["epoch"] == post_epoch][chosen_features].dropna().values

        if len(rest_mat) < len(chosen_features) + 1:
            continue

        mu_rest = np.mean(rest_mat, axis=0)
        cov_rest = np.cov(rest_mat, rowvar=False) + 1e-4 * np.eye(len(chosen_features))
        inv_cov = linalg.inv(cov_rest)

        def mahal_dist(X):
            diff = X - mu_rest
            return np.sqrt(np.sum(diff @ inv_cov * diff, axis=1))

        dm_rest = mahal_dist(rest_mat)
        dm_pre = mahal_dist(pre_mat) if len(pre_mat) > 0 else np.array([np.nan])
        dm_post = mahal_dist(post_mat) if len(post_mat) > 0 else np.array([np.nan])

        dm_records.append({
            "subject": s,
            "mean_dm_rest": np.mean(dm_rest),
            "mean_dm_pre": np.nanmean(dm_pre),
            "mean_dm_post": np.nanmean(dm_post),
            "pre_detect_rate": np.nanmean(dm_pre > FIXED_THRESHOLD_DM) * 100.0,
            "post_detect_rate": np.nanmean(dm_post > FIXED_THRESHOLD_DM) * 100.0
        })

    dm_df = pd.DataFrame(dm_records)
    dm_df.to_csv(os.path.join(OUTPUT_DIR, f"mahalanobis_detection_{win_ms}ms.csv"), index=False)

    print(f"[PHASE 2: STATE DEVIATION - {win_ms} ms]")
    print(f"Rest Noise Baseline D_M : {dm_df['mean_dm_rest'].mean():.2f}")
    print(f"Pre-Kinematic Mean D_M  : {dm_df['mean_dm_pre'].mean():.2f} (Detection Rate @ >3.0: {dm_df['pre_detect_rate'].mean():.1f}%)")
    print(f"Post-Kinematic Mean D_M : {dm_df['mean_dm_post'].mean():.2f} (Detection Rate @ >3.0: {dm_df['post_detect_rate'].mean():.1f}%)\n")

    # --------------------------------------------------------------------------
    # PHASE 3: Directional Decoding via Shrinkage LDA & LOSO-CV
    # --------------------------------------------------------------------------
    paired_pre = df_win[df_win["epoch"] == pre_epoch].copy()
    paired_post = df_win[df_win["epoch"] == post_epoch].copy()

    paired = pd.merge(
        paired_pre, paired_post,
        on=["subject", "repetition", "movement"],
        suffixes=("_pre", "_post")
    ).dropna(subset=[f"{c}_pre" for c in chosen_features] + [f"{c}_post" for c in chosen_features])

    X_pre = paired[[f"{c}_pre" for c in chosen_features]].values
    X_post = paired[[f"{c}_post" for c in chosen_features]].values
    X_delta = X_post - X_pre
    y = paired["movement"].values
    subjs = paired["subject"].values

    loso_records = []
    for s_out in np.unique(subjs):
        train_idx = subjs != s_out
        test_idx = subjs == s_out
        y_train, y_test = y[train_idx], y[test_idx]

        accs = {}
        for mode, X_mat in [("pre", X_pre), ("post", X_post), ("delta", X_delta)]:
            clf = LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto")
            clf.fit(X_mat[train_idx], y_train)
            preds = clf.predict(X_mat[test_idx])
            accs[mode] = accuracy_score(y_test, preds) * 100.0

        loso_records.append({
            "subject": s_out,
            "trials": len(y_test),
            "acc_pre": accs["pre"],
            "acc_post": accs["post"],
            "acc_delta": accs["delta"]
        })

    loso_df = pd.DataFrame(loso_records)
    loso_df.to_csv(os.path.join(OUTPUT_DIR, f"loso_lda_decoding_{win_ms}ms.csv"), index=False)

    print(f"[PHASE 3: LOSO-CV DIRECTIONAL DECODING ACCURACY (M5 vs M6) - {win_ms} ms]")
    print(f"Pre-Kinematic Window  : {loso_df['acc_pre'].mean():.1f}% ± {loso_df['acc_pre'].std():.1f}%")
    print(f"Post-Kinematic Window : {loso_df['acc_post'].mean():.1f}% ± {loso_df['acc_post'].std():.1f}%")
    print(f"Rate-of-Rise (Delta)  : {loso_df['acc_delta'].mean():.1f}% ± {loso_df['acc_delta'].std():.1f}%\n")

    return {
        "win_ms": win_ms,
        "chosen_features": chosen_features,
        "mean_dm_rest": dm_df["mean_dm_rest"].mean(),
        "mean_dm_pre": dm_df["mean_dm_pre"].mean(),
        "mean_dm_post": dm_df["mean_dm_post"].mean(),
        "pre_detect_rate": dm_df["pre_detect_rate"].mean(),
        "post_detect_rate": dm_df["post_detect_rate"].mean(),
        "acc_pre": loso_df["acc_pre"].mean(),
        "acc_pre_std": loso_df["acc_pre"].std(),
        "acc_post": loso_df["acc_post"].mean(),
        "acc_post_std": loso_df["acc_post"].std(),
        "acc_delta": loso_df["acc_delta"].mean(),
        "acc_delta_std": loso_df["acc_delta"].std(),
    }


# ==============================================================================
# RUN ACROSS 30 ms, 40 ms, AND 50 ms WINDOWS
# ==============================================================================
window_results = {}
for w in WINDOW_SIZES_MS:
    window_results[w] = run_angle_window_pipeline(w, raw_df)

# ==============================================================================
# COMPILE MULTI-WINDOW COMPARISON TABLE
# ==============================================================================
print("=" * 90)
print("MULTI-WINDOW HEAD-TO-HEAD COMPARISON (t_kin ANGLE)")
print("=" * 90)

comp_rows = [
    {
        "Evaluation Dimension": "Subspace Triplet Selected",
        "30 ms Window": ", ".join(window_results[30]["chosen_features"]),
        "40 ms Window": ", ".join(window_results[40]["chosen_features"]),
        "50 ms Window": ", ".join(window_results[50]["chosen_features"]),
    },
    {
        "Evaluation Dimension": "Rest Noise Baseline (D_M)",
        "30 ms Window": f"{window_results[30]['mean_dm_rest']:.2f}",
        "40 ms Window": f"{window_results[40]['mean_dm_rest']:.2f}",
        "50 ms Window": f"{window_results[50]['mean_dm_rest']:.2f}",
    },
    {
        "Evaluation Dimension": "Pre-Window Departure (Mean D_M)",
        "30 ms Window": f"{window_results[30]['mean_dm_pre']:.2f}",
        "40 ms Window": f"{window_results[40]['mean_dm_pre']:.2f}",
        "50 ms Window": f"{window_results[50]['mean_dm_pre']:.2f}",
    },
    {
        "Evaluation Dimension": "Pre-Window Detection Rate (D_M > 3.0)",
        "30 ms Window": f"{window_results[30]['pre_detect_rate']:.1f}%",
        "40 ms Window": f"{window_results[40]['pre_detect_rate']:.1f}%",
        "50 ms Window": f"{window_results[50]['pre_detect_rate']:.1f}%",
    },
    {
        "Evaluation Dimension": "Post-Window Detection Rate (D_M > 3.0)",
        "30 ms Window": f"{window_results[30]['post_detect_rate']:.1f}%",
        "40 ms Window": f"{window_results[40]['post_detect_rate']:.1f}%",
        "50 ms Window": f"{window_results[50]['post_detect_rate']:.1f}%",
    },
    {
        "Evaluation Dimension": "Zero-Latency Decoding (Pre-tkin LOSO Acc)",
        "30 ms Window": f"{window_results[30]['acc_pre']:.1f}% ± {window_results[30]['acc_pre_std']:.1f}%",
        "40 ms Window": f"{window_results[40]['acc_pre']:.1f}% ± {window_results[40]['acc_pre_std']:.1f}%",
        "50 ms Window": f"{window_results[50]['acc_pre']:.1f}% ± {window_results[50]['acc_pre_std']:.1f}%",
    },
    {
        "Evaluation Dimension": "Post-Latency Decoding (Post-tkin LOSO Acc)",
        "30 ms Window": f"{window_results[30]['acc_post']:.1f}% ± {window_results[30]['acc_post_std']:.1f}%",
        "40 ms Window": f"{window_results[40]['acc_post']:.1f}% ± {window_results[40]['acc_post_std']:.1f}%",
        "50 ms Window": f"{window_results[50]['acc_post']:.1f}% ± {window_results[50]['acc_post_std']:.1f}%",
    },
    {
        "Evaluation Dimension": "Dynamic Rate-of-Rise (Delta LOSO Acc)",
        "30 ms Window": f"{window_results[30]['acc_delta']:.1f}% ± {window_results[30]['acc_delta_std']:.1f}%",
        "40 ms Window": f"{window_results[40]['acc_delta']:.1f}% ± {window_results[40]['acc_delta_std']:.1f}%",
        "50 ms Window": f"{window_results[50]['acc_delta']:.1f}% ± {window_results[50]['acc_delta_std']:.1f}%",
    },
]

comp_df = pd.DataFrame(comp_rows)
comp_csv = os.path.join(OUTPUT_DIR, "multi_window_head_to_head_comparison.csv")
comp_df.to_csv(comp_csv, index=False)

print(f"{'Evaluation Dimension':<42} | {'30 ms Window':<15} | {'40 ms Window':<15} | {'50 ms Window':<15}")
print("-" * 94)
for _, r in comp_df.iterrows():
    print(f"{r['Evaluation Dimension']:<42} | {r['30 ms Window']:<15} | {r['40 ms Window']:<15} | {r['50 ms Window']:<15}")
print("=" * 94)
print(f"All tables, window-specific CSVs, and logs saved in: {OUTPUT_DIR}")

INGESTING DATA (11 SUBJECTS | ANGLE-DERIVED ONSETS | WINDOWS: [30, 40, 50] ms)
Total rows ingested   : 1188
Usable feature metrics: 42
Active subjects       : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12]

                          EVALUATION WINDOW: 30 ms (t_kin ANGLE)                          

[PHASE 1: TOP 5 CANDIDATE METRICS - 30 ms]
Feature                      d(Pre)     d(Post)    LME t-stat   Fisher Dir
--------------------------------------------------------------------------
edc_Hjorth_Mobility          0.62       0.65       6.88         0.11      
edc_WL                       0.62       0.64       6.28         0.09      
edc_Hjorth_Complexity        -0.73      -0.53      -5.77        0.13      
fds_LOG                      0.89       1.10       5.52         0.00      
edc_RMS                      0.68       0.60       5.77         0.00      

Selected Subspace Triplet (30 ms): ['antagonist_diff_ratio', 'edc_Hjorth_Mobility', 'fds_LOG']

[PHASE 2: STATE DEVIATION - 30 ms]
Rest Noise B

In [ ]:
import os
import sys
import warnings
import numpy as np
import pandas as pd
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import accuracy_score

warnings.filterwarnings("ignore")

# ==============================================================================
# CONFIGURATION
# ==============================================================================
BASE_DIR = "datasets/DB2"
SUBJECTS = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12]  # Discard S11
TARGET_WIN_MS = 40                              # Focused specifically on 40 ms
TARGET_EPOCH_POST = "post_tkin_angle"
TARGET_EPOCH_PRE = "pre_tkin_angle"

OUTPUT_DIR = os.path.join(BASE_DIR, "feature_triplet_variant_eval")
os.makedirs(OUTPUT_DIR, exist_ok=True)
LOG_FILE = os.path.join(OUTPUT_DIR, "variant_comparison_log.txt")


class Logger:
    """Tees stdout to both terminal and a persistent log file."""
    def __init__(self, filepath):
        self.terminal = sys.stdout
        self.logfile = open(filepath, "w", encoding="utf-8")

    def write(self, message):
        self.terminal.write(message)
        self.logfile.write(message)

    def flush(self):
        self.terminal.flush()
        self.logfile.flush()


sys.stdout = Logger(LOG_FILE)

# ==============================================================================
# FEATURE SET VARIANTS TO EVALUATE
# ==============================================================================
FEATURE_VARIANTS = {
    "Baseline Triplet": [
        "antagonist_diff_ratio",
        "edc_Hjorth_Mobility",
        "fds_LOG"
    ],
    "Variant 1 (Swap LOG -> WAMP)": [
        "antagonist_diff_ratio",
        "edc_Hjorth_Mobility",
        "fds_WAMP"
    ],
    "Variant 2 (Add Complexity to Baseline)": [
        "antagonist_diff_ratio",
        "edc_Hjorth_Mobility",
        "fds_LOG",
        "edc_Hjorth_Complexity"
    ],
    "Variant 3 (WAMP + Complexity Quad)": [
        "antagonist_diff_ratio",
        "edc_Hjorth_Mobility",
        "fds_WAMP",
        "edc_Hjorth_Complexity"
    ],
    "Variant 4 (Swap Mobility -> WL)": [
        "antagonist_diff_ratio",
        "edc_WL",
        "fds_LOG"
    ],
    "Variant 5 (Top-Tier 5-Feature Subspace)": [
        "antagonist_diff_ratio",
        "edc_Hjorth_Mobility",
        "edc_Hjorth_Complexity",
        "edc_WL",
        "fds_WAMP"
    ]
}

# ==============================================================================
# 1. DATA INGESTION & MATCHING
# ==============================================================================
print("=" * 95)
print(f"EVALUATING FEATURE VARIANTS | {len(SUBJECTS)} SUBJECTS | 40 ms POST-TAAttribute/ANGLE WINDOW")
print("=" * 95)

dfs = []
for s in SUBJECTS:
    candidates = [
        os.path.join(BASE_DIR, f"DB2_s{s}", f"S{s}_E1_A1_pruned_emg_features.csv"),
        os.path.join(BASE_DIR, f"s{s}", f"S{s}_E1_A1_pruned_emg_features.csv"),
    ]
    csv_path = next((p for p in candidates if os.path.isfile(p)), None)

    if csv_path is None:
        print(f"[WARN] File not found for Subject {s}. Skipping.")
        continue

    df_sub = pd.read_csv(csv_path)
    df_sub["subject"] = s
    dfs.append(df_sub)

if not dfs:
    raise FileNotFoundError(f"No feature CSV files found in {BASE_DIR}.")

raw_df = pd.concat(dfs, ignore_index=True)

# Filter by window length and target epochs
df_win = raw_df[
    (raw_df["window_ms"] == TARGET_WIN_MS) &
    (raw_df["epoch"].isin([TARGET_EPOCH_PRE, TARGET_EPOCH_POST]))
].copy()

# Pair pre and post epochs per trial to guarantee balanced trial counts
paired_pre = df_win[df_win["epoch"] == TARGET_EPOCH_PRE].copy()
paired_post = df_win[df_win["epoch"] == TARGET_EPOCH_POST].copy()

paired = pd.merge(
    paired_pre, paired_post,
    on=["subject", "repetition", "movement"],
    suffixes=("_pre", "_post")
)

print(f"Total matched trials available across 11 subjects: {len(paired)}")
print(f"Movements evaluated: {sorted(paired['movement'].unique().tolist())} (M5=Open, M6=Fist)\n")

# ==============================================================================
# 2. LEAVE-ONE-SUBJECT-OUT (LOSO-CV) LDA EVALUATION
# ==============================================================================
y = paired["movement"].values
subjects = paired["subject"].values
unique_subjs = np.unique(subjects)

variant_summary = []
per_subject_records = []

for variant_name, feat_list in FEATURE_VARIANTS.items():
    # Verify all features exist in dataset
    missing_feats = [f for f in feat_list if f"{f}_post" not in paired.columns]
    if missing_feats:
        print(f"[ERROR] Skipping '{variant_name}': Missing columns {missing_feats}")
        continue

    X_post = paired[[f"{f}_post" for f in feat_list]].values
    X_pre = paired[[f"{f}_pre" for f in feat_list]].values

    loso_accs_post = []
    loso_accs_pre = []

    for s_test in unique_subjs:
        train_mask = subjects != s_test
        test_mask = subjects == s_test

        y_train, y_test = y[train_mask], y[test_mask]

        # Fit regularized Shrinkage LDA
        clf_post = LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto")
        clf_post.fit(X_post[train_mask], y_train)
        acc_post = accuracy_score(y_test, clf_post.predict(X_post[test_mask])) * 100.0
        loso_accs_post.append(acc_post)

        clf_pre = LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto")
        clf_pre.fit(X_pre[train_mask], y_train)
        acc_pre = accuracy_score(y_test, clf_pre.predict(X_pre[test_mask])) * 100.0
        loso_accs_pre.append(acc_pre)

        per_subject_records.append({
            "variant": variant_name,
            "subject": s_test,
            "n_trials": len(y_test),
            "acc_pre": acc_pre,
            "acc_post": acc_post
        })

    mean_post = np.mean(loso_accs_post)
    std_post = np.std(loso_accs_post)
    mean_pre = np.mean(loso_accs_pre)
    std_pre = np.std(loso_accs_pre)

    n_above_70 = sum(a >= 70.0 for a in loso_accs_post)
    n_above_80 = sum(a >= 80.0 for a in loso_accs_post)

    variant_summary.append({
        "Variant Name": variant_name,
        "Features Used": ", ".join(feat_list),
        "Num Feats": len(feat_list),
        "Pre-Window Acc (Mean ± SD)": f"{mean_pre:.1f}% ± {std_pre:.1f}%",
        "Post-Window Acc (Mean ± SD)": f"{mean_post:.1f}% ± {std_post:.1f}%",
        "Mean Post Acc": mean_post,
        "Std Post Acc": std_post,
        "Subjects >= 70%": f"{n_above_70}/{len(unique_subjs)}",
        "Subjects >= 80%": f"{n_above_80}/{len(unique_subjs)}"
    })

# Save detailed per-subject runs
per_subj_df = pd.DataFrame(per_subject_records)
per_subj_df.to_csv(os.path.join(OUTPUT_DIR, "loso_per_subject_by_variant.csv"), index=False)

# ==============================================================================
# 3. RESULTS DISPLAY & BENCHMARKING
# ==============================================================================
summary_df = pd.DataFrame(variant_summary).sort_values("Mean Post Acc", ascending=False).reset_index(drop=True)
summary_df.to_csv(os.path.join(OUTPUT_DIR, "variant_summary_comparison.csv"), index=False)

print("=" * 95)
print("LOSO-CV DIRECTIONAL DECODING BENCHMARK (M5 vs M6 | 40 ms WINDOWS)")
print("=" * 95)
print(f"{'Variant':<36} | {'Features':<5} | {'Pre-tkin Acc':<16} | {'Post-tkin Acc':<16} | {'Subjs >=70%':<10}")
print("-" * 95)
for _, r in summary_df.iterrows():
    print(f"{r['Variant Name']:<36} | {r['Num Feats']:<5} | {r['Pre-Window Acc (Mean ± SD)']:<16} | "
          f"{r['Post-Window Acc (Mean ± SD)']:<16} | {r['Subjects >= 70%']:<10}")
print("=" * 95)

# Subject-by-Subject Pivot Table for Post-Window Accuracy
pivot_post = per_subj_df.pivot(index="subject", columns="variant", values="acc_post")
pivot_post.to_csv(os.path.join(OUTPUT_DIR, "subject_post_accuracy_matrix.csv"))

print("\nPER-SUBJECT POST-WINDOW (40 ms) ACCURACY MATRIX (%):")
print("-" * 95)
print(pivot_post.to_string())
print("-" * 95)

# Benchmark against the 70% threshold
top_variant = summary_df.iloc[0]
print(f"\nOUTCOME SUMMARY:")
print(f"Top Performing Subspace : {top_variant['Variant Name']}")
print(f"Feature Composition     : {top_variant['Features Used']}")
print(f"Post-Window Mean Acc    : {top_variant['Post-Window Acc (Mean ± SD)']}")
if top_variant['Mean Post Acc'] >= 70.0:
    print(f"[SUCCESS] Reached target ceiling: {top_variant['Mean Post Acc']:.1f}% >= 70.0%")
else:
    gap = 70.0 - top_variant['Mean Post Acc']
    print(f"[MARGINAL] Falls short of 70% population ceiling by {gap:.1f}% "
          f"(Subjects meeting >=70%: {top_variant['Subjects >= 70%']})")
print(f"\nAll comparison matrices and CSV logs saved to: {OUTPUT_DIR}")
print("=" * 95)

EVALUATING FEATURE VARIANTS | 11 SUBJECTS | 40 ms POST-TAAttribute/ANGLE WINDOW
Total matched trials available across 11 subjects: 132
Movements evaluated: [5, 6] (M5=Open, M6=Fist)

LOSO-CV DIRECTIONAL DECODING BENCHMARK (M5 vs M6 | 40 ms WINDOWS)
Variant                              | Features | Pre-tkin Acc     | Post-tkin Acc    | Subjs >=70%
-----------------------------------------------------------------------------------------------
Variant 2 (Add Complexity to Baseline) | 4     | 59.8% ± 16.2%    | 65.9% ± 17.6%    | 4/11      
Baseline Triplet                     | 3     | 59.8% ± 17.7%    | 65.2% ± 15.8%    | 4/11      
Variant 4 (Swap Mobility -> WL)      | 3     | 59.1% ± 18.6%    | 64.4% ± 15.9%    | 4/11      
Variant 1 (Swap LOG -> WAMP)         | 3     | 56.8% ± 15.8%    | 62.9% ± 14.8%    | 3/11      
Variant 3 (WAMP + Complexity Quad)   | 4     | 57.6% ± 17.2%    | 62.9% ± 16.0%    | 3/11      
Variant 5 (Top-Tier 5-Feature Subspace) | 5     | 57.6% ± 17.2%    | 62.9

FEW-SHOT CALIBRATION EVALUATION (1/6 TRIALS FOR TRAINING | 5/6 FOR TESTING)
Window: 40 ms | Epoch: post_tkin_angle | Subjects: 11
Total records ingested: 132
Movements: [5, 6] (M5=Open, M6=Fist)
Repetitions available: [1, 2, 3, 4, 5, 6]

-----------------------------------------------------------------------------------------------
EVALUATING: Variant 2 (Winning 4-Feat)
-----------------------------------------------------------------------------------------------
-----------------------------------------------------------------------------------------------
EVALUATING: Baseline Triplet (3-Feat)
-----------------------------------------------------------------------------------------------

CALIBRATION IMPACT BENCHMARK (1/6 TRIALS TRAIN vs 5/6 TRIALS TEST)
Feature Set                    : Variant 2 (Winning 4-Feat)
  • 0-Shot LOSO (Zero Calib)   : 65.9% ± 17.6%
  • 1-Shot Rep 1 Calib         : 64.5% ± 16.2%
  • 1-Shot 6-Fold Mean         : 64.8% ± 14.1%
  • Pooled Few-Shot LDA        :

In [3]:
import os
import sys
import warnings
import numpy as np
import pandas as pd
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import accuracy_score

warnings.filterwarnings("ignore")

# ==============================================================================
# CONFIGURATION
# ==============================================================================
BASE_DIR = "datasets/DB2"
SUBJECTS = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12]  # Discard S11
TARGET_WIN_MS = 40
TARGET_EPOCH = "post_tkin_angle"

OUTPUT_DIR = os.path.join(BASE_DIR, "calibration_1_6_eval")
os.makedirs(OUTPUT_DIR, exist_ok=True)
LOG_FILE = os.path.join(OUTPUT_DIR, "calibration_analysis_log.txt")


class Logger:
    """Tees stdout to terminal and log file."""
    def __init__(self, filepath):
        self.terminal = sys.stdout
        self.logfile = open(filepath, "w", encoding="utf-8")

    def write(self, message):
        self.terminal.write(message)
        self.logfile.write(message)

    def flush(self):
        self.terminal.flush()
        self.logfile.flush()


sys.stdout = Logger(LOG_FILE)

# Subspaces to evaluate
FEATURE_SETS = {
    "Variant 2 (Winning 4-Feat)": [
        "antagonist_diff_ratio",
        "edc_Hjorth_Mobility",
        "fds_LOG",
        "edc_Hjorth_Complexity"
    ],
    "Baseline Triplet (3-Feat)": [
        "antagonist_diff_ratio",
        "edc_Hjorth_Mobility",
        "fds_LOG"
    ]
}

# ==============================================================================
# 1. DATA INGESTION
# ==============================================================================
print("=" * 95)
print(f"FEW-SHOT CALIBRATION EVALUATION (1/6 TRIALS FOR TRAINING | 5/6 FOR TESTING)")
print(f"Window: {TARGET_WIN_MS} ms | Epoch: {TARGET_EPOCH} | Subjects: {len(SUBJECTS)}")
print("=" * 95)

dfs = []
for s in SUBJECTS:
    candidates = [
        os.path.join(BASE_DIR, f"DB2_s{s}", f"S{s}_E1_A1_pruned_emg_features.csv"),
        os.path.join(BASE_DIR, f"s{s}", f"S{s}_E1_A1_pruned_emg_features.csv"),
    ]
    csv_path = next((p for p in candidates if os.path.isfile(p)), None)
    if csv_path is None:
        print(f"[WARN] File not found for Subject {s}. Skipping.")
        continue

    df_sub = pd.read_csv(csv_path)
    df_sub["subject"] = s
    dfs.append(df_sub)

if not dfs:
    raise FileNotFoundError(f"No feature CSV files found in {BASE_DIR}.")

raw_df = pd.concat(dfs, ignore_index=True)
df = raw_df[(raw_df["window_ms"] == TARGET_WIN_MS) & (raw_df["epoch"] == TARGET_EPOCH)].copy()

print(f"Total records ingested: {len(df)}")
print(f"Movements: {sorted(df['movement'].unique().tolist())} (M5=Open, M6=Fist)")
print(f"Repetitions available: {sorted(df['repetition'].unique().tolist())}\n")

# ==============================================================================
# 2. CALIBRATION & ADAPTATION PIPELINE
# ==============================================================================
results_summary = []
subject_matrix_records = []

for set_name, feats in FEATURE_SETS.items():
    print("-" * 95)
    print(f"EVALUATING: {set_name}")
    print("-" * 95)

    accs_0shot = []
    accs_rep1_adapt = []
    accs_6fold_adapt = []
    accs_pooled = []

    for s_test in SUBJECTS:
        train_pop = df[df["subject"] != s_test].dropna(subset=feats)
        test_subj = df[df["subject"] == s_test].dropna(subset=feats)

        # ----------------------------------------------------------------------
        # A. 0-Shot Baseline (Standard LOSO)
        # ----------------------------------------------------------------------
        clf_base = LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto")
        clf_base.fit(train_pop[feats], train_pop["movement"])
        preds_0shot = clf_base.predict(test_subj[feats])
        acc_0shot = accuracy_score(test_subj["movement"], preds_0shot) * 100.0
        accs_0shot.append(acc_0shot)

        # ----------------------------------------------------------------------
        # B. 1-Shot Clinical Protocol (Calibrate on Rep 1, Test on Reps 2-6)
        # ----------------------------------------------------------------------
        cal_rep1 = test_subj[test_subj["repetition"] == 1]
        eval_rep26 = test_subj[test_subj["repetition"] != 1]

        # Population orientation vector
        w = clf_base.coef_[0]

        # Adapt midpoint and verify directional polarity
        x5_rep1 = cal_rep1[cal_rep1["movement"] == 5][feats].values
        x6_rep1 = cal_rep1[cal_rep1["movement"] == 6][feats].values

        if len(x5_rep1) > 0 and len(x6_rep1) > 0 and len(eval_rep26) > 0:
            x5 = x5_rep1[0]
            x6 = x6_rep1[0]

            # In scikit-learn binary LDA: decision_function > 0 predicts class 6 (M6)
            polarity = np.sign(np.dot(w, x6 - x5))
            if polarity == 0:
                polarity = 1.0
            w_eff = w * polarity

            # Midpoint threshold adaptation
            midpoint = 0.5 * (x5 + x6)
            b_eff = -np.dot(w_eff, midpoint)

            scores = np.dot(eval_rep26[feats].values, w_eff) + b_eff
            preds_rep1 = np.where(scores > 0, 6, 5)
            acc_rep1 = accuracy_score(eval_rep26["movement"], preds_rep1) * 100.0
        else:
            acc_rep1 = acc_0shot
        accs_rep1_adapt.append(acc_rep1)

        # ----------------------------------------------------------------------
        # C. 1-Shot 6-Fold Rotational Calibration (Unbiased Mean)
        # ----------------------------------------------------------------------
        rep_fold_accs = []
        pooled_fold_accs = []
        reps_available = sorted(test_subj["repetition"].unique())

        for r_cal in reps_available:
            cal_df = test_subj[test_subj["repetition"] == r_cal]
            eval_df = test_subj[test_subj["repetition"] != r_cal]

            x5_c = cal_df[cal_df["movement"] == 5][feats].values
            x6_c = cal_df[cal_df["movement"] == 6][feats].values

            if len(x5_c) > 0 and len(x6_c) > 0 and len(eval_df) > 0:
                # 1. Midpoint Adaptation
                pol = np.sign(np.dot(w, x6_c[0] - x5_c[0]))
                if pol == 0:
                    pol = 1.0
                w_f = w * pol
                b_f = -np.dot(w_f, 0.5 * (x5_c[0] + x6_c[0]))
                sc = np.dot(eval_df[feats].values, w_f) + b_f
                pr = np.where(sc > 0, 6, 5)
                rep_fold_accs.append(accuracy_score(eval_df["movement"], pr) * 100.0)

                # 2. Pooled Few-Shot Model
                train_comb = pd.concat([train_pop, cal_df], ignore_index=True)
                clf_comb = LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto")
                clf_comb.fit(train_comb[feats], train_comb["movement"])
                pr_comb = clf_comb.predict(eval_df[feats])
                pooled_fold_accs.append(accuracy_score(eval_df["movement"], pr_comb) * 100.0)

        acc_6fold = float(np.mean(rep_fold_accs)) if rep_fold_accs else acc_0shot
        acc_pool = float(np.mean(pooled_fold_accs)) if pooled_fold_accs else acc_0shot
        accs_6fold_adapt.append(acc_6fold)
        accs_pooled.append(acc_pool)

        subject_matrix_records.append({
            "feature_set": set_name,
            "subject": s_test,
            "acc_0shot": acc_0shot,
            "acc_rep1_adapt": acc_rep1,
            "acc_6fold_adapt": acc_6fold,
            "acc_pooled": acc_pool
        })

    results_summary.append({
        "Feature Set": set_name,
        "0-Shot LOSO (No Calib)": f"{np.mean(accs_0shot):.1f}% ± {np.std(accs_0shot):.1f}%",
        "1-Shot Rep 1 Calib (Test Reps 2-6)": f"{np.mean(accs_rep1_adapt):.1f}% ± {np.std(accs_rep1_adapt):.1f}%",
        "1-Shot 6-Fold Rotational Mean": f"{np.mean(accs_6fold_adapt):.1f}% ± {np.std(accs_6fold_adapt):.1f}%",
        "Pooled Few-Shot LDA": f"{np.mean(accs_pooled):.1f}% ± {np.std(accs_pooled):.1f}%"
    })

# ==============================================================================
# 3. DISPLAY RESULTS & SAVE MATRIX
# ==============================================================================
sum_df = pd.DataFrame(results_summary)
sum_df.to_csv(os.path.join(OUTPUT_DIR, "calibration_summary_comparison.csv"), index=False)

subj_df = pd.DataFrame(subject_matrix_records)
subj_df.to_csv(os.path.join(OUTPUT_DIR, "calibration_per_subject_matrix.csv"), index=False)

print("\n" + "=" * 95)
print("CALIBRATION IMPACT BENCHMARK (1/6 TRIALS TRAIN vs 5/6 TRIALS TEST)")
print("=" * 95)
for _, r in sum_df.iterrows():
    print(f"Feature Set                    : {r['Feature Set']}")
    print(f"  • 0-Shot LOSO (Zero Calib)   : {r['0-Shot LOSO (No Calib)']}")
    print(f"  • 1-Shot Rep 1 Calib         : {r['1-Shot Rep 1 Calib (Test Reps 2-6)']}")
    print(f"  • 1-Shot 6-Fold Mean         : {r['1-Shot 6-Fold Rotational Mean']}")
    print(f"  • Pooled Few-Shot LDA        : {r['Pooled Few-Shot LDA']}\n")
print("=" * 95)

# Display per-subject breakdown for the winning Variant 2
v2_matrix = subj_df[subj_df["feature_set"] == "Variant 2 (Winning 4-Feat)"].copy()
v2_matrix = v2_matrix.rename(columns={
    "acc_0shot": "0-Shot",
    "acc_rep1_adapt": "1-Shot (Rep 1)",
    "acc_6fold_adapt": "1-Shot (6-Fold)",
    "acc_pooled": "Pooled LDA"
})
print("\nPER-SUBJECT BREAKDOWN: VARIANT 2 (4-FEATURE SET) [% ACCURACY]")
print("-" * 75)
print(v2_matrix[["subject", "0-Shot", "1-Shot (Rep 1)", "1-Shot (6-Fold)", "Pooled LDA"]].to_string(index=False))
print("-" * 75)
print(f"All calibration logs and tables written to: {OUTPUT_DIR}")
print("=" * 95)

In [4]:
import os
import sys
import warnings
import numpy as np
import pandas as pd
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import accuracy_score

warnings.filterwarnings("ignore")

# ==============================================================================
# 1. ENVIRONMENT DETECTION & REAL-TIME LOGGING
# ==============================================================================
if os.path.isdir("datasets/DB2"):
    BASE_DIR = "datasets/DB2"
elif os.path.isdir("DB2"):
    BASE_DIR = "DB2"
elif os.path.exists("DB2_s1") or os.path.exists("s1"):
    BASE_DIR = "."
else:
    BASE_DIR = "datasets/DB2"

OUTPUT_DIR = os.path.join(BASE_DIR, "calibration_optimization_eval")
os.makedirs(OUTPUT_DIR, exist_ok=True)
LOG_FILE = os.path.join(OUTPUT_DIR, "optimization_run_log.txt")


class SafeLogger:
    """Tees stdout to terminal and disk with immediate per-line flushing."""
    def __init__(self, filepath):
        self.terminal = sys.stdout
        self.logfile = open(filepath, "w", encoding="utf-8")

    def write(self, message):
        self.terminal.write(message)
        self.terminal.flush()
        self.logfile.write(message)
        self.logfile.flush()

    def flush(self):
        self.terminal.flush()
        self.logfile.flush()

    def isatty(self):
        return getattr(self.terminal, "isatty", lambda: False)()


sys.stdout = SafeLogger(LOG_FILE)

print("=" * 95, flush=True)
print("FEW-SHOT CALIBRATION OPTIMIZATION: SHRINKAGE BLENDING & SAMPLE WEIGHTING", flush=True)
print(f"Working Directory : {os.path.abspath(BASE_DIR)}", flush=True)
print("=" * 95, flush=True)

# ==============================================================================
# CONFIGURATION & HYPERPARAMETER SWEEPS
# ==============================================================================
SUBJECTS = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12]  # Exclude S11
TARGET_WIN_MS = 40
TARGET_EPOCH = "post_tkin_angle"

# Evaluation Grids
ALPHA_GRID = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
WEIGHT_GRID = [1, 2, 5, 8, 10, 15, 20]

FEATURE_SETS = {
    "Variant 2 (Winning 4-Feat)": [
        "antagonist_diff_ratio",
        "edc_Hjorth_Mobility",
        "fds_LOG",
        "edc_Hjorth_Complexity"
    ],
    "Baseline Triplet (3-Feat)": [
        "antagonist_diff_ratio",
        "edc_Hjorth_Mobility",
        "fds_LOG"
    ]
}

# ==============================================================================
# 2. DATA INGESTION
# ==============================================================================
print("Ingesting multi-subject feature matrices...", flush=True)
dfs = []
for s in SUBJECTS:
    candidates = [
        os.path.join(BASE_DIR, f"DB2_s{s}", f"S{s}_E1_A1_pruned_emg_features.csv"),
        os.path.join(BASE_DIR, f"s{s}", f"S{s}_E1_A1_pruned_emg_features.csv"),
        os.path.join(BASE_DIR, f"S{s}_E1_A1_pruned_emg_features.csv"),
    ]
    csv_path = next((p for p in candidates if os.path.isfile(p)), None)
    if csv_path is None:
        print(f"[WARN] S{s} feature file not found. Skipping.", flush=True)
        continue

    df_sub = pd.read_csv(csv_path)
    df_sub["subject"] = s
    dfs.append(df_sub)
    print(f"  [+] Loaded S{s} ({len(df_sub)} rows)", flush=True)

if not dfs:
    print("[ERROR] No dataset files found. Verify directory path.", flush=True)
    sys.exit(1)

raw_df = pd.concat(dfs, ignore_index=True)
df = raw_df[(raw_df["window_ms"] == TARGET_WIN_MS) & (raw_df["epoch"] == TARGET_EPOCH)].copy()

print(f"\nUsable dataset: {len(df)} trials | Epoch: {TARGET_EPOCH} | Window: {TARGET_WIN_MS} ms\n", flush=True)

# ==============================================================================
# 3. EXPERIMENT A: REGULARIZED INTERCEPT SHRINKAGE SWEEP (alpha)
# ==============================================================================
print("=" * 95, flush=True)
print("EXPERIMENT A: REGULARIZED INTERCEPT SHRINKAGE SWEEP", flush=True)
print("Formula: b_hybrid = (1 - alpha) * b_pop + alpha * b_calib", flush=True)
print("=" * 95, flush=True)

shrinkage_records = []
shrinkage_subject_details = []

for set_name, feats in FEATURE_SETS.items():
    for alpha in ALPHA_GRID:
        accs_rep1 = []
        accs_6fold = []

        for s_test in SUBJECTS:
            train_pop = df[df["subject"] != s_test].dropna(subset=feats)
            test_subj = df[df["subject"] == s_test].dropna(subset=feats)

            if len(test_subj) == 0:
                continue

            # Fit Population LDA
            clf_pop = LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto")
            clf_pop.fit(train_pop[feats], train_pop["movement"])
            w = clf_pop.coef_[0]
            b_pop = clf_pop.intercept_[0]

            # --- Clinical Rep 1 Protocol ---
            cal_rep1 = test_subj[test_subj["repetition"] == 1]
            eval_rep26 = test_subj[test_subj["repetition"] != 1]

            x5_r1 = cal_rep1[cal_rep1["movement"] == 5][feats].values
            x6_r1 = cal_rep1[cal_rep1["movement"] == 6][feats].values

            if len(x5_r1) > 0 and len(x6_r1) > 0 and len(eval_rep26) > 0:
                pol = np.sign(np.dot(w, x6_r1[0] - x5_r1[0]))
                if pol == 0: pol = 1.0

                w_eff = w * pol
                b_pop_eff = b_pop * pol
                b_cal = -np.dot(w_eff, 0.5 * (x5_r1[0] + x6_r1[0]))

                # Blend population prior with individual calibration
                b_hybrid = (1.0 - alpha) * b_pop_eff + alpha * b_cal

                scores = np.dot(eval_rep26[feats].values, w_eff) + b_hybrid
                preds_r1 = np.where(scores > 0, 6, 5)
                acc_r1 = accuracy_score(eval_rep26["movement"], preds_r1) * 100.0
            else:
                acc_r1 = np.nan
            accs_rep1.append(acc_r1)

            # --- 6-Fold Rotational Calibration ---
            fold_accs = []
            for r_cal in sorted(test_subj["repetition"].unique()):
                c_df = test_subj[test_subj["repetition"] == r_cal]
                e_df = test_subj[test_subj["repetition"] != r_cal]

                x5_c = c_df[c_df["movement"] == 5][feats].values
                x6_c = c_df[c_df["movement"] == 6][feats].values

                if len(x5_c) > 0 and len(x6_c) > 0 and len(e_df) > 0:
                    p = np.sign(np.dot(w, x6_c[0] - x5_c[0]))
                    if p == 0: p = 1.0
                    w_f = w * p
                    b_p_f = b_pop * p
                    b_c = -np.dot(w_f, 0.5 * (x5_c[0] + x6_c[0]))

                    b_h = (1.0 - alpha) * b_p_f + alpha * b_c
                    sc = np.dot(e_df[feats].values, w_f) + b_h
                    pr = np.where(sc > 0, 6, 5)
                    fold_accs.append(accuracy_score(e_df["movement"], pr) * 100.0)

            acc_6f = float(np.mean(fold_accs)) if fold_accs else np.nan
            accs_6fold.append(acc_6f)

            shrinkage_subject_details.append({
                "feature_set": set_name,
                "alpha": alpha,
                "subject": s_test,
                "acc_rep1": acc_r1,
                "acc_6fold": acc_6f
            })

        shrinkage_records.append({
            "Feature Set": set_name,
            "Alpha": alpha,
            "Rep 1 Calib Acc (Mean ± SD)": f"{np.nanmean(accs_rep1):.1f}% ± {np.nanstd(accs_rep1):.1f}%",
            "6-Fold Rotational Acc (Mean ± SD)": f"{np.nanmean(accs_6fold):.1f}% ± {np.nanstd(accs_6fold):.1f}%",
            "Mean_Rep1": np.nanmean(accs_rep1),
            "Mean_6Fold": np.nanmean(accs_6fold)
        })

df_shrink = pd.DataFrame(shrinkage_records)
df_shrink.to_csv(os.path.join(OUTPUT_DIR, "shrinkage_alpha_sweep_summary.csv"), index=False)

df_shrink_subj = pd.DataFrame(shrinkage_subject_details)
df_shrink_subj.to_csv(os.path.join(OUTPUT_DIR, "shrinkage_alpha_per_subject.csv"), index=False)

# Display Shrinkage Summary
print(f"{'Feature Set':<28} | {'Alpha':<5} | {'Rep 1 Test Acc':<18} | {'6-Fold Mean Acc':<18}")
print("-" * 75)
for _, r in df_shrink[df_shrink["Feature Set"] == "Variant 2 (Winning 4-Feat)"].iterrows():
    print(f"{r['Feature Set']:<28} | {r['Alpha']:<5.1f} | {r['Rep 1 Calib Acc (Mean ± SD)']:<18} | {r['6-Fold Rotational Acc (Mean ± SD)']:<18}")
print("-" * 75)

# ==============================================================================
# 4. EXPERIMENT B: SAMPLE-WEIGHTED POOLED LDA SWEEP (K multiplier)
# ==============================================================================
print("\n" + "=" * 95, flush=True)
print("EXPERIMENT B: SAMPLE-WEIGHTED POOLED LDA SWEEP", flush=True)
print("Mechanism: Replicating individual calibration samples K times into population pool", flush=True)
print("=" * 95, flush=True)

weighted_records = []
weighted_subject_details = []

for set_name, feats in FEATURE_SETS.items():
    for k_weight in WEIGHT_GRID:
        accs_rep1 = []
        accs_6fold = []

        for s_test in SUBJECTS:
            train_pop = df[df["subject"] != s_test].dropna(subset=feats)
            test_subj = df[df["subject"] == s_test].dropna(subset=feats)

            if len(test_subj) == 0:
                continue

            # --- Clinical Rep 1 Protocol ---
            cal_rep1 = test_subj[test_subj["repetition"] == 1]
            eval_rep26 = test_subj[test_subj["repetition"] != 1]

            if len(cal_rep1) > 0 and len(eval_rep26) > 0:
                # Weight target subject calibration trials by integer replication factor K
                train_weighted = pd.concat([train_pop] + [cal_rep1] * k_weight, ignore_index=True)
                clf_w = LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto")
                clf_w.fit(train_weighted[feats], train_weighted["movement"])
                preds_w = clf_w.predict(eval_rep26[feats])
                acc_r1 = accuracy_score(eval_rep26["movement"], preds_w) * 100.0
            else:
                acc_r1 = np.nan
            accs_rep1.append(acc_r1)

            # --- 6-Fold Rotational Calibration ---
            fold_accs = []
            for r_cal in sorted(test_subj["repetition"].unique()):
                c_df = test_subj[test_subj["repetition"] == r_cal]
                e_df = test_subj[test_subj["repetition"] != r_cal]

                if len(c_df) > 0 and len(e_df) > 0:
                    train_w_fold = pd.concat([train_pop] + [c_df] * k_weight, ignore_index=True)
                    clf_wf = LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto")
                    clf_wf.fit(train_w_fold[feats], train_w_fold["movement"])
                    pr_wf = clf_wf.predict(e_df[feats])
                    fold_accs.append(accuracy_score(e_df["movement"], pr_wf) * 100.0)

            acc_6f = float(np.mean(fold_accs)) if fold_accs else np.nan
            accs_6fold.append(acc_6f)

            weighted_subject_details.append({
                "feature_set": set_name,
                "weight_k": k_weight,
                "subject": s_test,
                "acc_rep1": acc_r1,
                "acc_6fold": acc_6f
            })

        weighted_records.append({
            "Feature Set": set_name,
            "Weight K": k_weight,
            "Rep 1 Calib Acc (Mean ± SD)": f"{np.nanmean(accs_rep1):.1f}% ± {np.nanstd(accs_rep1):.1f}%",
            "6-Fold Rotational Acc (Mean ± SD)": f"{np.nanmean(accs_6fold):.1f}% ± {np.nanstd(accs_6fold):.1f}%",
            "Mean_Rep1": np.nanmean(accs_rep1),
            "Mean_6Fold": np.nanmean(accs_6fold)
        })

df_weight = pd.DataFrame(weighted_records)
df_weight.to_csv(os.path.join(OUTPUT_DIR, "weighted_pooled_lda_summary.csv"), index=False)

df_weight_subj = pd.DataFrame(weighted_subject_details)
df_weight_subj.to_csv(os.path.join(OUTPUT_DIR, "weighted_pooled_lda_per_subject.csv"), index=False)

# Display Weight Summary
print(f"{'Feature Set':<28} | {'Weight K':<8} | {'Rep 1 Test Acc':<18} | {'6-Fold Mean Acc':<18}")
print("-" * 75)
for _, r in df_weight[df_weight["Feature Set"] == "Variant 2 (Winning 4-Feat)"].iterrows():
    print(f"{r['Feature Set']:<28} | {r['Weight K']:<8} | {r['Rep 1 Calib Acc (Mean ± SD)']:<18} | {r['6-Fold Rotational Acc (Mean ± SD)']:<18}")
print("-" * 75)

# ==============================================================================
# 5. PER-SUBJECT OPTIMAL BREAKDOWN
# ==============================================================================
# Identify optimal alpha and optimal weight K for Variant 2
v2_shrink = df_shrink_subj[df_shrink_subj["feature_set"] == "Variant 2 (Winning 4-Feat)"]
v2_weight = df_weight_subj[df_weight_subj["feature_set"] == "Variant 2 (Winning 4-Feat)"]

best_alpha = df_shrink[df_shrink["Feature Set"] == "Variant 2 (Winning 4-Feat)"].sort_values("Mean_6Fold", ascending=False).iloc[0]["Alpha"]
best_k = df_weight[df_weight["Feature Set"] == "Variant 2 (Winning 4-Feat)"].sort_values("Mean_6Fold", ascending=False).iloc[0]["Weight K"]

print("\n" + "=" * 95, flush=True)
print(f"PER-SUBJECT COMPARISON AT OPTIMAL HYPERPARAMETERS (Variant 2 | 4-Features)", flush=True)
print(f"Optimal Alpha = {best_alpha:.2f} | Optimal Weight K = {best_k}", flush=True)
print("=" * 95, flush=True)

pivot_a0 = v2_shrink[v2_shrink["alpha"] == 0.0].set_index("subject")["acc_6fold"]
pivot_a1 = v2_shrink[v2_shrink["alpha"] == 1.0].set_index("subject")["acc_6fold"]
pivot_abest = v2_shrink[v2_shrink["alpha"] == best_alpha].set_index("subject")["acc_6fold"]
pivot_kbest = v2_weight[v2_weight["weight_k"] == best_k].set_index("subject")["acc_6fold"]

subj_summary = pd.DataFrame({
    "Subject": pivot_a0.index,
    "0-Shot (alpha=0)": pivot_a0.values,
    "Hard 1-Shot (alpha=1)": pivot_a1.values,
    f"Shrinkage (alpha={best_alpha:.1f})": pivot_abest.values,
    f"Weighted Pooled (K={best_k})": pivot_kbest.values
})

subj_summary["Delta (Shrink - 0Shot)"] = subj_summary[f"Shrinkage (alpha={best_alpha:.1f})"] - subj_summary["0-Shot (alpha=0)"]
subj_summary.to_csv(os.path.join(OUTPUT_DIR, "optimal_subject_breakdown.csv"), index=False)

print(subj_summary.to_string(index=False), flush=True)
print("-" * 95, flush=True)
print(f"All summary comparisons, logs, and matrices saved to: {OUTPUT_DIR}", flush=True)
print("=" * 95, flush=True)

In [6]:
"""
================================================================================
MASTER EMG-KINEMATIC ANALYSIS ENGINE: PHASE SCREENING & LASSO LOSO MODELING
Ingests: S{id}_E1_A1_feature_metrics.csv (Batch processed via MATLAB)
Excludes: Subject 11
Outputs: Consolidated single report ('emg_kinematics_master_analysis.txt')
================================================================================
"""

import os
import glob
import warnings
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.linear_model import LassoCV
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.feature_selection import mutual_info_regression

warnings.filterwarnings("ignore")

OUTPUT_REPORT = "emg_kinematics_master_analysis.txt"
EXCLUDE_SUBJECTS = {11}

# ------------------------------------------------------------------------------
# 1. FILE INGESTION & COHORT CONCATENATION
# ------------------------------------------------------------------------------
search_paths = [
    os.path.join("datasets", "DB2", "DB2_s*", "*_feature_metrics.csv"),
    os.path.join("..", "datasets", "DB2", "DB2_s*", "*_feature_metrics.csv"),
    os.path.join("**", "*_feature_metrics.csv")
]

csv_files = []
for p in search_paths:
    found = glob.glob(p, recursive=True)
    if found:
        csv_files.extend(found)

csv_files = sorted(list(set(csv_files)))
if not csv_files:
    raise FileNotFoundError("No '*_feature_metrics.csv' files found. Run the MATLAB feature extractor first.")

df_list = []
for f in csv_files:
    temp_df = pd.read_csv(f)
    # Ensure Subject 11 is strictly excluded even if file exists on disk
    temp_df = temp_df[~temp_df['subject_id'].isin(EXCLUDE_SUBJECTS)]
    df_list.append(temp_df)

df = pd.concat(df_list, ignore_index=True).dropna()
subject_list = sorted(df['subject_id'].unique().tolist())
print(f"[INGESTION] Cohort: {len(subject_list)} subjects {subject_list} | Total Valid Trials: {len(df)}")

# ------------------------------------------------------------------------------
# 2. ANALYSIS & REPORT GENERATION
# ------------------------------------------------------------------------------
lines = []
def log(s=""):
    print(s)
    lines.append(s)

log("=" * 88)
log("CONSOLIDATED EMG-KINEMATIC RELATIONSHIP REPORT (EXCLUDING SUBJECT 11)")
log("=" * 88)
log(f"Cohort Analyzed  : {len(subject_list)} subjects {subject_list}")
log(f"Validated Trials : {len(df)} total\n")

# --- SECTION 1: ELECTROMECHANICAL DELAY (EMD) DISTRIBUTIONS ---
log("-" * 88)
log("SECTION 1: PHYSIOLOGICAL ELECTROMECHANICAL DELAY (EMD) PROFILES")
log("-" * 88)
on_mean, on_std, on_med = df['emd_onset_ms'].mean(), df['emd_onset_ms'].std(), df['emd_onset_ms'].median()
off_mean, off_std, off_med = df['emd_offset_ms'].mean(), df['emd_offset_ms'].std(), df['emd_offset_ms'].median()

log(f"Onset EMD  (t1 - t_myo)  : Mean = {on_mean:>6.2f} ms | Median = {on_med:>6.2f} ms | Std = {on_std:>5.2f} ms")
log(f"Offset EMD (t3 - t_off)  : Mean = {off_mean:>6.2f} ms | Median = {off_med:>6.2f} ms | Std = {off_std:>5.2f} ms")
log("Physiological Note: Negative onset indicates sEMG neural drive leads kinematic takeoff.\n")

# --- SECTION 2: PHASE-GATED UNIVARIATE SCREENING & INFORMATION METRICS ---
target_predictor_map = {
    'Rise Phase (Velocity & Acceleration)': {
        'targets': ['kin_rise_vel', 'kin_peak_acc'],
        'predictors': ['emg_red_max', 'emg_red_mean', 'emg_early_iemg', 'emg_rise_wl']
    },
    'Hold Phase (Displacement & Stability)': {
        'targets': ['kin_excursion', 'kin_mean_hold_ang', 'kin_hold_wobble'],
        'predictors': ['emg_hold_mean_rms', 'emg_hold_peak_rms', 'emg_hold_mnf', 'emg_mobility', 'emg_cci', 'emg_ndi']
    },
    'Release Phase (Decay & Braking)': {
        'targets': ['kin_fall_vel', 'kin_fall_dur_ms'],
        'predictors': ['emg_decay_rate', 'emg_antagonist_brake']
    }
}

log("-" * 88)
log("SECTION 2: PHASE-GATED CORRELATIONS (PEARSON r, SPEARMAN rho, MUTUAL INFORMATION)")
log("-" * 88)
log(f"{'Kinematic Target':<20} | {'sEMG Metric':<20} | {'Pearson r':<10} | {'Spearman rho':<12} | {'MI (nats)':<10}")
log("-" * 88)

for phase_name, pdict in target_predictor_map.items():
    log(f"[{phase_name.upper()}]")
    for tgt in pdict['targets']:
        y = df[tgt].values
        for pred in pdict['predictors']:
            x = df[pred].values
            r_val, _ = stats.pearsonr(x, y)
            rho_val, _ = stats.spearmanr(x, y)
            mi_val = mutual_info_regression(x.reshape(-1, 1), y, random_state=42)[0]
            log(f"  {tgt:<18} | {pred:<20} | {r_val:>9.3f} | {rho_val:>12.3f} | {mi_val:>10.3f}")
    log()

# --- SECTION 3: COLLINEARITY & REDUNDANCY AUDIT ---
log("-" * 88)
log("SECTION 3: COLLINEARITY & REDUNDANCY AUDIT (|Pearson r| >= 0.80)")
log("-" * 88)
all_preds = [
    'emg_red_max', 'emg_red_mean', 'emg_early_iemg', 'emg_rise_wl',
    'emg_hold_mean_rms', 'emg_hold_peak_rms', 'emg_hold_mnf', 'emg_mobility',
    'emg_cci', 'emg_ndi', 'emg_decay_rate', 'emg_antagonist_brake'
]
corr_mat = df[all_preds].corr().abs()
flagged_pairs = []
for i in range(len(all_preds)):
    for j in range(i + 1, len(all_preds)):
        r_val = corr_mat.iloc[i, j]
        if r_val >= 0.80:
            flagged_pairs.append((all_preds[i], all_preds[j], r_val))

if flagged_pairs:
    for p1, p2, r_val in flagged_pairs:
        log(f"  [REDUNDANCY] {p1:<20} <---> {p2:<20} | r = {r_val:.3f}")
else:
    log("  No severe pairwise collinearity (|r| >= 0.80) detected among predictors.")
log()

# --- SECTION 4: LASSO SHRINKAGE & LEAVE-ONE-SUBJECT-OUT (LOSO) GENERALIZATION ---
log("-" * 88)
log("SECTION 4: LASSO SHRINKAGE & LEAVE-ONE-SUBJECT-OUT (LOSO) VALIDATION")
log("-" * 88)

tasks = [
    ('Rise Velocity', 'kin_rise_vel', ['emg_red_max', 'emg_red_mean', 'emg_early_iemg', 'emg_rise_wl']),
    ('Max Excursion', 'kin_excursion', ['emg_hold_mean_rms', 'emg_hold_peak_rms', 'emg_cci', 'emg_ndi']),
    ('Hold Stability', 'kin_hold_wobble', ['emg_cci', 'emg_hold_mnf', 'emg_mobility']),
    ('Fall Velocity', 'kin_fall_vel', ['emg_decay_rate', 'emg_antagonist_brake'])
]

logo = LeaveOneGroupOut()
groups = df['subject_id'].values

log(f"{'Kinematic Task':<18} | {'LOSO R^2':<9} | {'RMSE':<9} | {'MAE':<9} | {'Retained Features (Non-Zero Coefficients)'}")
log("-" * 88)

for task_label, tgt_col, pred_cols in tasks:
    X = df[pred_cols].values
    y = df[tgt_col].values
    y_preds = np.zeros_like(y)
    
    # 1. Evaluate generalization error via strictly normalized LOSO
    for train_idx, test_idx in logo.split(X, y, groups=groups):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train = y[train_idx]
        
        scaler_mean = np.mean(X_train, axis=0)
        scaler_std = np.std(X_train, axis=0) + 1e-8
        
        X_tr_norm = (X_train - scaler_mean) / scaler_std
        X_te_norm = (X_test - scaler_mean) / scaler_std
        
        lasso = LassoCV(cv=5, random_state=42, max_iter=5000).fit(X_tr_norm, y_train)
        y_preds[test_idx] = lasso.predict(X_te_norm)
        
    r2 = r2_score(y, y_preds)
    rmse = np.sqrt(mean_squared_error(y, y_preds))
    mae = mean_absolute_error(y, y_preds)
    
    # 2. Global LASSO fit to audit parameter pruning
    X_glob_norm = (X - np.mean(X, axis=0)) / (np.std(X, axis=0) + 1e-8)
    global_lasso = LassoCV(cv=5, random_state=42, max_iter=5000).fit(X_glob_norm, y)
    
    retained = [
        f"{col} (b={coef:+.2f})"
        for col, coef in zip(pred_cols, global_lasso.coef_)
        if abs(coef) > 1e-4
    ]
    retained_str = ", ".join(retained) if retained else "All features shrunk to 0"
    
    log(f"{task_label:<18} | {r2:>9.3f} | {rmse:>9.2f} | {mae:>9.2f} | {retained_str}")

log("\n" + "=" * 88)
log("END OF MULTI-PHASE STATISTICAL REPORT")
log("=" * 88)

# ------------------------------------------------------------------------------
# 3. CONSOLIDATED FILE DUMP
# ------------------------------------------------------------------------------
with open(OUTPUT_REPORT, 'w') as f:
    f.write("\n".join(lines))

print(f"\n[DONE] Full consolidated master analysis written to: {os.path.abspath(OUTPUT_REPORT)}")

[INGESTION] Cohort: 11 subjects [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12] | Total Valid Trials: 132
CONSOLIDATED EMG-KINEMATIC RELATIONSHIP REPORT (EXCLUDING SUBJECT 11)
Cohort Analyzed  : 11 subjects [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12]
Validated Trials : 132 total

----------------------------------------------------------------------------------------
SECTION 1: PHYSIOLOGICAL ELECTROMECHANICAL DELAY (EMD) PROFILES
----------------------------------------------------------------------------------------
Onset EMD  (t1 - t_myo)  : Mean = -50.66 ms | Median = -46.50 ms | Std = 81.92 ms
Offset EMD (t3 - t_off)  : Mean = -27.67 ms | Median =   0.00 ms | Std = 94.70 ms
Physiological Note: Negative onset indicates sEMG neural drive leads kinematic takeoff.

----------------------------------------------------------------------------------------
SECTION 2: PHASE-GATED CORRELATIONS (PEARSON r, SPEARMAN rho, MUTUAL INFORMATION)
-------------------------------------------------------------------------

In [1]:
import os
import glob
import warnings
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.linear_model import LassoCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.feature_selection import mutual_info_regression

warnings.filterwarnings("ignore")

OUTPUT_REPORT = "emg_kinematics_master_analysis.txt"
EXCLUDE_SUBJECTS = {11}

# ------------------------------------------------------------------------------
# 1. FILE INGESTION & WITHIN-SUBJECT DYNAMIC NORMALIZATION
# ------------------------------------------------------------------------------
search_paths = [
    os.path.join("datasets", "DB2", "DB2_s*", "*_feature_metrics.csv"),
    os.path.join("..", "datasets", "DB2", "DB2_s*", "*_feature_metrics.csv"),
    os.path.join("**", "*_feature_metrics.csv")
]

csv_files = []
for p in search_paths:
    found = glob.glob(p, recursive=True)
    if found:
        csv_files.extend(found)
csv_files = sorted(list(set(csv_files)))

if not csv_files:
    raise FileNotFoundError("No '*_feature_metrics.csv' files found. Run the updated MATLAB extractor first.")

raw_dfs = []
for f in csv_files:
    t_df = pd.read_csv(f)
    t_df = t_df[~t_df['subject_id'].isin(EXCLUDE_SUBJECTS)]
    raw_dfs.append(t_df)

df = pd.concat(raw_dfs, ignore_index=True).dropna()
subject_list = sorted(df['subject_id'].unique().tolist())

# Amplitude-dependent features sensitive to skin impedance and sensor contact
amp_features = [
    'emg_red_max', 'emg_red_mean', 'emg_early_iemg', 'emg_rise_wl',
    'emg_hold_mean_rms', 'emg_decay_rate', 'emg_antagonist_brake'
]

# Compute subject-normalized variants (relative dynamic activation: [0, 1])
for feat in amp_features:
    norm_col = f"{feat}_norm"
    df[norm_col] = df.groupby('subject_id')[feat].transform(
        lambda x: (x - x.min()) / (np.percentile(x, 95) - x.min() + 1e-6)
    )

# ------------------------------------------------------------------------------
# 2. ANALYSIS & REPORT GENERATION
# ------------------------------------------------------------------------------
lines = []
def log(s=""):
    print(s)
    lines.append(s)

log("=" * 90)
log("MASTER EMG-KINEMATIC ANALYSIS: RECTIFIED TARGETS & RELATIVE DRIVE MODELING")
log("=" * 90)
log(f"Cohort Analyzed  : {len(subject_list)} subjects {subject_list} (Excluded: 11)")
log(f"Validated Trials : {len(df)} total\n")

# --- SECTION 1: EMD DISTRIBUTIONS ---
log("-" * 90)
log("SECTION 1: PHYSIOLOGICAL LATENCY (EMD) PROFILES")
log("-" * 90)
log(f"Onset EMD  (t1 - t_myo) : Mean = {df['emd_onset_ms'].mean():>6.2f} ms | Median = {df['emd_onset_ms'].median():>6.2f} ms | Std = {df['emd_onset_ms'].std():>5.2f} ms")
log(f"Offset EMD (t3 - t_off) : Mean = {df['emd_offset_ms'].mean():>6.2f} ms | Median = {df['emd_offset_ms'].median():>6.2f} ms | Std = {df['emd_offset_ms'].std():>5.2f} ms\n")

# --- SECTION 2: PHASE-GATED UNIVARIATE SCREENING ---
target_predictor_map = {
    'Rise Phase (Rectified Speed & Acceleration)': {
        'targets': ['kin_rise_speed', 'kin_peak_acc'],
        'predictors': ['emg_red_max_norm', 'emg_red_mean_norm', 'emg_early_iemg_norm', 'emg_rise_wl_norm']
    },
    'Hold Phase (Displacement & Stability)': {
        'targets': ['kin_excursion', 'kin_hold_disp', 'kin_hold_wobble'],
        'predictors': ['emg_hold_mean_rms_norm', 'emg_mobility', 'emg_complexity', 'emg_cci', 'emg_ndi']
    },
    'Release Phase (Rectified Fall & Braking)': {
        'targets': ['kin_fall_speed', 'kin_fall_dur_ms'],
        'predictors': ['emg_decay_rate_norm', 'emg_antagonist_brake_norm']
    }
}

log("-" * 90)
log("SECTION 2: PHASE-GATED CORRELATIONS (RELATIVE EMG DRIVE VS. RECTIFIED TARGETS)")
log("-" * 90)
log(f"{'Kinematic Target':<20} | {'Normalized sEMG Metric':<25} | {'Pearson r':<10} | {'Spearman rho':<12} | {'MI (nats)':<10}")
log("-" * 90)

for phase_name, pdict in target_predictor_map.items():
    log(f"[{phase_name.upper()}]")
    for tgt in pdict['targets']:
        y = df[tgt].values
        for pred in pdict['predictors']:
            x = df[pred].values
            r_val, _ = stats.pearsonr(x, y)
            rho_val, _ = stats.spearmanr(x, y)
            mi_val = mutual_info_regression(x.reshape(-1, 1), y, random_state=42)[0]
            log(f"  {tgt:<18} | {pred:<25} | {r_val:>9.3f} | {rho_val:>12.3f} | {mi_val:>10.3f}")
    log()

# --- SECTION 3: LINEAR VS. NON-LINEAR LOSO PREDICTIVE BENCHMARK ---
log("-" * 90)
log("SECTION 3: MULTIVARIATE LEAVE-ONE-SUBJECT-OUT (LOSO) GENERALIZATION BENCHMARK")
log("-" * 90)

tasks = [
    ('Rise Speed', 'kin_rise_speed', ['emg_red_max_norm', 'emg_red_mean_norm', 'emg_early_iemg_norm', 'emg_rise_wl_norm']),
    ('Hold Displacement', 'kin_hold_disp', ['emg_hold_mean_rms_norm', 'emg_mobility', 'emg_cci', 'emg_ndi']),
    ('Hold Stability', 'kin_hold_wobble', ['emg_cci', 'emg_mobility', 'emg_complexity']),
    ('Fall Duration', 'kin_fall_dur_ms', ['emg_decay_rate_norm', 'emg_antagonist_brake_norm'])
]

logo = LeaveOneGroupOut()
groups = df['subject_id'].values

log(f"{'Kinematic Task':<18} | {'Model':<12} | {'LOSO R^2':<9} | {'RMSE':<9} | {'MAE':<9} | {'Dominant / Active Predictor'}")
log("-" * 90)

for task_label, tgt_col, pred_cols in tasks:
    X = df[pred_cols].values
    y = df[tgt_col].values
    
    y_preds_lasso = np.zeros_like(y)
    y_preds_rf    = np.zeros_like(y)
    
    for train_idx, test_idx in logo.split(X, y, groups=groups):
        X_tr, X_te = X[train_idx], X[test_idx]
        y_tr = y[train_idx]
        
        s_mean = np.mean(X_tr, axis=0)
        s_std  = np.std(X_tr, axis=0) + 1e-8
        
        X_tr_n = (X_tr - s_mean) / s_std
        X_te_n = (X_te - s_mean) / s_std
        
        # 1. Linear LASSO
        lasso = LassoCV(cv=5, random_state=42, max_iter=5000).fit(X_tr_n, y_tr)
        y_preds_lasso[test_idx] = lasso.predict(X_te_n)
        
        # 2. Non-Linear Random Forest (Captures high Mutual Information)
        rf = RandomForestRegressor(n_estimators=100, max_depth=4, random_state=42).fit(X_tr, y_tr)
        y_preds_rf[test_idx] = rf.predict(X_te)
        
    # LASSO Metrics
    r2_l = r2_score(y, y_preds_lasso)
    rmse_l = np.sqrt(mean_squared_error(y, y_preds_lasso))
    mae_l = mean_absolute_error(y, y_preds_lasso)
    
    # Random Forest Metrics
    r2_rf = r2_score(y, y_preds_rf)
    rmse_rf = np.sqrt(mean_squared_error(y, y_preds_rf))
    mae_rf = mean_absolute_error(y, y_preds_rf)
    
    # Determine dominant global predictors
    glob_rf = RandomForestRegressor(n_estimators=100, max_depth=4, random_state=42).fit(X, y)
    top_rf_feat = pred_cols[np.argmax(glob_rf.feature_importances_)]
    
    glob_lasso = LassoCV(cv=5, random_state=42).fit((X - np.mean(X, 0)) / (np.std(X, 0) + 1e-8), y)
    top_lasso_feat = pred_cols[np.argmax(np.abs(glob_lasso.coef_))] if any(abs(glob_lasso.coef_) > 1e-4) else "Shrunk to 0"
    
    log(f"{task_label:<18} | {'LASSO':<12} | {r2_l:>9.3f} | {rmse_l:>9.2f} | {mae_l:>9.2f} | {top_lasso_feat}")
    log(f"{'':<18} | {'RandomForest':<12} | {r2_rf:>9.3f} | {rmse_rf:>9.2f} | {mae_rf:>9.2f} | {top_rf_feat} (Top Importance)")
    log("-" * 90)

log("\n" + "=" * 90)
log("END OF REVISED STATISTICAL REPORT")
log("=" * 90)

with open(OUTPUT_REPORT, 'w') as f:
    f.write("\n".join(lines))

print(f"\n[COMPLETE] Revised report generated at: {os.path.abspath(OUTPUT_REPORT)}")

MASTER EMG-KINEMATIC ANALYSIS: RECTIFIED TARGETS & RELATIVE DRIVE MODELING
Cohort Analyzed  : 11 subjects [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12] (Excluded: 11)
Validated Trials : 132 total

------------------------------------------------------------------------------------------
SECTION 1: PHYSIOLOGICAL LATENCY (EMD) PROFILES
------------------------------------------------------------------------------------------
Onset EMD  (t1 - t_myo) : Mean = -50.66 ms | Median = -46.50 ms | Std = 81.92 ms
Offset EMD (t3 - t_off) : Mean = -27.67 ms | Median =   0.00 ms | Std = 94.70 ms

------------------------------------------------------------------------------------------
SECTION 2: PHASE-GATED CORRELATIONS (RELATIVE EMG DRIVE VS. RECTIFIED TARGETS)
------------------------------------------------------------------------------------------
Kinematic Target     | Normalized sEMG Metric    | Pearson r  | Spearman rho | MI (nats) 
---------------------------------------------------------------------

In [1]:
"""
================================================================================
EMPIRICAL FEATURE DISCOVERY & MULTI-DIMENSIONAL SUBSPACE SELECTION PIPELINE
(Patched: Scale-Invariant Ledoit-Wolf, LOGO-CV, Multi-Weight Grid & Subspace Cache)
================================================================================
"""

import os
import logging
import warnings
import itertools
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.covariance import LedoitWolf
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import accuracy_score
from sklearn.model_selection import LeaveOneGroupOut

# ==============================================================================
# CONFIGURATION & REPRODUCIBILITY SETUP
# ==============================================================================
BASE_DIR = "datasets/DB2"
SUBJECTS = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12]  # Subject 11 strictly excluded
WINDOW_SIZES_MS = [30, 40, 50]                   # Window lengths to benchmark

# Physiological Temporal Boundaries (relative to t_kin = 0 ms)
QUARANTINE_ZONE = (-200, -130)  # Pre-EMD Zone: False Alarm evaluation
EARLY_RAMP_ZONE = (-120, -60)   # Latent foot of recruitment
CRITICAL_EMD    = (-50, 0)      # Operational Lead Horizon (Motor Intent)
PEAK_FLIGHT     = (50, 150)     # Ballistic Flight Saturation
DM_TRIPWIRE     = 3.0           # Baseline Mahalanobis tripwire threshold

# Dynamic Weight Pools for Feature Screening (Applied to [0, 1] normalized metrics)
WEIGHT_POOLS = {
    "Balanced": {"contrast": 1.0, "spearman": 1.0, "smoothness": 1.0, "fisher": 1.0},
    "Trajectory_Heavy": {"contrast": 0.5, "spearman": 2.0, "smoothness": 2.0, "fisher": 0.5},
    "Separability_Heavy": {"contrast": 1.5, "spearman": 0.5, "smoothness": 0.5, "fisher": 3.0},
}

OUTPUT_DIR = os.path.join(BASE_DIR, "feature_discovery_empirical_results")
os.makedirs(OUTPUT_DIR, exist_ok=True)
LOG_FILE = os.path.join(OUTPUT_DIR, "empirical_discovery_master_log.txt")

# ==============================================================================
# PROPER LOGGING SETUP
# ==============================================================================
logger = logging.getLogger("FeatureDiscovery")
logger.setLevel(logging.INFO)
if logger.hasHandlers():
    logger.handlers.clear()

fh = logging.FileHandler(LOG_FILE, mode='w', encoding='utf-8')
ch = logging.StreamHandler()
formatter = logging.Formatter('%(message)s')
fh.setFormatter(formatter)
ch.setFormatter(formatter)
logger.addHandler(fh)
logger.addHandler(ch)

# ==============================================================================
# 1. INGESTION & COHORT CONSOLIDATION
# ==============================================================================
logger.info("=" * 115)
logger.info(f"INGESTING MULTI-WINDOW SLIDING DATASETS ({len(SUBJECTS)} SUBJECTS | 400 ms HORIZON)")
logger.info("=" * 115)

dfs = []
for s in SUBJECTS:
    candidates = [
        os.path.join(BASE_DIR, f"DB2_s{s}", f"S{s}_E1_A1_pruned_emg_sliding_features.csv"),
        os.path.join(BASE_DIR, f"s{s}", f"S{s}_E1_A1_pruned_emg_sliding_features.csv"),
    ]
    p = next((c for c in candidates if os.path.isfile(c)), None)
    if p:
        d = pd.read_csv(p)
        d["subject"] = s
        dfs.append(d)
    else:
        logger.warning(f"[WARN] Missing sliding feature file for Subject {s}.")

if not dfs:
    raise FileNotFoundError(f"No feature files found in {BASE_DIR}. Run sliding extraction first.")

raw_df = pd.concat(dfs, ignore_index=True)
meta_cols = ["repetition", "movement", "epoch_type", "offset_ms", "window_ms", "idx_start", "idx_end", "subject"]
feature_cols = [c for c in raw_df.columns if c not in meta_cols and pd.api.types.is_numeric_dtype(raw_df[c])]

logger.info(f"Total Rows Ingested     : {len(raw_df)}")
logger.info(f"Extracted EMG Metrics   : {len(feature_cols)}")
logger.info(f"Evaluated Subjects      : {sorted(raw_df['subject'].unique().tolist())}")
logger.info(f"Tested Window Sizes     : {sorted(raw_df['window_ms'].unique().tolist())} ms")
logger.info(f"Tested Temporal Offsets : {int(raw_df['offset_ms'].min())} ms to {int(raw_df['offset_ms'].max())} ms\n")


# ==============================================================================
# 2. ANALYSIS PIPELINE FOR A SPECIFIC WINDOW DURATION
# ==============================================================================
def analyze_window(win_ms, df_all):
    logger.info("=" * 115)
    logger.info(f"{f'COMPREHENSIVE MULTI-METRIC DISCOVERY: {win_ms} ms WINDOW':^115}")
    logger.info("=" * 115)

    df_win = df_all[df_all["window_ms"] == win_ms].copy()
    df_rest = df_win[df_win["epoch_type"] == "rest_gray"].copy()
    df_peri = df_win[df_win["epoch_type"] == "peri_tkin"].copy()

    # Step A: Per-Subject Resting Baseline Z-Score Normalization
    rest_stats = {}
    df_rest_norm = df_rest.copy()
    df_norm = df_peri.copy()

    for s in SUBJECTS:
        s_rest = df_rest[df_rest["subject"] == s][feature_cols]
        mu = s_rest.mean()
        std = s_rest.std().replace(0, 1e-6)
        rest_stats[s] = (mu, std)

        mask_rest = df_rest_norm["subject"] == s
        mask_peri = df_norm["subject"] == s
        
        if mask_rest.sum() > 0:
            df_rest_norm.loc[mask_rest, feature_cols] = (df_rest_norm.loc[mask_rest, feature_cols] - mu) / std
        if mask_peri.sum() > 0:
            df_norm.loc[mask_peri, feature_cols] = (df_norm.loc[mask_peri, feature_cols] - mu) / std

    offsets = sorted(df_norm["offset_ms"].unique())

    # Step B: Shape-Agnostic Non-Linear Trajectory Screening
    raw_metrics = []
    for feat in feature_cols:
        mean_traj = df_norm[feat].abs().groupby(df_norm["offset_ms"]).mean()

        trans_offsets = np.array([t for t in offsets if -120 <= t <= 60])
        trans_vals = np.array([mean_traj.get(t, 0) for t in trans_offsets])
        spearman_rho, _ = stats.spearmanr(trans_offsets, trans_vals) if len(trans_vals) > 4 else (0.0, 1.0)

        t_norm = (trans_offsets - trans_offsets.mean()) / (trans_offsets.std() + 1e-6)
        poly_coeffs = np.polyfit(t_norm, trans_vals, deg=2)
        poly_pred = np.polyval(poly_coeffs, t_norm)
        ss_tot = np.sum((trans_vals - np.mean(trans_vals))**2)
        ss_res = np.sum((trans_vals - poly_pred)**2)
        smoothness_r2 = 1.0 - (ss_res / (ss_tot + 1e-6)) if ss_tot > 0 else 0.0

        quar_noise = df_norm.loc[df_norm["offset_ms"] <= QUARANTINE_ZONE[1], feat].dropna().std()
        flight_power = df_norm.loc[df_norm["offset_ms"].between(PEAK_FLIGHT[0], PEAK_FLIGHT[1]), feat].dropna().abs().mean()
        contrast = flight_power / (quar_noise + 1e-6)

        dir_mask = df_norm["offset_ms"].between(-50, 100)
        m5_v = df_norm.loc[dir_mask & (df_norm["movement"] == 5), feat].dropna()
        m6_v = df_norm.loc[dir_mask & (df_norm["movement"] == 6), feat].dropna()
        fisher_dir = ((m5_v.mean() - m6_v.mean())**2 / (m5_v.var() + m6_v.var() + 1e-6)) if len(m5_v) > 1 and len(m6_v) > 1 else 0.0

        if "diff_ratio" in feat or "co_contraction" in feat:
            domain = "Relational"
        elif any(k in feat for k in ["_RMS", "_MAV", "_VAR", "_SSI", "_LOG"]):
            domain = "Amplitude"
        elif any(k in feat for k in ["_Mobility", "_MNF", "_MDF"]):
            domain = "Spectral"
        elif "TKEO" in feat:
            domain = "Nonlinear"
        else:
            domain = "Morphological"

        raw_metrics.append({
            "feature": feat, "domain": domain, "contrast": contrast, 
            "spearman_rho": max(0.0, spearman_rho),
            "smoothness_r2": max(0.0, smoothness_r2), "fisher_dir": fisher_dir
        })

    metrics_df = pd.DataFrame(raw_metrics)

    for col in ["contrast", "spearman_rho", "smoothness_r2", "fisher_dir"]:
        c_min, c_max = metrics_df[col].min(), metrics_df[col].max()
        metrics_df[f"{col}_norm"] = (metrics_df[col] - c_min) / (c_max - c_min + 1e-6)

    logo = LeaveOneGroupOut()
    df_emd = df_norm[df_norm["offset_ms"].between(CRITICAL_EMD[0], CRITICAL_EMD[1])].copy()
    df_flt = df_norm[df_norm["offset_ms"].between(PEAK_FLIGHT[0], PEAK_FLIGHT[1])].copy()

    # Evaluation Cache to eliminate redundant fits across overlapping weight pools
    combo_cache = {}
    best_pool_results = []

    # Step C & D: Combinatorial Search Across Weight Pools
    for pool_name, weights in WEIGHT_POOLS.items():
        metrics_df["composite_score"] = (
            (metrics_df["contrast_norm"] * weights["contrast"]) +
            (metrics_df["spearman_rho_norm"] * weights["spearman"]) +
            (metrics_df["smoothness_r2_norm"] * weights["smoothness"]) +
            (metrics_df["fisher_dir_norm"] * weights["fisher"])
        )

        ranked_df = metrics_df.sort_values("composite_score", ascending=False).reset_index(drop=True)
        
        stratified_pool = []
        for dom in ["Relational", "Amplitude", "Spectral", "Nonlinear", "Morphological"]:
            stratified_pool.extend(ranked_df[ranked_df["domain"] == dom].head(2)["feature"].tolist())

        corr_m = df_norm[stratified_pool].corr().abs()
        final_pool = []
        for f in stratified_pool:
            if not any(corr_m.loc[f, ex] > 0.85 for ex in final_pool):
                final_pool.append(f)

        all_combos = []
        for k in [3, 4, 5]:
            all_combos.extend(list(itertools.combinations(final_pool, k)))

        combo_evals = []
        
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            
            for combo in all_combos:
                subspace = list(combo)
                sub_key = tuple(sorted(subspace))
                
                # Check cache first
                if sub_key in combo_cache:
                    combo_evals.append(combo_cache[sub_key])
                    continue

                dim = len(subspace)

                # 1. Scale-Invariant Ledoit-Wolf on Z-Scored Baseline
                fa_trips, total_quar = 0, 0
                flt_dm, rest_dm = [], []

                for s in SUBJECTS:
                    s_rest = df_rest_norm[df_rest_norm["subject"] == s][subspace].dropna().values
                    s_quar = df_norm[(df_norm["subject"] == s) & df_norm["offset_ms"].between(QUARANTINE_ZONE[0], QUARANTINE_ZONE[1])][subspace].dropna().values
                    s_flt_p = df_norm[(df_norm["subject"] == s) & (df_norm["offset_ms"] == 100)][subspace].dropna().values

                    if len(s_rest) > dim + 2 and len(s_quar) > 0:
                        lw = LedoitWolf().fit(s_rest)
                        mu_r, inv_cov_r = lw.location_, lw.precision_

                        diff_q = s_quar - mu_r
                        dm_q = np.sqrt(np.sum(diff_q @ inv_cov_r * diff_q, axis=1))
                        fa_trips += np.sum(dm_q > DM_TRIPWIRE)
                        total_quar += len(dm_q)

                        if len(s_flt_p) > 0:
                            diff_f = s_flt_p - mu_r
                            flt_dm.extend(np.sqrt(np.sum(diff_f @ inv_cov_r * diff_f, axis=1)))
                            rest_dm.extend(dm_q)

                far_pct = (fa_trips / max(1, total_quar)) * 100.0
                contrast = (np.mean(flt_dm) / (np.mean(rest_dm) + 1e-6)) if flt_dm else 0.0

                # 2. LOGO-CV Directional Accuracy
                def calculate_logo_acc(df_subset):
                    clean_df = df_subset.dropna(subset=subspace + ["movement", "subject"])
                    X, y, grps = clean_df[subspace].values, clean_df["movement"].values, clean_df["subject"].values
                    acc_scores = []
                    for tr, te in logo.split(X, y, groups=grps):
                        if len(np.unique(y[tr])) < 2: continue
                        clf = LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto").fit(X[tr], y[tr])
                        acc_scores.append(accuracy_score(y[te], clf.predict(X[te])) * 100.0)
                    return np.mean(acc_scores) if acc_scores else 0.0

                acc_emd = calculate_logo_acc(df_emd)
                acc_flt = calculate_logo_acc(df_flt)
                master_score = (acc_emd * 0.50) + (acc_flt * 0.15) + (contrast * 8.0) - (far_pct * 4.0)

                eval_record = {
                    "dimension": dim, "subspace": " + ".join(subspace),
                    "emd_lead_acc": acc_emd, "flight_acc": acc_flt,
                    "contrast_ratio": contrast, "false_alarm_rate_pct": far_pct,
                    "master_score": master_score
                }
                combo_cache[sub_key] = eval_record
                combo_evals.append(eval_record)

        res_df = pd.DataFrame(combo_evals).sort_values("master_score", ascending=False).reset_index(drop=True)
        best_in_pool = res_df.iloc[0].copy()
        best_in_pool["weight_pool"] = pool_name
        best_pool_results.append(best_in_pool)

    # Master Pool Consolidation
    all_evaluated_df = pd.DataFrame(list(combo_cache.values())).sort_values("master_score", ascending=False).reset_index(drop=True)
    all_evaluated_df.to_csv(os.path.join(OUTPUT_DIR, f"dimension_evaluations_{win_ms}ms.csv"), index=False)

    best_3d = all_evaluated_df[all_evaluated_df["dimension"] == 3].iloc[0]
    best_4d = all_evaluated_df[all_evaluated_df["dimension"] == 4].iloc[0]
    best_5d = all_evaluated_df[all_evaluated_df["dimension"] == 5].iloc[0]
    best_overall = all_evaluated_df.iloc[0]

    # Print Dimensional Breakdown for the window
    logger.info(f"\n{'Dim':<5} {'Rank':<5} {'Subspace Formulation':<72} {'EMD Acc':<10} {'Flt Acc':<10} {'Contrast':<10} {'FAR (%)':<10}")
    logger.info("-" * 126)
    for d in [3, 4, 5]:
        sub_d = all_evaluated_df[all_evaluated_df["dimension"] == d].head(2)
        for rank, (_, r) in enumerate(sub_d.iterrows(), 1):
            logger.info(f"{r['dimension']:<5} {rank:<5} {r['subspace']:<72} {r['emd_lead_acc']:<9.1f}% "
                        f"{r['flight_acc']:<9.1f}% {r['contrast_ratio']:<9.2f}x {r['false_alarm_rate_pct']:<9.1f}%")
        logger.info("-" * 126)

    gain_4d_acc = best_4d["emd_lead_acc"] - best_3d["emd_lead_acc"]
    gain_4d_far = best_3d["false_alarm_rate_pct"] - best_4d["false_alarm_rate_pct"]
    gain_5d_acc = best_5d["emd_lead_acc"] - best_3d["emd_lead_acc"]
    gain_5d_far = best_3d["false_alarm_rate_pct"] - best_5d["false_alarm_rate_pct"]

    logger.info(f"\n[DIMENSIONAL MARGINAL GAIN ANALYSIS - {win_ms} ms WINDOW]")
    logger.info(f"  • Best 3D: {best_3d['subspace']}")
    logger.info(f"    -> Acc: {best_3d['emd_lead_acc']:.1f}% | Contrast: {best_3d['contrast_ratio']:.2f}x | FAR: {best_3d['false_alarm_rate_pct']:.1f}%")
    logger.info(f"  • Best 4D: {best_4d['subspace']}")
    logger.info(f"    -> Acc: {best_4d['emd_lead_acc']:.1f}% | Contrast: {best_4d['contrast_ratio']:.2f}x | FAR: {best_4d['false_alarm_rate_pct']:.1f}%")
    logger.info(f"    -> Delta vs 3D: Acc: {gain_4d_acc:+.1f}%, FAR Shift: {-gain_4d_far:+.1f}%")
    logger.info(f"  • Best 5D: {best_5d['subspace']}")
    logger.info(f"    -> Acc: {best_5d['emd_lead_acc']:.1f}% | Contrast: {best_5d['contrast_ratio']:.2f}x | FAR: {best_5d['false_alarm_rate_pct']:.1f}%")
    logger.info(f"    -> Delta vs 3D: Acc: {gain_5d_acc:+.1f}%, FAR Shift: {-gain_5d_far:+.1f}%\n")

    return {
        "win_ms": win_ms,
        "best_overall_dim": best_overall["dimension"],
        "best_overall_model": best_overall["subspace"],
        "best_overall_acc": best_overall["emd_lead_acc"],
        "best_overall_contrast": best_overall["contrast_ratio"],
        "best_overall_far": best_overall["false_alarm_rate_pct"],
        "best_3d_acc": best_3d["emd_lead_acc"],
        "best_3d_far": best_3d["false_alarm_rate_pct"],
        "gain_4d_acc": gain_4d_acc,
        "gain_4d_far": gain_4d_far,
        "gain_5d_acc": gain_5d_acc,
        "gain_5d_far": gain_5d_far,
    }

# ==============================================================================
# 3. RUN BENCHMARK ACROSS WINDOW LENGTHS & COMPILE SUMMARY
# ==============================================================================
summary_records = []
for w in WINDOW_SIZES_MS:
    res = analyze_window(w, raw_df)
    summary_records.append(res)

logger.info("=" * 115)
logger.info("HEAD-TO-HEAD WINDOW SUMMARY (ACROSS 11 SUBJECTS)")
logger.info("=" * 115)
sum_df = pd.DataFrame(summary_records)
sum_df.to_csv(os.path.join(OUTPUT_DIR, "final_window_comparison_summary.csv"), index=False)

for _, r in sum_df.iterrows():
    logger.info(f"Window Duration: {r['win_ms']} ms")
    logger.info(f"  • Optimal Subspace ({r['best_overall_dim']}D): {r['best_overall_model']}")
    logger.info(f"    -> EMD Accuracy: {r['best_overall_acc']:.1f}% | Contrast: {r['best_overall_contrast']:.2f}x | FAR: {r['best_overall_far']:.1f}%")
    logger.info(f"  • 4D Marginal Benefit over 3D: Acc: {r['gain_4d_acc']:+.1f}%, FAR Reduction: {r['gain_4d_far']:+.1f}%")
    logger.info(f"  • 5D Marginal Benefit over 3D: Acc: {r['gain_5d_acc']:+.1f}%, FAR Reduction: {r['gain_5d_far']:+.1f}%")
    logger.info("-" * 115)

logger.info(f"\nAll logs and dimensional comparison CSVs successfully written to:\n  {OUTPUT_DIR}")

INGESTING MULTI-WINDOW SLIDING DATASETS (11 SUBJECTS | 400 ms HORIZON)
Total Rows Ingested     : 16632
Extracted EMG Metrics   : 42
Evaluated Subjects      : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12]
Tested Window Sizes     : [30, 40, 50] ms
Tested Temporal Offsets : -200 ms to 200 ms

                                COMPREHENSIVE MULTI-METRIC DISCOVERY: 30 ms WINDOW                                 
C:\Users\user\AppData\Local\Temp\ipykernel_2696\3331543153.py:123: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[ 0.3385016 -0.6770032 -0.6770032 -0.6770032  2.3695112  0.3385016
 -0.6770032  1.3540064 -0.6770032 -0.6770032  0.3385016 -0.6770032]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df_rest_norm.loc[mask_rest, feature_cols] = (df_rest_norm.loc[mask_rest, feature_cols] - mu) / std
C:\Users\user\AppData\Local\Temp\ipykernel_2696\3331543153.py:123: FutureWarning: S

In [2]:
"""
================================================================================
TRUE ONSET DETECTION DISCOVERY & TEMPORAL ACCURACY PIPELINE (RE-ARMED & PATCHED)
Focus: Pure Onset Detection, EMD Lead, Temporal Accuracy & Delay
================================================================================
"""

import os
import sys
import warnings
import itertools
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.covariance import LedoitWolf

warnings.filterwarnings("ignore")

# ==============================================================================
# CONFIGURATION
# ==============================================================================
BASE_DIR = "datasets/DB2"
SUBJECTS = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12]  # Subject 11 excluded
WINDOW_SIZES_MS = [30, 40, 50]

# Statistical Significance (p = 0.0027 -> ~3 sigma in 1D)
ALPHA_CONFIDENCE = 0.0027  

# Operational Horizons (relative to t_kin = 0 ms)
QUARANTINE_HORIZON = -130   # Detections <= -130 ms are logged as Resting False Alarms
ACTION_START_HORIZON = -120 # Earliest valid physiological EMD horizon
MAX_SEARCH_HORIZON = 150    # Cutoff for missed detection
PERSIST_HITS = 3            # Require 3 consecutive 10 ms hops (30 ms) to confirm burst

OUTPUT_DIR = os.path.join(BASE_DIR, "onset_discovery_results_rearmed")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ==============================================================================
# 1. INGESTION
# ==============================================================================
print("=" * 115)
print(f"INGESTING 400 ms SLIDING DATASETS ({len(SUBJECTS)} SUBJECTS) FOR ONSET DISCOVERY")
print("=" * 115)

dfs = []
for s in SUBJECTS:
    candidates = [
        os.path.join(BASE_DIR, f"DB2_s{s}", f"S{s}_E1_A1_pruned_emg_sliding_features.csv"),
        os.path.join(BASE_DIR, f"s{s}", f"S{s}_E1_A1_pruned_emg_sliding_features.csv"),
    ]
    p = next((c for c in candidates if os.path.isfile(c)), None)
    if p:
        d = pd.read_csv(p)
        d["subject"] = s
        dfs.append(d)

if not dfs:
    raise FileNotFoundError("Feature CSV files not found. Ensure sliding extraction was completed.")

raw_df = pd.concat(dfs, ignore_index=True)
meta_cols = ["repetition", "movement", "epoch_type", "offset_ms", "window_ms", "idx_start", "idx_end", "subject"]
feature_cols = [c for c in raw_df.columns if c not in meta_cols and pd.api.types.is_numeric_dtype(raw_df[c])]

print(f"Total Rows Ingested   : {len(raw_df)}")
print(f"Extracted EMG Metrics : {len(feature_cols)}")
print(f"Active Subjects       : {sorted(raw_df['subject'].unique().tolist())}\n")


# ==============================================================================
# 2. ONSET DISCOVERY & SUBSPACE BENCHMARK
# ==============================================================================
def discover_onset_subspaces(win_ms, df_all):
    print("=" * 115)
    print(f"{f'ONSET DISCOVERY BENCHMARK: {win_ms} ms WINDOW (RE-ARMED ENGINE)':^115}")
    print("=" * 115)

    df_win = df_all[df_all["window_ms"] == win_ms].copy()
    df_rest = df_win[df_win["epoch_type"] == "rest_gray"].copy()
    df_peri = df_win[df_win["epoch_type"] == "peri_tkin"].copy()

    # Normalize relative to subject baseline rest
    df_rest_norm = df_rest.copy()
    df_norm = df_peri.copy()

    for s in SUBJECTS:
        s_rest = df_rest[df_rest["subject"] == s][feature_cols]
        mu = s_rest.mean()
        std = s_rest.std().replace(0, 1e-6)

        m_rest = df_rest_norm["subject"] == s
        m_peri = df_norm["subject"] == s
        if m_rest.sum() > 0:
            df_rest_norm.loc[m_rest, feature_cols] = (df_rest_norm.loc[m_rest, feature_cols] - mu) / std
        if m_peri.sum() > 0:
            df_norm.loc[m_peri, feature_cols] = (df_norm.loc[m_peri, feature_cols] - mu) / std

    # --------------------------------------------------------------------------
    # Step A: Screen Features for Bilateral Onset Sensitivity
    # --------------------------------------------------------------------------
    feat_screening = []
    
    for f in feature_cols:
        rest_noise = df_norm.loc[df_norm["offset_ms"] <= QUARANTINE_HORIZON, f].dropna().std()

        # Measure agonist departures in harmonized EMD window [-100, 0 ms]
        m5_emd = df_norm.loc[(df_norm["movement"] == 5) & df_norm["offset_ms"].between(-100, 0), f].dropna().abs().mean()
        m6_emd = df_norm.loc[(df_norm["movement"] == 6) & df_norm["offset_ms"].between(-100, 0), f].dropna().abs().mean()

        # Bilateral sensitivity (harmonic mean ensures feature responds to BOTH movements)
        bilateral_emd = (2.0 * m5_emd * m6_emd) / (m5_emd + m6_emd + 1e-6)
        onset_snr = bilateral_emd / (rest_noise + 1e-6)

        # Monotonic ramp leading to takeoff [-100, +40 ms]
        mean_traj = df_norm[f].abs().groupby(df_norm["offset_ms"]).mean()
        ramp_offsets = [t for t in sorted(df_norm["offset_ms"].unique()) if -100 <= t <= 40]
        ramp_vals = [mean_traj.get(t, 0) for t in ramp_offsets]
        
        if len(ramp_vals) > 4:
            rho_val, _ = stats.spearmanr(ramp_offsets, ramp_vals)
            spearman_rho = float(np.nan_to_num(rho_val, nan=0.0))
        else:
            spearman_rho = 0.0

        channel = "Bilateral" if ("diff_ratio" in f or "co_contraction" in f) else ("EDC" if "edc" in f else "FDS")

        feat_screening.append({
            "feature": f,
            "channel": channel,
            "onset_snr": onset_snr,
            "m5_departure": m5_emd,
            "m6_departure": m6_emd,
            "spearman_rho": max(0.0, spearman_rho),
            "onset_score": onset_snr * 0.5 + max(0.0, spearman_rho) * 3.0
        })

    scr_df = pd.DataFrame(feat_screening).sort_values("onset_score", ascending=False).reset_index(drop=True)
    
    print(f"\n[TOP 10 INDIVIDUAL ONSET DETECTOR CANDIDATES - {win_ms} ms WINDOW]")
    print(f"{'Rank':<5} {'Feature':<28} {'Channel':<10} {'Onset SNR':<11} {'M5 EMD (Z)':<11} {'M6 EMD (Z)':<11} {'Spearman':<10}")
    print("-" * 90)
    for idx, r in scr_df.head(10).iterrows():
        print(f"{idx+1:<5} {r['feature']:<28} {r['channel']:<10} {r['onset_snr']:<11.2f} "
              f"{r['m5_departure']:<11.2f} {r['m6_departure']:<11.2f} {r['spearman_rho']:<10.2f}")
    print()

    # --------------------------------------------------------------------------
    # Step B: Balanced Channel Selection & Collinearity Pruning
    # --------------------------------------------------------------------------
    candidate_pool = []
    candidate_pool.extend(scr_df[scr_df["channel"] == "EDC"].head(4)["feature"].tolist())
    candidate_pool.extend(scr_df[scr_df["channel"] == "FDS"].head(4)["feature"].tolist())
    candidate_pool.extend(scr_df[scr_df["channel"] == "Bilateral"].head(2)["feature"].tolist())

    corr_m = df_norm[candidate_pool].corr().abs()
    final_pool = []
    for f in candidate_pool:
        if not any(corr_m.loc[f, ex] > 0.85 for ex in final_pool):
            final_pool.append(f)

    print(f"Balanced Orthogonal Pool for Onset Subspaces ({len(final_pool)} features):")
    for f in final_pool:
        ch = scr_df.loc[scr_df['feature'] == f, 'channel'].values[0]
        print(f"  • [{ch:<9}] {f}")
    print()

    # --------------------------------------------------------------------------
    # Step C: Combinatorial Search with Persistent Re-Arming Logic
    # --------------------------------------------------------------------------
    all_combos = []
    for k in [2, 3, 4]:
        all_combos.extend(list(itertools.combinations(final_pool, k)))

    combo_evals = []

    for combo in all_combos:
        subspace = list(combo)
        dim = len(subspace)

        # Dimension-scaled statistical threshold: sqrt(chi2_inv(1 - alpha, df=dim))
        dm_thresh = np.sqrt(stats.chi2.ppf(1.0 - ALPHA_CONFIDENCE, df=dim))

        trial_deltas = []
        fa_count = 0
        missed_count = 0
        total_trials = 0

        for s in SUBJECTS:
            s_rest = df_rest_norm[df_rest_norm["subject"] == s][subspace].dropna().values
            if len(s_rest) < dim + 2:
                continue

            lw = LedoitWolf().fit(s_rest)
            mu_r, inv_cov_r = lw.location_, lw.precision_

            s_peri = df_norm[df_norm["subject"] == s]
            for (rep, mov), t_df in s_peri.groupby(["repetition", "movement"]):
                total_trials += 1
                t_df = t_df.sort_values("offset_ms").dropna(subset=subspace)
                offsets = t_df["offset_ms"].values
                X = t_df[subspace].values

                diff = X - mu_r
                dm_vals = np.sqrt(np.sum(diff @ inv_cov_r * diff, axis=1))

                # Streaming detector with 30 ms persistence and re-arming
                consec_hits = 0
                quarantine_fa_occurred = False
                detected_tau = None

                for b_idx in range(len(dm_vals)):
                    tau = offsets[b_idx]
                    if dm_vals[b_idx] >= dm_thresh:
                        consec_hits += 1
                        if consec_hits >= PERSIST_HITS and detected_tau is None:
                            candidate_tau = offsets[b_idx - (PERSIST_HITS - 1)]
                            if candidate_tau <= QUARANTINE_HORIZON:
                                quarantine_fa_occurred = True
                            elif candidate_tau >= ACTION_START_HORIZON:
                                detected_tau = candidate_tau
                                break
                    else:
                        consec_hits = 0

                if quarantine_fa_occurred:
                    fa_count += 1

                if detected_tau is not None and detected_tau <= MAX_SEARCH_HORIZON:
                    trial_deltas.append(detected_tau)
                else:
                    missed_count += 1

        # Statistical Metrics
        valid_count = len(trial_deltas)
        sensitivity = (valid_count / max(1, total_trials)) * 100.0
        far_pct = (fa_count / max(1, total_trials)) * 100.0
        
        hit_100ms = (np.sum(np.abs(trial_deltas) <= 100.0) / max(1, total_trials)) * 100.0
        hit_lead_80ms = (np.sum((-80.0 <= np.array(trial_deltas)) & (np.array(trial_deltas) <= 0.0)) / max(1, total_trials)) * 100.0

        median_delta = np.median(trial_deltas) if valid_count > 0 else np.nan
        mean_delta   = np.mean(trial_deltas) if valid_count > 0 else np.nan
        max_delay    = np.max(trial_deltas) if valid_count > 0 else np.nan
        max_lead     = np.min(trial_deltas) if valid_count > 0 else np.nan
        iqr_delta    = stats.iqr(trial_deltas) if valid_count > 0 else np.nan

        # Balanced Objective Function (with NaN safeguard)
        delay_penalty = max(0.0, max_delay) if not np.isnan(max_delay) else 0.0

        onset_perf_score = (
            (sensitivity * 1.0) + 
            (hit_100ms * 0.5) + 
            (hit_lead_80ms * 0.5) - 
            (far_pct * 1.0) - 
            (delay_penalty * 0.1)
        )

        combo_evals.append({
            "dimension": dim,
            "subspace": " + ".join(subspace),
            "dm_thresh": dm_thresh,
            "sensitivity_pct": sensitivity,
            "far_pct": far_pct,
            "hit_100ms_pct": hit_100ms,
            "hit_lead_80ms_pct": hit_lead_80ms,
            "median_delta_ms": median_delta,
            "mean_delta_ms": mean_delta,
            "max_lead_ms": max_lead,
            "max_delay_ms": max_delay,
            "iqr_delta_ms": iqr_delta,
            "score": onset_perf_score
        })

    eval_df = pd.DataFrame(combo_evals).sort_values("score", ascending=False).reset_index(drop=True)
    eval_df.to_csv(os.path.join(OUTPUT_DIR, f"onset_subspaces_rearmed_{win_ms}ms.csv"), index=False)

    print(f"\n[TOP 5 ONSET DETECTOR SUBSPACES - {win_ms} ms WINDOW]")
    print(f"{'Dim':<5} {'Subspace Formulation':<65} {'Sens (%)':<9} {'FAR (%)':<8} {'[-100,+100]':<12} {'Median Delta':<14} {'Max Delay':<10}")
    print("-" * 125)
    for idx, r in eval_df.head(5).iterrows():
        print(f"{r['dimension']:<5} {r['subspace']:<65} {r['sensitivity_pct']:<8.1f}% {r['far_pct']:<7.1f}% "
              f"{r['hit_100ms_pct']:<11.1f}% {r['median_delta_ms']:<+12.1f} ms {r['max_delay_ms']:<+8.1f} ms")
    print()

    # Dimension extraction with collinearity pruning safeguard
    best_2d = eval_df[eval_df["dimension"] == 2].iloc[0] if not eval_df[eval_df["dimension"] == 2].empty else None
    best_3d = eval_df[eval_df["dimension"] == 3].iloc[0] if not eval_df[eval_df["dimension"] == 3].empty else None
    best_4d = eval_df[eval_df["dimension"] == 4].iloc[0] if not eval_df[eval_df["dimension"] == 4].empty else None
    best_overall = eval_df.iloc[0]

    return {
        "win_ms": win_ms,
        "best_overall_dim": best_overall["dimension"],
        "best_overall_subspace": best_overall["subspace"],
        "best_overall_sens": best_overall["sensitivity_pct"],
        "best_overall_far": best_overall["far_pct"],
        "best_overall_hit100": best_overall["hit_100ms_pct"],
        "best_overall_median_delta": best_overall["median_delta_ms"],
        "best_overall_max_delay": best_overall["max_delay_ms"],
        "best_3d_subspace": best_3d["subspace"] if best_3d is not None else "N/A",
        "best_3d_sens": best_3d["sensitivity_pct"] if best_3d is not None else 0.0,
        "best_3d_hit100": best_3d["hit_100ms_pct"] if best_3d is not None else 0.0,
        "best_3d_median_delta": best_3d["median_delta_ms"] if best_3d is not None else np.nan,
    }


# ==============================================================================
# 3. RUN PIPELINE ACROSS ALL WINDOW LENGTHS
# ==============================================================================
summary_list = []
for w in WINDOW_SIZES_MS:
    res = discover_onset_subspaces(w, raw_df)
    summary_list.append(res)

sum_df = pd.DataFrame(summary_list)
sum_df.to_csv(os.path.join(OUTPUT_DIR, "onset_discovery_rearmed_master_summary.csv"), index=False)

print("=" * 115)
print("FINAL HEAD-TO-HEAD ONSET DETECTION SUMMARY (RE-ARMED ENGINE)")
print("=" * 115)
for _, r in sum_df.iterrows():
    print(f"Window Duration: {r['win_ms']} ms")
    print(f"  • Optimal Subspace ({r['best_overall_dim']}D): {r['best_overall_subspace']}")
    print(f"  • Detection Sensitivity       : {r['best_overall_sens']:.1f}%")
    print(f"  • Premature False Alarm Rate  : {r['best_overall_far']:.1f}%")
    print(f"  • Temporal Landing ([-100,+100]): {r['best_overall_hit100']:.1f}%")
    print(f"  • Median Activation Lead      : {r['best_overall_median_delta']:+.1f} ms relative to t_kin")
    print(f"  • Worst-Case Delay (Max Lag)  : {r['best_overall_max_delay']:+.1f} ms")
    print("-" * 115)

print(f"\nAll logs and comparison summaries written to:\n  {OUTPUT_DIR}")

INGESTING 400 ms SLIDING DATASETS (11 SUBJECTS) FOR ONSET DISCOVERY
Total Rows Ingested   : 16632
Extracted EMG Metrics : 42
Active Subjects       : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12]

                             ONSET DISCOVERY BENCHMARK: 30 ms WINDOW (RE-ARMED ENGINE)                             

[TOP 10 INDIVIDUAL ONSET DETECTOR CANDIDATES - 30 ms WINDOW]
Rank  Feature                      Channel    Onset SNR   M5 EMD (Z)  M6 EMD (Z)  Spearman  
------------------------------------------------------------------------------------------
1     edc_MAV                      EDC        0.95        2.66        1.73        0.95      
2     edc_RMS                      EDC        0.83        3.10        1.88        0.96      
3     edc_LOG                      EDC        1.00        2.08        1.49        0.94      
4     fds_WAMP                     FDS        0.91        2.34        1.34        0.94      
5     edc_Hjorth_Mobility          EDC        0.70        3.69        1.95      

In [3]:
"""
================================================================================
DIAGNOSTIC AUDIT & ADAPTIVE RECOVERY PIPELINE FOR MISSED ONSETS
Dataset: NinaPro DB2 (Subjects 1-10, 12 | EDC Ch 9 vs. FDS Ch 10 | M5 vs. M6)
Anchor: t_kin = 0 ms (PIP joint mechanical takeoff)
================================================================================
"""

import os
import sys
import warnings
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.covariance import LedoitWolf
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# ==============================================================================
# CONFIGURATION
# ==============================================================================
BASE_DIR = "datasets/DB2"
SUBJECTS = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12]  # Subject 11 excluded
WINDOW_SIZES_MS = [30, 40]                      # Primary operational window sizes

# Benchmark Models to Diagnose (Top Performers from Empirical Discovery)
EVAL_MODELS = {
    30: {
        "3D": ["edc_MAV", "edc_Hjorth_Mobility", "fds_WAMP"],
        "4D": ["edc_MAV", "edc_Hjorth_Mobility", "fds_WAMP", "fds_WL"],
    },
    40: {
        "3D": ["edc_MAV", "fds_SSC", "fds_WL"],
        "4D": ["edc_MAV", "edc_WL", "fds_SSC", "fds_WL"],
    }
}

# Standard Baseline Parameters
ALPHA_STRICT = 0.0027        # p = 0.0027 (~3 sigma in 1D)
QUARANTINE_HORIZON = -130    # <= -130 ms: Deep Rest Quarantine
ACTION_START_HORIZON = -120  # Earliest physiological activation lead
MAX_SEARCH_HORIZON = 150     # +150 ms relative to t_kin
PERSIST_HITS_STRICT = 3      # 3 hops x 10 ms = 30 ms sustained breach

# Dual-Tier Ramp Recovery Parameters
ALPHA_RELAXED = 0.02         # Lower threshold for sluggish contractions
PERSIST_HITS_RELAXED = 5     # 5 hops x 10 ms = 50 ms persistence required

OUTPUT_DIR = os.path.join(BASE_DIR, "missed_onsets_diagnostic_results")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ==============================================================================
# 1. DATA INGESTION
# ==============================================================================
print("=" * 115)
print("INGESTING DATASETS FOR MISSED ONSET AUDIT (11 SUBJECTS | 132 TRIALS)")
print("=" * 115)

dfs = []
for s in SUBJECTS:
    candidates = [
        os.path.join(BASE_DIR, f"DB2_s{s}", f"S{s}_E1_A1_pruned_emg_sliding_features.csv"),
        os.path.join(BASE_DIR, f"s{s}", f"S{s}_E1_A1_pruned_emg_sliding_features.csv"),
    ]
    p = next((c for c in candidates if os.path.isfile(c)), None)
    if p:
        d = pd.read_csv(p)
        d["subject"] = s
        dfs.append(d)

if not dfs:
    raise FileNotFoundError("Feature CSV files not found. Check BASE_DIR path.")

raw_df = pd.concat(dfs, ignore_index=True)
meta_cols = ["repetition", "movement", "epoch_type", "offset_ms", "window_ms", "idx_start", "idx_end", "subject"]
feature_cols = [c for c in raw_df.columns if c not in meta_cols and pd.api.types.is_numeric_dtype(raw_df[c])]

print(f"Total Rows Ingested : {len(raw_df)}")
print(f"Active Subjects     : {sorted(raw_df['subject'].unique().tolist())}\n")


# ==============================================================================
# 2. TRIAL-BY-TRIAL FAILURE MODE AUDIT ENGINE
# ==============================================================================
def audit_missed_onsets(win_ms, dim_name, subspace, df_all):
    dim = len(subspace)
    tau_strict = np.sqrt(stats.chi2.ppf(1.0 - ALPHA_STRICT, df=dim))
    tau_relaxed = np.sqrt(stats.chi2.ppf(1.0 - ALPHA_RELAXED, df=dim))

    df_win = df_all[df_all["window_ms"] == win_ms].copy()
    df_rest = df_win[df_win["epoch_type"] == "rest_gray"].copy()
    df_peri = df_win[df_win["epoch_type"] == "peri_tkin"].copy()

    # Per-subject baseline standardization
    df_rest_norm = df_rest.copy()
    df_norm = df_peri.copy()

    for s in SUBJECTS:
        s_rest = df_rest[df_rest["subject"] == s][feature_cols]
        mu = s_rest.mean()
        std = s_rest.std().replace(0, 1e-6)

        m_rest = df_rest_norm["subject"] == s
        m_peri = df_norm["subject"] == s
        if m_rest.sum() > 0:
            df_rest_norm.loc[m_rest, feature_cols] = (df_rest_norm.loc[m_rest, feature_cols] - mu) / std
        if m_peri.sum() > 0:
            df_norm.loc[m_peri, feature_cols] = (df_norm.loc[m_peri, feature_cols] - mu) / std

    trial_records = []

    for s in SUBJECTS:
        s_rest = df_rest_norm[df_rest_norm["subject"] == s][subspace].dropna().values
        if len(s_rest) < dim + 2:
            continue

        lw = LedoitWolf().fit(s_rest)
        mu_r, inv_cov_r = lw.location_, lw.precision_

        s_peri = df_norm[df_norm["subject"] == s]
        for (rep, mov), t_df in s_peri.groupby(["repetition", "movement"]):
            t_df = t_df.sort_values("offset_ms").dropna(subset=subspace)
            offsets = t_df["offset_ms"].values
            X = t_df[subspace].values

            diff = X - mu_r
            dm_vals = np.sqrt(np.sum(diff @ inv_cov_r * diff, axis=1))

            # Quantify Contraction Dynamics
            # Deep rest baseline noise (tau <= -130 ms)
            mask_quar = offsets <= QUARANTINE_HORIZON
            rest_dm_peak = np.max(dm_vals[mask_quar]) if np.any(mask_quar) else 0.0

            # Active action corridor (-100 ms to +150 ms)
            mask_action = (offsets >= -100) & (offsets <= MAX_SEARCH_HORIZON)
            action_dm_peak = np.max(dm_vals[mask_action]) if np.any(mask_action) else 0.0
            action_dm_mean = np.mean(dm_vals[mask_action]) if np.any(mask_action) else 0.0

            # Standard Detector Simulation (Strict Threshold)
            consec_strict = 0
            quarantine_fa = False
            detected_tau_strict = None

            for b_idx in range(len(dm_vals)):
                if dm_vals[b_idx] >= tau_strict:
                    consec_strict += 1
                    if consec_strict >= PERSIST_HITS_STRICT and detected_tau_strict is None:
                        cand_tau = offsets[b_idx - (PERSIST_HITS_STRICT - 1)]
                        if cand_tau <= QUARANTINE_HORIZON:
                            quarantine_fa = True
                        elif cand_tau >= ACTION_START_HORIZON:
                            detected_tau_strict = cand_tau
                            break
                else:
                    consec_strict = 0

            # Dual-Tier Recovery Simulation (Standard Strict + Persistent Relaxed)
            consec_dual_strict = 0
            consec_dual_relaxed = 0
            dual_quarantine_fa = False
            detected_tau_dual = None

            for b_idx in range(len(dm_vals)):
                # Tier 1: Strict Threshold (3 hops)
                if dm_vals[b_idx] >= tau_strict:
                    consec_dual_strict += 1
                    if consec_dual_strict >= PERSIST_HITS_STRICT and detected_tau_dual is None:
                        cand_tau = offsets[b_idx - (PERSIST_HITS_STRICT - 1)]
                        if cand_tau <= QUARANTINE_HORIZON:
                            dual_quarantine_fa = True
                        elif cand_tau >= ACTION_START_HORIZON:
                            detected_tau_dual = cand_tau
                            break
                else:
                    consec_dual_strict = 0

                # Tier 2: Relaxed Threshold for Sluggish Contractions (Requires 5 consecutive hops)
                if dm_vals[b_idx] >= tau_relaxed:
                    consec_dual_relaxed += 1
                    if consec_dual_relaxed >= PERSIST_HITS_RELAXED and detected_tau_dual is None:
                        cand_tau = offsets[b_idx - (PERSIST_HITS_RELAXED - 1)]
                        if cand_tau <= QUARANTINE_HORIZON:
                            dual_quarantine_fa = True
                        elif cand_tau >= ACTION_START_HORIZON:
                            detected_tau_dual = cand_tau
                            break
                else:
                    consec_dual_relaxed = 0

            # Determine Failure Mode
            if detected_tau_strict is not None and detected_tau_strict <= MAX_SEARCH_HORIZON:
                failure_mode = "DETECTED_VALID"
            elif action_dm_peak < tau_strict:
                failure_mode = "SUBTHRESHOLD_AMPLITUDE"  # Never reached activation threshold
            else:
                # Reached threshold, but did not sustain 3 consecutive hops
                hops_above = np.sum(dm_vals[mask_action] >= tau_strict)
                if hops_above > 0 and hops_above < PERSIST_HITS_STRICT:
                    failure_mode = "TRANSIENT_INSUFFICIENT_PERSISTENCE"
                else:
                    failure_mode = "TIMING_EXCEEDED_HORIZON"

            trial_records.append({
                "window_ms": win_ms,
                "model": f"{win_ms}ms_{dim_name}",
                "subject": s,
                "repetition": rep,
                "movement": mov,
                "movement_name": "Hand Open (EDC)" if mov == 5 else "Fist Clench (FDS)",
                "action_dm_peak": action_dm_peak,
                "action_dm_mean": action_dm_mean,
                "rest_dm_peak": rest_dm_peak,
                "tau_strict": tau_strict,
                "tau_relaxed": tau_relaxed,
                "detected_strict": detected_tau_strict is not None and detected_tau_strict <= MAX_SEARCH_HORIZON,
                "tau_detected_strict": detected_tau_strict,
                "detected_dual": detected_tau_dual is not None and detected_tau_dual <= MAX_SEARCH_HORIZON,
                "tau_detected_dual": detected_tau_dual,
                "quarantine_fa_strict": quarantine_fa,
                "quarantine_fa_dual": dual_quarantine_fa,
                "failure_mode": failure_mode
            })

    return pd.DataFrame(trial_records)


# ==============================================================================
# 3. RUN AUDIT ACROSS TARGET MODELS
# ==============================================================================
all_audit_dfs = []
for w, models in EVAL_MODELS.items():
    for d_name, feats in models.items():
        res_df = audit_missed_onsets(w, d_name, feats, raw_df)
        all_audit_dfs.append(res_df)

audit_df = pd.concat(all_audit_dfs, ignore_index=True)
audit_df.to_csv(os.path.join(OUTPUT_DIR, "trial_level_missed_onsets_audit.csv"), index=False)

# ==============================================================================
# 4. STRUCTURED REPORTING & STATISTICAL BREAKDOWN
# ==============================================================================
print("=" * 115)
print("TRIAL-LEVEL FAILURE MODE TAXONOMY ACROSS BENCHMARK MODELS")
print("=" * 115)

# Breakdown by Model
for model_name, m_df in audit_df.groupby("model"):
    total_trials = len(m_df)
    counts = m_df["failure_mode"].value_counts()
    sens_base = (m_df["detected_strict"].sum() / total_trials) * 100.0
    sens_dual = (m_df["detected_dual"].sum() / total_trials) * 100.0
    fa_base = (m_df["quarantine_fa_strict"].sum() / total_trials) * 100.0
    fa_dual = (m_df["quarantine_fa_dual"].sum() / total_trials) * 100.0

    print(f"\nMODEL: {model_name} (Total Trials: {total_trials})")
    print(f"  • Standard Sensitivity (Strict): {sens_base:.1f}% | Quarantine FAR: {fa_base:.1f}%")
    print(f"  • Dual-Tier Sensitivity (Ramp) : {sens_dual:.1f}% | Quarantine FAR: {fa_dual:.1f}%")
    print(f"  • Absolute Sensitivity Gain    : {sens_dual - sens_base:+.1f}%")
    print("  • Failure Mode Distribution (Standard Engine):")
    for f_mode, cnt in counts.items():
        print(f"      - {f_mode:<35}: {cnt:>3} trials ({cnt/total_trials*100:.1f}%)")

# Detailed Movement and Subject Matrix for the Winning 30ms 4D Model
w30_4d = audit_df[audit_df["model"] == "30ms_4D"].copy()

print("\n" + "=" * 115)
print("DETAILED BREAKDOWN FOR PRIMARY DETECTOR: 30ms 4D (edc_MAV + edc_Hjorth_Mobility + fds_WAMP + fds_WL)")
print("=" * 115)

# Movement Bias (M5 Hand Open vs. M6 Fist Clench)
mov_summary = w30_4d.groupby("movement_name").agg(
    total=("detected_strict", "count"),
    detected_base=("detected_strict", "sum"),
    detected_dual=("detected_dual", "sum"),
    peak_dm_mean=("action_dm_peak", "mean")
)
mov_summary["sens_base_pct"] = (mov_summary["detected_base"] / mov_summary["total"]) * 100.0
mov_summary["sens_dual_pct"] = (mov_summary["detected_dual"] / mov_summary["total"]) * 100.0

print("\n1. Compartment / Movement Performance:")
for mov_name, r in mov_summary.iterrows():
    print(f"  • {mov_name:<24}: Base Sens = {r['sens_base_pct']:.1f}% -> Dual Recovered = {r['sens_dual_pct']:.1f}% | Mean Peak D_M = {r['peak_dm_mean']:.2f}")

# Subject-Specific Sensitivity & Peak D_M Audit
sub_summary = w30_4d.groupby("subject").agg(
    total=("detected_strict", "count"),
    detected_base=("detected_strict", "sum"),
    detected_dual=("detected_dual", "sum"),
    subthreshold_count=("failure_mode", lambda x: (x == "SUBTHRESHOLD_AMPLITUDE").sum()),
    peak_dm_mean=("action_dm_peak", "mean"),
    rest_dm_mean=("rest_dm_peak", "mean")
)
sub_summary["sens_base_pct"] = (sub_summary["detected_base"] / sub_summary["total"]) * 100.0
sub_summary["sens_dual_pct"] = (sub_summary["detected_dual"] / sub_summary["total"]) * 100.0

print("\n2. Subject-by-Subject Audit Matrix:")
print(f"{'Subject':<10} | {'Base Sens':<10} | {'Dual Sens':<10} | {'Subthreshold':<14} | {'Mean Peak D_M':<15} | {'Mean Rest D_M':<15}")
print("-" * 85)
for s_id, r in sub_summary.iterrows():
    print(f"Subject {s_id:<2} | {r['sens_base_pct']:>8.1f}% | {r['sens_dual_pct']:>8.1f}% | {int(r['subthreshold_count']):>12} | {r['peak_dm_mean']:>13.2f}  | {r['rest_dm_mean']:>13.2f}")

# Extract Peak D_M Statistics for Missed vs. Detected Trials
missed_trials = w30_4d[~w30_4d["detected_strict"]]
detected_trials = w30_4d[w30_4d["detected_strict"]]

print("\n3. Contraction Dynamic Range Comparison (30ms 4D Model):")
print(f"  • Detected Trials (n={len(detected_trials)}): Peak D_M Mean = {detected_trials['action_dm_peak'].mean():.2f} (Median: {detected_trials['action_dm_peak'].median():.2f})")
print(f"  • Missed Trials   (n={len(missed_trials)}): Peak D_M Mean = {missed_trials['action_dm_peak'].mean():.2f} (Median: {missed_trials['action_dm_peak'].median():.2f})")
print(f"  • Strict Tripwire Threshold (tau_strict) : {w30_4d['tau_strict'].iloc[0]:.2f}")
print(f"  • Relaxed Ramp Threshold   (tau_relaxed): {w30_4d['tau_relaxed'].iloc[0]:.2f}")

# ==============================================================================
# 5. DIAGNOSTIC VISUALIZATION
# ==============================================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot A: Failure Mode Distribution
mode_counts = w30_4d["failure_mode"].value_counts()
mode_labels = [m.replace("_", "\n") for m in mode_counts.index]
colors = ["#2ca02c" if "DETECTED" in m else "#d62728" for m in mode_counts.index]
axes[0].bar(mode_labels, mode_counts.values, color=colors, edgecolor="black", alpha=0.85)
axes[0].set_ylabel("Number of Trials (out of 132)")
axes[0].set_title("Failure Mode Distribution (30ms 4D Model)")
axes[0].grid(axis="y", linestyle="--", alpha=0.7)

# Plot B: Peak Contraction D_M Distribution (Detected vs. Missed)
axes[1].hist(detected_trials["action_dm_peak"], bins=15, alpha=0.6, color="#1f77b4", label=f"Detected (n={len(detected_trials)})", edgecolor="black")
axes[1].hist(missed_trials["action_dm_peak"], bins=15, alpha=0.7, color="#ff7f0e", label=f"Missed (n={len(missed_trials)})", edgecolor="black")
axes[1].axvline(w30_4d["tau_strict"].iloc[0], color="red", linestyle="--", linewidth=2, label=r"Strict Threshold ($\tau_{strict}=4.03$)")
axes[1].axvline(w30_4d["tau_relaxed"].iloc[0], color="purple", linestyle=":", linewidth=2, label=r"Relaxed Ramp ($\tau_{relaxed}=3.12$)")
axes[1].set_xlabel("Peak Mahalanobis Distance ($D_M$) in Action Window")
axes[1].set_ylabel("Trial Count")
axes[1].set_title("Peak $D_M$ Separation: Detected vs. Missed")
axes[1].legend()

# Plot C: Sensitivity Recovery by Subject (Strict vs. Dual-Tier)
x_indices = np.arange(len(sub_summary))
bar_width = 0.38
axes[2].bar(x_indices - bar_width/2, sub_summary["sens_base_pct"], width=bar_width, label="Standard Strict Engine", color="#1f77b4", alpha=0.85, edgecolor="black")
axes[2].bar(x_indices + bar_width/2, sub_summary["sens_dual_pct"], width=bar_width, label="Dual-Tier Ramp Recovery", color="#2ca02c", alpha=0.85, edgecolor="black")
axes[2].set_xticks(x_indices)
axes[2].set_xticklabels([f"S{s}" for s in sub_summary.index])
axes[2].set_ylabel("Onset Sensitivity (%)")
axes[2].set_title("Subject Sensitivity: Standard vs. Dual-Tier Recovery")
axes[2].set_ylim(0, 105)
axes[2].axhline(100, color="gray", linestyle="--", alpha=0.5)
axes[2].legend()

plt.tight_layout()
plot_path = os.path.join(OUTPUT_DIR, "missed_onsets_diagnostic_analysis.png")
plt.savefig(plot_path, dpi=300)
plt.close()

print(f"\nDiagnostic plots and audit CSV successfully saved to:\n  {OUTPUT_DIR}\n")

INGESTING DATASETS FOR MISSED ONSET AUDIT (11 SUBJECTS | 132 TRIALS)
Total Rows Ingested : 16632
Active Subjects     : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12]

TRIAL-LEVEL FAILURE MODE TAXONOMY ACROSS BENCHMARK MODELS

MODEL: 30ms_3D (Total Trials: 132)
  • Standard Sensitivity (Strict): 71.2% | Quarantine FAR: 19.7%
  • Dual-Tier Sensitivity (Ramp) : 73.5% | Quarantine FAR: 21.2%
  • Absolute Sensitivity Gain    : +2.3%
  • Failure Mode Distribution (Standard Engine):
      - DETECTED_VALID                     :  94 trials (71.2%)
      - SUBTHRESHOLD_AMPLITUDE             :  21 trials (15.9%)
      - TRANSIENT_INSUFFICIENT_PERSISTENCE :  11 trials (8.3%)
      - TIMING_EXCEEDED_HORIZON            :   6 trials (4.5%)

MODEL: 30ms_4D (Total Trials: 132)
  • Standard Sensitivity (Strict): 76.5% | Quarantine FAR: 23.5%
  • Dual-Tier Sensitivity (Ramp) : 77.3% | Quarantine FAR: 24.2%
  • Absolute Sensitivity Gain    : +0.8%
  • Failure Mode Distribution (Standard Engine):
      - DETECTED_VAL

In [4]:
"""
================================================================================
REMEDY VALIDATION & ROBUST ONSET DETECTION BENCHMARK
Dataset: NinaPro DB2 (Subjects 1-10, 12 | EDC Ch 9 vs. FDS Ch 10 | M5 vs. M6)
Focus: Resolving Subthreshold Contractions, Channel Noise (S7), and Persistence
================================================================================

SCIENTIFIC CONTEXT & ARCHITECTURAL MOTIVATION:
---------------------------------------------
In the previous audit, 23.5% (31/132) of trials failed detection:
  1. Subthreshold Amplitude (16 trials): Peak D_M in the action window failed 
     to breach strict statistical thresholds (prevalent in Subject 7 and 8).
  2. Transient Insufficient Persistence (9 trials): Signal breached threshold 
     intermittently but failed the unbroken 3-hop (30 ms) persistence rule 
     (Subject 7 accounted for 5 of these 9 trials due to heavy FDS baseline noise).
  3. Timing Exceeded Horizon (6 trials): Contractions were sluggish, reaching 
     sustained takeoff between +150 ms and +200 ms past t_kin.

PHYSICAL ROOT CAUSE FOR SUBJECT 7 & CORRUPTED CHANNELS:
-------------------------------------------------------
In Subject 7, the FDS channel (Channel 10) exhibits continuous, high-amplitude 
broadband noise / electrode impedance saturation ("red strip end-to-end with 
no major bumps or downs").
When using a Monolithic 4D Covariance Matrix [EDC_1, EDC_2, FDS_1, FDS_2]:
  - FDS baseline noise inflates the resting covariance matrix, attenuating the 
    effective dynamic range of the entire Mahalanobis distance vector.
  - For Hand Open (EDC agonist), clean extensor departure is diluted by 
    stochastic noise fluctuations in FDS, causing intermittent dropouts that 
    fail the strict 3-hop persistence test.
  - For Fist Clench (FDS agonist), the signal cannot emerge above the high noise 
    floor, collapsing into subthreshold failure.

EVALUATED REMEDIES IN THIS BENCHMARK:
------------------------------------
1. Baseline: Standard Monolithic 4D Detector (Strict Chi2, 30 ms unbroken persistence).
2. Remedy 1 (Leaky Integrator / CUSUM Accumulator): Replaces hard consecutive-frame 
   rules with a continuous evidence accumulator S_t = max(0, S_{t-1} + D_M - drift). 
   Bridges single-frame micro-dips caused by noisy channels.
3. Remedy 2 (Extended Action Horizon to +200 ms): Captures sluggish and delayed 
   motor unit recruitment ramps without penalizing slow reaction times.
4. Remedy 3 (Decoupled Bilateral OR-Gate): Evaluates EDC (2D) and FDS (2D) independently:
   D_trigger = max( D_EDC / tau_2D, D_FDS / tau_2D ). Prevents a noisy FDS channel 
   from corrupting or diluting clean EDC contractions.
5. Remedy 4 (Robust Median/MAD Baseline Normalization): Replaces sample mean/std 
   with median and median absolute deviation (MAD) to prevent baseline artifacts 
   from desensitizing z-score scaling.
6. Remedy 5 (Unified Robust Pipeline): Integrates Decoupled Bilateral OR-gate + 
   Leaky Energy Accumulator + Extended Horizon.
================================================================================
"""

import os
import sys
import warnings
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.covariance import LedoitWolf
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# ==============================================================================
# CONFIGURATION
# ==============================================================================
BASE_DIR = "datasets/DB2"
SUBJECTS = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12]  # Subject 11 strictly excluded
WIN_MS = 30                                      # Primary 30 ms window

# Target Feature Subspaces
MONOLITHIC_4D = ["edc_MAV", "edc_Hjorth_Mobility", "fds_WAMP", "fds_WL"]
DECOUPLED_EDC_2D = ["edc_MAV", "edc_Hjorth_Mobility"]
DECOUPLED_FDS_2D = ["fds_WAMP", "fds_WL"]

# Threshold Significance (p = 0.0027 -> ~3 sigma equivalent)
ALPHA_CONF = 0.0027

# Temporal Horizon Definitions
QUARANTINE_HORIZON = -130     # <= -130 ms: Deep Rest Quarantine
ACTION_START_HORIZON = -120   # Earliest physiological lead
BASE_MAX_HORIZON = 150        # Standard cutoff (+150 ms)
EXT_MAX_HORIZON = 200         # Extended cutoff (+200 ms)

OUTPUT_DIR = os.path.join(BASE_DIR, "remedy_evaluation_results")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ==============================================================================
# 1. DATA INGESTION
# ==============================================================================
print("=" * 115)
print("INGESTING DATASETS FOR REMEDY BENCHMARKING (11 SUBJECTS | 132 TRIALS)")
print("=" * 115)

dfs = []
for s in SUBJECTS:
    candidates = [
        os.path.join(BASE_DIR, f"DB2_s{s}", f"S{s}_E1_A1_pruned_emg_sliding_features.csv"),
        os.path.join(BASE_DIR, f"s{s}", f"S{s}_E1_A1_pruned_emg_sliding_features.csv"),
    ]
    p = next((c for c in candidates if os.path.isfile(c)), None)
    if p:
        d = pd.read_csv(p)
        d["subject"] = s
        dfs.append(d)

if not dfs:
    raise FileNotFoundError("Feature CSV files not found. Check BASE_DIR path.")

raw_df = pd.concat(dfs, ignore_index=True)
raw_df = raw_df[raw_df["window_ms"] == WIN_MS].copy()
meta_cols = ["repetition", "movement", "epoch_type", "offset_ms", "window_ms", "idx_start", "idx_end", "subject"]
feature_cols = [c for c in raw_df.columns if c not in meta_cols and pd.api.types.is_numeric_dtype(raw_df[c])]

print(f"Total Window Rows Ingested : {len(raw_df)} (Window: {WIN_MS} ms)")
print(f"Active Subjects             : {sorted(raw_df['subject'].unique().tolist())}\n")


# ==============================================================================
# 2. BASELINE AND ROBUST NORMALIZATION FUNCTIONS
# ==============================================================================
def compute_subject_normalizations(df_all):
    df_rest = df_all[df_all["epoch_type"] == "rest_gray"].copy()
    df_peri = df_all[df_all["epoch_type"] == "peri_tkin"].copy()

    # Standard Gaussian Z-score (Mean / Std)
    df_norm_std = df_peri.copy()
    df_rest_std = df_rest.copy()

    # Robust Normalization (Median / MAD)
    df_norm_rob = df_peri.copy()
    df_rest_rob = df_rest.copy()

    for s in SUBJECTS:
        s_rest = df_rest[df_rest["subject"] == s][feature_cols]

        # Standard parameters
        mu = s_rest.mean()
        std = s_rest.std().replace(0, 1e-6)

        m_rest = df_rest["subject"] == s
        m_peri = df_peri["subject"] == s

        df_rest_std.loc[m_rest, feature_cols] = (df_rest_std.loc[m_rest, feature_cols] - mu) / std
        df_norm_std.loc[m_peri, feature_cols] = (df_norm_std.loc[m_peri, feature_cols] - mu) / std

        # Robust parameters (Median & MAD with 1.4826 normal consistency scale)
        med = s_rest.median()
        mad = (s_rest - med).abs().median() * 1.4826
        mad = mad.replace(0, 1e-6)

        df_rest_rob.loc[m_rest, feature_cols] = (df_rest_rob.loc[m_rest, feature_cols] - med) / mad
        df_norm_rob.loc[m_peri, feature_cols] = (df_norm_rob.loc[m_peri, feature_cols] - med) / mad

    return (df_rest_std, df_norm_std), (df_rest_rob, df_norm_rob)


(df_rest_std, df_norm_std), (df_rest_rob, df_norm_rob) = compute_subject_normalizations(raw_df)

# Compute dimension-scaled Chi-square thresholds
TAU_4D = np.sqrt(stats.chi2.ppf(1.0 - ALPHA_CONF, df=4))  # ~4.03
TAU_2D = np.sqrt(stats.chi2.ppf(1.0 - ALPHA_CONF, df=2))  # ~3.44


# ==============================================================================
# 3. DETECTION ENGINES
# ==============================================================================
def run_detector_pipeline(config_name, use_robust_norm, use_decoupled, use_cusum, max_horizon):
    """Executes full streaming onset detection under specified remedy configuration."""
    df_rest = df_rest_rob if use_robust_norm else df_rest_std
    df_norm = df_norm_rob if use_robust_norm else df_norm_std

    trial_log = []

    for s in SUBJECTS:
        s_rest = df_rest[df_rest["subject"] == s]
        s_peri = df_norm[df_norm["subject"] == s]

        # Prepare covariance models
        if use_decoupled:
            lw_edc = LedoitWolf().fit(s_rest[DECOUPLED_EDC_2D].dropna().values)
            mu_edc, inv_edc = lw_edc.location_, lw_edc.precision_

            lw_fds = LedoitWolf().fit(s_rest[DECOUPLED_FDS_2D].dropna().values)
            mu_fds, inv_fds = lw_fds.location_, lw_fds.precision_
        else:
            lw_mono = LedoitWolf().fit(s_rest[MONOLITHIC_4D].dropna().values)
            mu_mono, inv_mono = lw_mono.location_, lw_mono.precision_

        for (rep, mov), t_df in s_peri.groupby(["repetition", "movement"]):
            t_df = t_df.sort_values("offset_ms")
            offsets = t_df["offset_ms"].values

            # Calculate Distance Trajectories
            if use_decoupled:
                diff_edc = t_df[DECOUPLED_EDC_2D].values - mu_edc
                dm_edc = np.sqrt(np.sum(diff_edc @ inv_edc * diff_edc, axis=1))

                diff_fds = t_df[DECOUPLED_FDS_2D].values - mu_fds
                dm_fds = np.sqrt(np.sum(diff_fds @ inv_fds * diff_fds, axis=1))

                # Normalized evidence ratio relative to dimension threshold
                ratio_edc = dm_edc / TAU_2D
                ratio_fds = dm_fds / TAU_2D
                joint_ratio = np.maximum(ratio_edc, ratio_fds)
                dm_eval = joint_ratio
                thresh_eval = 1.0  # Normalized trigger boundary
            else:
                diff_mono = t_df[MONOLITHIC_4D].values - mu_mono
                dm_mono = np.sqrt(np.sum(diff_mono @ inv_mono * diff_mono, axis=1))
                dm_eval = dm_mono
                thresh_eval = TAU_4D

            # Streaming State Machine
            consec_hits = 0
            quarantine_fa = False
            detected_tau = None

            # CUSUM parameters (Accumulator threshold S_thresh, drift allowance k)
            cusum_s = 0.0
            cusum_drift = 0.65 * thresh_eval
            cusum_thresh = 2.0 * thresh_eval
            cusum_onset_idx = None

            for i in range(len(offsets)):
                tau = offsets[i]
                val = dm_eval[i]

                if use_cusum:
                    # Leaky Integrator / Page-Hinkley Accumulator
                    if val > cusum_drift:
                        if cusum_s == 0.0:
                            cusum_onset_idx = i
                        cusum_s += (val - cusum_drift)
                    else:
                        cusum_s = max(0.0, cusum_s - (cusum_drift - val) * 0.5)
                        if cusum_s == 0.0:
                            cusum_onset_idx = None

                    if cusum_s >= cusum_thresh and detected_tau is None:
                        first_tau = offsets[cusum_onset_idx] if cusum_onset_idx is not None else tau
                        if first_tau <= QUARANTINE_HORIZON:
                            quarantine_fa = True
                        elif first_tau >= ACTION_START_HORIZON:
                            detected_tau = first_tau
                            break
                else:
                    # Standard Consecutive Frame Rule (3 consecutive hops = 30 ms)
                    if val >= thresh_eval:
                        consec_hits += 1
                        if consec_hits >= 3 and detected_tau is None:
                            first_tau = offsets[i - 2]
                            if first_tau <= QUARANTINE_HORIZON:
                                quarantine_fa = True
                            elif first_tau >= ACTION_START_HORIZON:
                                detected_tau = first_tau
                                break
                    else:
                        consec_hits = 0

            # Evaluate Outcome
            is_valid = (detected_tau is not None and detected_tau <= max_horizon)
            delta_t = detected_tau if is_valid else np.nan

            trial_log.append({
                "config": config_name,
                "subject": s,
                "repetition": rep,
                "movement": mov,
                "movement_name": "Hand Open (EDC)" if mov == 5 else "Fist Clench (FDS)",
                "detected": is_valid,
                "detected_tau": delta_t,
                "quarantine_fa": quarantine_fa,
                "is_hit_100ms": abs(delta_t) <= 100.0 if is_valid else False,
                "is_hit_lead": (-80.0 <= delta_t <= 0.0) if is_valid else False
            })

    return pd.DataFrame(trial_log)


# ==============================================================================
# 4. EXECUTE COMPARATIVE BENCHMARK
# ==============================================================================
CONFIGURATIONS = [
    {
        "name": "1. Baseline (Monolithic 4D)",
        "rob_norm": False, "decoupled": False, "cusum": False, "max_h": BASE_MAX_HORIZON
    },
    {
        "name": "2. Remedy A (CUSUM Leaky Integrator)",
        "rob_norm": False, "decoupled": False, "cusum": True, "max_h": BASE_MAX_HORIZON
    },
    {
        "name": "3. Remedy B (Extended Horizon +200ms)",
        "rob_norm": False, "decoupled": False, "cusum": False, "max_h": EXT_MAX_HORIZON
    },
    {
        "name": "4. Remedy C (Decoupled Bilateral OR-Gate)",
        "rob_norm": False, "decoupled": True, "cusum": False, "max_h": BASE_MAX_HORIZON
    },
    {
        "name": "5. Remedy D (Robust Median/MAD Baseline)",
        "rob_norm": True, "decoupled": False, "cusum": False, "max_h": BASE_MAX_HORIZON
    },
    {
        "name": "6. Unified Robust Pipeline (C+A+B)",
        "rob_norm": False, "decoupled": True, "cusum": True, "max_h": EXT_MAX_HORIZON
    },
]

results = []
for cfg in CONFIGURATIONS:
    df_res = run_detector_pipeline(
        cfg["name"], cfg["rob_norm"], cfg["decoupled"], cfg["cusum"], cfg["max_h"]
    )
    results.append(df_res)

all_results_df = pd.concat(results, ignore_index=True)
all_results_df.to_csv(os.path.join(OUTPUT_DIR, "remedy_comparison_trial_log.csv"), index=False)


# ==============================================================================
# 5. STATISTICAL SUMMARIZATION & REPORTING
# ==============================================================================
print("=" * 115)
print("OVERALL PERFORMANCE SUMMARY ACROSS ALL 11 SUBJECTS (132 TRIALS)")
print("=" * 115)
print(f"{'Configuration':<38} | {'Sens (%)':<9} | {'FAR (%)':<8} | {'[-100,+100]':<12} | {'Lead [-80,0]':<13} | {'Median Lead':<12}")
print("-" * 115)

summary_rows = []
for cfg_name, grp in all_results_df.groupby("config", sort=False):
    total = len(grp)
    sens = (grp["detected"].sum() / total) * 100.0
    far = (grp["quarantine_fa"].sum() / total) * 100.0
    hit100 = (grp["is_hit_100ms"].sum() / total) * 100.0
    hitlead = (grp["is_hit_lead"].sum() / total) * 100.0
    valid_taus = grp[grp["detected"]]["detected_tau"].dropna()
    med_tau = np.median(valid_taus) if len(valid_taus) > 0 else np.nan

    summary_rows.append({
        "Configuration": cfg_name,
        "Sensitivity_pct": sens,
        "Quarantine_FAR_pct": far,
        "Hit_100ms_pct": hit100,
        "Hit_Lead_80ms_pct": hitlead,
        "Median_Lead_ms": med_tau
    })

    print(f"{cfg_name:<38} | {sens:>7.1f}% | {far:>6.1f}% | {hit100:>10.1f}% | {hitlead:>11.1f}% | {med_tau:>9.1f} ms")
print("=" * 115)

sum_df = pd.DataFrame(summary_rows)
sum_df.to_csv(os.path.join(OUTPUT_DIR, "remedy_summary_table.csv"), index=False)


# ==============================================================================
# 6. SUBJECT 7 SPECIFIC DEEP DIVE AUDIT
# ==============================================================================
print("\n" + "=" * 115)
print("SUBJECT 7 DEEP-DIVE AUDIT (CHANNEL 10 FDS CONTINUOUS NOISE CORRUPTION)")
print("=" * 115)
s7_df = all_results_df[all_results_df["subject"] == 7]

print(f"{'Configuration':<38} | {'S7 Sens (%)':<12} | {'Hand Open (EDC)':<18} | {'Fist Clench (FDS)':<18} | {'S7 FAR (%)':<10}")
print("-" * 115)
for cfg_name, grp in s7_df.groupby("config", sort=False):
    total = len(grp)
    sens = (grp["detected"].sum() / total) * 100.0
    far = (grp["quarantine_fa"].sum() / total) * 100.0

    m5_grp = grp[grp["movement"] == 5]
    m6_grp = grp[grp["movement"] == 6]

    m5_sens = (m5_grp["detected"].sum() / len(m5_grp)) * 100.0
    m6_sens = (m6_grp["detected"].sum() / len(m6_grp)) * 100.0

    print(f"{cfg_name:<38} | {sens:>10.1f}% | {m5_sens:>10.1f}% ({m5_grp['detected'].sum()}/6) | {m6_sens:>10.1f}% ({m6_grp['detected'].sum()}/6) | {far:>8.1f}%")
print("=" * 115)


# ==============================================================================
# 7. PER-SUBJECT SENSITIVITY BREAKDOWN TABLE
# ==============================================================================
print("\n" + "=" * 115)
print("SUBJECT-BY-SUBJECT SENSITIVITY BREAKDOWN: BASELINE vs. UNIFIED PIPELINE")
print("=" * 115)

base_sub = all_results_df[all_results_df["config"] == "1. Baseline (Monolithic 4D)"].groupby("subject")["detected"].mean() * 100.0
unif_sub = all_results_df[all_results_df["config"] == "6. Unified Robust Pipeline (C+A+B)"].groupby("subject")["detected"].mean() * 100.0

print(f"{'Subject':<12} | {'Baseline Sensitivity':<22} | {'Unified Pipeline Sensitivity':<28} | {'Absolute Recovery Gain':<22}")
print("-" * 95)
for s in SUBJECTS:
    b_val = base_sub.get(s, 0.0)
    u_val = unif_sub.get(s, 0.0)
    print(f"Subject {s:<4} | {b_val:>19.1f}% | {u_val:>25.1f}% | {u_val - b_val:>+19.1f}%")
print("-" * 95)
print(f"Cohort Mean  | {base_sub.mean():>19.1f}% | {unif_sub.mean():>25.1f}% | {unif_sub.mean() - base_sub.mean():>+19.1f}%\n")


# ==============================================================================
# 8. DIAGNOSTIC VISUALIZATION
# ==============================================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot A: Overall Cohort Sensitivity vs. False Alarm Rate
cfg_labels = [c.split(". ")[1].split(" (")[0] for c in sum_df["Configuration"]]
axes[0].bar(np.arange(len(sum_df)) - 0.18, sum_df["Sensitivity_pct"], width=0.36, label="Sensitivity (TPR %)", color="#1f77b4", edgecolor="black")
axes[0].bar(np.arange(len(sum_df)) + 0.18, sum_df["Quarantine_FAR_pct"], width=0.36, label="Quarantine FAR (%)", color="#d62728", edgecolor="black")
axes[0].set_xticks(range(len(sum_df)))
axes[0].set_xticklabels(cfg_labels, rotation=25, ha="right")
axes[0].set_ylabel("Percentage (%)")
axes[0].set_title("Cohort Sensitivity vs. FAR across Remedies")
axes[0].set_ylim(0, 105)
axes[0].legend()
axes[0].grid(axis="y", linestyle="--", alpha=0.6)

# Plot B: Subject 7 Recovery Audit
s7_cfg = s7_df.groupby("config", sort=False).agg(
    sens=("detected", lambda x: x.mean() * 100.0),
    m5_sens=("detected", lambda x: x[s7_df.loc[x.index, "movement"] == 5].mean() * 100.0),
    m6_sens=("detected", lambda x: x[s7_df.loc[x.index, "movement"] == 6].mean() * 100.0),
)
axes[1].plot(cfg_labels, s7_cfg["sens"], marker="o", linewidth=2.5, color="black", label="S7 Total Sensitivity")
axes[1].plot(cfg_labels, s7_cfg["m5_sens"], marker="s", linestyle="--", linewidth=1.8, color="#2ca02c", label="S7 Hand Open (EDC)")
axes[1].plot(cfg_labels, s7_cfg["m6_sens"], marker="^", linestyle=":", linewidth=1.8, color="#ff7f0e", label="S7 Fist Clench (FDS)")
axes[1].set_xticklabels(cfg_labels, rotation=25, ha="right")
axes[1].set_ylabel("Onset Sensitivity (%)")
axes[1].set_title("Subject 7 Recovery (Noisy FDS Channel)")
axes[1].set_ylim(0, 105)
axes[1].legend()
axes[1].grid(True, linestyle="--", alpha=0.6)

# Plot C: Per-Subject Recovery (Baseline vs. Unified Pipeline)
x_sub = np.arange(len(SUBJECTS))
w_bar = 0.38
axes[2].bar(x_sub - w_bar/2, [base_sub.get(s, 0.0) for s in SUBJECTS], width=w_bar, label="Baseline Monolithic", color="#7f7f7f", edgecolor="black")
axes[2].bar(x_sub + w_bar/2, [unif_sub.get(s, 0.0) for s in SUBJECTS], width=w_bar, label="Unified Pipeline", color="#2ca02c", edgecolor="black")
axes[2].set_xticks(x_sub)
axes[2].set_xticklabels([f"S{s}" for s in SUBJECTS])
axes[2].set_ylabel("Sensitivity (%)")
axes[2].set_title("Per-Subject Recovery: Baseline vs. Unified")
axes[2].set_ylim(0, 105)
axes[2].axhline(100, color="gray", linestyle="--", alpha=0.5)
axes[2].legend()
axes[2].grid(axis="y", linestyle="--", alpha=0.6)

plt.tight_layout()
plot_path = os.path.join(OUTPUT_DIR, "remedies_benchmarking_analysis.png")
plt.savefig(plot_path, dpi=300)
plt.close()

print(f"Benchmark plot successfully saved to: {plot_path}")
print(f"All summary tables and trial logs saved in: {OUTPUT_DIR}\n")

INGESTING DATASETS FOR REMEDY BENCHMARKING (11 SUBJECTS | 132 TRIALS)
Total Window Rows Ingested : 5544 (Window: 30 ms)
Active Subjects             : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12]

OVERALL PERFORMANCE SUMMARY ACROSS ALL 11 SUBJECTS (132 TRIALS)
Configuration                          | Sens (%)  | FAR (%)  | [-100,+100]  | Lead [-80,0]  | Median Lead 
-------------------------------------------------------------------------------------------------------------------
1. Baseline (Monolithic 4D)            |    76.5% |   23.5% |       51.5% |        31.1% |     -70.0 ms
2. Remedy A (CUSUM Leaky Integrator)   |    39.4% |   45.5% |       33.3% |        18.2% |     -50.0 ms
3. Remedy B (Extended Horizon +200ms)  |    78.0% |   23.5% |       51.5% |        31.1% |     -70.0 ms
4. Remedy C (Decoupled Bilateral OR-Gate) |    75.8% |   25.8% |       50.8% |        30.3% |     -70.0 ms
5. Remedy D (Robust Median/MAD Baseline) |    72.0% |   20.5% |       51.5% |        30.3% |     -70.0 ms


In [2]:
#!/usr/bin/env python3
"""
==============================================================================
CONTINUOUS sEMG VELOCITY REGRESSION: MULTI-MODEL BENCHMARKING & AUDIT (PYTHON)
Cohort      : Subjects 1 to 12 (Excluding Subject 11; Excluding S4 Trial 6)
Filtering   : Drops all 'is_hold_plateau == 1' static grasp/open periods
Partitioning: Repetitions 1 to 4 (Train) | Repetitions 5 to 6 (Holdout Test)
Models      : 1. Ridge Regression (L2 Linear with Group-CV)
              2. Support Vector Regression (SVR - Linear & RBF)
              3. Random Forest Regressor (Constrained Depth)
              4. HistGradientBoosting (Fast GBDT with L2 & Early Stopping)
              5. Multi-Layer Perceptron (MLP Neural Net with Weight Decay)
Outputs     : Formatted Console Log + Text File Dump ('cohort_model_validation_report.txt')
==============================================================================
"""

import os
import sys
import numpy as np
import pandas as pd
from scipy import signal, stats
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV
from sklearn.svm import SVR
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import GroupKFold

# --- 1. Experimental Cohort & Validation Constants ---
COHORT = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12]  # Subject 11 excluded
TRAIN_REPS = [1, 2, 3, 4]                      # Repetitions 1-4 for training
TEST_REPS = [5, 6]                             # Repetitions 5-6 for unseen holdout testing
V_DEADBAND = 4.0                               # deg/s threshold for resting jitter (FAR)
LOG_FILE = "cohort_model_validation_report.txt"

MODEL_FAMILIES = ["Ridge", "SVR_Linear", "SVR_RBF", "Random_Forest", "Hist_Gradient_Boost", "MLP_NeuralNet"]

# --- 2. Evaluation Metrics Calculation Helper ---
def evaluate_predictions(y_true, y_pred, v_db=V_DEADBAND, hop_ms=10.0):
    """
    Computes Pearson r, R^2 (VAF), RMSE, Peak Lag (ms), and Resting Jitter FAR (%).
    """
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    valid = ~(np.isnan(y_true) | np.isnan(y_pred))
    yt = y_true[valid]
    yp = y_pred[valid]

    if len(yt) < 10 or np.std(yt) == 0 or np.std(yp) == 0:
        return 0.0, 0.0, float("inf"), float("nan"), 100.0

    # 1. Pearson Correlation Coefficient
    r_val, _ = stats.pearsonr(yt, yp)

    # 2. Variance Accounted For (VAF / R^2)
    sse = np.sum((yt - yp) ** 2)
    sst = np.sum((yt - np.mean(yt)) ** 2)
    r2_val = max(0.0, 1.0 - (sse / sst)) if sst > 0 else 0.0

    # 3. Root Mean Squared Error (RMSE)
    rmse_val = np.sqrt(np.mean((yt - yp) ** 2))

    # 4. Cross-Correlation Peak Lag (Lead/Lag in ms)
    # A negative lag means predicted velocity leads true kinematics (EMD lead)
    max_lag_pts = 30  # +/- 300 ms window at 100 Hz cadence
    yp_detrend = yp - np.mean(yp)
    yt_detrend = yt - np.mean(yt)
    xcorr = signal.correlate(yp_detrend, yt_detrend, mode="full")
    lags = signal.correlation_lags(len(yp_detrend), len(yt_detrend), mode="full")

    # Constrain lag search window
    lag_mask = (lags >= -max_lag_pts) & (lags <= max_lag_pts)
    valid_lags = lags[lag_mask]
    valid_xcorr = xcorr[lag_mask]
    peak_lag_idx = np.argmax(valid_xcorr)
    lag_ms = valid_lags[peak_lag_idx] * hop_ms

    # 5. Resting Jitter / False Actuation Rate (FAR)
    # Evaluates true rest frames (|y_true| <= 2 deg/s) where model commands motion > deadband
    rest_mask = np.abs(yt) <= 2.0
    if np.any(rest_mask):
        false_actuations = np.abs(yp[rest_mask]) > v_db
        far_pct = (np.sum(false_actuations) / np.sum(rest_mask)) * 100.0
    else:
        far_pct = 0.0

    return r_val, r2_val, rmse_val, lag_ms, far_pct

def log_print(msg, file_handle):
    print(msg, end="")
    file_handle.write(msg)

# --- 3. Main Cohort Training and Benchmark Loop ---
def main():
    f_log = open(LOG_FILE, "w", encoding="utf-8")

    header = (
        "========================================================================================================\n"
        "                NINAPRO DB2 CONTINUOUS sEMG VELOCITY REGRESSION BENCHMARKING REPORT                     \n"
        "========================================================================================================\n"
        f" Cohort: {len(COHORT)} Subjects | Train Reps: [1-4] | Holdout Test Reps: [5-6]\n"
        " Exclusions: Subject 11 (Entirely) | Subject 4 Trial 6 (Rep 6)\n"
        " Hold Plateau Filtering: is_hold_plateau == 1 strictly excluded from velocity mapping\n"
        "========================================================================================================\n\n"
    )
    log_print(header, f_log)

    # Master results dictionaries: [subject_id][model_name] -> metrics dict
    cohort_results = {sub: {} for sub in COHORT}

    for sub_id in COHORT:
        # Resolve CSV path
        csv_path = os.path.join("datasets", "DB2", f"DB2_s{sub_id}", f"S{sub_id}_E1_A1_dynamic_velocity_features.csv")
        if not os.path.exists(csv_path):
            csv_path = f"S{sub_id}_E1_A1_dynamic_velocity_features.csv"

        if not os.path.exists(csv_path):
            log_print(f"[WARN] Subject {sub_id:02d}: CSV not found at {csv_path}. Skipping.\n", f_log)
            continue

        df = pd.read_csv(csv_path)

        # A. Filter out isometric hold plateaus
        df = df[df["is_hold_plateau"] == 0].copy()

        # B. Strictly exclude Subject 4 and 7 Trial 6
        if sub_id == 4:
            df = df[df["repetition"] != 6].copy()

        if sub_id == 7:
            df = df[df["repetition"] != 6].copy()

        meta_cols = ["timestamp_s", "theta_deg", "theta_vel_deg_s", "stimulus", "repetition", "is_hold_plateau"]
        feat_cols = [c for c in df.columns if c not in meta_cols]

        # C. Temporal Partitioning (Train: Reps 1-4 | Test: Reps 5-6)
        train_mask = df["repetition"].isin(TRAIN_REPS) | ((df["stimulus"] == 0) & (df["repetition"] == 0))
        test_mask = df["repetition"].isin(TEST_REPS)

        train_df = df[train_mask].copy()
        test_df = df[test_mask].copy()

        if len(train_df) == 0 or len(test_df) == 0:
            log_print(f"[WARN] Subject {sub_id:02d}: Insufficient train/test partition size. Skipping.\n", f_log)
            continue

        X_train = train_df[feat_cols].values
        y_train = train_df["theta_vel_deg_s"].values
        reps_train = train_df["repetition"].values

        X_test = test_df[feat_cols].values
        y_test = test_df["theta_vel_deg_s"].values

        # D. Anti-Leakage Feature Standardization
        # Fit scaler ONLY on train data; apply forward to test partition
        scaler = StandardScaler()
        X_train_z = scaler.fit_transform(X_train)
        X_test_z = scaler.transform(X_test)
        # Outlier clamping: protect against floating-point sensor bursts in test
        X_test_z = np.clip(X_test_z, -20.0, 20.0)

        print(f">>> Processing Subject {sub_id:02d} (Train: {len(X_train_z)} frames | Test: {len(X_test_z)} frames)...")

        # Repetition-stratified cross-validation generator for hyperparameter tuning
        # Ensures validation folds evaluate across unseen trials, preventing auto-correlation leakage
        unique_reps = np.unique(reps_train[reps_train > 0])
        n_splits = min(4, len(unique_reps))
        gkf = GroupKFold(n_splits=n_splits)
        cv_groups = list(gkf.split(X_train_z, y_train, groups=reps_train))

        # ----------------------------------------------------------------------
        # MODEL 1: Ridge Regression (L2 Regularized Linear Model)
        # ----------------------------------------------------------------------
        ridge_alphas = np.logspace(-3, 3, 13)
        ridge = RidgeCV(alphas=ridge_alphas, cv=cv_groups)
        ridge.fit(X_train_z, y_train)
        y_pred_ridge = ridge.predict(X_test_z)
        r, r2, rmse, lag, far = evaluate_predictions(y_test, y_pred_ridge)
        cohort_results[sub_id]["Ridge"] = {
            "r": r, "r2": r2, "rmse": rmse, "lag": lag, "far": far,
            "cfg": f"alpha={ridge.alpha_:.2e}"
        }

        # ----------------------------------------------------------------------
        # MODEL 2: Support Vector Regression (Linear Kernel)
        # ----------------------------------------------------------------------
        # Subsample if training frames > 4000 to keep QP solver execution fast
        sub_step = max(1, len(X_train_z) // 3500)
        svr_lin = SVR(kernel="linear", C=1.0, epsilon=1.5)
        svr_lin.fit(X_train_z[::sub_step], y_train[::sub_step])
        y_pred_svr_lin = svr_lin.predict(X_test_z)
        r, r2, rmse, lag, far = evaluate_predictions(y_test, y_pred_svr_lin)
        cohort_results[sub_id]["SVR_Linear"] = {
            "r": r, "r2": r2, "rmse": rmse, "lag": lag, "far": far,
            "cfg": "C=1.0, eps=1.5"
        }

        # ----------------------------------------------------------------------
        # MODEL 3: Support Vector Regression (RBF Kernel)
        # ----------------------------------------------------------------------
        svr_rbf = SVR(kernel="rbf", C=2.0, epsilon=1.0, gamma="scale")
        svr_rbf.fit(X_train_z[::sub_step], y_train[::sub_step])
        y_pred_svr_rbf = svr_rbf.predict(X_test_z)
        r, r2, rmse, lag, far = evaluate_predictions(y_test, y_pred_svr_rbf)
        cohort_results[sub_id]["SVR_RBF"] = {
            "r": r, "r2": r2, "rmse": rmse, "lag": lag, "far": far,
            "cfg": "C=2.0, eps=1.0"
        }

        # ----------------------------------------------------------------------
        # MODEL 4: Random Forest (Constrained Depth to Prevent Overfitting)
        # ----------------------------------------------------------------------
        rf = RandomForestRegressor(
            n_estimators=60,
            max_depth=10,
            min_samples_leaf=8,
            max_features="sqrt",
            n_jobs=-1,
            random_state=42
        )
        rf.fit(X_train_z, y_train)
        y_pred_rf = rf.predict(X_test_z)
        r, r2, rmse, lag, far = evaluate_predictions(y_test, y_pred_rf)
        cohort_results[sub_id]["Random_Forest"] = {
            "r": r, "r2": r2, "rmse": rmse, "lag": lag, "far": far,
            "cfg": "trees=60, depth=10"
        }

        # ----------------------------------------------------------------------
        # MODEL 5: HistGradientBoosting (Fast GBDT with L2 & Early Stopping)
        # ----------------------------------------------------------------------
        hgb = HistGradientBoostingRegressor(
            max_iter=100,
            learning_rate=0.08,
            max_leaf_nodes=31,
            min_samples_leaf=15,
            l2_regularization=1.5,
            early_stopping=True,
            n_iter_no_change=10,
            random_state=42
        )
        hgb.fit(X_train_z, y_train)
        y_pred_hgb = hgb.predict(X_test_z)
        r, r2, rmse, lag, far = evaluate_predictions(y_test, y_pred_hgb)
        cohort_results[sub_id]["Hist_Gradient_Boost"] = {
            "r": r, "r2": r2, "rmse": rmse, "lag": lag, "far": far,
            "cfg": f"iters={hgb.n_iter_}, lr=0.08, l2=1.5"
        }

        # ----------------------------------------------------------------------
        # MODEL 6: Multi-Layer Perceptron (Shallow Neural Network)
        # ----------------------------------------------------------------------
        mlp = MLPRegressor(
            hidden_layer_sizes=(64, 32),
            activation="relu",
            solver="adam",
            alpha=0.05,             # Weight decay penalty (L2)
            learning_rate_init=0.003,
            max_iter=200,
            early_stopping=True,    # Halts training when validation plateau reached
            n_iter_no_change=12,
            random_state=42
        )
        mlp.fit(X_train_z, y_train)
        y_pred_mlp = mlp.predict(X_test_z)
        r, r2, rmse, lag, far = evaluate_predictions(y_test, y_pred_mlp)
        cohort_results[sub_id]["MLP_NeuralNet"] = {
            "r": r, "r2": r2, "rmse": rmse, "lag": lag, "far": far,
            "cfg": f"MLP[64,32] (iters={mlp.n_iter_})"
        }

    # --- 4. Format & Log Summary Reports ---
    summary_header = (
        "\n========================================================================================================\n"
        "                              HOLDOUT VALIDATION SUMMARY ACROSS COHORT                                  \n"
        "========================================================================================================\n"
        " Subj | Best Model Family     | Test Pearson r | Test R^2 (VAF) | RMSE (deg/s) | Peak Lag (ms) | Rest Jitter % \n"
        "--------------------------------------------------------------------------------------------------------\n"
    )
    log_print(summary_header, f_log)

    for sub_id in COHORT:
        sub_dict = cohort_results.get(sub_id, {})
        if not sub_dict:
            log_print(f"  S{sub_id:02d} | [DATA NOT PROCESSED]\n", f_log)
            continue

        # Select best model based on Pearson r
        best_model_name = max(sub_dict.keys(), key=lambda m: sub_dict[m]["r"])
        bm = sub_dict[best_model_name]

        line = (
            f"  S{sub_id:02d} | {best_model_name:<21} |     {bm['r']:6.4f}     |     {bm['r2']:6.4f}     "
            f"|    {bm['rmse']:6.2f}    |    {bm['lag']:+6.1f} ms   |     {bm['far']:4.2f}%\n"
        )
        log_print(line, f_log)

    log_print("--------------------------------------------------------------------------------------------------------\n\n", f_log)

    # --- 5. Aggregate Model Family Comparison Table ---
    agg_header = (
        "========================================================================================================\n"
        "                         OVERALL MODEL FAMILY PERFORMANCE (COHORT MEAN +/- STD)                         \n"
        "========================================================================================================\n"
        " Model Architecture    | Test Pearson r   | Test R^2 (VAF)   | RMSE (deg/s)     | Peak Lag (ms)  | Jitter (FAR) \n"
        "--------------------------------------------------------------------------------------------------------\n"
    )
    log_print(agg_header, f_log)

    for m_name in MODEL_FAMILIES:
        r_list = [cohort_results[s][m_name]["r"] for s in COHORT if m_name in cohort_results[s]]
        r2_list = [cohort_results[s][m_name]["r2"] for s in COHORT if m_name in cohort_results[s]]
        rmse_list = [cohort_results[s][m_name]["rmse"] for s in COHORT if m_name in cohort_results[s]]
        lag_list = [cohort_results[s][m_name]["lag"] for s in COHORT if m_name in cohort_results[s]]
        far_list = [cohort_results[s][m_name]["far"] for s in COHORT if m_name in cohort_results[s]]

        if not r_list:
            continue

        agg_line = (
            f" {m_name:<22} |  {np.mean(r_list):5.3f} +/- {np.std(r_list):4.3f} |  "
            f"{np.mean(r2_list):5.3f} +/- {np.std(r2_list):4.3f} |  "
            f"{np.mean(rmse_list):5.2f} +/- {np.std(rmse_list):4.2f}  | "
            f"{np.mean(lag_list):+5.1f} +/- {np.std(lag_list):4.1f} |  "
            f"{np.mean(far_list):4.2f} +/- {np.std(far_list):4.2f}%\n"
        )
        log_print(agg_line, f_log)

    log_print("========================================================================================================\n", f_log)
    print(f"\n[SAVED] Comprehensive benchmark log successfully dumped to: {LOG_FILE}\n")
    f_log.close()

if __name__ == "__main__":
    main()

                NINAPRO DB2 CONTINUOUS sEMG VELOCITY REGRESSION BENCHMARKING REPORT                     
 Cohort: 11 Subjects | Train Reps: [1-4] | Holdout Test Reps: [5-6]
 Exclusions: Subject 11 (Entirely) | Subject 4 Trial 6 (Rep 6)
 Hold Plateau Filtering: is_hold_plateau == 1 strictly excluded from velocity mapping

>>> Processing Subject 01 (Train: 5488 frames | Test: 2975 frames)...
>>> Processing Subject 02 (Train: 6043 frames | Test: 3206 frames)...
>>> Processing Subject 03 (Train: 5461 frames | Test: 2723 frames)...
>>> Processing Subject 04 (Train: 5886 frames | Test: 1203 frames)...
>>> Processing Subject 05 (Train: 5723 frames | Test: 3026 frames)...
>>> Processing Subject 06 (Train: 5928 frames | Test: 2712 frames)...
>>> Processing Subject 07 (Train: 5904 frames | Test: 1483 frames)...
>>> Processing Subject 08 (Train: 5599 frames | Test: 2931 frames)...
>>> Processing Subject 09 (Train: 5175 frames | Test: 2788 frames)...
>>> Processing Subject 10 (Train: 5359 frames |

In [3]:
#!/usr/bin/env python3
"""
==============================================================================
COMPREHENSIVE sEMG VELOCITY REGRESSION PIPELINE (PYTHON)
Architecture: Contextual Multi-Lag Stacking + EMD Optimization [-200ms, +200ms]
Cohort      : Subjects 1 to 12 (Excluding S11, S4 Rep 6, and S7 Rep 6)
Filtering   : is_hold_plateau == 1 strictly dropped across all trials
Partitioning: Interleaved Cross-Trial Split:
              - Train: Repetitions [1, 3, 4, 6]
              - Test : Repetitions [2, 5]
Models      : 1. Ridge Regression (L2 Regularized Linear)
              2. SVR Linear Kernel
              3. SVR Gaussian (RBF) Kernel
              4. Random Forest Regressor (Constrained Depth)
              5. HistGradientBoosting (Fast GBDT with L2 Regularization)
              6. Multi-Layer Perceptron (MLP Neural Net with Weight Decay)
Outputs     : Formatted Console Log + Text File Dump ('cohort_model_validation_report.txt')
==============================================================================
"""

import os
import sys
import numpy as np
import pandas as pd
from scipy import signal, stats
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV
from sklearn.svm import SVR
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import GroupKFold

# --- 1. Cohort & Partitioning Definitions ---
COHORT = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12]  # Subject 11 strictly excluded
TRAIN_REPS = [1, 3, 4, 6]                      # Interleaved training trials
TEST_REPS = [2, 5]                             # Interleaved holdout test trials (resolves duplicate typo)
V_DEADBAND = 4.0                               # deg/s threshold for resting jitter (FAR)
HOP_MS = 10.0                                  # 10 ms hop (100 Hz loop)
LOG_FILE = "cohort_model_validation_report.txt"

MODEL_FAMILIES = [
    "Ridge",
    "SVR_Linear",
    "SVR_RBF",
    "Random_Forest",
    "Hist_Gradient_Boost",
    "MLP_NeuralNet"
]

# --- 2. Contextual Feature Stacking (Problem 1 Fix) ---
def build_contextual_matrix(df, feat_cols, n_lags=3, include_vel_lags=True):
    """
    Constructs a rich spatiotemporal context matrix:
      - sEMG multi-metric features across historical lags: x(t), x(t-1), x(t-2), x(t-3)
      - First-order rate-of-change (derivative): Delta x(t) = x(t) - x(t-1)
      - Autoregressive kinematic context: v(t-1), v(t-2)
    Resets history across timestamp discontinuities (> 25 ms) caused by plateau exclusions.
    """
    X_raw = df[feat_cols].values
    v_raw = df["theta_vel_deg_s"].values
    t_raw = df["timestamp_s"].values
    n_samples, n_feats = X_raw.shape

    # Preallocate lagged containers
    stacked_list = [X_raw]
    col_names_stacked = [f"{c}_lag0" for c in feat_cols]

    # Detect discontinuous boundaries (where plateaus were dropped or cues switched)
    dt = np.diff(t_raw, prepend=t_raw[0])
    is_break = dt > 0.025  # Any gap larger than 25 ms resets context

    # 1. Historical sEMG Lags & First Difference
    for lag in range(1, n_lags + 1):
        X_lag = np.zeros_like(X_raw)
        for i in range(n_samples):
            if i < lag or np.any(is_break[max(0, i - lag + 1): i + 1]):
                X_lag[i] = X_raw[i]  # Stationary fallback across discontinuity
            else:
                X_lag[i] = X_raw[i - lag]
        stacked_list.append(X_lag)
        col_names_stacked.extend([f"{c}_lag{lag}" for c in feat_cols])

    # First difference (rate of recruitment / velocity of features)
    delta_X = X_raw - stacked_list[1]
    stacked_list.append(delta_X)
    col_names_stacked.extend([f"{c}_delta" for c in feat_cols])

    # 2. Kinematic Velocity Context (Past Motion State)
    if include_vel_lags:
        v_lags = np.zeros((n_samples, 2), dtype=np.float64)
        for i in range(n_samples):
            if i < 1 or is_break[i]:
                v_lags[i, 0] = 0.0
            else:
                v_lags[i, 0] = v_raw[i - 1]

            if i < 2 or is_break[i] or is_break[i - 1]:
                v_lags[i, 1] = 0.0
            else:
                v_lags[i, 1] = v_raw[i - 2]

        stacked_list.append(v_lags)
        col_names_stacked.extend(["vel_lag1", "vel_lag2"])

    X_context = np.hstack(stacked_list)
    return X_context, col_names_stacked

# --- 3. EMD Slider: Grid Search Optimal Shift [-200ms to +200ms] ---
def find_optimal_emd_shift(df_train, feat_cols, slider_range_ms=(-200, 200), step_ms=10):
    """
    Sweeps a temporal slider across [-200ms, +200ms] in 10 ms steps to locate the
    physiological electromechanical delay (EMD) that maximizes cross-correlation
    between primary forearm recruitment power (EDC_MAV + FDS_MAV) and joint velocity.
    """
    mav_edc_col = "EDC_MAV" if "EDC_MAV" in feat_cols else feat_cols[0]
    mav_fds_col = "FDS_MAV" if "FDS_MAV" in feat_cols else feat_cols[1]

    emg_power = df_train[mav_edc_col].values + df_train[mav_fds_col].values
    vel = df_train["theta_vel_deg_s"].values

    emg_norm = (emg_power - np.mean(emg_power)) / (np.std(emg_power) + 1e-12)
    vel_norm = np.abs(vel) - np.mean(np.abs(vel))
    vel_norm = vel_norm / (np.std(vel_norm) + 1e-12)

    min_ms, max_ms = slider_range_ms
    shift_steps = np.arange(int(min_ms / step_ms), int(max_ms / step_ms) + 1)

    best_corr = -1.0
    best_shift_hops = 0

    for s in shift_steps:
        if s > 0:
            e_sub = emg_norm[:-s]
            v_sub = vel_norm[s:]
        elif s < 0:
            s_abs = abs(s)
            e_sub = emg_norm[s_abs:]
            v_sub = vel_norm[:-s_abs]
        else:
            e_sub = emg_norm
            v_sub = vel_norm

        if len(e_sub) > 50:
            r_val, _ = stats.pearsonr(e_sub, v_sub)
            if r_val > best_corr:
                best_corr = r_val
                best_shift_hops = s

    optimal_emd_ms = best_shift_hops * step_ms
    return best_shift_hops, optimal_emd_ms

def apply_temporal_shift(X, y, shift_hops):
    """
    Shifts the kinematic velocity target relative to sEMG features by shift_hops.
    shift_hops > 0: predicts impending velocity v(t + k) from EMG(t) (EMD forward projection).
    """
    if shift_hops > 0:
        X_adj = X[:-shift_hops]
        y_adj = y[shift_hops:]
    elif shift_hops < 0:
        k_abs = abs(shift_hops)
        X_adj = X[k_abs:]
        y_adj = y[:-k_abs]
    else:
        X_adj = X
        y_adj = y
    return X_adj, y_adj

# --- 4. Comprehensive Evaluation Metrics ---
def evaluate_predictions(y_true, y_pred, v_db=V_DEADBAND, hop_ms=HOP_MS):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    valid = ~(np.isnan(y_true) | np.isnan(y_pred))
    yt = y_true[valid]
    yp = y_pred[valid]

    if len(yt) < 10 or np.std(yt) == 0 or np.std(yp) == 0:
        return 0.0, 0.0, float("inf"), float("nan"), 100.0

    # 1. Pearson Correlation
    r_val, _ = stats.pearsonr(yt, yp)

    # 2. Variance Accounted For (VAF / R^2)
    sse = np.sum((yt - yp) ** 2)
    sst = np.sum((yt - np.mean(yt)) ** 2)
    r2_val = max(0.0, 1.0 - (sse / sst)) if sst > 0 else 0.0

    # 3. RMSE
    rmse_val = np.sqrt(np.mean((yt - yp) ** 2))

    # 4. Cross-Correlation Peak Lag across expanded +/- 350 ms horizon
    max_lag_pts = 35  # +/- 350 ms at 100 Hz loop
    yp_detrend = yp - np.mean(yp)
    yt_detrend = yt - np.mean(yt)
    xcorr = signal.correlate(yp_detrend, yt_detrend, mode="full")
    lags = signal.correlation_lags(len(yp_detrend), len(yt_detrend), mode="full")

    lag_mask = (lags >= -max_lag_pts) & (lags <= max_lag_pts)
    valid_lags = lags[lag_mask]
    valid_xcorr = xcorr[lag_mask]
    peak_lag_idx = np.argmax(valid_xcorr)
    lag_ms = valid_lags[peak_lag_idx] * hop_ms

    # 5. Resting Jitter / False Actuation Rate (FAR)
    rest_mask = np.abs(yt) <= 2.0
    if np.any(rest_mask):
        false_actuations = np.abs(yp[rest_mask]) > v_db
        far_pct = (np.sum(false_actuations) / np.sum(rest_mask)) * 100.0
    else:
        far_pct = 0.0

    return r_val, r2_val, rmse_val, lag_ms, far_pct

def log_print(msg, file_handle):
    print(msg, end="")
    file_handle.write(msg)

# --- 5. Main Execution Loop ---
def main():
    f_log = open(LOG_FILE, "w", encoding="utf-8")

    header = (
        "========================================================================================================\n"
        "             COMPREHENSIVE MULTI-LAG sEMG VELOCITY REGRESSION BENCHMARKING REPORT                        \n"
        "========================================================================================================\n"
        f" Cohort: {len(COHORT)} Subjects | Train Reps: {TRAIN_REPS} | Holdout Test Reps: {TEST_REPS}\n"
        " Exclusions: Subject 11 (Entirely) | S4 Rep 6 | S7 Rep 6 | is_hold_plateau == 1\n"
        " Context Architecture: 4-Step sEMG Lag History + Delta Features + Kinematic Velocity Feedback\n"
        " EMD Alignment Slider: Dynamically Swept across [-200 ms, +200 ms]\n"
        "========================================================================================================\n\n"
    )
    log_print(header, f_log)

    cohort_results = {sub: {} for sub in COHORT}
    cohort_emd_log = {}

    for sub_id in COHORT:
        csv_path = os.path.join("datasets", "DB2", f"DB2_s{sub_id}", f"S{sub_id}_E1_A1_dynamic_velocity_features.csv")
        if not os.path.exists(csv_path):
            csv_path = f"S{sub_id}_E1_A1_dynamic_velocity_features.csv"

        if not os.path.exists(csv_path):
            log_print(f"[WARN] Subject {sub_id:02d}: CSV not found. Skipping.\n", f_log)
            continue

        df = pd.read_csv(csv_path)

        # A. Filter static hold plateaus
        df = df[df["is_hold_plateau"] == 0].copy()

        # B. Enforce Subject-Specific Trial Exclusions
        if sub_id == 4:
            df = df[df["repetition"] != 6].copy()
        if sub_id == 7:
            df = df[df["repetition"] != 6].copy()

        meta_cols = ["timestamp_s", "theta_deg", "theta_vel_deg_s", "stimulus", "repetition", "is_hold_plateau"]
        feat_cols = [c for c in df.columns if c not in meta_cols]

        # C. Interleaved Split Partitioning
        train_mask = df["repetition"].isin(TRAIN_REPS) | ((df["stimulus"] == 0) & (df["repetition"] == 0))
        test_mask = df["repetition"].isin(TEST_REPS)

        train_df = df[train_mask].copy()
        test_df = df[test_mask].copy()

        if len(train_df) == 0 or len(test_df) == 0:
            log_print(f"[WARN] Subject {sub_id:02d}: Missing rows in train or test partition. Skipping.\n", f_log)
            continue

        # D. Dynamic EMD Optimization on Training Data [-200ms, +200ms]
        shift_hops, opt_emd_ms = find_optimal_emd_shift(train_df, feat_cols, slider_range_ms=(-200, 200))
        cohort_emd_log[sub_id] = opt_emd_ms

        # E. Contextual Spatiotemporal Feature Extraction
        X_train_ctx, ctx_cols = build_contextual_matrix(train_df, feat_cols, n_lags=3, include_vel_lags=True)
        y_train_raw = train_df["theta_vel_deg_s"].values
        reps_train = train_df["repetition"].values

        X_test_ctx, _ = build_contextual_matrix(test_df, feat_cols, n_lags=3, include_vel_lags=True)
        y_test_raw = test_df["theta_vel_deg_s"].values

        # Apply optimal EMD forward/backward shift
        X_tr_adj, y_tr_adj = apply_temporal_shift(X_train_ctx, y_train_raw, shift_hops)
        X_te_adj, y_te_adj = apply_temporal_shift(X_test_ctx, y_test_raw, shift_hops)

        # Adjust repetition labels for GroupKFold after shifting
        if shift_hops > 0:
            reps_tr_adj = reps_train[:-shift_hops]
        elif shift_hops < 0:
            reps_tr_adj = reps_train[abs(shift_hops):]
        else:
            reps_tr_adj = reps_train

        # F. Anti-Leakage Feature Standardization
        scaler = StandardScaler()
        X_tr_z = scaler.fit_transform(X_tr_adj)
        X_te_z = scaler.transform(X_te_adj)
        X_te_z = np.clip(X_te_z, -20.0, 20.0)

        print(f">>> Subject {sub_id:02d} | EMD Shift: {opt_emd_ms:+4.0f} ms | Context Features: {X_tr_z.shape[1]} | Train: {len(X_tr_z)} | Test: {len(X_te_z)}")

        # Cross-validation generator grouped on repetition
        gkf = GroupKFold(n_splits=min(4, len(np.unique(reps_tr_adj))))
        cv_groups = list(gkf.split(X_tr_z, y_tr_adj, groups=reps_tr_adj))

        # ----------------------------------------------------------------------
        # MODEL 1: Ridge Regression
        # ----------------------------------------------------------------------
        ridge_alphas = np.logspace(-2, 4, 13)
        ridge = RidgeCV(alphas=ridge_alphas, cv=cv_groups)
        ridge.fit(X_tr_z, y_tr_adj)
        y_pred_ridge = ridge.predict(X_te_z)
        r, r2, rmse, lag, far = evaluate_predictions(y_te_adj, y_pred_ridge)
        cohort_results[sub_id]["Ridge"] = {
            "r": r, "r2": r2, "rmse": rmse, "lag": lag, "far": far,
            "cfg": f"alpha={ridge.alpha_:.2e}"
        }

        # ----------------------------------------------------------------------
        # MODEL 2: Support Vector Regression (Linear)
        # ----------------------------------------------------------------------
        sub_step = max(1, len(X_tr_z) // 3500)
        svr_lin = SVR(kernel="linear", C=1.0, epsilon=1.5)
        svr_lin.fit(X_tr_z[::sub_step], y_tr_adj[::sub_step])
        y_pred_svr_lin = svr_lin.predict(X_te_z)
        r, r2, rmse, lag, far = evaluate_predictions(y_te_adj, y_pred_svr_lin)
        cohort_results[sub_id]["SVR_Linear"] = {
            "r": r, "r2": r2, "rmse": rmse, "lag": lag, "far": far,
            "cfg": "C=1.0, eps=1.5"
        }

        # ----------------------------------------------------------------------
        # MODEL 3: Support Vector Regression (RBF)
        # ----------------------------------------------------------------------
        svr_rbf = SVR(kernel="rbf", C=2.5, epsilon=1.0, gamma="scale")
        svr_rbf.fit(X_tr_z[::sub_step], y_tr_adj[::sub_step])
        y_pred_svr_rbf = svr_rbf.predict(X_te_z)
        r, r2, rmse, lag, far = evaluate_predictions(y_te_adj, y_pred_svr_rbf)
        cohort_results[sub_id]["SVR_RBF"] = {
            "r": r, "r2": r2, "rmse": rmse, "lag": lag, "far": far,
            "cfg": "C=2.5, eps=1.0"
        }

        # ----------------------------------------------------------------------
        # MODEL 4: Random Forest (Constrained Depth)
        # ----------------------------------------------------------------------
        rf = RandomForestRegressor(
            n_estimators=75,
            max_depth=12,
            min_samples_leaf=6,
            max_features="sqrt",
            n_jobs=-1,
            random_state=42
        )
        rf.fit(X_tr_z, y_tr_adj)
        y_pred_rf = rf.predict(X_te_z)
        r, r2, rmse, lag, far = evaluate_predictions(y_te_adj, y_pred_rf)
        cohort_results[sub_id]["Random_Forest"] = {
            "r": r, "r2": r2, "rmse": rmse, "lag": lag, "far": far,
            "cfg": "trees=75, depth=12"
        }

        # ----------------------------------------------------------------------
        # MODEL 5: HistGradientBoosting (Fast GBDT with L2 Regularization)
        # ----------------------------------------------------------------------
        hgb = HistGradientBoostingRegressor(
            max_iter=120,
            learning_rate=0.06,
            max_leaf_nodes=31,
            min_samples_leaf=15,
            l2_regularization=2.0,
            early_stopping=True,
            n_iter_no_change=10,
            random_state=42
        )
        hgb.fit(X_tr_z, y_tr_adj)
        y_pred_hgb = hgb.predict(X_te_z)
        r, r2, rmse, lag, far = evaluate_predictions(y_te_adj, y_pred_hgb)
        cohort_results[sub_id]["Hist_Gradient_Boost"] = {
            "r": r, "r2": r2, "rmse": rmse, "lag": lag, "far": far,
            "cfg": f"iters={hgb.n_iter_}, lr=0.06, l2=2.0"
        }

        # ----------------------------------------------------------------------
        # MODEL 6: Multi-Layer Perceptron (Neural Network)
        # ----------------------------------------------------------------------
        mlp = MLPRegressor(
            hidden_layer_sizes=(128, 64),
            activation="relu",
            solver="adam",
            alpha=0.08,             # Weight decay
            learning_rate_init=0.002,
            max_iter=200,
            early_stopping=True,
            n_iter_no_change=12,
            random_state=42
        )
        mlp.fit(X_tr_z, y_tr_adj)
        y_pred_mlp = mlp.predict(X_te_z)
        r, r2, rmse, lag, far = evaluate_predictions(y_te_adj, y_pred_mlp)
        cohort_results[sub_id]["MLP_NeuralNet"] = {
            "r": r, "r2": r2, "rmse": rmse, "lag": lag, "far": far,
            "cfg": f"MLP[128,64] (iters={mlp.n_iter_})"
        }

    # --- 6. Formulate & Log Summary Reports ---
    summary_header = (
        "\n========================================================================================================\n"
        "                              HOLDOUT VALIDATION SUMMARY ACROSS COHORT                                  \n"
        "========================================================================================================\n"
        " Subj | EMD Shift | Best Model Family     | Test Pearson r | Test R^2 (VAF) | RMSE (deg/s) | Peak Lag (ms) | Rest Jitter % \n"
        "--------------------------------------------------------------------------------------------------------\n"
    )
    log_print(summary_header, f_log)

    for sub_id in COHORT:
        sub_dict = cohort_results.get(sub_id, {})
        if not sub_dict:
            log_print(f"  S{sub_id:02d} |  [N/A]   | [DATA NOT PROCESSED]\n", f_log)
            continue

        best_model_name = max(sub_dict.keys(), key=lambda m: sub_dict[m]["r"])
        bm = sub_dict[best_model_name]
        emd_str = f"{cohort_emd_log.get(sub_id, 0):+4.0f} ms"

        line = (
            f"  S{sub_id:02d} | {emd_str:<9} | {best_model_name:<21} |     {bm['r']:6.4f}     |     {bm['r2']:6.4f}     "
            f"|    {bm['rmse']:6.2f}    |    {bm['lag']:+6.1f} ms   |     {bm['far']:4.2f}%\n"
        )
        log_print(line, f_log)

    log_print("--------------------------------------------------------------------------------------------------------\n\n", f_log)

    # --- 7. Aggregate Comparison Table Across Architectures ---
    agg_header = (
        "========================================================================================================\n"
        "                         OVERALL MODEL FAMILY PERFORMANCE (COHORT MEAN +/- STD)                         \n"
        "========================================================================================================\n"
        " Model Architecture    | Test Pearson r   | Test R^2 (VAF)   | RMSE (deg/s)     | Peak Lag (ms)  | Jitter (FAR) \n"
        "--------------------------------------------------------------------------------------------------------\n"
    )
    log_print(agg_header, f_log)

    for m_name in MODEL_FAMILIES:
        r_list = [cohort_results[s][m_name]["r"] for s in COHORT if m_name in cohort_results[s]]
        r2_list = [cohort_results[s][m_name]["r2"] for s in COHORT if m_name in cohort_results[s]]
        rmse_list = [cohort_results[s][m_name]["rmse"] for s in COHORT if m_name in cohort_results[s]]
        lag_list = [cohort_results[s][m_name]["lag"] for s in COHORT if m_name in cohort_results[s]]
        far_list = [cohort_results[s][m_name]["far"] for s in COHORT if m_name in cohort_results[s]]

        if not r_list:
            continue

        agg_line = (
            f" {m_name:<22} |  {np.mean(r_list):5.3f} +/- {np.std(r_list):4.3f} |  "
            f"{np.mean(r2_list):5.3f} +/- {np.std(r2_list):4.3f} |  "
            f"{np.mean(rmse_list):5.2f} +/- {np.std(rmse_list):4.2f}  | "
            f"{np.mean(lag_list):+5.1f} +/- {np.std(lag_list):4.1f} |  "
            f"{np.mean(far_list):4.2f} +/- {np.std(far_list):4.2f}%\n"
        )
        log_print(agg_line, f_log)

    log_print("========================================================================================================\n", f_log)
    print(f"\n[SAVED] Comprehensive report successfully dumped to: {LOG_FILE}\n")
    f_log.close()

if __name__ == "__main__":
    main()

             COMPREHENSIVE MULTI-LAG sEMG VELOCITY REGRESSION BENCHMARKING REPORT                        
 Cohort: 11 Subjects | Train Reps: [1, 3, 4, 6] | Holdout Test Reps: [2, 5]
 Exclusions: Subject 11 (Entirely) | S4 Rep 6 | S7 Rep 6 | is_hold_plateau == 1
 Context Architecture: 4-Step sEMG Lag History + Delta Features + Kinematic Velocity Feedback
 EMD Alignment Slider: Dynamically Swept across [-200 ms, +200 ms]

>>> Subject 01 | EMD Shift: -200 ms | Context Features: 152 | Train: 5485 | Test: 2938
>>> Subject 02 | EMD Shift: +200 ms | Context Features: 152 | Train: 6008 | Test: 3201
>>> Subject 03 | EMD Shift: -200 ms | Context Features: 152 | Train: 5565 | Test: 2579
>>> Subject 04 | EMD Shift: -200 ms | Context Features: 152 | Train: 4667 | Test: 2382
>>> Subject 05 | EMD Shift:   +0 ms | Context Features: 152 | Train: 5901 | Test: 2848
>>> Subject 06 | EMD Shift: -200 ms | Context Features: 152 | Train: 6192 | Test: 2408
>>> Subject 07 | EMD Shift: -200 ms | Context Features

In [4]:
#!/usr/bin/env python3
"""
==============================================================================
TRUTHFUL CONTINUOUS sEMG VELOCITY DECODER (PURE sEMG CONTEXT + CAUSAL EMD)
Cohort      : Subjects 1 to 12 (Excluding S11, S4 Rep 6, and S7 Rep 6)
Filtering   : is_hold_plateau == 1 strictly dropped across all trials
Partitioning: Interleaved Cross-Trial Split:
              - Train: Repetitions [1, 3, 4, 6]
              - Test : Repetitions [2, 5]
Features    : 100% Pure sEMG Context (150 Features):
              - 4-Step Multi-Metric History: x(t), x(t-1), x(t-2), x(t-3)
              - First-Order Derivative (Recruitment Rate): Delta x(t) = x(t) - x(t-1)
              - NO Kinematic Velocity Feedback (Zero Autoregressive Leakage)
EMD Range   : Causal Feedforward Sweep strictly within [0 ms, +100 ms]
Actuation   : Piecewise Non-Linear Deadband (4.0 deg/s) for Zero Resting Creep
Outputs     : Formatted Console Log + Text File Dump ('cohort_model_validation_report.txt')
==============================================================================
"""

import os
import sys
import numpy as np
import pandas as pd
from scipy import signal, stats
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV
from sklearn.svm import SVR
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import GroupKFold

# --- 1. Cohort & Protocol Constants ---
COHORT = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12]  # Subject 11 strictly excluded
TRAIN_REPS = [1, 3, 4, 6]                      # Interleaved training trials
TEST_REPS = [2, 5]                             # Interleaved holdout test trials
V_DEADBAND = 4.0                               # deg/s: Non-linear prosthetic deadband
HOP_MS = 10.0                                  # 10 ms hop (100 Hz inference loop)
LOG_FILE = "cohort_model_validation_report.txt"

MODEL_FAMILIES = [
    "Ridge",
    "SVR_Linear",
    "SVR_RBF",
    "Random_Forest",
    "Hist_Gradient_Boost",
    "MLP_NeuralNet",
]

# --- 2. Pure sEMG Spatiotemporal Context Builder ---
def build_pure_semg_context_matrix(df, feat_cols, n_lags=3):
    """
    Constructs an uncompromised 150-dimensional sEMG feature representation:
      - Raw 30 multi-domain features at frame t
      - 3 past historical frames: t-1 (10ms), t-2 (20ms), t-3 (30ms)
      - Instantaneous rate-of-change: Delta x(t) = x(t) - x(t-1)
    Strictly excludes true kinematic velocity from the feature matrix to ensure
    decoding relies 100% on physiological muscle recruitment.
    """
    X_raw = df[feat_cols].values
    t_raw = df["timestamp_s"].values
    n_samples, n_feats = X_raw.shape

    # Detect discontinuity breaks (> 25 ms) where hold plateaus were dropped
    dt = np.diff(t_raw, prepend=t_raw[0])
    is_break = dt > 0.025

    stacked_list = [X_raw]

    # Historical sEMG Lags
    for lag in range(1, n_lags + 1):
        X_lag = np.zeros_like(X_raw)
        for i in range(n_samples):
            if i < lag or np.any(is_break[max(0, i - lag + 1): i + 1]):
                X_lag[i] = X_raw[i]  # Clamp to current frame across boundary gap
            else:
                X_lag[i] = X_raw[i - lag]
        stacked_list.append(X_lag)

    # First Difference: Electrical Recruitment Rate (dEMG/dt)
    delta_X = np.zeros_like(X_raw)
    for i in range(n_samples):
        if i < 1 or is_break[i]:
            delta_X[i] = 0.0
        else:
            delta_X[i] = X_raw[i] - X_raw[i - 1]
    stacked_list.append(delta_X)

    X_context = np.hstack(stacked_list)
    return X_context

# --- 3. Causal Feedforward EMD Optimizer [0 ms to +100 ms] ---
def find_causal_emd_shift(df_train, feat_cols, slider_range_ms=(0, 100), step_ms=10):
    """
    Sweeps a strictly feedforward EMD window [0 ms, +100 ms] on training data.
    Finds the lead time where sEMG activation power best correlates with impending
    joint kinematics, preventing trailing-phase lockup.
    """
    mav_edc = df_train["EDC_MAV"].values if "EDC_MAV" in feat_cols else df_train[feat_cols[0]].values
    mav_fds = df_train["FDS_MAV"].values if "FDS_MAV" in feat_cols else df_train[feat_cols[1]].values
    emg_power = mav_edc + mav_fds
    vel_magnitude = np.abs(df_train["theta_vel_deg_s"].values)

    e_norm = (emg_power - np.mean(emg_power)) / (np.std(emg_power) + 1e-12)
    v_norm = (vel_magnitude - np.mean(vel_magnitude)) / (np.std(vel_magnitude) + 1e-12)

    min_ms, max_ms = slider_range_ms
    shift_steps = np.arange(int(min_ms / step_ms), int(max_ms / step_ms) + 1)

    best_corr = -1.0
    best_shift = 3  # Default fallback: 30 ms biological EMD

    for s in shift_steps:
        if s > 0:
            e_sub = e_norm[:-s]
            v_sub = v_norm[s:]
        else:
            e_sub = e_norm
            v_sub = v_norm

        if len(e_sub) > 50:
            r_val, _ = stats.pearsonr(e_sub, v_sub)
            if r_val > best_corr:
                best_corr = r_val
                best_shift = s

    return best_shift, best_shift * step_ms

def align_feedforward_target(X, y, shift_hops):
    """
    Aligns sEMG features at time t with impending velocity at time t + shift_hops.
    """
    if shift_hops > 0:
        return X[:-shift_hops], y[shift_hops:]
    return X, y

# --- 4. Piecewise Non-Linear Output Velocity Deadband ---
def apply_velocity_deadband(y_raw, v_db=V_DEADBAND):
    """
    Clamps resting motor jitter strictly to zero below v_db while maintaining
    continuous, proportional velocity modulation above threshold.
    """
    y_cmd = np.zeros_like(y_raw)
    active = np.abs(y_raw) > v_db
    y_cmd[active] = np.sign(y_raw[active]) * (np.abs(y_raw[active]) - v_db)
    return y_cmd

# --- 5. Diagnostic Evaluation Metrics ---
def evaluate_predictions(y_true, y_pred_raw, v_db=V_DEADBAND, hop_ms=HOP_MS):
    """
    Evaluates both raw and deadbanded predictions against ground truth.
    Reports commanded metrics that reflect actual closed-loop prosthetic actuation.
    """
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred_raw = np.asarray(y_pred_raw, dtype=np.float64)
    y_cmd = apply_velocity_deadband(y_pred_raw, v_db=v_db)

    valid = ~(np.isnan(y_true) | np.isnan(y_cmd))
    yt = y_true[valid]
    yp = y_cmd[valid]

    if len(yt) < 10 or np.std(yt) == 0 or np.std(yp) == 0:
        return 0.0, 0.0, float("inf"), float("nan"), 100.0

    # 1. Pearson Correlation (Commanded Profile Tracking)
    r_val, _ = stats.pearsonr(yt, yp)

    # 2. Variance Accounted For (VAF / R^2)
    sse = np.sum((yt - yp) ** 2)
    sst = np.sum((yt - np.mean(yt)) ** 2)
    r2_val = max(0.0, 1.0 - (sse / sst)) if sst > 0 else 0.0

    # 3. Root Mean Squared Error (deg/s)
    rmse_val = np.sqrt(np.mean((yt - yp) ** 2))

    # 4. Cross-Correlation Peak Lag (ms)
    max_lag_pts = 35  # +/- 350 ms horizon
    yp_detrend = yp - np.mean(yp)
    yt_detrend = yt - np.mean(yt)
    xcorr = signal.correlate(yp_detrend, yt_detrend, mode="full")
    lags = signal.correlation_lags(len(yp_detrend), len(yt_detrend), mode="full")

    lag_mask = (lags >= -max_lag_pts) & (lags <= max_lag_pts)
    valid_lags = lags[lag_mask]
    valid_xcorr = xcorr[lag_mask]
    peak_lag_idx = np.argmax(valid_xcorr)
    lag_ms = valid_lags[peak_lag_idx] * hop_ms

    # 5. Resting Jitter / False Actuation Rate (FAR %)
    # Percentage of true rest frames (|yt| <= 2.0 deg/s) where motor outputs non-zero velocity
    rest_mask = np.abs(yt) <= 2.0
    if np.any(rest_mask):
        false_actuations = np.abs(yp[rest_mask]) > 0.0
        far_pct = (np.sum(false_actuations) / np.sum(rest_mask)) * 100.0
    else:
        far_pct = 0.0

    return r_val, r2_val, rmse_val, lag_ms, far_pct

def log_print(msg, file_handle):
    print(msg, end="")
    file_handle.write(msg)

# --- 6. Main Cohort Training and Benchmark Loop ---
def main():
    f_log = open(LOG_FILE, "w", encoding="utf-8")

    header = (
        "========================================================================================================\n"
        "             TRUTHFUL MULTI-LAG sEMG VELOCITY REGRESSION BENCHMARKING REPORT                             \n"
        "========================================================================================================\n"
        f" Cohort: {len(COHORT)} Subjects | Train Reps: {TRAIN_REPS} | Holdout Test Reps: {TEST_REPS}\n"
        " Exclusions: Subject 11 (Entirely) | S4 Rep 6 | S7 Rep 6 | is_hold_plateau == 1\n"
        " Features  : 150 Pure sEMG Context Metrics (Lags 0-3 + Delta dEMG/dt | NO Velocity Feedback)\n"
        " Alignment : Causal Feedforward EMD Sweep strictly bounded in [0 ms, +100 ms]\n"
        " Actuation : Non-Linear Velocity Deadband (4.0 deg/s) applied to commanded outputs\n"
        "========================================================================================================\n\n"
    )
    log_print(header, f_log)

    cohort_results = {sub: {} for sub in COHORT}
    cohort_emd_log = {}

    for sub_id in COHORT:
        csv_path = os.path.join("datasets", "DB2", f"DB2_s{sub_id}", f"S{sub_id}_E1_A1_dynamic_velocity_features.csv")
        if not os.path.exists(csv_path):
            csv_path = f"S{sub_id}_E1_A1_dynamic_velocity_features.csv"

        if not os.path.exists(csv_path):
            log_print(f"[WARN] Subject {sub_id:02d}: CSV not found at {csv_path}. Skipping.\n", f_log)
            continue

        df = pd.read_csv(csv_path)

        # A. Filter static hold plateaus
        df = df[df["is_hold_plateau"] == 0].copy()

        # B. Enforce Subject-Specific Trial Exclusions
        if sub_id == 4:
            df = df[df["repetition"] != 6].copy()
        if sub_id == 7:
            df = df[df["repetition"] != 6].copy()

        meta_cols = ["timestamp_s", "theta_deg", "theta_vel_deg_s", "stimulus", "repetition", "is_hold_plateau"]
        feat_cols = [c for c in df.columns if c not in meta_cols]

        # C. Interleaved Split Partitioning
        train_mask = df["repetition"].isin(TRAIN_REPS) | ((df["stimulus"] == 0) & (df["repetition"] == 0))
        test_mask = df["repetition"].isin(TEST_REPS)

        train_df = df[train_mask].copy()
        test_df = df[test_mask].copy()

        if len(train_df) == 0 or len(test_df) == 0:
            log_print(f"[WARN] Subject {sub_id:02d}: Insufficient train or test partition size. Skipping.\n", f_log)
            continue

        # D. Causal EMD Optimization on Training Data [0 ms to +100 ms]
        shift_hops, opt_emd_ms = find_causal_emd_shift(train_df, feat_cols, slider_range_ms=(0, 100), step_ms=10)
        cohort_emd_log[sub_id] = opt_emd_ms

        # E. Pure sEMG Contextual Feature Stacking (150 Features)
        X_train_ctx = build_pure_semg_context_matrix(train_df, feat_cols, n_lags=3)
        y_train_raw = train_df["theta_vel_deg_s"].values
        reps_train = train_df["repetition"].values

        X_test_ctx = build_pure_semg_context_matrix(test_df, feat_cols, n_lags=3)
        y_test_raw = test_df["theta_vel_deg_s"].values

        # Apply Causal Feedforward Alignment
        X_tr_adj, y_tr_adj = align_feedforward_target(X_train_ctx, y_train_raw, shift_hops)
        X_te_adj, y_te_adj = align_feedforward_target(X_test_ctx, y_test_raw, shift_hops)

        # Synchronize repetition grouping vector
        reps_tr_adj = reps_train[:-shift_hops] if shift_hops > 0 else reps_train

        # F. Anti-Leakage Feature Standardization
        scaler = StandardScaler()
        X_tr_z = scaler.fit_transform(X_tr_adj)
        X_te_z = scaler.transform(X_te_adj)
        X_te_z = np.clip(X_te_z, -20.0, 20.0)

        print(f">>> Subject {sub_id:02d} | EMD Lead: +{opt_emd_ms:02d} ms | Features: {X_tr_z.shape[1]} | Train: {len(X_tr_z)} | Test: {len(X_te_z)}")

        # Repetition-stratified cross-validation generator
        gkf = GroupKFold(n_splits=min(4, len(np.unique(reps_tr_adj))))
        cv_groups = list(gkf.split(X_tr_z, y_tr_adj, groups=reps_tr_adj))

        # ----------------------------------------------------------------------
        # MODEL 1: Ridge Regression (L2 Linear)
        # ----------------------------------------------------------------------
        ridge_alphas = np.logspace(-1, 5, 15)
        ridge = RidgeCV(alphas=ridge_alphas, cv=cv_groups)
        ridge.fit(X_tr_z, y_tr_adj)
        y_pred_ridge = ridge.predict(X_te_z)
        r, r2, rmse, lag, far = evaluate_predictions(y_te_adj, y_pred_ridge)
        cohort_results[sub_id]["Ridge"] = {
            "r": r, "r2": r2, "rmse": rmse, "lag": lag, "far": far,
            "cfg": f"alpha={ridge.alpha_:.2e}"
        }

        # ----------------------------------------------------------------------
        # MODEL 2: Support Vector Regression (Linear Kernel)
        # ----------------------------------------------------------------------
        sub_step = max(1, len(X_tr_z) // 3500)
        svr_lin = SVR(kernel="linear", C=1.5, epsilon=2.0)
        svr_lin.fit(X_tr_z[::sub_step], y_tr_adj[::sub_step])
        y_pred_svr_lin = svr_lin.predict(X_te_z)
        r, r2, rmse, lag, far = evaluate_predictions(y_te_adj, y_pred_svr_lin)
        cohort_results[sub_id]["SVR_Linear"] = {
            "r": r, "r2": r2, "rmse": rmse, "lag": lag, "far": far,
            "cfg": "C=1.5, eps=2.0"
        }

        # ----------------------------------------------------------------------
        # MODEL 3: Support Vector Regression (RBF Kernel)
        # ----------------------------------------------------------------------
        svr_rbf = SVR(kernel="rbf", C=3.0, epsilon=1.5, gamma="scale")
        svr_rbf.fit(X_tr_z[::sub_step], y_tr_adj[::sub_step])
        y_pred_svr_rbf = svr_rbf.predict(X_te_z)
        r, r2, rmse, lag, far = evaluate_predictions(y_te_adj, y_pred_svr_rbf)
        cohort_results[sub_id]["SVR_RBF"] = {
            "r": r, "r2": r2, "rmse": rmse, "lag": lag, "far": far,
            "cfg": "C=3.0, eps=1.5"
        }

        # ----------------------------------------------------------------------
        # MODEL 4: Random Forest (Constrained Depth)
        # ----------------------------------------------------------------------
        rf = RandomForestRegressor(
            n_estimators=75,
            max_depth=12,
            min_samples_leaf=6,
            max_features="sqrt",
            n_jobs=-1,
            random_state=42
        )
        rf.fit(X_tr_z, y_tr_adj)
        y_pred_rf = rf.predict(X_te_z)
        r, r2, rmse, lag, far = evaluate_predictions(y_te_adj, y_pred_rf)
        cohort_results[sub_id]["Random_Forest"] = {
            "r": r, "r2": r2, "rmse": rmse, "lag": lag, "far": far,
            "cfg": "trees=75, depth=12"
        }

        # ----------------------------------------------------------------------
        # MODEL 5: HistGradientBoosting (Fast GBDT with L2 Regularization)
        # ----------------------------------------------------------------------
        hgb = HistGradientBoostingRegressor(
            max_iter=120,
            learning_rate=0.06,
            max_leaf_nodes=31,
            min_samples_leaf=15,
            l2_regularization=2.5,
            early_stopping=True,
            n_iter_no_change=10,
            random_state=42
        )
        hgb.fit(X_tr_z, y_tr_adj)
        y_pred_hgb = hgb.predict(X_te_z)
        r, r2, rmse, lag, far = evaluate_predictions(y_te_adj, y_pred_hgb)
        cohort_results[sub_id]["Hist_Gradient_Boost"] = {
            "r": r, "r2": r2, "rmse": rmse, "lag": lag, "far": far,
            "cfg": f"iters={hgb.n_iter_}, lr=0.06, l2=2.5"
        }

        # ----------------------------------------------------------------------
        # MODEL 6: Multi-Layer Perceptron (Shallow Neural Network)
        # ----------------------------------------------------------------------
        mlp = MLPRegressor(
            hidden_layer_sizes=(128, 64),
            activation="relu",
            solver="adam",
            alpha=0.10,             # L2 weight decay penalty
            learning_rate_init=0.002,
            max_iter=200,
            early_stopping=True,
            n_iter_no_change=12,
            random_state=42
        )
        mlp.fit(X_tr_z, y_tr_adj)
        y_pred_mlp = mlp.predict(X_te_z)
        r, r2, rmse, lag, far = evaluate_predictions(y_te_adj, y_pred_mlp)
        cohort_results[sub_id]["MLP_NeuralNet"] = {
            "r": r, "r2": r2, "rmse": rmse, "lag": lag, "far": far,
            "cfg": f"MLP[128,64] (iters={mlp.n_iter_})"
        }

    # --- 7. Format & Log Summary Reports ---
    summary_header = (
        "\n========================================================================================================\n"
        "                              HOLDOUT VALIDATION SUMMARY ACROSS COHORT                                  \n"
        "========================================================================================================\n"
        " Subj | EMD Lead  | Best Model Family     | Test Pearson r | Test R^2 (VAF) | RMSE (deg/s) | Peak Lag (ms) | Rest Jitter % \n"
        "--------------------------------------------------------------------------------------------------------\n"
    )
    log_print(summary_header, f_log)

    for sub_id in COHORT:
        sub_dict = cohort_results.get(sub_id, {})
        if not sub_dict:
            log_print(f"  S{sub_id:02d} |  [N/A]   | [DATA NOT PROCESSED]\n", f_log)
            continue

        best_model_name = max(sub_dict.keys(), key=lambda m: sub_dict[m]["r"])
        bm = sub_dict[best_model_name]
        emd_str = f"+{cohort_emd_log.get(sub_id, 0):02d} ms"

        line = (
            f"  S{sub_id:02d} | {emd_str:<9} | {best_model_name:<21} |     {bm['r']:6.4f}     |     {bm['r2']:6.4f}     "
            f"|    {bm['rmse']:6.2f}    |    {bm['lag']:+6.1f} ms   |     {bm['far']:4.2f}%\n"
        )
        log_print(line, f_log)

    log_print("--------------------------------------------------------------------------------------------------------\n\n", f_log)

    # --- 8. Aggregate Model Family Comparison Table ---
    agg_header = (
        "========================================================================================================\n"
        "                         OVERALL MODEL FAMILY PERFORMANCE (COHORT MEAN +/- STD)                         \n"
        "========================================================================================================\n"
        " Model Architecture    | Test Pearson r   | Test R^2 (VAF)   | RMSE (deg/s)     | Peak Lag (ms)  | Jitter (FAR) \n"
        "--------------------------------------------------------------------------------------------------------\n"
    )
    log_print(agg_header, f_log)

    for m_name in MODEL_FAMILIES:
        r_list = [cohort_results[s][m_name]["r"] for s in COHORT if m_name in cohort_results[s]]
        r2_list = [cohort_results[s][m_name]["r2"] for s in COHORT if m_name in cohort_results[s]]
        rmse_list = [cohort_results[s][m_name]["rmse"] for s in COHORT if m_name in cohort_results[s]]
        lag_list = [cohort_results[s][m_name]["lag"] for s in COHORT if m_name in cohort_results[s]]
        far_list = [cohort_results[s][m_name]["far"] for s in COHORT if m_name in cohort_results[s]]

        if not r_list:
            continue

        agg_line = (
            f" {m_name:<22} |  {np.mean(r_list):5.3f} +/- {np.std(r_list):4.3f} |  "
            f"{np.mean(r2_list):5.3f} +/- {np.std(r2_list):4.3f} |  "
            f"{np.mean(rmse_list):5.2f} +/- {np.std(rmse_list):4.2f}  | "
            f"{np.mean(lag_list):+5.1f} +/- {np.std(lag_list):4.1f} |  "
            f"{np.mean(far_list):4.2f} +/- {np.std(far_list):4.2f}%\n"
        )
        log_print(agg_line, f_log)

    log_print("========================================================================================================\n", f_log)
    print(f"\n[SAVED] Truthful evaluation report successfully dumped to: {LOG_FILE}\n")
    f_log.close()

if __name__ == "__main__":
    main()

             TRUTHFUL MULTI-LAG sEMG VELOCITY REGRESSION BENCHMARKING REPORT                             
 Cohort: 11 Subjects | Train Reps: [1, 3, 4, 6] | Holdout Test Reps: [2, 5]
 Exclusions: Subject 11 (Entirely) | S4 Rep 6 | S7 Rep 6 | is_hold_plateau == 1
 Features  : 150 Pure sEMG Context Metrics (Lags 0-3 + Delta dEMG/dt | NO Velocity Feedback)
 Alignment : Causal Feedforward EMD Sweep strictly bounded in [0 ms, +100 ms]
 Actuation : Non-Linear Velocity Deadband (4.0 deg/s) applied to commanded outputs

>>> Subject 01 | EMD Lead: +00 ms | Features: 150 | Train: 5505 | Test: 2958
>>> Subject 02 | EMD Lead: +10 ms | Features: 150 | Train: 6027 | Test: 3220
>>> Subject 03 | EMD Lead: +00 ms | Features: 150 | Train: 5585 | Test: 2599
>>> Subject 04 | EMD Lead: +00 ms | Features: 150 | Train: 4687 | Test: 2402
>>> Subject 05 | EMD Lead: +00 ms | Features: 150 | Train: 5901 | Test: 2848
>>> Subject 06 | EMD Lead: +00 ms | Features: 150 | Train: 6212 | Test: 2428
>>> Subject 07 | EMD 

In [ ]:
#!/usr/bin/env python3
"""
==============================================================================
TRUTHFUL CONTINUOUS sEMG VELOCITY REGRESSION PIPELINE (PYTHON)
Architecture: Midpoint-Centered Pure sEMG Context + Causal EMD [0 to +60 ms]
Cohort      : Subjects 1 to 12 (Excluding S11, S4 Rep 6, and S7 Rep 6)
Filtering   : is_hold_plateau == 1 strictly dropped across all trials
Partitioning: Interleaved Cross-Trial Split:
              - Train: Repetitions [1, 3, 4, 6]
              - Test : Repetitions [2, 5]
Features    : 90 Pure sEMG Context Features:
              - Lag 0 (Synchronous 30 Metrics)
              - Lag 2 (20 ms Historical Context)
              - Delta (Instantaneous Recruitment Rate: Lag 0 - Lag 1)
              - NO Kinematic Velocity Feedback
EMD Search  : Causal Feedforward Sweep strictly within [0 ms, +60 ms]
Actuation   : Piecewise Non-Linear Deadband (4.0 deg/s) for Zero Resting Jitter
Outputs     : Formatted Console Log + Text File Dump ('cohort_model_validation_report.txt')
==============================================================================
"""

import os
import sys
import numpy as np
import pandas as pd
from scipy import signal, stats
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV
from sklearn.svm import SVR
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import GroupKFold

# Optional SOTA Gradient Boosters (graceful fallback if not installed)
try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except ImportError:
    HAS_XGB = False

try:
    from catboost import CatBoostRegressor
    HAS_CAT = True
except ImportError:
    HAS_CAT = False

# --- 1. Cohort & Protocol Constants ---
COHORT = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12]  # Subject 11 strictly excluded
TRAIN_REPS = [1, 3, 4, 6]                      # Interleaved training trials
TEST_REPS = [2, 5]                             # Interleaved holdout test trials
V_DEADBAND = 4.0                               # deg/s: Non-linear prosthetic deadband
HOP_MS = 10.0                                  # 10 ms hop (100 Hz cadence)
LOG_FILE = "cohort_model_validation_report.txt"

# Dynamic Model Suite
MODEL_FAMILIES = ["Ridge", "SVR_Linear", "SVR_RBF", "Random_Forest", "Hist_Gradient_Boost", "MLP_NeuralNet"]
if HAS_XGB:
    MODEL_FAMILIES.append("XGBoost")
if HAS_CAT:
    MODEL_FAMILIES.append("CatBoost")

# --- 2. Compact Pure sEMG Spatiotemporal Context Builder (90 Features) ---
def build_compact_semg_context(df, feat_cols):
    """
    Constructs an orthogonal 90-dimensional feature space from 2 channels:
      - Raw 30 metrics at time t (Lag 0)
      - Raw 30 metrics at time t - 20 ms (Lag 2)
      - First-order derivative Delta x(t) = x(t) - x(t - 10 ms)
    Clamps state across timestamp breaks (> 25 ms) where hold plateaus were dropped.
    """
    X_raw = df[feat_cols].values
    t_raw = df["timestamp_s"].values
    n_samples, n_feats = X_raw.shape

    dt = np.diff(t_raw, prepend=t_raw[0])
    is_break = dt > 0.025

    # Lag 1 (10 ms prior) for computing Delta
    X_lag1 = np.zeros_like(X_raw)
    for i in range(n_samples):
        if i < 1 or is_break[i]:
            X_lag1[i] = X_raw[i]
        else:
            X_lag1[i] = X_raw[i - 1]

    # Lag 2 (20 ms prior) for temporal baseline reference
    X_lag2 = np.zeros_like(X_raw)
    for i in range(n_samples):
        if i < 2 or is_break[i] or is_break[i - 1]:
            X_lag2[i] = X_raw[i]
        else:
            X_lag2[i] = X_raw[i - 2]

    # Delta: Instantaneous rate-of-recruitment
    delta_X = X_raw - X_lag1

    # Stack: [Lag 0 (30), Lag 2 (30), Delta (30)] = 90 features
    X_context = np.hstack([X_raw, X_lag2, delta_X])
    return X_context

# --- 3. Causal Feedforward EMD Optimizer [0 ms to +60 ms] ---
def find_causal_emd_shift(df_train, feat_cols, slider_range_ms=(0, 60), step_ms=10):
    """
    Sweeps a strictly feedforward EMD window [0 ms, +60 ms] on training data.
    Aligns sEMG recruitment envelope with impending joint velocity.
    """
    mav_edc = df_train["EDC_MAV"].values if "EDC_MAV" in feat_cols else df_train[feat_cols[0]].values
    mav_fds = df_train["FDS_MAV"].values if "FDS_MAV" in feat_cols else df_train[feat_cols[1]].values
    emg_power = mav_edc + mav_fds
    vel_magnitude = np.abs(df_train["theta_vel_deg_s"].values)

    e_norm = (emg_power - np.mean(emg_power)) / (np.std(emg_power) + 1e-12)
    v_norm = (vel_magnitude - np.mean(vel_magnitude)) / (np.std(vel_magnitude) + 1e-12)

    min_ms, max_ms = slider_range_ms
    shift_steps = np.arange(int(min_ms / step_ms), int(max_ms / step_ms) + 1)

    best_corr = -1.0
    best_shift = 2  # Default fallback: 20 ms EMD

    for s in shift_steps:
        if s > 0:
            e_sub = e_norm[:-s]
            v_sub = v_norm[s:]
        else:
            e_sub = e_norm
            v_sub = v_norm

        if len(e_sub) > 50:
            r_val, _ = stats.pearsonr(e_sub, v_sub)
            if r_val > best_corr:
                best_corr = r_val
                best_shift = s

    return best_shift, best_shift * step_ms

def align_feedforward_target(X, y, shift_hops):
    if shift_hops > 0:
        return X[:-shift_hops], y[shift_hops:]
    return X, y

# --- 4. Piecewise Non-Linear Output Velocity Deadband ---
def apply_velocity_deadband(y_raw, v_db=V_DEADBAND):
    y_cmd = np.zeros_like(y_raw)
    active = np.abs(y_raw) > v_db
    y_cmd[active] = np.sign(y_raw[active]) * (np.abs(y_raw[active]) - v_db)
    return y_cmd

# --- 5. Diagnostic Evaluation Metrics ---
def evaluate_predictions(y_true, y_pred_raw, v_db=V_DEADBAND, hop_ms=HOP_MS):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred_raw = np.asarray(y_pred_raw, dtype=np.float64)
    y_cmd = apply_velocity_deadband(y_pred_raw, v_db=v_db)

    valid = ~(np.isnan(y_true) | np.isnan(y_cmd))
    yt = y_true[valid]
    yp = y_cmd[valid]

    if len(yt) < 10 or np.std(yt) == 0 or np.std(yp) == 0:
        return 0.0, 0.0, float("inf"), float("nan"), 100.0

    # 1. Pearson Correlation
    r_val, _ = stats.pearsonr(yt, yp)

    # 2. Variance Accounted For (VAF / R^2)
    sse = np.sum((yt - yp) ** 2)
    sst = np.sum((yt - np.mean(yt)) ** 2)
    r2_val = max(0.0, 1.0 - (sse / sst)) if sst > 0 else 0.0

    # 3. Root Mean Squared Error (deg/s)
    rmse_val = np.sqrt(np.mean((yt - yp) ** 2))

    # 4. Cross-Correlation Peak Lag (ms)
    max_lag_pts = 35  # +/- 350 ms horizon
    yp_detrend = yp - np.mean(yp)
    yt_detrend = yt - np.mean(yt)
    xcorr = signal.correlate(yp_detrend, yt_detrend, mode="full")
    lags = signal.correlation_lags(len(yp_detrend), len(yt_detrend), mode="full")

    lag_mask = (lags >= -max_lag_pts) & (lags <= max_lag_pts)
    valid_lags = lags[lag_mask]
    valid_xcorr = xcorr[lag_mask]
    peak_lag_idx = np.argmax(valid_xcorr)
    lag_ms = valid_lags[peak_lag_idx] * hop_ms

    # 5. Resting Jitter / False Actuation Rate (FAR %)
    rest_mask = np.abs(yt) <= 2.0
    if np.any(rest_mask):
        false_actuations = np.abs(yp[rest_mask]) > 0.0
        far_pct = (np.sum(false_actuations) / np.sum(rest_mask)) * 100.0
    else:
        far_pct = 0.0

    return r_val, r2_val, rmse_val, lag_ms, far_pct

def log_print(msg, file_handle):
    print(msg, end="")
    file_handle.write(msg)

# --- 6. Main Cohort Training and Benchmark Loop ---
def main():
    f_log = open(LOG_FILE, "w", encoding="utf-8")

    header = (
        "========================================================================================================\n"
        "             TRUTHFUL CONTINUOUS sEMG VELOCITY REGRESSION BENCHMARKING REPORT                            \n"
        "========================================================================================================\n"
        f" Cohort: {len(COHORT)} Subjects | Train Reps: {TRAIN_REPS} | Holdout Test Reps: {TEST_REPS}\n"
        " Exclusions: Subject 11 (Entirely) | S4 Rep 6 | S7 Rep 6 | is_hold_plateau == 1\n"
        " Alignment : Midpoint-Centered Kinematics (MATLAB mid_idx) + Causal EMD [0 ms to +60 ms]\n"
        " Features  : 90 Pure sEMG Context Metrics (Lags 0 & 2 + Delta Rate | Pure Muscle Decoding)\n"
        " Actuation : Piecewise Non-Linear Velocity Deadband (4.0 deg/s) applied to outputs\n"
        "========================================================================================================\n\n"
    )
    log_print(header, f_log)

    cohort_results = {sub: {} for sub in COHORT}
    cohort_emd_log = {}

    for sub_id in COHORT:
        csv_path = os.path.join("datasets", "DB2", f"DB2_s{sub_id}", f"S{sub_id}_E1_A1_dynamic_velocity_features.csv")
        if not os.path.exists(csv_path):
            csv_path = f"S{sub_id}_E1_A1_dynamic_velocity_features.csv"

        if not os.path.exists(csv_path):
            log_print(f"[WARN] Subject {sub_id:02d}: CSV not found at {csv_path}. Skipping.\n", f_log)
            continue

        df = pd.read_csv(csv_path)

        # A. Filter static hold plateaus
        df = df[df["is_hold_plateau"] == 0].copy()

        # B. Enforce Subject-Specific Trial Exclusions
        if sub_id == 4:
            df = df[df["repetition"] != 6].copy()
        if sub_id == 7:
            df = df[df["repetition"] != 6].copy()

        meta_cols = ["timestamp_s", "theta_deg", "theta_vel_deg_s", "stimulus", "repetition", "is_hold_plateau"]
        feat_cols = [c for c in df.columns if c not in meta_cols]

        # C. Interleaved Partitioning
        train_mask = df["repetition"].isin(TRAIN_REPS) | ((df["stimulus"] == 0) & (df["repetition"] == 0))
        test_mask = df["repetition"].isin(TEST_REPS)

        train_df = df[train_mask].copy()
        test_df = df[test_mask].copy()

        if len(train_df) == 0 or len(test_df) == 0:
            log_print(f"[WARN] Subject {sub_id:02d}: Insufficient partition size. Skipping.\n", f_log)
            continue

        # D. Causal EMD Optimization on Training Partition [0 ms to +60 ms]
        shift_hops, opt_emd_ms = find_causal_emd_shift(train_df, feat_cols, slider_range_ms=(0, 60), step_ms=10)
        cohort_emd_log[sub_id] = opt_emd_ms

        # E. Extract Pure sEMG Context (90 Features)
        X_train_ctx = build_compact_semg_context(train_df, feat_cols)
        y_train_raw = train_df["theta_vel_deg_s"].values
        reps_train = train_df["repetition"].values

        X_test_ctx = build_compact_semg_context(test_df, feat_cols)
        y_test_raw = test_df["theta_vel_deg_s"].values

        # Apply Causal Feedforward EMD Shift
        X_tr_adj, y_tr_adj = align_feedforward_target(X_train_ctx, y_train_raw, shift_hops)
        X_te_adj, y_te_adj = align_feedforward_target(X_test_ctx, y_test_raw, shift_hops)

        reps_tr_adj = reps_train[:-shift_hops] if shift_hops > 0 else reps_train

        # F. Anti-Leakage Feature Standardization
        scaler = StandardScaler()
        X_tr_z = scaler.fit_transform(X_tr_adj)
        X_te_z = scaler.transform(X_te_adj)
        X_te_z = np.clip(X_te_z, -20.0, 20.0)

        print(f">>> Subject {sub_id:02d} | EMD Lead: +{opt_emd_ms:02d} ms | Features: {X_tr_z.shape[1]} | Train: {len(X_tr_z)} | Test: {len(X_te_z)}")

        # Group-KFold CV grouped on repetition
        gkf = GroupKFold(n_splits=min(4, len(np.unique(reps_tr_adj))))
        cv_groups = list(gkf.split(X_tr_z, y_tr_adj, groups=reps_tr_adj))

        # ----------------------------------------------------------------------
        # MODEL 1: Ridge Regression (L2 Linear)
        # ----------------------------------------------------------------------
        ridge_alphas = np.logspace(0, 5, 12)
        ridge = RidgeCV(alphas=ridge_alphas, cv=cv_groups)
        ridge.fit(X_tr_z, y_tr_adj)
        y_pred_ridge = ridge.predict(X_te_z)
        r, r2, rmse, lag, far = evaluate_predictions(y_te_adj, y_pred_ridge)
        cohort_results[sub_id]["Ridge"] = {
            "r": r, "r2": r2, "rmse": rmse, "lag": lag, "far": far,
            "cfg": f"alpha={ridge.alpha_:.2e}"
        }

        # ----------------------------------------------------------------------
        # MODEL 2: Support Vector Regression (Linear Kernel)
        # ----------------------------------------------------------------------
        sub_step = max(1, len(X_tr_z) // 3500)
        svr_lin = SVR(kernel="linear", C=1.5, epsilon=2.0)
        svr_lin.fit(X_tr_z[::sub_step], y_tr_adj[::sub_step])
        y_pred_svr_lin = svr_lin.predict(X_te_z)
        r, r2, rmse, lag, far = evaluate_predictions(y_te_adj, y_pred_svr_lin)
        cohort_results[sub_id]["SVR_Linear"] = {
            "r": r, "r2": r2, "rmse": rmse, "lag": lag, "far": far,
            "cfg": "C=1.5, eps=2.0"
        }

        # ----------------------------------------------------------------------
        # MODEL 3: Support Vector Regression (RBF Kernel)
        # ----------------------------------------------------------------------
        svr_rbf = SVR(kernel="rbf", C=3.0, epsilon=1.5, gamma="scale")
        svr_rbf.fit(X_tr_z[::sub_step], y_tr_adj[::sub_step])
        y_pred_svr_rbf = svr_rbf.predict(X_te_z)
        r, r2, rmse, lag, far = evaluate_predictions(y_te_adj, y_pred_svr_rbf)
        cohort_results[sub_id]["SVR_RBF"] = {
            "r": r, "r2": r2, "rmse": rmse, "lag": lag, "far": far,
            "cfg": "C=3.0, eps=1.5"
        }

        # ----------------------------------------------------------------------
        # MODEL 4: Random Forest (Subsampled Features per Split)
        # ----------------------------------------------------------------------
        rf = RandomForestRegressor(
            n_estimators=80,
            max_depth=12,
            min_samples_leaf=6,
            max_features="sqrt",
            n_jobs=-1,
            random_state=42
        )
        rf.fit(X_tr_z, y_tr_adj)
        y_pred_rf = rf.predict(X_te_z)
        r, r2, rmse, lag, far = evaluate_predictions(y_te_adj, y_pred_rf)
        cohort_results[sub_id]["Random_Forest"] = {
            "r": r, "r2": r2, "rmse": rmse, "lag": lag, "far": far,
            "cfg": "trees=80, depth=12"
        }

        # ----------------------------------------------------------------------
        # MODEL 5: HistGradientBoosting (Fast GBDT with L2 Regularization)
        # ----------------------------------------------------------------------
        hgb = HistGradientBoostingRegressor(
            max_iter=130,
            learning_rate=0.06,
            max_leaf_nodes=31,
            min_samples_leaf=15,
            l2_regularization=2.5,
            early_stopping=True,
            n_iter_no_change=10,
            random_state=42
        )
        hgb.fit(X_tr_z, y_tr_adj)
        y_pred_hgb = hgb.predict(X_te_z)
        r, r2, rmse, lag, far = evaluate_predictions(y_te_adj, y_pred_hgb)
        cohort_results[sub_id]["Hist_Gradient_Boost"] = {
            "r": r, "r2": r2, "rmse": rmse, "lag": lag, "far": far,
            "cfg": f"iters={hgb.n_iter_}, lr=0.06, l2=2.5"
        }

        # ----------------------------------------------------------------------
        # MODEL 6: Multi-Layer Perceptron (Shallow Neural Network)
        # ----------------------------------------------------------------------
        mlp = MLPRegressor(
            hidden_layer_sizes=(96, 48),
            activation="relu",
            solver="adam",
            alpha=0.12,             # L2 weight decay to suppress resting baseline drift
            learning_rate_init=0.002,
            max_iter=200,
            early_stopping=True,
            n_iter_no_change=12,
            random_state=42
        )
        mlp.fit(X_tr_z, y_tr_adj)
        y_pred_mlp = mlp.predict(X_te_z)
        r, r2, rmse, lag, far = evaluate_predictions(y_te_adj, y_pred_mlp)
        cohort_results[sub_id]["MLP_NeuralNet"] = {
            "r": r, "r2": r2, "rmse": rmse, "lag": lag, "far": far,
            "cfg": f"MLP[96,48] (iters={mlp.n_iter_})"
        }

        # ----------------------------------------------------------------------
        # MODEL 7 (OPTIONAL): XGBoost Regressor (L1 Sparsity + L2 Regularization)
        # ----------------------------------------------------------------------
        if HAS_XGB:
            xgb = XGBRegressor(
                n_estimators=120,
                max_depth=5,
                learning_rate=0.06,
                reg_alpha=1.0,      # L1 Lasso penalty selects best orthogonal metrics
                reg_lambda=2.0,     # L2 Ridge penalty prevents tree extreme leaf weights
                subsample=0.85,
                colsample_bytree=0.7,
                n_jobs=-1,
                random_state=42
            )
            xgb.fit(X_tr_z, y_tr_adj)
            y_pred_xgb = xgb.predict(X_te_z)
            r, r2, rmse, lag, far = evaluate_predictions(y_te_adj, y_pred_xgb)
            cohort_results[sub_id]["XGBoost"] = {
                "r": r, "r2": r2, "rmse": rmse, "lag": lag, "far": far,
                "cfg": "depth=5, L1=1.0, L2=2.0"
            }

        # ----------------------------------------------------------------------
        # MODEL 8 (OPTIONAL): CatBoost Regressor (Oblivious Trees)
        # ----------------------------------------------------------------------
        if HAS_CAT:
            cb = CatBoostRegressor(
                iterations=150,
                depth=6,
                learning_rate=0.06,
                l2_leaf_reg=4.0,
                verbose=0,
                random_seed=42
            )
            cb.fit(X_tr_z, y_tr_adj)
            y_pred_cb = cb.predict(X_te_z)
            r, r2, rmse, lag, far = evaluate_predictions(y_te_adj, y_pred_cb)
            cohort_results[sub_id]["CatBoost"] = {
                "r": r, "r2": r2, "rmse": rmse, "lag": lag, "far": far,
                "cfg": "depth=6, L2=4.0"
            }

    # --- 7. Format & Log Summary Reports ---
    summary_header = (
        "\n========================================================================================================\n"
        "                              HOLDOUT VALIDATION SUMMARY ACROSS COHORT                                  \n"
        "========================================================================================================\n"
        " Subj | EMD Lead  | Best Model Family     | Test Pearson r | Test R^2 (VAF) | RMSE (deg/s) | Peak Lag (ms) | Rest Jitter % \n"
        "--------------------------------------------------------------------------------------------------------\n"
    )
    log_print(summary_header, f_log)

    for sub_id in COHORT:
        sub_dict = cohort_results.get(sub_id, {})
        if not sub_dict:
            log_print(f"  S{sub_id:02d} |  [N/A]   | [DATA NOT PROCESSED]\n", f_log)
            continue

        best_model_name = max(sub_dict.keys(), key=lambda m: sub_dict[m]["r"])
        bm = sub_dict[best_model_name]
        emd_str = f"+{cohort_emd_log.get(sub_id, 0):02d} ms"

        line = (
            f"  S{sub_id:02d} | {emd_str:<9} | {best_model_name:<21} |     {bm['r']:6.4f}     |     {bm['r2']:6.4f}     "
            f"|    {bm['rmse']:6.2f}    |    {bm['lag']:+6.1f} ms   |     {bm['far']:4.2f}%\n"
        )
        log_print(line, f_log)

    log_print("--------------------------------------------------------------------------------------------------------\n\n", f_log)

    # --- 8. Aggregate Model Family Comparison Table ---
    agg_header = (
        "========================================================================================================\n"
        "                         OVERALL MODEL FAMILY PERFORMANCE (COHORT MEAN +/- STD)                         \n"
        "========================================================================================================\n"
        " Model Architecture    | Test Pearson r   | Test R^2 (VAF)   | RMSE (deg/s)     | Peak Lag (ms)  | Jitter (FAR) \n"
        "--------------------------------------------------------------------------------------------------------\n"
    )
    log_print(agg_header, f_log)

    for m_name in MODEL_FAMILIES:
        r_list = [cohort_results[s][m_name]["r"] for s in COHORT if m_name in cohort_results[s]]
        r2_list = [cohort_results[s][m_name]["r2"] for s in COHORT if m_name in cohort_results[s]]
        rmse_list = [cohort_results[s][m_name]["rmse"] for s in COHORT if m_name in cohort_results[s]]
        lag_list = [cohort_results[s][m_name]["lag"] for s in COHORT if m_name in cohort_results[s]]
        far_list = [cohort_results[s][m_name]["far"] for s in COHORT if m_name in cohort_results[s]]

        if not r_list:
            continue

        agg_line = (
            f" {m_name:<22} |  {np.mean(r_list):5.3f} +/- {np.std(r_list):4.3f} |  "
            f"{np.mean(r2_list):5.3f} +/- {np.std(r2_list):4.3f} |  "
            f"{np.mean(rmse_list):5.2f} +/- {np.std(rmse_list):4.2f}  | "
            f"{np.mean(lag_list):+5.1f} +/- {np.std(lag_list):4.1f} |  "
            f"{np.mean(far_list):4.2f} +/- {np.std(far_list):4.2f}%\n"
        )
        log_print(agg_line, f_log)

    log_print("========================================================================================================\n", f_log)
    print(f"\n[SAVED] Comprehensive truthful benchmark report dumped to: {LOG_FILE}\n")
    f_log.close()

if __name__ == "__main__":
    main()

             TRUTHFUL CONTINUOUS sEMG VELOCITY REGRESSION BENCHMARKING REPORT                            
 Cohort: 11 Subjects | Train Reps: [1, 3, 4, 6] | Holdout Test Reps: [2, 5]
 Exclusions: Subject 11 (Entirely) | S4 Rep 6 | S7 Rep 6 | is_hold_plateau == 1
 Alignment : Midpoint-Centered Kinematics (MATLAB mid_idx) + Causal EMD [0 ms to +60 ms]
 Features  : 90 Pure sEMG Context Metrics (Lags 0 & 2 + Delta Rate | Pure Muscle Decoding)
 Actuation : Piecewise Non-Linear Velocity Deadband (4.0 deg/s) applied to outputs

>>> Subject 01 | EMD Lead: +00 ms | Features: 90 | Train: 5505 | Test: 2958
>>> Subject 02 | EMD Lead: +30 ms | Features: 90 | Train: 6025 | Test: 3218
>>> Subject 03 | EMD Lead: +00 ms | Features: 90 | Train: 5585 | Test: 2599
>>> Subject 04 | EMD Lead: +00 ms | Features: 90 | Train: 4687 | Test: 2402
>>> Subject 05 | EMD Lead: +10 ms | Features: 90 | Train: 5900 | Test: 2847
>>> Subject 06 | EMD Lead: +00 ms | Features: 90 | Train: 6212 | Test: 2428
>>> Subject 07 | E

In [6]:
#!/usr/bin/env python3
"""
==============================================================================
CONTINUOUS sEMG VELOCITY REGRESSION: FULL-TIMELINE TRAIN / BOUNDED TEST AUDIT
Cohort      : Subjects 1 to 12 (Excluding S11, S4 Rep 6, and S7 Rep 6)
Filtering   : is_hold_plateau == 1 strictly dropped across all data
Testing     : Bounded Epochs [t_cue - 1.0s, t_cue_end + 1.0s] on Reps [2, 6]
              (Automatic fallback to Rep 5 for S04 and S07)
Training    : Entire remaining continuous timeline (Prompted + Natural Movements)
Features    : 90 Pure sEMG Context Metrics (Lags 0 & 2 + Delta Rate | NO Velocity Feedback)
EMD Range   : Causal feedforward sweep strictly within [0 ms, +60 ms]
Actuation   : Piecewise Non-Linear Deadband (4.0 deg/s) + Dual-Tier Envelope Gate
Outputs     : Formatted Console Log + Text File Dump ('cohort_model_validation_report.txt')
==============================================================================
"""

import os
import sys
import numpy as np
import pandas as pd
from scipy import signal, stats
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV
from sklearn.svm import SVR
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import KFold

# Optional gradient boosters
try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except ImportError:
    HAS_XGB = False

try:
    from catboost import CatBoostRegressor
    HAS_CAT = True
except ImportError:
    HAS_CAT = False

# --- 1. Experimental Constants ---
COHORT = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12]  # Subject 11 strictly excluded
TARGET_MOVEMENTS = [5, 6]                      # Movement 5 (Open), Movement 6 (Fist)
V_DEADBAND = 4.0                               # deg/s: Velocity deadband
HOP_MS = 10.0                                  # 10 ms hop (100 Hz loop)
LOG_FILE = "cohort_model_validation_report.txt"

MODEL_FAMILIES = ["Ridge", "SVR_Linear", "SVR_RBF", "Random_Forest", "Hist_Gradient_Boost", "MLP_NeuralNet"]
if HAS_XGB:
    MODEL_FAMILIES.append("XGBoost")
if HAS_CAT:
    MODEL_FAMILIES.append("CatBoost")

# --- 2. Bounded Test Slice & Timeline Partitioning ---
def partition_timeline_bounded(df, sub_id, buffer_sec=1.0):
    """
    Carves out [t_cue - 1.0s, t_cue_end + 1.0s] test windows around Reps 2 & 6.
    Falls back to Rep 5 for S04 and S07 where Rep 6 is corrupted.
    Assigns all other continuous samples (prompted + natural movements) to train.
    """
    # Select test reps
    if sub_id in [4, 7]:
        test_reps = [2, 5]
    else:
        test_reps = [2, 6]

    is_test = np.zeros(len(df), dtype=bool)

    for m in TARGET_MOVEMENTS:
        for r in test_reps:
            cue_mask = (df["stimulus"] == m) & (df["repetition"] == r)
            if not np.any(cue_mask):
                continue

            t_cue_start = df.loc[cue_mask, "timestamp_s"].min()
            t_cue_end = df.loc[cue_mask, "timestamp_s"].max()

            # Expand bounds by buffer_sec
            t_eval_start = t_cue_start - buffer_sec
            t_eval_end = t_cue_end + buffer_sec

            slice_mask = (df["timestamp_s"] >= t_eval_start) & (df["timestamp_s"] <= t_eval_end)
            is_test[slice_mask] = True

    # Corrupted trial exclusion: drop Rep 6 entirely for S04 and S07
    is_corrupt = np.zeros(len(df), dtype=bool)
    if sub_id in [4, 7]:
        is_corrupt = (df["repetition"] == 6).values

    # Drop static hold plateaus
    is_plateau = (df["is_hold_plateau"] == 1).values

    valid_mask = ~is_corrupt & ~is_plateau
    train_mask = valid_mask & ~is_test
    test_mask = valid_mask & is_test

    return train_mask, test_mask, test_reps

# --- 3. Compact Pure sEMG Context Builder (90 Features) ---
def build_compact_semg_context(df, feat_cols):
    """
    Constructs an orthogonal 90-dimensional feature space from 2 channels:
      - Raw 30 metrics at frame t (Lag 0)
      - Raw 30 metrics at frame t - 20 ms (Lag 2)
      - Instantaneous rate-of-change: Delta x(t) = x(t) - x(t - 10 ms)
    """
    X_raw = df[feat_cols].values
    t_raw = df["timestamp_s"].values
    n_samples, _ = X_raw.shape

    dt = np.diff(t_raw, prepend=t_raw[0])
    is_break = dt > 0.025

    X_lag1 = np.zeros_like(X_raw)
    for i in range(n_samples):
        if i < 1 or is_break[i]:
            X_lag1[i] = X_raw[i]
        else:
            X_lag1[i] = X_raw[i - 1]

    X_lag2 = np.zeros_like(X_raw)
    for i in range(n_samples):
        if i < 2 or is_break[i] or is_break[i - 1]:
            X_lag2[i] = X_raw[i]
        else:
            X_lag2[i] = X_raw[i - 2]

    delta_X = X_raw - X_lag1
    return np.hstack([X_raw, X_lag2, delta_X])

# --- 4. Causal Feedforward EMD Optimizer [0 ms to +60 ms] ---
def find_causal_emd_shift(df_train, feat_cols, slider_range_ms=(0, 60), step_ms=10):
    mav_edc = df_train["EDC_MAV"].values if "EDC_MAV" in feat_cols else df_train[feat_cols[0]].values
    mav_fds = df_train["FDS_MAV"].values if "FDS_MAV" in feat_cols else df_train[feat_cols[1]].values
    emg_power = mav_edc + mav_fds
    vel_magnitude = np.abs(df_train["theta_vel_deg_s"].values)

    e_norm = (emg_power - np.mean(emg_power)) / (np.std(emg_power) + 1e-12)
    v_norm = (vel_magnitude - np.mean(vel_magnitude)) / (np.std(vel_magnitude) + 1e-12)

    min_ms, max_ms = slider_range_ms
    shift_steps = np.arange(int(min_ms / step_ms), int(max_ms / step_ms) + 1)

    best_corr = -1.0
    best_shift = 2

    for s in shift_steps:
        if s > 0:
            e_sub = e_norm[:-s]
            v_sub = v_norm[s:]
        else:
            e_sub = e_norm
            v_sub = v_norm

        if len(e_sub) > 50:
            r_val, _ = stats.pearsonr(e_sub, v_sub)
            if r_val > best_corr:
                best_corr = r_val
                best_shift = s

    return best_shift, best_shift * step_ms

def align_feedforward_target(X, y, shift_hops):
    if shift_hops > 0:
        return X[:-shift_hops], y[shift_hops:]
    return X, y

# --- 5. Non-Linear Velocity Deadband & Dual-Tier Gating ---
def apply_actuation_postprocessing(y_raw, mav_total, mav_rest_floor, v_db=V_DEADBAND, use_dual_tier=True):
    """
    Applies piecewise non-linear velocity deadband and dual-tier electrical MAV gating.
    """
    y_cmd = np.zeros_like(y_raw)
    active = np.abs(y_raw) > v_db
    y_cmd[active] = np.sign(y_raw[active]) * (np.abs(y_raw[active]) - v_db)

    if use_dual_tier:
        is_electrically_quiet = mav_total < (1.5 * mav_rest_floor)
        y_cmd[is_electrically_quiet] = 0.0

    return y_cmd

# --- 6. Comprehensive Metric Evaluator ---
def evaluate_predictions(y_true, y_cmd, hop_ms=HOP_MS):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_cmd = np.asarray(y_cmd, dtype=np.float64)

    valid = ~(np.isnan(y_true) | np.isnan(y_cmd))
    yt = y_true[valid]
    yp = y_cmd[valid]

    if len(yt) < 10 or np.std(yt) == 0 or np.std(yp) == 0:
        return 0.0, 0.0, float("inf"), float("nan"), 100.0

    # 1. Pearson Correlation
    r_val, _ = stats.pearsonr(yt, yp)

    # 2. Variance Accounted For (VAF / R^2)
    sse = np.sum((yt - yp) ** 2)
    sst = np.sum((yt - np.mean(yt)) ** 2)
    r2_val = max(0.0, 1.0 - (sse / sst)) if sst > 0 else 0.0

    # 3. RMSE (deg/s)
    rmse_val = np.sqrt(np.mean((yt - yp) ** 2))

    # 4. Cross-Correlation Peak Lag (ms)
    max_lag_pts = 35
    yp_detrend = yp - np.mean(yp)
    yt_detrend = yt - np.mean(yt)
    xcorr = signal.correlate(yp_detrend, yt_detrend, mode="full")
    lags = signal.correlation_lags(len(yp_detrend), len(yt_detrend), mode="full")

    lag_mask = (lags >= -max_lag_pts) & (lags <= max_lag_pts)
    valid_lags = lags[lag_mask]
    valid_xcorr = xcorr[lag_mask]
    peak_lag_idx = np.argmax(valid_xcorr)
    lag_ms = valid_lags[peak_lag_idx] * hop_ms

    # 5. True Rest Jitter (FAR %) on verified resting frames (|y_true| <= 2.0 deg/s)
    rest_mask = np.abs(yt) <= 2.0
    if np.any(rest_mask):
        false_actuations = np.abs(yp[rest_mask]) > 0.0
        far_pct = (np.sum(false_actuations) / np.sum(rest_mask)) * 100.0
    else:
        far_pct = 0.0

    return r_val, r2_val, rmse_val, lag_ms, far_pct

def log_print(msg, file_handle):
    print(msg, end="")
    file_handle.write(msg)

# --- 7. Main Cohort Training and Benchmark Loop ---
def main():
    f_log = open(LOG_FILE, "w", encoding="utf-8")

    header = (
        "========================================================================================================\n"
        "        CONTINUOUS sEMG VELOCITY REGRESSION: FULL TIMELINE TRAIN / BOUNDED TEST REPORT                  \n"
        "========================================================================================================\n"
        f" Cohort: {len(COHORT)} Subjects | Exclusions: Subject 11 (Entirely) | S4 Rep 6 | S7 Rep 6\n"
        " Strategy: Training on full continuous timeline (prompted + natural movements, minus plateaus)\n"
        " Testing : Bounded task epochs [t_cue - 1.0s, t_cue_end + 1.0s] on Reps [2, 6] (Fallback Rep 5 for S4/S7)\n"
        " Features: 90 Pure sEMG Context Metrics (Lags 0 & 2 + Delta Rate | Pure Muscle Decoding)\n"
        " Actuation: Non-Linear Deadband (4.0 deg/s) + Dual-Tier Electrical MAV Envelope Gating\n"
        "========================================================================================================\n\n"
    )
    log_print(header, f_log)

    cohort_results = {sub: {} for sub in COHORT}
    cohort_emd_log = {}
    cohort_test_reps_log = {}

    for sub_id in COHORT:
        csv_path = os.path.join("datasets", "DB2", f"DB2_s{sub_id}", f"S{sub_id}_E1_A1_dynamic_velocity_features.csv")
        if not os.path.exists(csv_path):
            csv_path = f"S{sub_id}_E1_A1_dynamic_velocity_features.csv"

        if not os.path.exists(csv_path):
            log_print(f"[WARN] Subject {sub_id:02d}: CSV not found. Skipping.\n", f_log)
            continue

        df = pd.read_csv(csv_path)

        # 1. Partition Timeline: Bounded Test Slices vs Full Continuous Train
        train_mask, test_mask, test_reps = partition_timeline_bounded(df, sub_id, buffer_sec=1.0)
        cohort_test_reps_log[sub_id] = test_reps

        train_df = df[train_mask].copy()
        test_df = df[test_mask].copy()

        if len(train_df) == 0 or len(test_df) == 0:
            log_print(f"[WARN] Subject {sub_id:02d}: Insufficient partition size. Skipping.\n", f_log)
            continue

        meta_cols = ["timestamp_s", "theta_deg", "theta_vel_deg_s", "stimulus", "repetition", "is_hold_plateau"]
        feat_cols = [c for c in df.columns if c not in meta_cols]

        # 2. Causal EMD Optimization on Training Data [0 ms to +60 ms]
        shift_hops, opt_emd_ms = find_causal_emd_shift(train_df, feat_cols, slider_range_ms=(0, 60), step_ms=10)
        cohort_emd_log[sub_id] = opt_emd_ms

        # 3. Extract 90 Pure sEMG Features
        X_train_ctx = build_compact_semg_context(train_df, feat_cols)
        y_train_raw = train_df["theta_vel_deg_s"].values

        X_test_ctx = build_compact_semg_context(test_df, feat_cols)
        y_test_raw = test_df["theta_vel_deg_s"].values

        # Compute MAV total for dual-tier gating
        mav_total_train = (train_df["EDC_MAV"] + train_df["FDS_MAV"]).values
        mav_total_test = (test_df["EDC_MAV"] + test_df["FDS_MAV"]).values
        # Baseline noise floor: 10th percentile of training MAV
        mav_rest_floor = np.percentile(mav_total_train, 10)

        # Apply Causal Feedforward Shift
        X_tr_adj, y_tr_adj = align_feedforward_target(X_train_ctx, y_train_raw, shift_hops)
        X_te_adj, y_te_adj = align_feedforward_target(X_test_ctx, y_test_raw, shift_hops)
        mav_te_adj = mav_total_test[shift_hops:] if shift_hops > 0 else mav_total_test

        # 4. Anti-Leakage Feature Standardization
        scaler = StandardScaler()
        X_tr_z = scaler.fit_transform(X_tr_adj)
        X_te_z = scaler.transform(X_te_adj)
        X_te_z = np.clip(X_te_z, -20.0, 20.0)

        print(f">>> Subject {sub_id:02d} | Test Reps: {test_reps} | EMD Lead: +{opt_emd_ms:02d} ms | Train: {len(X_tr_z)} | Test: {len(X_te_z)}")

        cv_splits = KFold(n_splits=4, shuffle=True, random_state=42)

        # ----------------------------------------------------------------------
        # MODEL 1: Ridge Regression
        # ----------------------------------------------------------------------
        ridge = RidgeCV(alphas=np.logspace(0, 5, 12), cv=cv_splits)
        ridge.fit(X_tr_z, y_tr_adj)
        y_pred_ridge = ridge.predict(X_te_z)
        y_cmd_ridge = apply_actuation_postprocessing(y_pred_ridge, mav_te_adj, mav_rest_floor)
        r, r2, rmse, lag, far = evaluate_predictions(y_te_adj, y_cmd_ridge)
        cohort_results[sub_id]["Ridge"] = {"r": r, "r2": r2, "rmse": rmse, "lag": lag, "far": far}

        # ----------------------------------------------------------------------
        # MODEL 2: Support Vector Regression (Linear)
        # ----------------------------------------------------------------------
        sub_step = max(1, len(X_tr_z) // 4000)
        svr_lin = SVR(kernel="linear", C=1.5, epsilon=2.0)
        svr_lin.fit(X_tr_z[::sub_step], y_tr_adj[::sub_step])
        y_pred_svr_lin = svr_lin.predict(X_te_z)
        y_cmd_svr_lin = apply_actuation_postprocessing(y_pred_svr_lin, mav_te_adj, mav_rest_floor)
        r, r2, rmse, lag, far = evaluate_predictions(y_te_adj, y_cmd_svr_lin)
        cohort_results[sub_id]["SVR_Linear"] = {"r": r, "r2": r2, "rmse": rmse, "lag": lag, "far": far}

        # ----------------------------------------------------------------------
        # MODEL 3: Support Vector Regression (RBF)
        # ----------------------------------------------------------------------
        svr_rbf = SVR(kernel="rbf", C=3.0, epsilon=1.5, gamma="scale")
        svr_rbf.fit(X_tr_z[::sub_step], y_tr_adj[::sub_step])
        y_pred_svr_rbf = svr_rbf.predict(X_te_z)
        y_cmd_svr_rbf = apply_actuation_postprocessing(y_pred_svr_rbf, mav_te_adj, mav_rest_floor)
        r, r2, rmse, lag, far = evaluate_predictions(y_te_adj, y_cmd_svr_rbf)
        cohort_results[sub_id]["SVR_RBF"] = {"r": r, "r2": r2, "rmse": rmse, "lag": lag, "far": far}

        # ----------------------------------------------------------------------
        # MODEL 4: Random Forest
        # ----------------------------------------------------------------------
        rf = RandomForestRegressor(n_estimators=80, max_depth=12, min_samples_leaf=6, max_features="sqrt", n_jobs=-1, random_state=42)
        rf.fit(X_tr_z, y_tr_adj)
        y_pred_rf = rf.predict(X_te_z)
        y_cmd_rf = apply_actuation_postprocessing(y_pred_rf, mav_te_adj, mav_rest_floor)
        r, r2, rmse, lag, far = evaluate_predictions(y_te_adj, y_cmd_rf)
        cohort_results[sub_id]["Random_Forest"] = {"r": r, "r2": r2, "rmse": rmse, "lag": lag, "far": far}

        # ----------------------------------------------------------------------
        # MODEL 5: HistGradientBoosting
        # ----------------------------------------------------------------------
        hgb = HistGradientBoostingRegressor(max_iter=130, learning_rate=0.06, max_leaf_nodes=31, min_samples_leaf=15, l2_regularization=2.5, early_stopping=True, n_iter_no_change=10, random_state=42)
        hgb.fit(X_tr_z, y_tr_adj)
        y_pred_hgb = hgb.predict(X_te_z)
        y_cmd_hgb = apply_actuation_postprocessing(y_pred_hgb, mav_te_adj, mav_rest_floor)
        r, r2, rmse, lag, far = evaluate_predictions(y_te_adj, y_cmd_hgb)
        cohort_results[sub_id]["Hist_Gradient_Boost"] = {"r": r, "r2": r2, "rmse": rmse, "lag": lag, "far": far}

        # ----------------------------------------------------------------------
        # MODEL 6: Multi-Layer Perceptron (Shallow Neural Network)
        # ----------------------------------------------------------------------
        mlp = MLPRegressor(hidden_layer_sizes=(96, 48), activation="relu", solver="adam", alpha=0.12, learning_rate_init=0.002, max_iter=200, early_stopping=True, n_iter_no_change=12, random_state=42)
        mlp.fit(X_tr_z, y_tr_adj)
        y_pred_mlp = mlp.predict(X_te_z)
        y_cmd_mlp = apply_actuation_postprocessing(y_pred_mlp, mav_te_adj, mav_rest_floor)
        r, r2, rmse, lag, far = evaluate_predictions(y_te_adj, y_cmd_mlp)
        cohort_results[sub_id]["MLP_NeuralNet"] = {"r": r, "r2": r2, "rmse": rmse, "lag": lag, "far": far}

        # ----------------------------------------------------------------------
        # MODEL 7 (OPTIONAL): XGBoost
        # ----------------------------------------------------------------------
        if HAS_XGB:
            xgb = XGBRegressor(n_estimators=120, max_depth=5, learning_rate=0.06, reg_alpha=1.0, reg_lambda=2.0, subsample=0.85, colsample_bytree=0.7, n_jobs=-1, random_state=42)
            xgb.fit(X_tr_z, y_tr_adj)
            y_pred_xgb = xgb.predict(X_te_z)
            y_cmd_xgb = apply_actuation_postprocessing(y_pred_xgb, mav_te_adj, mav_rest_floor)
            r, r2, rmse, lag, far = evaluate_predictions(y_te_adj, y_cmd_xgb)
            cohort_results[sub_id]["XGBoost"] = {"r": r, "r2": r2, "rmse": rmse, "lag": lag, "far": far}

        # ----------------------------------------------------------------------
        # MODEL 8 (OPTIONAL): CatBoost
        # ----------------------------------------------------------------------
        if HAS_CAT:
            cb = CatBoostRegressor(iterations=150, depth=6, learning_rate=0.06, l2_leaf_reg=4.0, verbose=0, random_seed=42)
            cb.fit(X_tr_z, y_tr_adj)
            y_pred_cb = cb.predict(X_te_z)
            y_cmd_cb = apply_actuation_postprocessing(y_pred_cb, mav_te_adj, mav_rest_floor)
            r, r2, rmse, lag, far = evaluate_predictions(y_te_adj, y_cmd_cb)
            cohort_results[sub_id]["CatBoost"] = {"r": r, "r2": r2, "rmse": rmse, "lag": lag, "far": far}

    # --- 8. Format & Log Summary Reports ---
    summary_header = (
        "\n========================================================================================================\n"
        "                              HOLDOUT VALIDATION SUMMARY ACROSS COHORT                                  \n"
        "========================================================================================================\n"
        " Subj | Test Reps | EMD Lead  | Best Model Family     | Test Pearson r | Test R^2 (VAF) | RMSE (deg/s) | Peak Lag (ms) | Rest Jitter % \n"
        "--------------------------------------------------------------------------------------------------------\n"
    )
    log_print(summary_header, f_log)

    for sub_id in COHORT:
        sub_dict = cohort_results.get(sub_id, {})
        if not sub_dict:
            log_print(f"  S{sub_id:02d} |    --     |  [N/A]   | [DATA NOT PROCESSED]\n", f_log)
            continue

        best_model_name = max(sub_dict.keys(), key=lambda m: sub_dict[m]["r"])
        bm = sub_dict[best_model_name]
        reps_str = str(cohort_test_reps_log.get(sub_id, [2, 6]))
        emd_str = f"+{cohort_emd_log.get(sub_id, 0):02d} ms"

        line = (
            f"  S{sub_id:02d} | {reps_str:<9} | {emd_str:<9} | {best_model_name:<21} |     {bm['r']:6.4f}     |     {bm['r2']:6.4f}     "
            f"|    {bm['rmse']:6.2f}    |    {bm['lag']:+6.1f} ms   |     {bm['far']:4.2f}%\n"
        )
        log_print(line, f_log)

    log_print("--------------------------------------------------------------------------------------------------------\n\n", f_log)

    # --- 9. Aggregate Model Family Comparison Table ---
    agg_header = (
        "========================================================================================================\n"
        "                         OVERALL MODEL FAMILY PERFORMANCE (COHORT MEAN +/- STD)                         \n"
        "========================================================================================================\n"
        " Model Architecture    | Test Pearson r   | Test R^2 (VAF)   | RMSE (deg/s)     | Peak Lag (ms)  | Jitter (FAR) \n"
        "--------------------------------------------------------------------------------------------------------\n"
    )
    log_print(agg_header, f_log)

    for m_name in MODEL_FAMILIES:
        r_list = [cohort_results[s][m_name]["r"] for s in COHORT if m_name in cohort_results[s]]
        r2_list = [cohort_results[s][m_name]["r2"] for s in COHORT if m_name in cohort_results[s]]
        rmse_list = [cohort_results[s][m_name]["rmse"] for s in COHORT if m_name in cohort_results[s]]
        lag_list = [cohort_results[s][m_name]["lag"] for s in COHORT if m_name in cohort_results[s]]
        far_list = [cohort_results[s][m_name]["far"] for s in COHORT if m_name in cohort_results[s]]

        if not r_list:
            continue

        agg_line = (
            f" {m_name:<22} |  {np.mean(r_list):5.3f} +/- {np.std(r_list):4.3f} |  "
            f"{np.mean(r2_list):5.3f} +/- {np.std(r2_list):4.3f} |  "
            f"{np.mean(rmse_list):5.2f} +/- {np.std(rmse_list):4.2f}  | "
            f"{np.mean(lag_list):+5.1f} +/- {np.std(lag_list):4.1f} |  "
            f"{np.mean(far_list):4.2f} +/- {np.std(far_list):4.2f}%\n"
        )
        log_print(agg_line, f_log)

    log_print("========================================================================================================\n", f_log)
    print(f"\n[SAVED] Benchmark report dumped to: {LOG_FILE}\n")
    f_log.close()

if __name__ == "__main__":
    main()

        CONTINUOUS sEMG VELOCITY REGRESSION: FULL TIMELINE TRAIN / BOUNDED TEST REPORT                  
 Cohort: 11 Subjects | Exclusions: Subject 11 (Entirely) | S4 Rep 6 | S7 Rep 6
 Strategy: Training on full continuous timeline (prompted + natural movements, minus plateaus)
 Testing : Bounded task epochs [t_cue - 1.0s, t_cue_end + 1.0s] on Reps [2, 6] (Fallback Rep 5 for S4/S7)
 Features: 90 Pure sEMG Context Metrics (Lags 0 & 2 + Delta Rate | Pure Muscle Decoding)
 Actuation: Non-Linear Deadband (4.0 deg/s) + Dual-Tier Electrical MAV Envelope Gating

>>> Subject 01 | Test Reps: [2, 6] | EMD Lead: +00 ms | Train: 7149 | Test: 1314
>>> Subject 02 | Test Reps: [2, 6] | EMD Lead: +20 ms | Train: 7245 | Test: 2000
>>> Subject 03 | Test Reps: [2, 6] | EMD Lead: +00 ms | Train: 6361 | Test: 1823
>>> Subject 04 | Test Reps: [2, 5] | EMD Lead: +00 ms | Train: 5256 | Test: 1833
>>> Subject 05 | Test Reps: [2, 6] | EMD Lead: +20 ms | Train: 7454 | Test: 1291
>>> Subject 06 | Test Reps: [2, 6

In [7]:
#!/usr/bin/env python3
"""
==============================================================================
CONTINUOUS sEMG VELOCITY REGRESSION: 100 ms LAG TRADE-OFF EXPERIMENT
Cohort      : Subjects 1 to 12 (Excluding S11 AND S07; Inverting S05)
Temporal    : 80 ms Causal Receptive Field (Lags 0, 20, 40, 60 ms + Delta Rate)
Alignment   : Causal Feedforward EMD Sweep extended to [0 ms, +100 ms]
Actuation   : Velocity Deadband (4 deg/s) + Pre-Cue Resting MAV Hurdle +
              Causal Leaky-Integrator (alpha=0.65, ~40 ms intentional smoothing lag)
Partitioning: Full continuous timeline train (minus plateaus) | Bounded test [t-1s, t+1s]
Outputs     : Formatted Console Log + Text File Dump ('cohort_model_validation_report.txt')
==============================================================================
"""

import os
import sys
import numpy as np
import pandas as pd
from scipy import signal, stats
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV
from sklearn.svm import SVR
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import KFold

try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except ImportError:
    HAS_XGB = False

try:
    from catboost import CatBoostRegressor
    HAS_CAT = True
except ImportError:
    HAS_CAT = False

# --- 1. Cohort & Protocol Constants ---
# Subject 07 (electrode failure) and Subject 11 strictly excluded
COHORT = [1, 2, 3, 4, 5, 6, 8, 9, 10, 12]
TARGET_MOVEMENTS = [5, 6]
V_DEADBAND = 4.0        # deg/s: Non-linear deadband
SMOOTH_ALPHA = 0.65     # Leaky-integrator smoothing coefficient (~40 ms causal lag)
HOP_MS = 10.0           # 10 ms hop (100 Hz cadence)
LOG_FILE = "cohort_model_validation_report.txt"

# Set to True ONLY if you have NOT already swapped Ch 9/10 in MATLAB for Subject 5
INVERT_S05_IN_PYTHON = False

MODEL_FAMILIES = ["Ridge", "SVR_Linear", "SVR_RBF", "Random_Forest", "Hist_Gradient_Boost", "MLP_NeuralNet"]
if HAS_XGB:
    MODEL_FAMILIES.append("XGBoost")
if HAS_CAT:
    MODEL_FAMILIES.append("CatBoost")

# --- 2. Timeline Slicing & Bounded Evaluation Mask ---
def partition_timeline_bounded(df, sub_id, buffer_sec=1.0):
    test_reps = [2, 5] if sub_id == 4 else [2, 6]
    is_test = np.zeros(len(df), dtype=bool)

    for m in TARGET_MOVEMENTS:
        for r in test_reps:
            cue_mask = (df["stimulus"] == m) & (df["repetition"] == r)
            if not np.any(cue_mask):
                continue

            t_cue_start = df.loc[cue_mask, "timestamp_s"].min()
            t_cue_end = df.loc[cue_mask, "timestamp_s"].max()

            t_eval_start = t_cue_start - buffer_sec
            t_eval_end = t_cue_end + buffer_sec

            slice_mask = (df["timestamp_s"] >= t_eval_start) & (df["timestamp_s"] <= t_eval_end)
            is_test[slice_mask] = True

    is_corrupt = (df["repetition"] == 6).values if sub_id == 4 else np.zeros(len(df), dtype=bool)
    is_plateau = (df["is_hold_plateau"] == 1).values

    valid_mask = ~is_corrupt & ~is_plateau
    train_mask = valid_mask & ~is_test
    test_mask = valid_mask & is_test

    return train_mask, test_mask, test_reps

# --- 3. 80 ms Causal Receptive Field Feature Builder (120 Features) ---
def build_expanded_causal_context(df, feat_cols):
    """
    Constructs a 120-dimensional causal representation:
      - Synchronous metrics at t (Lag 0: 30 feats)
      - History at t - 20 ms (Lag 2: 30 feats)
      - History at t - 40 ms (Lag 4: 30 feats)
      - Rate-of-change Delta x(t) = x(t) - x(t - 10 ms) (30 feats)
    """
    X_raw = df[feat_cols].values
    t_raw = df["timestamp_s"].values
    n_samples, _ = X_raw.shape

    dt = np.diff(t_raw, prepend=t_raw[0])
    is_break = dt > 0.025

    def get_lag(k):
        X_lag = np.zeros_like(X_raw)
        for i in range(n_samples):
            if i < k or np.any(is_break[max(0, i - k + 1): i + 1]):
                X_lag[i] = X_raw[i]
            else:
                X_lag[i] = X_raw[i - k]
        return X_lag

    X_lag1 = get_lag(1)
    X_lag2 = get_lag(2)  # -20 ms
    X_lag4 = get_lag(4)  # -40 ms
    delta_X = X_raw - X_lag1

    return np.hstack([X_raw, X_lag2, X_lag4, delta_X])

# --- 4. Causal Feedforward EMD Optimizer [0 ms to +100 ms] ---
def find_causal_emd_shift(df_train, feat_cols, slider_range_ms=(0, 100), step_ms=10):
    mav_edc = df_train["EDC_MAV"].values if "EDC_MAV" in feat_cols else df_train[feat_cols[0]].values
    mav_fds = df_train["FDS_MAV"].values if "FDS_MAV" in feat_cols else df_train[feat_cols[1]].values
    emg_power = mav_edc + mav_fds
    vel_magnitude = np.abs(df_train["theta_vel_deg_s"].values)

    e_norm = (emg_power - np.mean(emg_power)) / (np.std(emg_power) + 1e-12)
    v_norm = (vel_magnitude - np.mean(vel_magnitude)) / (np.std(vel_magnitude) + 1e-12)

    min_ms, max_ms = slider_range_ms
    shift_steps = np.arange(int(min_ms / step_ms), int(max_ms / step_ms) + 1)

    best_corr = -1.0
    best_shift = 4  # Default: 40 ms forward lead

    for s in shift_steps:
        if s > 0:
            e_sub = e_norm[:-s]
            v_sub = v_norm[s:]
        else:
            e_sub = e_norm
            v_sub = v_norm

        if len(e_sub) > 50:
            r_val, _ = stats.pearsonr(e_sub, v_sub)
            if r_val > best_corr:
                best_corr = r_val
                best_shift = s

    return best_shift, best_shift * step_ms

def align_feedforward_target(X, y, shift_hops):
    if shift_hops > 0:
        return X[:-shift_hops], y[shift_hops:]
    return X, y

# --- 5. Verified Pre-Cue Resting Floor Calculator ---
def compute_precue_rest_floor(df_train):
    """
    Extracts MAV strictly from verified pre-cue baseline intervals
    (1 second before visual cue prompt) to eliminate post-activation potentiation bias.
    """
    precue_mask = np.zeros(len(df_train), dtype=bool)
    t_vals = df_train["timestamp_s"].values

    for m in TARGET_MOVEMENTS:
        cue_idx = np.where(df_train["stimulus"].values == m)[0]
        if len(cue_idx) == 0:
            continue
        breaks = np.where(np.diff(cue_idx) > 1)[0]
        starts = np.insert(cue_idx[breaks + 1], 0, cue_idx[0])

        for s_idx in starts:
            t_start = t_vals[s_idx]
            pre_window = (t_vals >= t_start - 1.0) & (t_vals < t_start)
            precue_mask |= pre_window

    mav_total = (df_train["EDC_MAV"] + df_train["FDS_MAV"]).values
    if np.sum(precue_mask) > 20:
        return np.median(mav_total[precue_mask])
    return np.percentile(mav_total, 10)

# --- 6. Actuator Filter: Deadband + Rest Hurdle + Causal Leaky Integrator ---
def apply_smoothed_actuation(y_raw, mav_total, mav_floor, v_db=V_DEADBAND, alpha=SMOOTH_ALPHA):
    """
    Applies:
      1. Piecewise velocity deadband (4 deg/s)
      2. Dual-tier electrical MAV quiet gate
      3. Causal low-pass leaky integrator (trading ~40 ms of lag for jitter suppression)
    """
    n_samples = len(y_raw)
    y_db = np.zeros(n_samples)
    active = np.abs(y_raw) > v_db
    y_db[active] = np.sign(y_raw[active]) * (np.abs(y_raw[active]) - v_db)

    # Front-end electrical quiet clamp
    is_quiet = mav_total < (1.5 * mav_floor)
    y_db[is_quiet] = 0.0

    # Causal recursive smoother: y_cmd[i] = alpha * y_cmd[i-1] + (1 - alpha) * y_db[i]
    y_cmd = np.zeros(n_samples)
    curr = 0.0
    for i in range(n_samples):
        if is_quiet[i]:
            curr = 0.0
        else:
            curr = alpha * curr + (1.0 - alpha) * y_db[i]
        y_cmd[i] = curr

    return y_cmd

# --- 7. Diagnostic Metric Evaluator ---
def evaluate_predictions(y_true, y_cmd, hop_ms=HOP_MS):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_cmd = np.asarray(y_cmd, dtype=np.float64)

    valid = ~(np.isnan(y_true) | np.isnan(y_cmd))
    yt = y_true[valid]
    yp = y_cmd[valid]

    if len(yt) < 10 or np.std(yt) == 0 or np.std(yp) == 0:
        return 0.0, 0.0, float("inf"), float("nan"), 100.0

    r_val, _ = stats.pearsonr(yt, yp)

    sse = np.sum((yt - yp) ** 2)
    sst = np.sum((yt - np.mean(yt)) ** 2)
    r2_val = max(0.0, 1.0 - (sse / sst)) if sst > 0 else 0.0
    rmse_val = np.sqrt(np.mean((yt - yp) ** 2))

    max_lag_pts = 35
    yp_detrend = yp - np.mean(yp)
    yt_detrend = yt - np.mean(yt)
    xcorr = signal.correlate(yp_detrend, yt_detrend, mode="full")
    lags = signal.correlation_lags(len(yp_detrend), len(yt_detrend), mode="full")

    lag_mask = (lags >= -max_lag_pts) & (lags <= max_lag_pts)
    valid_lags = lags[lag_mask]
    valid_xcorr = xcorr[lag_mask]
    peak_lag_idx = np.argmax(valid_xcorr)
    lag_ms = valid_lags[peak_lag_idx] * hop_ms

    rest_mask = np.abs(yt) <= 2.0
    if np.any(rest_mask):
        far_pct = (np.sum(np.abs(yp[rest_mask]) > 0.0) / np.sum(rest_mask)) * 100.0
    else:
        far_pct = 0.0

    return r_val, r2_val, rmse_val, lag_ms, far_pct

def log_print(msg, file_handle):
    print(msg, end="")
    file_handle.write(msg)

# --- 8. Main Benchmark Loop ---
def main():
    f_log = open(LOG_FILE, "w", encoding="utf-8")

    header = (
        "========================================================================================================\n"
        "        CONTINUOUS sEMG VELOCITY REGRESSION: 100 ms LAG TRADE-OFF BENCHMARK REPORT                      \n"
        "========================================================================================================\n"
        f" Cohort: {len(COHORT)} Subjects | Exclusions: Subject 11 & Subject 07 (Entirely) | S4 Rep 6\n"
        " Strategy: Full-timeline training | Bounded task testing [t_cue - 1.0s, t_cue_end + 1.0s]\n"
        " Context : 120 Causal Features (Lags 0, 20, 40 ms + Delta Rate | 80 ms Temporal Receptive Field)\n"
        " Alignment: Causal Feedforward EMD Sweep extended to [0 ms, +100 ms]\n"
        " Actuator : Deadband (4 deg/s) + Pre-cue MAV Hurdle + Causal Smoother (alpha=0.65, ~40 ms intentional lag)\n"
        "========================================================================================================\n\n"
    )
    log_print(header, f_log)

    cohort_results = {sub: {} for sub in COHORT}
    cohort_emd_log = {}
    cohort_reps_log = {}

    for sub_id in COHORT:
        csv_path = os.path.join("datasets", "DB2", f"DB2_s{sub_id}", f"S{sub_id}_E1_A1_dynamic_velocity_features.csv")
        if not os.path.exists(csv_path):
            csv_path = f"S{sub_id}_E1_A1_dynamic_velocity_features.csv"

        if not os.path.exists(csv_path):
            log_print(f"[WARN] Subject {sub_id:02d}: CSV not found. Skipping.\n", f_log)
            continue

        df = pd.read_csv(csv_path)

        # Invert S05 in Python if not already fixed in MATLAB
        if sub_id == 5 and INVERT_S05_IN_PYTHON:
            edc_cols = [c for c in df.columns if c.startswith("EDC_")]
            fds_cols = [c for c in df.columns if c.startswith("FDS_")]
            for e_col, f_col in zip(edc_cols, fds_cols):
                df[e_col], df[f_col] = df[f_col].copy(), df[e_col].copy()
            for ndi_col in ["NDI_MAV", "NDI_RMS", "NDI_WL"]:
                if ndi_col in df.columns:
                    df[ndi_col] = -df[ndi_col]
            print(f">>> Subject {sub_id:02d}: Inverted EDC/FDS features in Python.")

        # 1. Timeline Partitioning
        train_mask, test_mask, test_reps = partition_timeline_bounded(df, sub_id, buffer_sec=1.0)
        cohort_reps_log[sub_id] = test_reps

        train_df = df[train_mask].copy()
        test_df = df[test_mask].copy()

        if len(train_df) == 0 or len(test_df) == 0:
            log_print(f"[WARN] Subject {sub_id:02d}: Insufficient partition size. Skipping.\n", f_log)
            continue

        meta_cols = ["timestamp_s", "theta_deg", "theta_vel_deg_s", "stimulus", "repetition", "is_hold_plateau"]
        feat_cols = [c for c in df.columns if c not in meta_cols]

        # 2. Compute Pre-Cue Resting MAV Floor
        mav_rest_floor = compute_precue_rest_floor(train_df)

        # 3. Causal EMD Sweep [0 ms to +100 ms]
        shift_hops, opt_emd_ms = find_causal_emd_shift(train_df, feat_cols, slider_range_ms=(0, 100), step_ms=10)
        cohort_emd_log[sub_id] = opt_emd_ms

        # 4. Extract 120 Causal Features
        X_train_ctx = build_expanded_causal_context(train_df, feat_cols)
        y_train_raw = train_df["theta_vel_deg_s"].values

        X_test_ctx = build_expanded_causal_context(test_df, feat_cols)
        y_test_raw = test_df["theta_vel_deg_s"].values

        mav_test_total = (test_df["EDC_MAV"] + test_df["FDS_MAV"]).values

        # Apply Causal Feedforward Alignment
        X_tr_adj, y_tr_adj = align_feedforward_target(X_train_ctx, y_train_raw, shift_hops)
        X_te_adj, y_te_adj = align_feedforward_target(X_test_ctx, y_test_raw, shift_hops)
        mav_te_adj = mav_test_total[shift_hops:] if shift_hops > 0 else mav_test_total

        # 5. Normalization
        scaler = StandardScaler()
        X_tr_z = scaler.fit_transform(X_tr_adj)
        X_te_z = scaler.transform(X_te_adj)
        X_te_z = np.clip(X_te_z, -20.0, 20.0)

        print(f">>> Subject {sub_id:02d} | Test Reps: {test_reps} | EMD Lead: +{opt_emd_ms:02d} ms | Train: {len(X_tr_z)} | Test: {len(X_te_z)}")

        cv_splits = KFold(n_splits=4, shuffle=True, random_state=42)

        # ----------------------------------------------------------------------
        # MODEL 1: Ridge Regression
        # ----------------------------------------------------------------------
        ridge = RidgeCV(alphas=np.logspace(0, 5, 12), cv=cv_splits)
        ridge.fit(X_tr_z, y_tr_adj)
        y_pred_ridge = ridge.predict(X_te_z)
        y_cmd_ridge = apply_smoothed_actuation(y_pred_ridge, mav_te_adj, mav_rest_floor)
        r, r2, rmse, lag, far = evaluate_predictions(y_te_adj, y_cmd_ridge)
        cohort_results[sub_id]["Ridge"] = {"r": r, "r2": r2, "rmse": rmse, "lag": lag, "far": far}

        # ----------------------------------------------------------------------
        # MODEL 2: Support Vector Regression (Linear)
        # ----------------------------------------------------------------------
        sub_step = max(1, len(X_tr_z) // 4000)
        svr_lin = SVR(kernel="linear", C=1.5, epsilon=2.0)
        svr_lin.fit(X_tr_z[::sub_step], y_tr_adj[::sub_step])
        y_pred_svr_lin = svr_lin.predict(X_te_z)
        y_cmd_svr_lin = apply_smoothed_actuation(y_pred_svr_lin, mav_te_adj, mav_rest_floor)
        r, r2, rmse, lag, far = evaluate_predictions(y_te_adj, y_cmd_svr_lin)
        cohort_results[sub_id]["SVR_Linear"] = {"r": r, "r2": r2, "rmse": rmse, "lag": lag, "far": far}

        # ----------------------------------------------------------------------
        # MODEL 3: Support Vector Regression (RBF)
        # ----------------------------------------------------------------------
        svr_rbf = SVR(kernel="rbf", C=3.0, epsilon=1.5, gamma="scale")
        svr_rbf.fit(X_tr_z[::sub_step], y_tr_adj[::sub_step])
        y_pred_svr_rbf = svr_rbf.predict(X_te_z)
        y_cmd_svr_rbf = apply_smoothed_actuation(y_pred_svr_rbf, mav_te_adj, mav_rest_floor)
        r, r2, rmse, lag, far = evaluate_predictions(y_te_adj, y_cmd_svr_rbf)
        cohort_results[sub_id]["SVR_RBF"] = {"r": r, "r2": r2, "rmse": rmse, "lag": lag, "far": far}

        # ----------------------------------------------------------------------
        # MODEL 4: Random Forest
        # ----------------------------------------------------------------------
        rf = RandomForestRegressor(n_estimators=85, max_depth=12, min_samples_leaf=6, max_features="sqrt", n_jobs=-1, random_state=42)
        rf.fit(X_tr_z, y_tr_adj)
        y_pred_rf = rf.predict(X_te_z)
        y_cmd_rf = apply_smoothed_actuation(y_pred_rf, mav_te_adj, mav_rest_floor)
        r, r2, rmse, lag, far = evaluate_predictions(y_te_adj, y_cmd_rf)
        cohort_results[sub_id]["Random_Forest"] = {"r": r, "r2": r2, "rmse": rmse, "lag": lag, "far": far}

        # ----------------------------------------------------------------------
        # MODEL 5: HistGradientBoosting
        # ----------------------------------------------------------------------
        hgb = HistGradientBoostingRegressor(max_iter=140, learning_rate=0.06, max_leaf_nodes=31, min_samples_leaf=15, l2_regularization=2.5, early_stopping=True, n_iter_no_change=10, random_state=42)
        hgb.fit(X_tr_z, y_tr_adj)
        y_pred_hgb = hgb.predict(X_te_z)
        y_cmd_hgb = apply_smoothed_actuation(y_pred_hgb, mav_te_adj, mav_rest_floor)
        r, r2, rmse, lag, far = evaluate_predictions(y_te_adj, y_cmd_hgb)
        cohort_results[sub_id]["Hist_Gradient_Boost"] = {"r": r, "r2": r2, "rmse": rmse, "lag": lag, "far": far}

        # ----------------------------------------------------------------------
        # MODEL 6: Multi-Layer Perceptron
        # ----------------------------------------------------------------------
        mlp = MLPRegressor(hidden_layer_sizes=(96, 48), activation="relu", solver="adam", alpha=0.15, learning_rate_init=0.002, max_iter=200, early_stopping=True, n_iter_no_change=12, random_state=42)
        mlp.fit(X_tr_z, y_tr_adj)
        y_pred_mlp = mlp.predict(X_te_z)
        y_cmd_mlp = apply_smoothed_actuation(y_pred_mlp, mav_te_adj, mav_rest_floor)
        r, r2, rmse, lag, far = evaluate_predictions(y_te_adj, y_cmd_mlp)
        cohort_results[sub_id]["MLP_NeuralNet"] = {"r": r, "r2": r2, "rmse": rmse, "lag": lag, "far": far}

        # ----------------------------------------------------------------------
        # MODEL 7 (OPTIONAL): XGBoost
        # ----------------------------------------------------------------------
        if HAS_XGB:
            xgb = XGBRegressor(n_estimators=130, max_depth=5, learning_rate=0.06, reg_alpha=1.0, reg_lambda=2.0, subsample=0.85, colsample_bytree=0.7, n_jobs=-1, random_state=42)
            xgb.fit(X_tr_z, y_tr_adj)
            y_pred_xgb = xgb.predict(X_te_z)
            y_cmd_xgb = apply_smoothed_actuation(y_pred_xgb, mav_te_adj, mav_rest_floor)
            r, r2, rmse, lag, far = evaluate_predictions(y_te_adj, y_cmd_xgb)
            cohort_results[sub_id]["XGBoost"] = {"r": r, "r2": r2, "rmse": rmse, "lag": lag, "far": far}

        # ----------------------------------------------------------------------
        # MODEL 8 (OPTIONAL): CatBoost
        # ----------------------------------------------------------------------
        if HAS_CAT:
            cb = CatBoostRegressor(iterations=160, depth=6, learning_rate=0.06, l2_leaf_reg=4.0, verbose=0, random_seed=42)
            cb.fit(X_tr_z, y_tr_adj)
            y_pred_cb = cb.predict(X_te_z)
            y_cmd_cb = apply_smoothed_actuation(y_pred_cb, mav_te_adj, mav_rest_floor)
            r, r2, rmse, lag, far = evaluate_predictions(y_te_adj, y_cmd_cb)
            cohort_results[sub_id]["CatBoost"] = {"r": r, "r2": r2, "rmse": rmse, "lag": lag, "far": far}

    # --- 9. Format & Log Summary Reports ---
    summary_header = (
        "\n========================================================================================================\n"
        "                              HOLDOUT VALIDATION SUMMARY ACROSS COHORT                                  \n"
        "========================================================================================================\n"
        " Subj | Test Reps | EMD Lead  | Best Model Family     | Test Pearson r | Test R^2 (VAF) | RMSE (deg/s) | Peak Lag (ms) | Rest Jitter % \n"
        "--------------------------------------------------------------------------------------------------------\n"
    )
    log_print(summary_header, f_log)

    for sub_id in COHORT:
        sub_dict = cohort_results.get(sub_id, {})
        if not sub_dict:
            log_print(f"  S{sub_id:02d} |    --     |  [N/A]   | [DATA NOT PROCESSED]\n", f_log)
            continue

        best_model_name = max(sub_dict.keys(), key=lambda m: sub_dict[m]["r"])
        bm = sub_dict[best_model_name]
        reps_str = str(cohort_reps_log.get(sub_id, [2, 6]))
        emd_str = f"+{cohort_emd_log.get(sub_id, 0):02d} ms"

        line = (
            f"  S{sub_id:02d} | {reps_str:<9} | {emd_str:<9} | {best_model_name:<21} |     {bm['r']:6.4f}     |     {bm['r2']:6.4f}     "
            f"|    {bm['rmse']:6.2f}    |    {bm['lag']:+6.1f} ms   |     {bm['far']:4.2f}%\n"
        )
        log_print(line, f_log)

    log_print("--------------------------------------------------------------------------------------------------------\n\n", f_log)

    # --- 10. Aggregate Model Family Comparison Table ---
    agg_header = (
        "========================================================================================================\n"
        "                         OVERALL MODEL FAMILY PERFORMANCE (COHORT MEAN +/- STD)                         \n"
        "========================================================================================================\n"
        " Model Architecture    | Test Pearson r   | Test R^2 (VAF)   | RMSE (deg/s)     | Peak Lag (ms)  | Jitter (FAR) \n"
        "--------------------------------------------------------------------------------------------------------\n"
    )
    log_print(agg_header, f_log)

    for m_name in MODEL_FAMILIES:
        r_list = [cohort_results[s][m_name]["r"] for s in COHORT if m_name in cohort_results[s]]
        r2_list = [cohort_results[s][m_name]["r2"] for s in COHORT if m_name in cohort_results[s]]
        rmse_list = [cohort_results[s][m_name]["rmse"] for s in COHORT if m_name in cohort_results[s]]
        lag_list = [cohort_results[s][m_name]["lag"] for s in COHORT if m_name in cohort_results[s]]
        far_list = [cohort_results[s][m_name]["far"] for s in COHORT if m_name in cohort_results[s]]

        if not r_list:
            continue

        agg_line = (
            f" {m_name:<22} |  {np.mean(r_list):5.3f} +/- {np.std(r_list):4.3f} |  "
            f"{np.mean(r2_list):5.3f} +/- {np.std(r2_list):4.3f} |  "
            f"{np.mean(rmse_list):5.2f} +/- {np.std(rmse_list):4.2f}  | "
            f"{np.mean(lag_list):+5.1f} +/- {np.std(lag_list):4.1f} |  "
            f"{np.mean(far_list):4.2f} +/- {np.std(far_list):4.2f}%\n"
        )
        log_print(agg_line, f_log)

    log_print("========================================================================================================\n", f_log)
    print(f"\n[SAVED] Benchmark report dumped to: {LOG_FILE}\n")
    f_log.close()

if __name__ == "__main__":
    main()

        CONTINUOUS sEMG VELOCITY REGRESSION: 100 ms LAG TRADE-OFF BENCHMARK REPORT                      
 Cohort: 10 Subjects | Exclusions: Subject 11 & Subject 07 (Entirely) | S4 Rep 6
 Strategy: Full-timeline training | Bounded task testing [t_cue - 1.0s, t_cue_end + 1.0s]
 Context : 120 Causal Features (Lags 0, 20, 40 ms + Delta Rate | 80 ms Temporal Receptive Field)
 Alignment: Causal Feedforward EMD Sweep extended to [0 ms, +100 ms]
 Actuator : Deadband (4 deg/s) + Pre-cue MAV Hurdle + Causal Smoother (alpha=0.65, ~40 ms intentional lag)

>>> Subject 01 | Test Reps: [2, 6] | EMD Lead: +00 ms | Train: 7149 | Test: 1314
>>> Subject 02 | Test Reps: [2, 6] | EMD Lead: +20 ms | Train: 7245 | Test: 2000
>>> Subject 03 | Test Reps: [2, 6] | EMD Lead: +00 ms | Train: 6361 | Test: 1823
>>> Subject 04 | Test Reps: [2, 5] | EMD Lead: +00 ms | Train: 5256 | Test: 1833
>>> Subject 05 | Test Reps: [2, 6] | EMD Lead: +20 ms | Train: 7454 | Test: 1291
>>> Subject 06 | Test Reps: [2, 6] | EMD Lead

In [8]:
#!/usr/bin/env python3
"""
==============================================================================
CONTINUOUS sEMG VELOCITY REGRESSION: 100 ms FIXED-LAG RETROSPECTIVE DECODER
Cohort      : Subjects 1 to 12 (Excluding S11 & S07; Inverting S05)
Strategy    : Fixed-Lag Smoothing (Predicting velocity at t - 100 ms using
              all causal sEMG received up to current real-time clock t).
Context     : 120 Features spanning 140 ms relative to target:
              - x(t - 140 ms) : -40 ms pre-velocity baseline
              - x(t - 100 ms) : Target synchronous sEMG
              - x(t - 50 ms)  : +50 ms post-velocity lookahead
              - x(t)          : +100 ms post-velocity newest sEMG
              - Rate-of-recruitment derivatives across the span
Actuation   : Piecewise Non-Linear Deadband (4.0 deg/s) + Pre-cue MAV Hurdle
Evaluation  : Bounded test slices [t_cue - 1.0s, t_cue_end + 1.0s] on Reps [2, 6]
==============================================================================
"""

import os
import sys
import numpy as np
import pandas as pd
from scipy import signal, stats
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV
from sklearn.svm import SVR
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import KFold

try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except ImportError:
    HAS_XGB = False

try:
    from catboost import CatBoostRegressor
    HAS_CAT = True
except ImportError:
    HAS_CAT = False

# --- 1. Cohort & Protocol Constants ---
COHORT = [1, 2, 3, 4, 5, 6, 8, 9, 10, 12]  # S07 and S11 permanently excluded
TARGET_MOVEMENTS = [5, 6]
V_DEADBAND = 4.0        # deg/s: Velocity deadband
HOP_MS = 10.0           # 10 ms hop (100 Hz cadence)
LAG_HOPS = 10           # 10 hops * 10 ms = 100 ms Retrospective Delay
LOG_FILE = "cohort_100ms_fixed_lag_report.txt"

# Set to True ONLY if you have NOT already swapped Ch 9/10 in MATLAB for Subject 5
INVERT_S05_IN_PYTHON = False

MODEL_FAMILIES = ["Ridge", "SVR_Linear", "SVR_RBF", "Random_Forest", "Hist_Gradient_Boost", "MLP_NeuralNet"]
if HAS_XGB:
    MODEL_FAMILIES.append("XGBoost")
if HAS_CAT:
    MODEL_FAMILIES.append("CatBoost")

# --- 2. Bounded Timeline Slicing ---
def partition_timeline_bounded(df, sub_id, buffer_sec=1.0):
    test_reps = [2, 5] if sub_id == 4 else [2, 6]
    is_test = np.zeros(len(df), dtype=bool)

    for m in TARGET_MOVEMENTS:
        for r in test_reps:
            cue_mask = (df["stimulus"] == m) & (df["repetition"] == r)
            if not np.any(cue_mask):
                continue

            t_cue_start = df.loc[cue_mask, "timestamp_s"].min()
            t_cue_end = df.loc[cue_mask, "timestamp_s"].max()

            t_eval_start = t_cue_start - buffer_sec
            t_eval_end = t_cue_end + buffer_sec

            slice_mask = (df["timestamp_s"] >= t_eval_start) & (df["timestamp_s"] <= t_eval_end)
            is_test[slice_mask] = True

    is_corrupt = (df["repetition"] == 6).values if sub_id == 4 else np.zeros(len(df), dtype=bool)
    is_plateau = (df["is_hold_plateau"] == 1).values

    valid_mask = ~is_corrupt & ~is_plateau
    train_mask = valid_mask & ~is_test
    test_mask = valid_mask & is_test

    return train_mask, test_mask, test_reps

# --- 3. Retrospective 140 ms Causal Context Matrix (120 Features) ---
def build_retrospective_context(df, feat_cols):
    """
    Constructs feature rows at current real-time clock t containing:
      - x(t)          : Newest sEMG (+100 ms relative to target)
      - x(t - 50 ms)  : Intermediate sEMG (+50 ms relative to target)
      - x(t - 100 ms) : Synchronous sEMG (coincident with target)
      - x(t - 140 ms) : Baseline sEMG (-40 ms before target)
      - Instantaneous recruitment delta: x(t) - x(t - 50 ms)
    """
    X_raw = df[feat_cols].values
    t_raw = df["timestamp_s"].values
    n_samples, _ = X_raw.shape

    dt = np.diff(t_raw, prepend=t_raw[0])
    is_break = dt > 0.025

    def get_lag(k):
        X_lag = np.zeros_like(X_raw)
        for i in range(n_samples):
            if i < k or np.any(is_break[max(0, i - k + 1): i + 1]):
                X_lag[i] = X_raw[i]
            else:
                X_lag[i] = X_raw[i - k]
        return X_lag

    X_lag0  = X_raw           # Current real-time t (+100 ms lookahead)
    X_lag5  = get_lag(5)      # t - 50 ms (+50 ms lookahead)
    X_lag10 = get_lag(10)     # t - 100 ms (Synchronous with target v)
    X_lag14 = get_lag(14)     # t - 140 ms (-40 ms prior baseline)

    delta_burst = X_lag0 - X_lag5

    # 30 * 4 = 120 features
    return np.hstack([X_lag0, X_lag5, X_lag10, delta_burst])

# --- 4. 100 ms Retrospective Alignment (Target = v at t - 100 ms) ---
def align_retrospective_100ms(X, y, mav_total, lag_hops=LAG_HOPS):
    """
    At index i (time t), target is y[i - lag_hops] (velocity at t - 100 ms).
    X[i] provides all sEMG collected up to time t.
    """
    X_aligned = X[lag_hops:]
    y_aligned = y[:-lag_hops]
    mav_aligned = mav_total[:-lag_hops]
    return X_aligned, y_aligned, mav_aligned

# --- 5. Verified Pre-Cue Baseline Calculator ---
def compute_precue_rest_floor(df_train):
    precue_mask = np.zeros(len(df_train), dtype=bool)
    t_vals = df_train["timestamp_s"].values

    for m in TARGET_MOVEMENTS:
        cue_idx = np.where(df_train["stimulus"].values == m)[0]
        if len(cue_idx) == 0:
            continue
        breaks = np.where(np.diff(cue_idx) > 1)[0]
        starts = np.insert(cue_idx[breaks + 1], 0, cue_idx[0])

        for s_idx in starts:
            t_start = t_vals[s_idx]
            pre_window = (t_vals >= t_start - 1.0) & (t_vals < t_start)
            precue_mask |= pre_window

    mav_total = (df_train["EDC_MAV"] + df_train["FDS_MAV"]).values
    if np.sum(precue_mask) > 20:
        return np.median(mav_total[precue_mask])
    return np.percentile(mav_total, 10)

# --- 6. Actuator Deadband & Dual-Tier Gate ---
def apply_retrospective_actuation(y_raw, mav_total, mav_floor, v_db=V_DEADBAND):
    y_cmd = np.zeros_like(y_raw)
    active = np.abs(y_raw) > v_db
    y_cmd[active] = np.sign(y_raw[active]) * (np.abs(y_raw[active]) - v_db)

    is_quiet = mav_total < (1.5 * mav_floor)
    y_cmd[is_quiet] = 0.0
    return y_cmd

# --- 7. Diagnostic Metric Evaluator ---
def evaluate_predictions(y_true, y_cmd, hop_ms=HOP_MS):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_cmd = np.asarray(y_cmd, dtype=np.float64)

    valid = ~(np.isnan(y_true) | np.isnan(y_cmd))
    yt = y_true[valid]
    yp = y_cmd[valid]

    if len(yt) < 10 or np.std(yt) == 0 or np.std(yp) == 0:
        return 0.0, 0.0, float("inf"), float("nan"), 100.0

    # Pearson correlation against the true retrospective velocity profile
    r_val, _ = stats.pearsonr(yt, yp)

    # Variance Accounted For (VAF / R^2)
    sse = np.sum((yt - yp) ** 2)
    sst = np.sum((yt - np.mean(yt)) ** 2)
    r2_val = max(0.0, 1.0 - (sse / sst)) if sst > 0 else 0.0
    rmse_val = np.sqrt(np.mean((yt - yp) ** 2))

    # Cross-correlation lag check relative to the retrospective target
    max_lag_pts = 35
    yp_detrend = yp - np.mean(yp)
    yt_detrend = yt - np.mean(yt)
    xcorr = signal.correlate(yp_detrend, yt_detrend, mode="full")
    lags = signal.correlation_lags(len(yp_detrend), len(yt_detrend), mode="full")

    lag_mask = (lags >= -max_lag_pts) & (lags <= max_lag_pts)
    valid_lags = lags[lag_mask]
    valid_xcorr = xcorr[lag_mask]
    peak_lag_idx = np.argmax(valid_xcorr)
    recon_lag_ms = valid_lags[peak_lag_idx] * hop_ms

    rest_mask = np.abs(yt) <= 2.0
    if np.any(rest_mask):
        far_pct = (np.sum(np.abs(yp[rest_mask]) > 0.0) / np.sum(rest_mask)) * 100.0
    else:
        far_pct = 0.0

    return r_val, r2_val, rmse_val, recon_lag_ms, far_pct

def log_print(msg, file_handle):
    print(msg, end="")
    file_handle.write(msg)

# --- 8. Main Training and Benchmark Loop ---
def main():
    f_log = open(LOG_FILE, "w", encoding="utf-8")

    header = (
        "========================================================================================================\n"
        "      CONTINUOUS sEMG VELOCITY REGRESSION: 100 ms FIXED-LAG RETROSPECTIVE BENCHMARK                     \n"
        "========================================================================================================\n"
        f" Cohort: {len(COHORT)} Subjects | Exclusions: Subject 11 & Subject 07 (Entirely) | S4 Rep 6\n"
        " Strategy: Fixed-Lag Decoding (predicting v(t - 100 ms) from 140 ms sEMG history spanning up to clock t)\n"
        " Context : 120 Features (Lags -40 ms, 0 ms, +50 ms, +100 ms relative to velocity target)\n"
        " Actuator: Velocity Deadband (4.0 deg/s) + Verified Pre-Cue MAV Quiet Hurdle\n"
        "========================================================================================================\n\n"
    )
    log_print(header, f_log)

    cohort_results = {sub: {} for sub in COHORT}
    cohort_reps_log = {}

    for sub_id in COHORT:
        csv_path = os.path.join("datasets", "DB2", f"DB2_s{sub_id}", f"S{sub_id}_E1_A1_dynamic_velocity_features.csv")
        if not os.path.exists(csv_path):
            csv_path = f"S{sub_id}_E1_A1_dynamic_velocity_features.csv"

        if not os.path.exists(csv_path):
            log_print(f"[WARN] Subject {sub_id:02d}: CSV not found. Skipping.\n", f_log)
            continue

        df = pd.read_csv(csv_path)

        if sub_id == 5 and INVERT_S05_IN_PYTHON:
            edc_cols = [c for c in df.columns if c.startswith("EDC_")]
            fds_cols = [c for c in df.columns if c.startswith("FDS_")]
            for e_col, f_col in zip(edc_cols, fds_cols):
                df[e_col], df[f_col] = df[f_col].copy(), df[e_col].copy()
            for ndi_col in ["NDI_MAV", "NDI_RMS", "NDI_WL"]:
                if ndi_col in df.columns:
                    df[ndi_col] = -df[ndi_col]
            print(f">>> Subject {sub_id:02d}: Inverted EDC/FDS features in Python.")

        # 1. Partition Timeline
        train_mask, test_mask, test_reps = partition_timeline_bounded(df, sub_id, buffer_sec=1.0)
        cohort_reps_log[sub_id] = test_reps

        train_df = df[train_mask].copy()
        test_df = df[test_mask].copy()

        if len(train_df) == 0 or len(test_df) == 0:
            log_print(f"[WARN] Subject {sub_id:02d}: Insufficient partition size. Skipping.\n", f_log)
            continue

        meta_cols = ["timestamp_s", "theta_deg", "theta_vel_deg_s", "stimulus", "repetition", "is_hold_plateau"]
        feat_cols = [c for c in df.columns if c not in meta_cols]

        mav_rest_floor = compute_precue_rest_floor(train_df)

        # 2. Extract Retrospective Context (120 Features)
        X_train_ctx = build_retrospective_context(train_df, feat_cols)
        y_train_raw = train_df["theta_vel_deg_s"].values
        mav_tr_total = (train_df["EDC_MAV"] + train_df["FDS_MAV"]).values

        X_test_ctx = build_retrospective_context(test_df, feat_cols)
        y_test_raw = test_df["theta_vel_deg_s"].values
        mav_te_total = (test_df["EDC_MAV"] + test_df["FDS_MAV"]).values

        # 3. Align to 100 ms Retrospective Target: y = v(t - 100 ms)
        X_tr_adj, y_tr_adj, mav_tr_adj = align_retrospective_100ms(X_train_ctx, y_train_raw, mav_tr_total, LAG_HOPS)
        X_te_adj, y_te_adj, mav_te_adj = align_retrospective_100ms(X_test_ctx, y_test_raw, mav_te_total, LAG_HOPS)

        # 4. Standardize Features
        scaler = StandardScaler()
        X_tr_z = scaler.fit_transform(X_tr_adj)
        X_te_z = scaler.transform(X_te_adj)
        X_te_z = np.clip(X_te_z, -20.0, 20.0)

        print(f">>> Subject {sub_id:02d} | Test Reps: {test_reps} | Features: {X_tr_z.shape[1]} | Train: {len(X_tr_z)} | Test: {len(X_te_z)}")

        cv_splits = KFold(n_splits=4, shuffle=True, random_state=42)

        # ----------------------------------------------------------------------
        # MODEL 1: Ridge Regression
        # ----------------------------------------------------------------------
        ridge = RidgeCV(alphas=np.logspace(0, 5, 12), cv=cv_splits)
        ridge.fit(X_tr_z, y_tr_adj)
        y_pred = ridge.predict(X_te_z)
        y_cmd = apply_retrospective_actuation(y_pred, mav_te_adj, mav_rest_floor)
        r, r2, rmse, lag, far = evaluate_predictions(y_te_adj, y_cmd)
        cohort_results[sub_id]["Ridge"] = {"r": r, "r2": r2, "rmse": rmse, "lag": lag, "far": far}

        # ----------------------------------------------------------------------
        # MODEL 2: Support Vector Regression (Linear)
        # ----------------------------------------------------------------------
        sub_step = max(1, len(X_tr_z) // 4000)
        svr_lin = SVR(kernel="linear", C=1.5, epsilon=2.0)
        svr_lin.fit(X_tr_z[::sub_step], y_tr_adj[::sub_step])
        y_pred = svr_lin.predict(X_te_z)
        y_cmd = apply_retrospective_actuation(y_pred, mav_te_adj, mav_rest_floor)
        r, r2, rmse, lag, far = evaluate_predictions(y_te_adj, y_cmd)
        cohort_results[sub_id]["SVR_Linear"] = {"r": r, "r2": r2, "rmse": rmse, "lag": lag, "far": far}

        # ----------------------------------------------------------------------
        # MODEL 3: Support Vector Regression (RBF)
        # ----------------------------------------------------------------------
        svr_rbf = SVR(kernel="rbf", C=3.0, epsilon=1.5, gamma="scale")
        svr_rbf.fit(X_tr_z[::sub_step], y_tr_adj[::sub_step])
        y_pred = svr_rbf.predict(X_te_z)
        y_cmd = apply_retrospective_actuation(y_pred, mav_te_adj, mav_rest_floor)
        r, r2, rmse, lag, far = evaluate_predictions(y_te_adj, y_cmd)
        cohort_results[sub_id]["SVR_RBF"] = {"r": r, "r2": r2, "rmse": rmse, "lag": lag, "far": far}

        # ----------------------------------------------------------------------
        # MODEL 4: Random Forest
        # ----------------------------------------------------------------------
        rf = RandomForestRegressor(n_estimators=85, max_depth=12, min_samples_leaf=6, max_features="sqrt", n_jobs=-1, random_state=42)
        rf.fit(X_tr_z, y_tr_adj)
        y_pred = rf.predict(X_te_z)
        y_cmd = apply_retrospective_actuation(y_pred, mav_te_adj, mav_rest_floor)
        r, r2, rmse, lag, far = evaluate_predictions(y_te_adj, y_cmd)
        cohort_results[sub_id]["Random_Forest"] = {"r": r, "r2": r2, "rmse": rmse, "lag": lag, "far": far}

        # ----------------------------------------------------------------------
        # MODEL 5: HistGradientBoosting
        # ----------------------------------------------------------------------
        hgb = HistGradientBoostingRegressor(max_iter=140, learning_rate=0.06, max_leaf_nodes=31, min_samples_leaf=15, l2_regularization=2.5, early_stopping=True, n_iter_no_change=10, random_state=42)
        hgb.fit(X_tr_z, y_tr_adj)
        y_pred = hgb.predict(X_te_z)
        y_cmd = apply_retrospective_actuation(y_pred, mav_te_adj, mav_rest_floor)
        r, r2, rmse, lag, far = evaluate_predictions(y_te_adj, y_cmd)
        cohort_results[sub_id]["Hist_Gradient_Boost"] = {"r": r, "r2": r2, "rmse": rmse, "lag": lag, "far": far}

        # ----------------------------------------------------------------------
        # MODEL 6: Multi-Layer Perceptron
        # ----------------------------------------------------------------------
        mlp = MLPRegressor(hidden_layer_sizes=(96, 48), activation="relu", solver="adam", alpha=0.15, learning_rate_init=0.002, max_iter=200, early_stopping=True, n_iter_no_change=12, random_state=42)
        mlp.fit(X_tr_z, y_tr_adj)
        y_pred = mlp.predict(X_te_z)
        y_cmd = apply_retrospective_actuation(y_pred, mav_te_adj, mav_rest_floor)
        r, r2, rmse, lag, far = evaluate_predictions(y_te_adj, y_cmd)
        cohort_results[sub_id]["MLP_NeuralNet"] = {"r": r, "r2": r2, "rmse": rmse, "lag": lag, "far": far}

        # ----------------------------------------------------------------------
        # MODEL 7 (OPTIONAL): XGBoost
        # ----------------------------------------------------------------------
        if HAS_XGB:
            xgb = XGBRegressor(n_estimators=130, max_depth=5, learning_rate=0.06, reg_alpha=1.0, reg_lambda=2.0, subsample=0.85, colsample_bytree=0.7, n_jobs=-1, random_state=42)
            xgb.fit(X_tr_z, y_tr_adj)
            y_pred = xgb.predict(X_te_z)
            y_cmd = apply_retrospective_actuation(y_pred, mav_te_adj, mav_rest_floor)
            r, r2, rmse, lag, far = evaluate_predictions(y_te_adj, y_cmd)
            cohort_results[sub_id]["XGBoost"] = {"r": r, "r2": r2, "rmse": rmse, "lag": lag, "far": far}

        # ----------------------------------------------------------------------
        # MODEL 8 (OPTIONAL): CatBoost
        # ----------------------------------------------------------------------
        if HAS_CAT:
            cb = CatBoostRegressor(iterations=160, depth=6, learning_rate=0.06, l2_leaf_reg=4.0, verbose=0, random_seed=42)
            cb.fit(X_tr_z, y_tr_adj)
            y_pred = cb.predict(X_te_z)
            y_cmd = apply_retrospective_actuation(y_pred, mav_te_adj, mav_rest_floor)
            r, r2, rmse, lag, far = evaluate_predictions(y_te_adj, y_cmd)
            cohort_results[sub_id]["CatBoost"] = {"r": r, "r2": r2, "rmse": rmse, "lag": lag, "far": far}

    # --- 9. Format & Log Summary Reports ---
    summary_header = (
        "\n========================================================================================================\n"
        "                              HOLDOUT VALIDATION SUMMARY ACROSS COHORT                                  \n"
        "========================================================================================================\n"
        " Subj | Test Reps | Best Model Family     | Test Pearson r | Test R^2 (VAF) | RMSE (deg/s) | Recon Lag (ms)| Rest Jitter % \n"
        "--------------------------------------------------------------------------------------------------------\n"
    )
    log_print(summary_header, f_log)

    for sub_id in COHORT:
        sub_dict = cohort_results.get(sub_id, {})
        if not sub_dict:
            log_print(f"  S{sub_id:02d} |    --     |  [N/A]   | [DATA NOT PROCESSED]\n", f_log)
            continue

        best_model_name = max(sub_dict.keys(), key=lambda m: sub_dict[m]["r"])
        bm = sub_dict[best_model_name]
        reps_str = str(cohort_reps_log.get(sub_id, [2, 6]))

        line = (
            f"  S{sub_id:02d} | {reps_str:<9} | {best_model_name:<21} |     {bm['r']:6.4f}     |     {bm['r2']:6.4f}     "
            f"|    {bm['rmse']:6.2f}    |    {bm['lag']:+6.1f} ms   |     {bm['far']:4.2f}%\n"
        )
        log_print(line, f_log)

    log_print("--------------------------------------------------------------------------------------------------------\n\n", f_log)

    # --- 10. Aggregate Model Family Comparison Table ---
    agg_header = (
        "========================================================================================================\n"
        "                         OVERALL MODEL FAMILY PERFORMANCE (COHORT MEAN +/- STD)                         \n"
        "========================================================================================================\n"
        " Model Architecture    | Test Pearson r   | Test R^2 (VAF)   | RMSE (deg/s)     | Recon Lag (ms) | Jitter (FAR) \n"
        "--------------------------------------------------------------------------------------------------------\n"
    )
    log_print(agg_header, f_log)

    for m_name in MODEL_FAMILIES:
        r_list = [cohort_results[s][m_name]["r"] for s in COHORT if m_name in cohort_results[s]]
        r2_list = [cohort_results[s][m_name]["r2"] for s in COHORT if m_name in cohort_results[s]]
        rmse_list = [cohort_results[s][m_name]["rmse"] for s in COHORT if m_name in cohort_results[s]]
        lag_list = [cohort_results[s][m_name]["lag"] for s in COHORT if m_name in cohort_results[s]]
        far_list = [cohort_results[s][m_name]["far"] for s in COHORT if m_name in cohort_results[s]]

        if not r_list:
            continue

        agg_line = (
            f" {m_name:<22} |  {np.mean(r_list):5.3f} +/- {np.std(r_list):4.3f} |  "
            f"{np.mean(r2_list):5.3f} +/- {np.std(r2_list):4.3f} |  "
            f"{np.mean(rmse_list):5.2f} +/- {np.std(rmse_list):4.2f}  | "
            f"{np.mean(lag_list):+5.1f} +/- {np.std(lag_list):4.1f} |  "
            f"{np.mean(far_list):4.2f} +/- {np.std(far_list):4.2f}%\n"
        )
        log_print(agg_line, f_log)

    log_print("========================================================================================================\n", f_log)
    print(f"\n[SAVED] 100 ms retrospective benchmark report dumped to: {LOG_FILE}\n")
    f_log.close()

if __name__ == "__main__":
    main()

      CONTINUOUS sEMG VELOCITY REGRESSION: 100 ms FIXED-LAG RETROSPECTIVE BENCHMARK                     
 Cohort: 10 Subjects | Exclusions: Subject 11 & Subject 07 (Entirely) | S4 Rep 6
 Strategy: Fixed-Lag Decoding (predicting v(t - 100 ms) from 140 ms sEMG history spanning up to clock t)
 Context : 120 Features (Lags -40 ms, 0 ms, +50 ms, +100 ms relative to velocity target)
 Actuator: Velocity Deadband (4.0 deg/s) + Verified Pre-Cue MAV Quiet Hurdle

>>> Subject 01 | Test Reps: [2, 6] | Features: 120 | Train: 7139 | Test: 1304
>>> Subject 02 | Test Reps: [2, 6] | Features: 120 | Train: 7237 | Test: 1992
>>> Subject 03 | Test Reps: [2, 6] | Features: 120 | Train: 6351 | Test: 1813
>>> Subject 04 | Test Reps: [2, 5] | Features: 120 | Train: 5246 | Test: 1823
>>> Subject 05 | Test Reps: [2, 6] | Features: 120 | Train: 7446 | Test: 1283
>>> Subject 06 | Test Reps: [2, 6] | Features: 120 | Train: 7152 | Test: 1468
>>> Subject 08 | Test Reps: [2, 6] | Features: 120 | Train: 7343 | Test: 1

In [9]:
#!/usr/bin/env python3
"""
==============================================================================
CONTINUOUS sEMG 100 ms RETROSPECTIVE BENCHMARK (AUDITED & PHYSICALLY SOUND)
Cohort      : Subjects 1 to 12 (Excluding S07 & S11)
Integrity   : Contiguous-segment slicing (Zero boundary leakage)
Gating      : Target-synchronous MAV silence floor (No premature truncation)
Angle Recon : Per-segment independent integration (No cumulative cross-trial drift)
CV Strategy : Chronological block K-Fold (Zero adjacent-frame leakage)
==============================================================================
"""

import os
import sys
import numpy as np
import pandas as pd
from scipy import signal, stats
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV
from sklearn.svm import SVR
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import KFold

try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except ImportError:
    HAS_XGB = False

try:
    from catboost import CatBoostRegressor
    HAS_CAT = True
except ImportError:
    HAS_CAT = False

# --- 1. Cohort & Protocol Constants ---
COHORT = [1, 2, 3, 4, 5, 6, 8, 9, 10, 12]  # S07 (sensor failure) & S11 excluded
TARGET_MOVEMENTS = [5, 6]
V_DEADBAND = 4.0        # deg/s: Noise suppression deadband
DT_SEC = 0.010          # 10 ms hop (100 Hz cadence)
LAG_HOPS = 10           # 100 ms Retrospective Delay
LOG_FILE = "cohort_100ms_audited_report.txt"

# Set to True ONLY if you have NOT already swapped Ch 9/10 in MATLAB for S05
INVERT_S05_IN_PYTHON = False

MODEL_FAMILIES = ["Ridge", "SVR_Linear", "SVR_RBF", "Random_Forest", "Hist_Gradient_Boost", "MLP_NeuralNet"]
if HAS_XGB:
    MODEL_FAMILIES.append("XGBoost")
if HAS_CAT:
    MODEL_FAMILIES.append("CatBoost")

# --- 2. Contiguous Block Slicer ---
def find_contiguous_segments(df, max_dt=0.018):
    """
    Identifies continuous chunks where sample spacing is strictly <= 18 ms.
    Drops any fragment shorter than 15 frames.
    """
    t = df["timestamp_s"].values
    if len(t) == 0:
        return []
    dt = np.diff(t)
    break_indices = np.where(dt > max_dt)[0] + 1
    segments = np.split(np.arange(len(df)), break_indices)
    return [seg for seg in segments if len(seg) >= LAG_HOPS + 5]

# --- 3. Retrospective Context Builder (Segment-Aware) ---
def extract_segment_retrospective_data(df, feat_cols, lag_hops=LAG_HOPS):
    """
    Extracts features and targets strictly within independent contiguous segments.
    Returns segmented data structures to ensure correct angle integration.
    """
    segments = find_contiguous_segments(df)
    
    seg_records = []
    X_raw_all = df[feat_cols].values
    y_raw_all = df["theta_vel_deg_s"].values
    theta_raw_all = df["theta_deg"].values
    mav_all = (df["EDC_MAV"] + df["FDS_MAV"]).values

    for seg in segments:
        X_seg = X_raw_all[seg]
        y_seg = y_raw_all[seg]
        mav_seg = mav_all[seg]
        th_seg = theta_raw_all[seg]
        L = len(seg)

        X_s, y_s, th_s, mav_s = [], [], [], []

        for i in range(lag_hops + 4, L):
            target_idx = i - lag_hops  # Exactly t - 100 ms

            # 4 Causal Taps spanning 140 ms relative to target
            tap0 = X_seg[i]               # t (Clock t: +100 ms lookahead)
            tap1 = X_seg[i - 5]           # t - 50 ms (+50 ms lookahead)
            tap2 = X_seg[target_idx]      # t - 100 ms (Synchronous with target v)
            tap3 = X_seg[target_idx - 4]  # t - 140 ms (-40 ms pre-movement reference)

            X_s.append(np.hstack([tap0, tap1, tap2, tap3]))
            y_s.append(y_seg[target_idx])
            th_s.append(th_seg[target_idx])
            # Gating must evaluate the target frame's electrical state
            mav_s.append(mav_seg[target_idx])

        if len(X_s) > 0:
            seg_records.append({
                "X": np.array(X_s),
                "y": np.array(y_s),
                "theta": np.array(th_s),
                "mav": np.array(mav_s)
            })

    return seg_records

# --- 4. Bounded Timeline Slicing ---
def partition_timeline_bounded(df, sub_id, buffer_sec=1.0):
    test_reps = [2, 5] if sub_id == 4 else [2, 6]
    is_test = np.zeros(len(df), dtype=bool)

    for m in TARGET_MOVEMENTS:
        for r in test_reps:
            cue_mask = (df["stimulus"] == m) & (df["repetition"] == r)
            if not np.any(cue_mask):
                continue

            t_cue_start = df.loc[cue_mask, "timestamp_s"].min()
            t_cue_end = df.loc[cue_mask, "timestamp_s"].max()

            t_eval_start = t_cue_start - buffer_sec
            t_eval_end = t_cue_end + buffer_sec

            slice_mask = (df["timestamp_s"] >= t_eval_start) & (df["timestamp_s"] <= t_eval_end)
            is_test[slice_mask] = True

    is_corrupt = (df["repetition"] == 6).values if sub_id == 4 else np.zeros(len(df), dtype=bool)
    is_plateau = (df["is_hold_plateau"] == 1).values

    valid_mask = ~is_corrupt & ~is_plateau
    train_mask = valid_mask & ~is_test
    test_mask = valid_mask & is_test

    return train_mask, test_mask, test_reps

# --- 5. Pure Pre-Cue Baseline Calculator (Contamination Guarded) ---
def compute_precue_rest_floor(df_raw):
    """
    Extracts baseline MAV strictly from unprompted rest periods (stimulus == 0)
    immediately preceding movement cues to eliminate active contamination.
    """
    t_vals = df_raw["timestamp_s"].values
    stim = df_raw["stimulus"].values
    rest_mask = np.zeros(len(df_raw), dtype=bool)

    for m in TARGET_MOVEMENTS:
        m_idx = np.where(stim == m)[0]
        if len(m_idx) == 0:
            continue
        # Find true cue initiations from quiet rest
        breaks = np.where(np.diff(m_idx) > 1)[0]
        cue_starts = np.insert(m_idx[breaks + 1], 0, m_idx[0])

        for c_idx in cue_starts:
            t_start = t_vals[c_idx]
            # Must be inside stimulus == 0 in the 1 second before cue start
            pre_window = (t_vals >= t_start - 1.0) & (t_vals < t_start) & (stim == 0)
            rest_mask |= pre_window

    mav_total = (df_raw["EDC_MAV"] + df_raw["FDS_MAV"]).values
    if np.sum(rest_mask) > 30:
        return np.median(mav_total[rest_mask])
    return np.percentile(mav_total, 10)

# --- 6. Unbiased Output Velocity Gating ---
def apply_unbiased_actuation(y_raw, mav_target, mav_floor, v_db=V_DEADBAND):
    y_cmd = np.copy(y_raw)
    # Piecewise continuous deadband: preserves slope without clipping shock
    active = np.abs(y_cmd) > v_db
    y_cmd[~active] = 0.0
    y_cmd[active] = np.sign(y_cmd[active]) * (np.abs(y_cmd[active]) - v_db)

    # Electrical silence hurdle
    is_quiet = mav_target < (1.5 * mav_floor)
    y_cmd[is_quiet] = 0.0
    return y_cmd

# --- 7. Dual Evaluator (Segment-Wise Angle Integration) ---
def evaluate_dual_domain(test_records, model, scaler, gamma, mav_floor, dt=DT_SEC):
    """
    Integrates angle trajectory INDEPENDENTLY per contiguous test segment,
    eliminating cumulative multi-second drift artifacts across separate trials.
    """
    all_y_true, all_y_cmd = [], []
    all_th_true, all_th_recon = [], []

    for seg in test_records:
        X_seg = np.clip(scaler.transform(seg["X"]), -20.0, 20.0)
        y_seg = seg["y"]
        th_seg = seg["theta"]
        mav_seg = seg["mav"]

        y_pred = model.predict(X_seg) * gamma
        y_cmd = apply_unbiased_actuation(y_pred, mav_seg, mav_floor)

        # Segment-specific independent integration
        # Each segment starts exactly at its own ground-truth initial angle
        theta_recon_seg = th_seg[0] + np.cumsum(y_cmd * dt)

        all_y_true.extend(y_seg)
        all_y_cmd.extend(y_cmd)
        all_th_true.extend(th_seg)
        all_th_recon.extend(theta_recon_seg)

    yt = np.array(all_y_true, dtype=np.float64)
    yp = np.array(all_y_cmd, dtype=np.float64)
    th_t = np.array(all_th_true, dtype=np.float64)
    th_p = np.array(all_th_recon, dtype=np.float64)

    # 1. Velocity Tracking Metrics
    r_vel, _ = stats.pearsonr(yt, yp)
    sse_v = np.sum((yt - yp) ** 2)
    sst_v = np.sum((yt - np.mean(yt)) ** 2)
    r2_vel = max(0.0, 1.0 - (sse_v / sst_v)) if sst_v > 0 else 0.0
    rmse_vel = np.sqrt(np.mean((yt - yp) ** 2))

    # 2. Angle Trajectory Metrics (True Functional Control)
    r_ang, _ = stats.pearsonr(th_t, th_p)
    sse_a = np.sum((th_t - th_p) ** 2)
    sst_a = np.sum((th_t - np.mean(th_t)) ** 2)
    r2_ang = max(0.0, 1.0 - (sse_a / sst_a)) if sst_a > 0 else 0.0
    rmse_ang = np.sqrt(np.mean((th_t - th_p) ** 2))

    # 3. Peak Cross-Correlation Reconstruction Lag
    yp_d = yp - np.mean(yp)
    yt_d = yt - np.mean(yt)
    xcorr = signal.correlate(yp_d, yt_d, mode="full")
    lags = signal.correlation_lags(len(yp_d), len(yt_d), mode="full")
    lag_mask = (lags >= -35) & (lags <= 35)
    lag_ms = lags[lag_mask][np.argmax(xcorr[lag_mask])] * (dt * 1000.0)

    # 4. Rest Jitter %
    rest_mask = np.abs(yt) <= 2.0
    far_pct = (np.sum(np.abs(yp[rest_mask]) > 0.0) / np.sum(rest_mask)) * 100.0 if np.any(rest_mask) else 0.0

    return r_vel, r2_vel, rmse_vel, r_ang, r2_ang, rmse_ang, lag_ms, far_pct

def log_print(msg, file_handle):
    print(msg, end="")
    file_handle.write(msg)

# --- 8. Main Benchmark Loop ---
def main():
    f_log = open(LOG_FILE, "w", encoding="utf-8")

    header = (
        "========================================================================================================================\n"
        "      CONTINUOUS sEMG 100 ms RETROSPECTIVE BENCHMARK: STRICT CONTIGUITY & DUAL EVALUATION (VELOCITY + ANGLE)           \n"
        "========================================================================================================================\n"
        f" Cohort: {len(COHORT)} Subjects | Exclusions: S07 & S11 (Entirely) | S4 Rep 6\n"
        " Integrity: Contiguous block segmentation (Zero pairing across dropped plateaus or trial boundaries)\n"
        " Features : 120 Causal Features across 140 ms receptive field (-40 ms, 0 ms, +50 ms, +100 ms relative to target)\n"
        " Metrics  : Instantaneous Velocity (r, R^2, RMSE) AND Integrated Excursion Angle (r_ang, R^2_ang, RMSE_deg)\n"
        "========================================================================================================================\n\n"
    )
    log_print(header, f_log)

    cohort_results = {sub: {} for sub in COHORT}

    for sub_id in COHORT:
        csv_path = os.path.join("datasets", "DB2", f"DB2_s{sub_id}", f"S{sub_id}_E1_A1_dynamic_velocity_features.csv")
        if not os.path.exists(csv_path):
            csv_path = f"S{sub_id}_E1_A1_dynamic_velocity_features.csv"

        if not os.path.exists(csv_path):
            log_print(f"[WARN] Subject {sub_id:02d}: CSV not found. Skipping.\n", f_log)
            continue

        df = pd.read_csv(csv_path)

        # Hardware correction: Invert Subject 5 ONLY if not already swapped in MATLAB
        if sub_id == 5 and INVERT_S05_IN_PYTHON:
            edc_cols = [c for c in df.columns if c.startswith("EDC_")]
            fds_cols = [c for c in df.columns if c.startswith("FDS_")]
            for e_col, f_col in zip(edc_cols, fds_cols):
                df[e_col], df[f_col] = df[f_col].copy(), df[e_col].copy()
            for ndi_col in ["NDI_MAV", "NDI_RMS", "NDI_WL"]:
                if ndi_col in df.columns:
                    df[ndi_col] = -df[ndi_col]

        # 1. Clean Pre-Cue Noise Floor from Unpruned Baseline
        mav_rest_floor = compute_precue_rest_floor(df)

        # 2. Partition Timeline
        train_mask, test_mask, test_reps = partition_timeline_bounded(df, sub_id, buffer_sec=1.0)
        train_df = df[train_mask].copy()
        test_df = df[test_mask].copy()

        meta_cols = ["timestamp_s", "theta_deg", "theta_vel_deg_s", "stimulus", "repetition", "is_hold_plateau"]
        feat_cols = [c for c in df.columns if c not in meta_cols]

        # 3. Extract Data Strictly within Contiguous Segments
        train_records = extract_segment_retrospective_data(train_df, feat_cols)
        test_records = extract_segment_retrospective_data(test_df, feat_cols)

        if len(train_records) == 0 or len(test_records) == 0:
            log_print(f"[WARN] Subject {sub_id:02d}: Insufficient contiguous data. Skipping.\n", f_log)
            continue

        # Flatten training segments into design matrix
        X_tr = np.vstack([rec["X"] for rec in train_records])
        y_tr = np.concatenate([rec["y"] for rec in train_records])

        # 4. Feature Standardization
        scaler = StandardScaler()
        X_tr_z = scaler.fit_transform(X_tr)

        n_test_frames = sum(len(r["y"]) for r in test_records)
        print(f">>> Subject {sub_id:02d} | Train Frames: {len(X_tr_z)} | Test Frames: {n_test_frames} | Pre-Cue Floor: {mav_rest_floor*1e6:.1f} uV")

        # Chronological non-shuffling cross-validation (Zero adjacent leakage)
        cv_splits = KFold(n_splits=4, shuffle=False)

        def run_model(model, m_name):
            model.fit(X_tr_z, y_tr)
            y_pred_tr = model.predict(X_tr_z)

            # Variance gain calibration to counteract tree leaf mean-compression
            denom = np.sum(y_pred_tr ** 2)
            gamma = np.sum(y_tr * y_pred_tr) / denom if denom > 1e-6 else 1.0
            gamma = np.clip(gamma, 0.9, 1.35)

            rv, r2v, rmsev, ra, r2a, rmsea, lag, far = evaluate_dual_domain(
                test_records, model, scaler, gamma, mav_rest_floor
            )
            cohort_results[sub_id][m_name] = {
                "r_vel": rv, "r2_vel": r2v, "rmse_vel": rmsev,
                "r_ang": ra, "r2_ang": r2a, "rmse_ang": rmsea,
                "lag": lag, "far": far
            }

        # Model Suite Evaluation
        run_model(RidgeCV(alphas=np.logspace(0, 5, 12), cv=cv_splits), "Ridge")
        
        sub_step = max(1, len(X_tr_z) // 3500)
        run_model(SVR(kernel="linear", C=1.5, epsilon=2.0), "SVR_Linear")
        run_model(SVR(kernel="rbf", C=3.0, epsilon=1.5, gamma="scale"), "SVR_RBF")
        run_model(RandomForestRegressor(n_estimators=85, max_depth=12, min_samples_leaf=6, max_features="sqrt", n_jobs=-1, random_state=42), "Random_Forest")
        run_model(HistGradientBoostingRegressor(max_iter=140, learning_rate=0.06, max_leaf_nodes=31, min_samples_leaf=15, l2_regularization=2.5, early_stopping=True, n_iter_no_change=10, random_state=42), "Hist_Gradient_Boost")
        run_model(MLPRegressor(hidden_layer_sizes=(96, 48), activation="relu", solver="adam", alpha=0.15, learning_rate_init=0.002, max_iter=200, early_stopping=True, n_iter_no_change=12, random_state=42), "MLP_NeuralNet")

        if HAS_XGB:
            run_model(XGBRegressor(n_estimators=130, max_depth=5, learning_rate=0.06, reg_alpha=1.0, reg_lambda=2.0, subsample=0.85, colsample_bytree=0.7, n_jobs=-1, random_state=42), "XGBoost")
        if HAS_CAT:
            run_model(CatBoostRegressor(iterations=160, depth=6, learning_rate=0.06, l2_leaf_reg=4.0, verbose=0, random_seed=42), "CatBoost")

    # --- 9. Format & Log Summary Reports ---
    summary_header = (
        "\n========================================================================================================================\n"
        "                                     HOLDOUT DUAL-DOMAIN VALIDATION SUMMARY                                             \n"
        "========================================================================================================================\n"
        " Subj | Best Model Family     | Vel Pearson r | Vel R^2 (VAF) | Angle Pearson r | Angle R^2 (VAF)| Recon Lag | Rest Jitter % \n"
        "------------------------------------------------------------------------------------------------------------------------\n"
    )
    log_print(summary_header, f_log)

    for sub_id in COHORT:
        sub_dict = cohort_results.get(sub_id, {})
        if not sub_dict:
            continue

        best_m = max(sub_dict.keys(), key=lambda m: sub_dict[m]["r_vel"])
        bm = sub_dict[best_m]

        line = (
            f"  S{sub_id:02d} | {best_m:<21} |     {bm['r_vel']:6.4f}    |     {bm['r2_vel']:6.4f}    "
            f"|      {bm['r_ang']:6.4f}     |     {bm['r2_ang']:6.4f}     |  {bm['lag']:+5.1f} ms  |     {bm['far']:4.2f}%\n"
        )
        log_print(line, f_log)

    log_print("------------------------------------------------------------------------------------------------------------------------\n\n", f_log)

    # --- 10. Aggregate Model Family Comparison Table ---
    agg_header = (
        "========================================================================================================================\n"
        "                         OVERALL MODEL FAMILY PERFORMANCE (COHORT MEAN +/- STD)                                         \n"
        "========================================================================================================================\n"
        " Model Architecture    | Vel Pearson r   | Vel R^2 (VAF)   | Angle Pearson r | Angle R^2 (VAF) | Recon Lag (ms) | Rest Jitter  \n"
        "------------------------------------------------------------------------------------------------------------------------\n"
    )
    log_print(agg_header, f_log)

    for m_name in MODEL_FAMILIES:
        rv_l = [cohort_results[s][m_name]["r_vel"] for s in COHORT if m_name in cohort_results[s]]
        r2v_l = [cohort_results[s][m_name]["r2_vel"] for s in COHORT if m_name in cohort_results[s]]
        ra_l = [cohort_results[s][m_name]["r_ang"] for s in COHORT if m_name in cohort_results[s]]
        r2a_l = [cohort_results[s][m_name]["r2_ang"] for s in COHORT if m_name in cohort_results[s]]
        lag_l = [cohort_results[s][m_name]["lag"] for s in COHORT if m_name in cohort_results[s]]
        far_l = [cohort_results[s][m_name]["far"] for s in COHORT if m_name in cohort_results[s]]

        if not rv_l:
            continue

        agg_line = (
            f" {m_name:<22} |  {np.mean(rv_l):5.3f} +/- {np.std(rv_l):4.3f} |  {np.mean(r2v_l):5.3f} +/- {np.std(r2v_l):4.3f} |  "
            f"{np.mean(ra_l):5.3f} +/- {np.std(ra_l):4.3f}  |  {np.mean(r2a_l):5.3f} +/- {np.std(r2a_l):4.3f}  | "
            f"{np.mean(lag_l):+5.1f} +/- {np.std(lag_l):4.1f}  |  {np.mean(far_l):4.2f} +/- {np.std(far_l):4.2f}%\n"
        )
        log_print(agg_line, f_log)

    log_print("========================================================================================================================\n", f_log)
    print(f"\n[SUCCESS] Audited benchmark report successfully generated at: {LOG_FILE}\n")
    f_log.close()

if __name__ == "__main__":
    main()

      CONTINUOUS sEMG 100 ms RETROSPECTIVE BENCHMARK: STRICT CONTIGUITY & DUAL EVALUATION (VELOCITY + ANGLE)           
 Cohort: 10 Subjects | Exclusions: S07 & S11 (Entirely) | S4 Rep 6
 Integrity: Contiguous block segmentation (Zero pairing across dropped plateaus or trial boundaries)
 Features : 120 Causal Features across 140 ms receptive field (-40 ms, 0 ms, +50 ms, +100 ms relative to target)
 Metrics  : Instantaneous Velocity (r, R^2, RMSE) AND Integrated Excursion Angle (r_ang, R^2_ang, RMSE_deg)

>>> Subject 01 | Train Frames: 6967 | Test Frames: 1202 | Pre-Cue Floor: 15.3 uV
>>> Subject 02 | Train Frames: 7065 | Test Frames: 1890 | Pre-Cue Floor: 12.3 uV
>>> Subject 03 | Train Frames: 6165 | Test Frames: 1711 | Pre-Cue Floor: 21.0 uV
>>> Subject 04 | Train Frames: 5102 | Test Frames: 1721 | Pre-Cue Floor: 17.0 uV
>>> Subject 05 | Train Frames: 7260 | Test Frames: 1167 | Pre-Cue Floor: 25.9 uV
>>> Subject 06 | Train Frames: 6980 | Test Frames: 1366 | Pre-Cue Floor: 13.0 uV
>>> 

In [1]:
#!/usr/bin/env python3
"""
====================================================================================================
CONTINUOUS sEMG UNIPOLAR CLOSING REGRESSOR (REST -> FIST SPECIALIZED BENCHMARK)
Cohort      : Subjects 1 to 12 (Excluding S07 & S11; Inverting S05)
Architecture: Unipolar Closing Speed Decoder (v_close >= 0.0 deg/s)
Target Logic:
  - M6 Takeoff (t1 -> t2)        : Active closing speed v = |theta_vel|
  - M5 Hand Open                 : Explicit negative training (v = 0.0 deg/s)
  - M6 Return Descent (t3 -> t4) : Explicit negative training (v = 0.0 deg/s)
  - Unprompted Baseline Rest     : Explicit negative training (v = 0.0 deg/s)
Evaluation  : 
  1. M6 Closing Excursion Tracking (r_vel, R^2_vel, r_ang, R^2_ang, Recon Lag)
  2. M5 Inhibitory Stress Test (M5 False Closing %: should be 0.0%)
  3. Pre-Cue Resting False Alarm Rate (Rest Jitter %)
====================================================================================================
"""

import os
import sys
import numpy as np
import pandas as pd
from scipy import signal, stats
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV
from sklearn.svm import SVR
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import KFold

try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except ImportError:
    HAS_XGB = False

try:
    from catboost import CatBoostRegressor
    HAS_CAT = True
except ImportError:
    HAS_CAT = False

# --- 1. Cohort & Protocol Constants ---
COHORT = [1, 2, 3, 4, 5, 6, 8, 9, 10, 12]  # S07 (sensor failure) & S11 permanently excluded
V_DEADBAND = 4.0        # deg/s: Unbiased threshold
DT_SEC = 0.010          # 10 ms hop (100 Hz cadence)
LAG_HOPS = 10           # 100 ms Retrospective Delay
LOG_FILE = "cohort_unipolar_closing_report.txt"

# Set to True ONLY if you have NOT already swapped Ch 9/10 in MATLAB for Subject 5
INVERT_S05_IN_PYTHON = False

MODEL_FAMILIES = ["Ridge", "SVR_Linear", "SVR_RBF", "Random_Forest", "Hist_Gradient_Boost", "MLP_NeuralNet"]
if HAS_XGB:
    MODEL_FAMILIES.append("XGBoost")
if HAS_CAT:
    MODEL_FAMILIES.append("CatBoost")

# --- 2. Unipolar Target & Phase Labeler ---
def construct_unipolar_closing_labels(df):
    y_close = np.zeros(len(df), dtype=np.float64)
    is_m6_takeoff = np.zeros(len(df), dtype=bool)
    is_m5_open = np.zeros(len(df), dtype=bool)

    stim = df["stimulus"].values
    rep = df["repetition"].values
    is_plat = df["is_hold_plateau"].values.astype(bool)
    v_raw = df["theta_vel_deg_s"].values

    # 1. Label M6 (Fist Clench) Takeoffs
    m6_reps = np.unique(rep[stim == 6])
    for r in m6_reps:
        trial_mask = (stim == 6) & (rep == r)
        trial_indices = np.where(trial_mask)[0]
        if len(trial_indices) == 0:
            continue

        plat_in_trial = np.where(trial_mask & is_plat)[0]
        if len(plat_in_trial) > 0:
            first_plat = plat_in_trial[0]
            takeoff_idx = [i for i in trial_indices if i < first_plat]
        else:
            cutoff = trial_indices[0] + int(0.40 * len(trial_indices))
            takeoff_idx = [i for i in trial_indices if i < cutoff]

        if len(takeoff_idx) > 0:
            is_m6_takeoff[takeoff_idx] = True
            y_close[takeoff_idx] = np.abs(v_raw[takeoff_idx])

    # 2. Label M5 (Hand Open)
    is_m5_open[stim == 5] = True

    df_labeled = df.copy()
    df_labeled["y_close"] = y_close
    df_labeled["is_m6_takeoff"] = is_m6_takeoff
    df_labeled["is_m5_open"] = is_m5_open
    return df_labeled

# --- 3. Contiguous Block Slicer ---
def find_contiguous_segments(df, max_dt=0.018):
    t = df["timestamp_s"].values
    if len(t) == 0:
        return []
    dt = np.diff(t)
    break_indices = np.where(dt > max_dt)[0] + 1
    segments = np.split(np.arange(len(df)), break_indices)
    return [seg for seg in segments if len(seg) >= LAG_HOPS + 5]

# --- 4. Retrospective Context Builder (Segment Aware) ---
def extract_segment_retrospective_data(df, feat_cols, lag_hops=LAG_HOPS):
    segments = find_contiguous_segments(df)
    seg_records = []

    X_raw_all = df[feat_cols].values
    y_close_all = df["y_close"].values
    theta_raw_all = df["theta_deg"].values
    is_m6_all = df["is_m6_takeoff"].values
    is_m5_all = df["is_m5_open"].values

    for seg in segments:
        X_seg = X_raw_all[seg]
        y_seg = y_close_all[seg]
        th_seg = theta_raw_all[seg]
        m6_seg = is_m6_all[seg]
        m5_seg = is_m5_all[seg]
        L = len(seg)

        X_s, y_s, th_s, m6_s, m5_s = [], [], [], [], []

        for i in range(lag_hops + 4, L):
            target_idx = i - lag_hops  # t - 100 ms

            tap0 = X_seg[i]               # t (Clock t: +100 ms lookahead)
            tap1 = X_seg[i - 5]           # t - 50 ms (+50 ms lookahead)
            tap2 = X_seg[target_idx]      # t - 100 ms (Synchronous with target)
            tap3 = X_seg[target_idx - 4]  # t - 140 ms (-40 ms pre-movement reference)

            X_s.append(np.hstack([tap0, tap1, tap2, tap3]))
            y_s.append(y_seg[target_idx])
            th_s.append(th_seg[target_idx])
            m6_s.append(m6_seg[target_idx])
            m5_s.append(m5_seg[target_idx])

        if len(X_s) > 0:
            seg_records.append({
                "X": np.array(X_s),
                "y": np.array(y_s),
                "theta": np.array(th_s),
                "is_m6_takeoff": np.array(m6_s),
                "is_m5_open": np.array(m5_s)
            })

    return seg_records

# --- 5. Timeline Partitioning ---
def partition_timeline_unipolar(df, sub_id, buffer_sec=1.0):
    test_reps = [2, 5] if sub_id == 4 else [2, 6]
    is_test = np.zeros(len(df), dtype=bool)

    for m in [5, 6]:
        for r in test_reps:
            cue_mask = (df["stimulus"] == m) & (df["repetition"] == r)
            if not np.any(cue_mask):
                continue
            t_start = df.loc[cue_mask, "timestamp_s"].min() - buffer_sec
            t_end = df.loc[cue_mask, "timestamp_s"].max() + buffer_sec
            is_test[(df["timestamp_s"] >= t_start) & (df["timestamp_s"] <= t_end)] = True

    is_corrupt = (df["repetition"] == 6).values if sub_id == 4 else np.zeros(len(df), dtype=bool)
    is_plateau = (df["is_hold_plateau"] == 1).values

    valid_mask = ~is_corrupt & ~is_plateau
    train_mask = valid_mask & ~is_test
    test_mask = valid_mask & is_test

    return train_mask, test_mask, test_reps

# --- 6. Unbiased Deadband Actuator ---
def apply_unbiased_deadband(y_raw, v_db=V_DEADBAND):
    y_cmd = np.maximum(0.0, np.copy(y_raw))
    y_cmd[y_cmd <= v_db] = 0.0
    return y_cmd

# --- 7. Dual-Domain & Inhibitory Stress Evaluator ---
def evaluate_unipolar_performance(test_records, model, scaler, gamma, dt=DT_SEC):
    m6_y_true, m6_y_cmd = [], []
    m6_th_true, m6_th_recon = [], []
    m5_cmds = []
    rest_cmds = []

    for seg in test_records:
        X_seg = np.clip(scaler.transform(seg["X"]), -20.0, 20.0)
        y_seg = seg["y"]
        th_seg = seg["theta"]
        m6_mask = seg["is_m6_takeoff"]
        m5_mask = seg["is_m5_open"]

        y_pred = model.predict(X_seg) * gamma
        y_cmd = apply_unbiased_deadband(y_pred)

        # 1. Evaluate Active M6 Closing Trajectory
        if np.any(m6_mask):
            idx_m6 = np.where(m6_mask)[0]
            yt_sub = y_seg[idx_m6]
            yp_sub = y_cmd[idx_m6]
            th_sub = th_seg[idx_m6]

            direction = np.sign(th_sub[-1] - th_sub[0]) if len(th_sub) > 1 else 1.0
            if direction == 0.0:
                direction = 1.0
            th_recon_sub = th_sub[0] + direction * np.cumsum(yp_sub * dt)

            m6_y_true.extend(yt_sub)
            m6_y_cmd.extend(yp_sub)
            m6_th_true.extend(th_sub)
            m6_th_recon.extend(th_recon_sub)

        # 2. Evaluate M5 Inhibitory Suppression
        if np.any(m5_mask):
            m5_cmds.extend(y_cmd[m5_mask])

        # 3. Evaluate Resting Baseline
        rest_mask = (~m6_mask) & (~m5_mask) & (y_seg <= 2.0)
        if np.any(rest_mask):
            rest_cmds.extend(y_cmd[rest_mask])

    yt_m6 = np.array(m6_y_true, dtype=np.float64)
    yp_m6 = np.array(m6_y_cmd, dtype=np.float64)
    th_t = np.array(m6_th_true, dtype=np.float64)
    th_p = np.array(m6_th_recon, dtype=np.float64)

    # Closing Velocity Metrics
    if len(yt_m6) > 10 and np.std(yp_m6) > 0 and np.std(yt_m6) > 0:
        r_vel, _ = stats.pearsonr(yt_m6, yp_m6)
        sse_v = np.sum((yt_m6 - yp_m6) ** 2)
        sst_v = np.sum((yt_m6 - np.mean(yt_m6)) ** 2)
        r2_vel = max(0.0, 1.0 - (sse_v / sst_v)) if sst_v > 0 else 0.0
    else:
        r_vel, r2_vel = 0.0, 0.0

    # Angle Excursion Metrics
    if len(th_t) > 10 and np.std(th_p) > 0 and np.std(th_t) > 0:
        r_ang, _ = stats.pearsonr(th_t, th_p)
        sse_a = np.sum((th_t - th_p) ** 2)
        sst_a = np.sum((th_t - np.mean(th_t)) ** 2)
        r2_ang = max(0.0, 1.0 - (sse_a / sst_a)) if sst_a > 0 else 0.0
    else:
        r_ang, r2_ang = 0.0, 0.0

    # Cross-Correlation Lag on M6 Closing
    if len(yt_m6) > 20 and np.std(yp_m6) > 0:
        yp_d = yp_m6 - np.mean(yp_m6)
        yt_d = yt_m6 - np.mean(yt_m6)
        xcorr = signal.correlate(yp_d, yt_d, mode="full")
        lags = signal.correlation_lags(len(yp_d), len(yt_d), mode="full")
        lag_mask = (lags >= -35) & (lags <= 35)
        lag_ms = lags[lag_mask][np.argmax(xcorr[lag_mask])] * (dt * 1000.0)
    else:
        lag_ms = float("nan")

    # M5 False Closing Rate
    m5_cmds = np.array(m5_cmds)
    m5_false_alarm_pct = (np.sum(m5_cmds > 0.0) / len(m5_cmds) * 100.0) if len(m5_cmds) > 0 else 0.0

    # Rest Jitter %
    rest_cmds = np.array(rest_cmds)
    rest_jitter_pct = (np.sum(rest_cmds > 0.0) / len(rest_cmds) * 100.0) if len(rest_cmds) > 0 else 0.0

    return r_vel, r2_vel, r_ang, r2_ang, lag_ms, m5_false_alarm_pct, rest_jitter_pct

def log_print(msg, file_handle):
    print(msg, end="")
    file_handle.write(msg)

# --- 8. Main Benchmark Loop ---
def main():
    f_log = open(LOG_FILE, "w", encoding="utf-8")

    header = (
        "========================================================================================================================\n"
        "      CONTINUOUS sEMG UNIPOLAR CLOSING BENCHMARK: INHIBITORY M5 GROUNDING & ANGLE EXCURSION                             \n"
        "========================================================================================================================\n"
        f" Cohort: {len(COHORT)} Subjects | Exclusions: S07 & S11 (Entirely) | S4 Rep 6\n"
        " Target  : Unipolar closing speed v_close >= 0 deg/s (M6 Takeoff = |v|; M5 Open = 0; M6 Return = 0; Rest = 0)\n"
        " Features: 120 Causal Features across 140 ms receptive field (-40 ms, 0 ms, +50 ms, +100 ms relative to target)\n"
        " Metrics : M6 Closing Flight (r_vel, R^2_vel, r_ang, R^2_ang) | M5 False Closing % | Rest Jitter %\n"
        "========================================================================================================================\n\n"
    )
    log_print(header, f_log)

    cohort_results = {sub: {} for sub in COHORT}

    for sub_id in COHORT:
        csv_path = os.path.join("datasets", "DB2", f"DB2_s{sub_id}", f"S{sub_id}_E1_A1_dynamic_velocity_features.csv")
        if not os.path.exists(csv_path):
            csv_path = f"S{sub_id}_E1_A1_dynamic_velocity_features.csv"

        if not os.path.exists(csv_path):
            log_print(f"[WARN] Subject {sub_id:02d}: CSV not found. Skipping.\n", f_log)
            continue

        df_raw = pd.read_csv(csv_path)

        if sub_id == 5 and INVERT_S05_IN_PYTHON:
            edc_cols = [c for c in df_raw.columns if c.startswith("EDC_")]
            fds_cols = [c for c in df_raw.columns if c.startswith("FDS_")]
            for e_col, f_col in zip(edc_cols, fds_cols):
                df_raw[e_col], df_raw[f_col] = df_raw[f_col].copy(), df_raw[e_col].copy()
            for ndi_col in ["NDI_MAV", "NDI_RMS", "NDI_WL"]:
                if ndi_col in df_raw.columns:
                    df_raw[ndi_col] = -df_raw[ndi_col]

        # 1. Label Targets
        df_labeled = construct_unipolar_closing_labels(df_raw)

        # 2. Partition Timeline
        train_mask, test_mask, test_reps = partition_timeline_unipolar(df_labeled, sub_id, buffer_sec=1.0)
        train_df = df_labeled[train_mask].copy()
        test_df = df_labeled[test_mask].copy()

        meta_cols = ["timestamp_s", "theta_deg", "theta_vel_deg_s", "stimulus", "repetition", 
                     "is_hold_plateau", "y_close", "is_m6_takeoff", "is_m5_open"]
        feat_cols = [c for c in df_raw.columns if c not in meta_cols]

        # 3. Extract Segments
        train_records = extract_segment_retrospective_data(train_df, feat_cols)
        test_records = extract_segment_retrospective_data(test_df, feat_cols)

        if len(train_records) == 0 or len(test_records) == 0:
            log_print(f"[WARN] Subject {sub_id:02d}: Insufficient contiguous data. Skipping.\n", f_log)
            continue

        X_tr = np.vstack([rec["X"] for rec in train_records])
        y_tr = np.concatenate([rec["y"] for rec in train_records])

        # 4. Standardize Features
        scaler = StandardScaler()
        X_tr_z = scaler.fit_transform(X_tr)

        print(f">>> Subject {sub_id:02d} | Train: {len(X_tr_z)} frames (M6 Takeoffs + Inhibitory M5/Return) | Test Segments: {len(test_records)}")

        cv_splits = KFold(n_splits=4, shuffle=False)

        # Subsampling step specifically for SVR to prevent execution hang
        sub_step_svr = max(1, len(X_tr_z) // 3000)

        def run_model(model, m_name, subsample=1):
            if subsample > 1:
                model.fit(X_tr_z[::subsample], y_tr[::subsample])
            else:
                model.fit(X_tr_z, y_tr)

            # Restrict variance calibration strictly to active closing frames
            active_tr = y_tr > 0.0
            if np.any(active_tr):
                y_pred_active = np.maximum(0.0, model.predict(X_tr_z[active_tr]))
                denom = np.sum(y_pred_active ** 2)
                gamma = np.sum(y_tr[active_tr] * y_pred_active) / denom if denom > 1e-6 else 1.0
            else:
                gamma = 1.0
            gamma = np.clip(gamma, 0.9, 1.4)

            rv, r2v, ra, r2a, lag, m5_fa, rj = evaluate_unipolar_performance(
                test_records, model, scaler, gamma
            )
            cohort_results[sub_id][m_name] = {
                "r_vel": rv, "r2_vel": r2v,
                "r_ang": ra, "r2_ang": r2a,
                "lag": lag, "m5_fa": m5_fa, "rj": rj
            }

        # Model Suite Evaluation
        run_model(RidgeCV(alphas=np.logspace(0, 5, 12), cv=cv_splits), "Ridge")
        run_model(SVR(kernel="linear", C=1.5, epsilon=2.0), "SVR_Linear", subsample=sub_step_svr)
        run_model(SVR(kernel="rbf", C=3.0, epsilon=1.5, gamma="scale"), "SVR_RBF", subsample=sub_step_svr)
        run_model(RandomForestRegressor(n_estimators=85, max_depth=12, min_samples_leaf=6, max_features="sqrt", n_jobs=-1, random_state=42), "Random_Forest")
        run_model(HistGradientBoostingRegressor(max_iter=140, learning_rate=0.06, max_leaf_nodes=31, min_samples_leaf=15, l2_regularization=2.5, early_stopping=True, n_iter_no_change=10, random_state=42), "Hist_Gradient_Boost")
        run_model(MLPRegressor(hidden_layer_sizes=(96, 48), activation="relu", solver="adam", alpha=0.15, learning_rate_init=0.002, max_iter=200, early_stopping=True, n_iter_no_change=12, random_state=42), "MLP_NeuralNet")

        if HAS_XGB:
            run_model(XGBRegressor(n_estimators=130, max_depth=5, learning_rate=0.06, reg_alpha=1.0, reg_lambda=2.0, subsample=0.85, colsample_bytree=0.7, n_jobs=-1, random_state=42), "XGBoost")
        if HAS_CAT:
            run_model(CatBoostRegressor(iterations=160, depth=6, learning_rate=0.06, l2_leaf_reg=4.0, verbose=0, random_seed=42), "CatBoost")

    # --- 9. Format & Log Summary Reports ---
    summary_header = (
        "\n========================================================================================================================\n"
        "                              HOLDOUT UNIPOLAR CLOSING & INHIBITORY VALIDATION SUMMARY                                  \n"
        "========================================================================================================================\n"
        " Subj | Best Model Family     | Vel Pearson r | Vel R^2 (VAF) | Angle Pearson r | Angle R^2 (VAF)| Recon Lag | M5 False Cls% | Rest Jitter% \n"
        "------------------------------------------------------------------------------------------------------------------------\n"
    )
    log_print(summary_header, f_log)

    for sub_id in COHORT:
        sub_dict = cohort_results.get(sub_id, {})
        if not sub_dict:
            continue

        best_m = max(sub_dict.keys(), key=lambda m: sub_dict[m]["r_vel"])
        bm = sub_dict[best_m]

        line = (
            f"  S{sub_id:02d} | {best_m:<21} |     {bm['r_vel']:6.4f}    |     {bm['r2_vel']:6.4f}    "
            f"|      {bm['r_ang']:6.4f}     |     {bm['r2_ang']:6.4f}     |  {bm['lag']:+5.1f} ms  "
            f"|     {bm['m5_fa']:4.2f}%    |    {bm['rj']:4.2f}%\n"
        )
        log_print(line, f_log)

    log_print("------------------------------------------------------------------------------------------------------------------------\n\n", f_log)

    # --- 10. Aggregate Model Family Comparison Table ---
    agg_header = (
        "========================================================================================================================\n"
        "                         OVERALL MODEL FAMILY PERFORMANCE (COHORT MEAN +/- STD)                                         \n"
        "========================================================================================================================\n"
        " Model Architecture    | Vel Pearson r   | Vel R^2 (VAF)   | Angle Pearson r | Angle R^2 (VAF) | M5 False Cls% | Rest Jitter  \n"
        "------------------------------------------------------------------------------------------------------------------------\n"
    )
    log_print(agg_header, f_log)

    for m_name in MODEL_FAMILIES:
        rv_l = [cohort_results[s][m_name]["r_vel"] for s in COHORT if m_name in cohort_results[s]]
        r2v_l = [cohort_results[s][m_name]["r2_vel"] for s in COHORT if m_name in cohort_results[s]]
        ra_l = [cohort_results[s][m_name]["r_ang"] for s in COHORT if m_name in cohort_results[s]]
        r2a_l = [cohort_results[s][m_name]["r2_ang"] for s in COHORT if m_name in cohort_results[s]]
        m5_fa_l = [cohort_results[s][m_name]["m5_fa"] for s in COHORT if m_name in cohort_results[s]]
        rj_l = [cohort_results[s][m_name]["rj"] for s in COHORT if m_name in cohort_results[s]]

        if not rv_l:
            continue

        agg_line = (
            f" {m_name:<22} |  {np.mean(rv_l):5.3f} +/- {np.std(rv_l):4.3f} |  {np.mean(r2v_l):5.3f} +/- {np.std(r2v_l):4.3f} |  "
            f"{np.mean(ra_l):5.3f} +/- {np.std(ra_l):4.3f}  |  {np.mean(r2a_l):5.3f} +/- {np.std(r2a_l):4.3f}  |  "
            f"{np.mean(m5_fa_l):4.2f} +/- {np.std(m5_fa_l):4.2f}% |  {np.mean(rj_l):4.2f} +/- {np.std(rj_l):4.2f}%\n"
        )
        log_print(agg_line, f_log)

    log_print("========================================================================================================================\n", f_log)
    print(f"\n[SUCCESS] Unipolar benchmark report saved to: {LOG_FILE}\n")
    f_log.close()

if __name__ == "__main__":
    main()

      CONTINUOUS sEMG UNIPOLAR CLOSING BENCHMARK: INHIBITORY M5 GROUNDING & ANGLE EXCURSION                             
 Cohort: 10 Subjects | Exclusions: S07 & S11 (Entirely) | S4 Rep 6
 Target  : Unipolar closing speed v_close >= 0 deg/s (M6 Takeoff = |v|; M5 Open = 0; M6 Return = 0; Rest = 0)
 Features: 120 Causal Features across 140 ms receptive field (-40 ms, 0 ms, +50 ms, +100 ms relative to target)
 Metrics : M6 Closing Flight (r_vel, R^2_vel, r_ang, R^2_ang) | M5 False Closing % | Rest Jitter %

>>> Subject 01 | Train: 6967 frames (M6 Takeoffs + Inhibitory M5/Return) | Test Segments: 8
>>> Subject 02 | Train: 7065 frames (M6 Takeoffs + Inhibitory M5/Return) | Test Segments: 8
>>> Subject 03 | Train: 6165 frames (M6 Takeoffs + Inhibitory M5/Return) | Test Segments: 8
>>> Subject 04 | Train: 5102 frames (M6 Takeoffs + Inhibitory M5/Return) | Test Segments: 8
>>> Subject 05 | Train: 7260 frames (M6 Takeoffs + Inhibitory M5/Return) | Test Segments: 9
>>> Subject 06 | Train: 6980 f

In [1]:
#!/usr/bin/env python3
"""
====================================================================================================
CONTINUOUS sEMG UNIPOLAR CLOSING: PARAMETRIC DELAY SWEEP (0 ms TO 100 ms)
Cohort      : Subjects 1 to 12 (Excluding S07 & S11; Inverting S05)
Sweep Grid  : tau in [0, 20, 40, 60, 80, 100] ms (0 to 10 hops at 10 ms/hop)
Focus       : Multi-Layer Perceptron (MLP_Standard & MLP_Compact)
Metrics     : Vel r, Vel R^2, Angle r, Angle R^2, Recon Lag, Total Latency, M5 False Cls%, Rest Jitter%
====================================================================================================
"""

import os
import sys
import numpy as np
import pandas as pd
from scipy import signal, stats
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import KFold

# --- 1. Cohort & Protocol Constants ---
COHORT = [1, 2, 3, 4, 5, 6, 8, 9, 10, 12]  # S07 & S11 permanently excluded
V_DEADBAND = 4.0        # deg/s: Unbiased threshold
DT_SEC = 0.010          # 10 ms hop (100 Hz cadence)
DELAY_SWEEP_MS = [0, 20, 40, 60, 80, 100]  # Parametric sweep grid
LOG_FILE = "cohort_delay_sweep_mlp_report.txt"

# Set to True: S05 needs polarity inversion in Python
INVERT_S05_IN_PYTHON = True

# --- 2. Unipolar Target & Phase Labeler ---
def construct_unipolar_closing_labels(df):
    y_close = np.zeros(len(df), dtype=np.float64)
    is_m6_takeoff = np.zeros(len(df), dtype=bool)
    is_m5_open = np.zeros(len(df), dtype=bool)

    stim = df["stimulus"].values
    rep = df["repetition"].values
    is_plat = df["is_hold_plateau"].values.astype(bool)
    v_raw = df["theta_vel_deg_s"].values

    # 1. Label M6 (Fist Clench) Takeoffs
    m6_reps = np.unique(rep[stim == 6])
    for r in m6_reps:
        trial_mask = (stim == 6) & (rep == r)
        trial_indices = np.where(trial_mask)[0]
        if len(trial_indices) == 0:
            continue

        plat_in_trial = np.where(trial_mask & is_plat)[0]
        if len(plat_in_trial) > 0:
            first_plat = plat_in_trial[0]
            takeoff_idx = [i for i in trial_indices if i < first_plat]
        else:
            cutoff = trial_indices[0] + int(0.40 * len(trial_indices))
            takeoff_idx = [i for i in trial_indices if i < cutoff]

        if len(takeoff_idx) > 0:
            is_m6_takeoff[takeoff_idx] = True
            y_close[takeoff_idx] = np.abs(v_raw[takeoff_idx])

    # 2. Label M5 (Hand Open) as Negative Inhibitory Data
    is_m5_open[stim == 5] = True

    df_labeled = df.copy()
    df_labeled["y_close"] = y_close
    df_labeled["is_m6_takeoff"] = is_m6_takeoff
    df_labeled["is_m5_open"] = is_m5_open
    return df_labeled

# --- 3. Contiguous Block Slicer ---
def find_contiguous_segments(df, max_dt=0.018, min_len=20):
    t = df["timestamp_s"].values
    if len(t) == 0:
        return []
    dt = np.diff(t)
    break_indices = np.where(dt > max_dt)[0] + 1
    segments = np.split(np.arange(len(df)), break_indices)
    return [seg for seg in segments if len(seg) >= min_len]

# --- 4. Causal Feature & Dynamic Lag Target Builder ---
def extract_delay_sweep_data(df, feat_cols, lag_ms, max_history_hops=12):
    """
    Fixed Causal Feature Buffer at clock t:
      - Tap 0: t
      - Tap 1: t - 40 ms
      - Tap 2: t - 80 ms
      - Tap 3: t - 120 ms
    Target Velocity:
      - Evaluated at t - lag_ms
    """
    lag_hops = int(round(lag_ms / (DT_SEC * 1000.0)))
    segments = find_contiguous_segments(df, min_len=max_history_hops + lag_hops + 5)
    seg_records = []

    X_raw_all = df[feat_cols].values
    y_close_all = df["y_close"].values
    theta_raw_all = df["theta_deg"].values
    is_m6_all = df["is_m6_takeoff"].values
    is_m5_all = df["is_m5_open"].values

    for seg in segments:
        X_seg = X_raw_all[seg]
        y_seg = y_close_all[seg]
        th_seg = theta_raw_all[seg]
        m6_seg = is_m6_all[seg]
        m5_seg = is_m5_all[seg]
        L = len(seg)

        X_s, y_s, th_s, m6_s, m5_s = [], [], [], [], []

        # Start loop where all causal taps are in-bounds
        start_idx = max(max_history_hops, lag_hops)
        for i in range(start_idx, L):
            target_idx = i - lag_hops  # Retrospective target: t - lag_ms

            # 4 Causal Taps spanning 120 ms relative to clock t
            tap0 = X_seg[i]               # t (Current real-time)
            tap1 = X_seg[i - 4]           # t - 40 ms
            tap2 = X_seg[i - 8]           # t - 80 ms
            tap3 = X_seg[i - 12]          # t - 120 ms

            X_s.append(np.hstack([tap0, tap1, tap2, tap3]))
            y_s.append(y_seg[target_idx])
            th_s.append(th_seg[target_idx])
            m6_s.append(m6_seg[target_idx])
            m5_s.append(m5_seg[target_idx])

        if len(X_s) > 0:
            seg_records.append({
                "X": np.array(X_s),
                "y": np.array(y_s),
                "theta": np.array(th_s),
                "is_m6_takeoff": np.array(m6_s),
                "is_m5_open": np.array(m5_s)
            })

    return seg_records

# --- 5. Timeline Partitioning ---
def partition_timeline_unipolar(df, sub_id, buffer_sec=1.0):
    test_reps = [2, 5] if sub_id == 4 else [2, 6]
    is_test = np.zeros(len(df), dtype=bool)

    for m in [5, 6]:
        for r in test_reps:
            cue_mask = (df["stimulus"] == m) & (df["repetition"] == r)
            if not np.any(cue_mask):
                continue
            t_start = df.loc[cue_mask, "timestamp_s"].min() - buffer_sec
            t_end = df.loc[cue_mask, "timestamp_s"].max() + buffer_sec
            is_test[(df["timestamp_s"] >= t_start) & (df["timestamp_s"] <= t_end)] = True

    is_corrupt = (df["repetition"] == 6).values if sub_id == 4 else np.zeros(len(df), dtype=bool)
    is_plateau = (df["is_hold_plateau"] == 1).values

    valid_mask = ~is_corrupt & ~is_plateau
    train_mask = valid_mask & ~is_test
    test_mask = valid_mask & is_test

    return train_mask, test_mask, test_reps

# --- 6. Unbiased Deadband Actuator ---
def apply_unbiased_deadband(y_raw, v_db=V_DEADBAND):
    y_cmd = np.maximum(0.0, np.copy(y_raw))
    y_cmd[y_cmd <= v_db] = 0.0
    return y_cmd

# --- 7. Dual-Domain Evaluator ---
def evaluate_delay_performance(test_records, model, scaler, gamma, lag_ms, dt=DT_SEC):
    m6_y_true, m6_y_cmd = [], []
    m6_th_true, m6_th_recon = [], []
    m5_cmds, rest_cmds = [], []

    for seg in test_records:
        X_seg = np.clip(scaler.transform(seg["X"]), -20.0, 20.0)
        y_seg = seg["y"]
        th_seg = seg["theta"]
        m6_mask = seg["is_m6_takeoff"]
        m5_mask = seg["is_m5_open"]

        y_pred = model.predict(X_seg) * gamma
        y_cmd = apply_unbiased_deadband(y_pred)

        # 1. Active Closing Excursion
        if np.any(m6_mask):
            idx_m6 = np.where(m6_mask)[0]
            yt_sub = y_seg[idx_m6]
            yp_sub = y_cmd[idx_m6]
            th_sub = th_seg[idx_m6]

            direction = np.sign(th_sub[-1] - th_sub[0]) if len(th_sub) > 1 else 1.0
            if direction == 0.0:
                direction = 1.0
            th_recon_sub = th_sub[0] + direction * np.cumsum(yp_sub * dt)

            m6_y_true.extend(yt_sub)
            m6_y_cmd.extend(yp_sub)
            m6_th_true.extend(th_sub)
            m6_th_recon.extend(th_recon_sub)

        # 2. Inhibitory Hand-Open Suppression
        if np.any(m5_mask):
            m5_cmds.extend(y_cmd[m5_mask])

        # 3. Resting Baseline Silence
        rest_mask = (~m6_mask) & (~m5_mask) & (y_seg <= 2.0)
        if np.any(rest_mask):
            rest_cmds.extend(y_cmd[rest_mask])

    yt_m6 = np.array(m6_y_true, dtype=np.float64)
    yp_m6 = np.array(m6_y_cmd, dtype=np.float64)
    th_t = np.array(m6_th_true, dtype=np.float64)
    th_p = np.array(m6_th_recon, dtype=np.float64)

    # Velocity Metrics
    if len(yt_m6) > 10 and np.std(yp_m6) > 0 and np.std(yt_m6) > 0:
        r_vel, _ = stats.pearsonr(yt_m6, yp_m6)
        sse_v = np.sum((yt_m6 - yp_m6) ** 2)
        sst_v = np.sum((yt_m6 - np.mean(yt_m6)) ** 2)
        r2_vel = max(0.0, 1.0 - (sse_v / sst_v)) if sst_v > 0 else 0.0
    else:
        r_vel, r2_vel = 0.0, 0.0

    # Angle Metrics
    if len(th_t) > 10 and np.std(th_p) > 0 and np.std(th_t) > 0:
        r_ang, _ = stats.pearsonr(th_t, th_p)
        sse_a = np.sum((th_t - th_p) ** 2)
        sst_a = np.sum((th_t - np.mean(th_t)) ** 2)
        r2_ang = max(0.0, 1.0 - (sse_a / sst_a)) if sst_a > 0 else 0.0
    else:
        r_ang, r2_ang = 0.0, 0.0

    # Cross-Correlation Reconstruction Lag
    if len(yt_m6) > 20 and np.std(yp_m6) > 0:
        yp_d = yp_m6 - np.mean(yp_m6)
        yt_d = yt_m6 - np.mean(yt_m6)
        xcorr = signal.correlate(yp_d, yt_d, mode="full")
        lags = signal.correlation_lags(len(yp_d), len(yt_d), mode="full")
        lag_mask = (lags >= -35) & (lags <= 35)
        recon_lag_ms = lags[lag_mask][np.argmax(xcorr[lag_mask])] * (dt * 1000.0)
    else:
        recon_lag_ms = 0.0

    total_latency_ms = lag_ms + recon_lag_ms

    m5_cmds = np.array(m5_cmds)
    m5_false_alarm_pct = (np.sum(m5_cmds > 0.0) / len(m5_cmds) * 100.0) if len(m5_cmds) > 0 else 0.0

    rest_cmds = np.array(rest_cmds)
    rest_jitter_pct = (np.sum(rest_cmds > 0.0) / len(rest_cmds) * 100.0) if len(rest_cmds) > 0 else 0.0

    return r_vel, r2_vel, r_ang, r2_ang, recon_lag_ms, total_latency_ms, m5_false_alarm_pct, rest_jitter_pct

def log_print(msg, file_handle):
    print(msg, end="")
    file_handle.write(msg)

# --- 8. Main Sweep Loop ---
def main():
    f_log = open(LOG_FILE, "w", encoding="utf-8")

    header = (
        "========================================================================================================================\n"
        "      PARAMETRIC DELAY SWEEP ON UNIPOLAR CLOSING: NEURAL NETWORK (MLP) PERFORMANCE VS. INDUCED DELAY                   \n"
        "========================================================================================================================\n"
        f" Cohort     : {len(COHORT)} Subjects | Exclusions: S07 & S11 (Entirely) | S4 Rep 6\n"
        f" Delay Grid : {DELAY_SWEEP_MS} ms\n"
        " Networks   : MLP_Standard (96, 48) and MLP_Compact (32, 16)\n"
        " Goal       : Identify optimal latency tradeoff where Angle R^2 is maximized before delay is felt\n"
        "========================================================================================================================\n\n"
    )
    log_print(header, f_log)

    # sweep_results[model_name][delay_ms][sub_id] = metrics
    sweep_results = {
        "MLP_Standard": {d: {} for d in DELAY_SWEEP_MS},
        "MLP_Compact":  {d: {} for d in DELAY_SWEEP_MS}
    }

    for sub_id in COHORT:
        csv_path = os.path.join("datasets", "DB2", f"DB2_s{sub_id}", f"S{sub_id}_E1_A1_dynamic_velocity_features.csv")
        if not os.path.exists(csv_path):
            csv_path = f"S{sub_id}_E1_A1_dynamic_velocity_features.csv"

        if not os.path.exists(csv_path):
            log_print(f"[WARN] Subject {sub_id:02d}: CSV not found. Skipping.\n", f_log)
            continue

        df_raw = pd.read_csv(csv_path)

        if sub_id == 5 and INVERT_S05_IN_PYTHON:
            edc_cols = [c for c in df_raw.columns if c.startswith("EDC_")]
            fds_cols = [c for c in df_raw.columns if c.startswith("FDS_")]
            for e_col, f_col in zip(edc_cols, fds_cols):
                df_raw[e_col], df_raw[f_col] = df_raw[f_col].copy(), df_raw[e_col].copy()
            for ndi_col in ["NDI_MAV", "NDI_RMS", "NDI_WL"]:
                if ndi_col in df_raw.columns:
                    df_raw[ndi_col] = -df_raw[ndi_col]

        df_labeled = construct_unipolar_closing_labels(df_raw)
        train_mask, test_mask, _ = partition_timeline_unipolar(df_labeled, sub_id, buffer_sec=1.0)
        train_df = df_labeled[train_mask].copy()
        test_df = df_labeled[test_mask].copy()

        meta_cols = ["timestamp_s", "theta_deg", "theta_vel_deg_s", "stimulus", "repetition", 
                     "is_hold_plateau", "y_close", "is_m6_takeoff", "is_m5_open"]
        feat_cols = [c for c in df_raw.columns if c not in meta_cols]

        print(f"\n>>> Processing Subject {sub_id:02d} across {len(DELAY_SWEEP_MS)} delay points...")

        for lag_ms in DELAY_SWEEP_MS:
            train_records = extract_delay_sweep_data(train_df, feat_cols, lag_ms)
            test_records = extract_delay_sweep_data(test_df, feat_cols, lag_ms)

            if len(train_records) == 0 or len(test_records) == 0:
                continue

            X_tr = np.vstack([rec["X"] for rec in train_records])
            y_tr = np.concatenate([rec["y"] for rec in train_records])

            scaler = StandardScaler()
            X_tr_z = scaler.fit_transform(X_tr)

            models = {
                "MLP_Standard": MLPRegressor(
                    hidden_layer_sizes=(96, 48), activation="relu", solver="adam", 
                    alpha=0.15, learning_rate_init=0.002, max_iter=200, 
                    early_stopping=True, n_iter_no_change=12, random_state=42
                ),
                "MLP_Compact": MLPRegressor(
                    hidden_layer_sizes=(32, 16), activation="relu", solver="adam", 
                    alpha=0.20, learning_rate_init=0.003, max_iter=200, 
                    early_stopping=True, n_iter_no_change=12, random_state=42
                )
            }

            for m_name, model in models.items():
                model.fit(X_tr_z, y_tr)

                # Variance gain on active closing frames
                active_tr = y_tr > 0.0
                if np.any(active_tr):
                    y_pred_act = np.maximum(0.0, model.predict(X_tr_z[active_tr]))
                    denom = np.sum(y_pred_act ** 2)
                    gamma = np.sum(y_tr[active_tr] * y_pred_act) / denom if denom > 1e-6 else 1.0
                else:
                    gamma = 1.0
                gamma = np.clip(gamma, 0.9, 1.4)

                rv, r2v, ra, r2a, r_lag, tot_lat, m5_fa, rj = evaluate_delay_performance(
                    test_records, model, scaler, gamma, lag_ms
                )

                sweep_results[m_name][lag_ms][sub_id] = {
                    "r_vel": rv, "r2_vel": r2v,
                    "r_ang": ra, "r2_ang": r2a,
                    "recon_lag": r_lag, "total_latency": tot_lat,
                    "m5_fa": m5_fa, "rj": rj
                }

    # --- 9. Format Aggregate Delay Curve Tables ---
    for m_name in ["MLP_Standard", "MLP_Compact"]:
        table_header = (
            f"\n========================================================================================================================\n"
            f"               COHORT AGGREGATE DELAY CURVE: {m_name.upper()} (MEAN +/- STD ACROSS 10 SUBJECTS)                       \n"
            f"========================================================================================================================\n"
            f" Induced Delay | Vel Pearson r   | Angle Pearson r | Angle R^2 (VAF) | Recon Lag (ms) | Total Latency (ms)| M5 False Cls% | Rest Jitter  \n"
            f"------------------------------------------------------------------------------------------------------------------------\n"
        )
        log_print(table_header, f_log)

        for lag_ms in DELAY_SWEEP_MS:
            res_dict = sweep_results[m_name][lag_ms]
            subs_available = [s for s in COHORT if s in res_dict]

            if not subs_available:
                continue

            rv_l  = [res_dict[s]["r_vel"] for s in subs_available]
            ra_l  = [res_dict[s]["r_ang"] for s in subs_available]
            r2a_l = [res_dict[s]["r2_ang"] for s in subs_available]
            rl_l  = [res_dict[s]["recon_lag"] for s in subs_available]
            tot_l = [res_dict[s]["total_latency"] for s in subs_available]
            m5_l  = [res_dict[s]["m5_fa"] for s in subs_available]
            rj_l  = [res_dict[s]["rj"] for s in subs_available]

            line = (
                f"  tau = {lag_ms:3d} ms |  {np.mean(rv_l):5.3f} +/- {np.std(rv_l):4.3f} |  "
                f"{np.mean(ra_l):5.3f} +/- {np.std(ra_l):4.3f} |  "
                f"{np.mean(r2a_l):5.3f} +/- {np.std(r2a_l):4.3f} | "
                f"{np.mean(rl_l):+5.1f} +/- {np.std(rl_l):4.1f} |  "
                f"{np.mean(tot_l):5.1f} +/- {np.std(tot_l):4.1f} ms |  "
                f"{np.mean(m5_l):4.2f} +/- {np.std(m5_l):4.2f}% |  "
                f"{np.mean(rj_l):4.2f} +/- {np.std(rj_l):4.2f}%\n"
            )
            log_print(line, f_log)

        log_print("========================================================================================================================\n", f_log)

    # --- 10. Subject-by-Subject Detailed Breakdown for MLP_Standard ---
    sub_header = (
        "\n========================================================================================================================\n"
        "                  DETAILED SUBJECT BREAKDOWN ACROSS DELAYS: MLP_STANDARD (ANGLE R^2 / TOTAL LATENCY)                    \n"
        "========================================================================================================================\n"
        " Subj | tau = 0 ms       | tau = 20 ms      | tau = 40 ms      | tau = 60 ms      | tau = 80 ms      | tau = 100 ms     \n"
        "------------------------------------------------------------------------------------------------------------------------\n"
    )
    log_print(sub_header, f_log)

    for sub_id in COHORT:
        row = f"  S{sub_id:02d} |"
        for lag_ms in DELAY_SWEEP_MS:
            if sub_id in sweep_results["MLP_Standard"][lag_ms]:
                d = sweep_results["MLP_Standard"][lag_ms][sub_id]
                cell = f" {d['r2_ang']:5.2f} ({d['total_latency']:+3.0f}ms)"
            else:
                cell = "   N/A       "
            row += f" {cell:<16} |"
        row = row.rstrip("|") + "\n"
        log_print(row, f_log)

    log_print("========================================================================================================================\n", f_log)
    print(f"\n[SUCCESS] Delay sweep report successfully generated at: {LOG_FILE}\n")
    f_log.close()

if __name__ == "__main__":
    main()

      PARAMETRIC DELAY SWEEP ON UNIPOLAR CLOSING: NEURAL NETWORK (MLP) PERFORMANCE VS. INDUCED DELAY                   
 Cohort     : 10 Subjects | Exclusions: S07 & S11 (Entirely) | S4 Rep 6
 Delay Grid : [0, 20, 40, 60, 80, 100] ms
 Networks   : MLP_Standard (96, 48) and MLP_Compact (32, 16)
 Goal       : Identify optimal latency tradeoff where Angle R^2 is maximized before delay is felt


>>> Processing Subject 01 across 6 delay points...

>>> Processing Subject 02 across 6 delay points...

>>> Processing Subject 03 across 6 delay points...

>>> Processing Subject 04 across 6 delay points...

>>> Processing Subject 05 across 6 delay points...

>>> Processing Subject 06 across 6 delay points...

>>> Processing Subject 08 across 6 delay points...

>>> Processing Subject 09 across 6 delay points...

>>> Processing Subject 10 across 6 delay points...

>>> Processing Subject 12 across 6 delay points...

               COHORT AGGREGATE DELAY CURVE: MLP_STANDARD (MEAN +/- STD ACROSS 10 SUB

In [1]:
#!/usr/bin/env python3
"""
====================================================================================================
CONTINUOUS sEMG UNIPOLAR CLOSING: ARCHITECTURE SEARCH & INDEPENDENT FIX ABLATION
Cohort      : Subjects 1 to 12 (Excluding S07 & S11; Inverting S05)
Sweep Mode  : 
  Part 1: Architecture Comparison (MLP_Standard, MLP_Compact, MLP_DeepNarrow, TwoHeaded, Residual)
  Part 2: Independent Fix Ablation Matrix (Buffer Pruning, Smooth Deadband, Quench, Peak Expansion)
Target Lag  : tau = 60 ms
Cadence     : 10 ms hop (100 Hz)
====================================================================================================
"""

import os
import sys
import numpy as np
import pandas as pd
from scipy import signal, stats
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.neural_network import MLPRegressor, MLPClassifier

# --- 1. Cohort & Protocol Constants ---
COHORT = [1, 2, 3, 4, 5, 6, 8, 9, 10, 12]
V_DEADBAND = 4.0        # deg/s
DT_SEC = 0.010          # 10 ms hop
TAU_LAG_MS = 60         # 60 ms retrospective lag
LAG_HOPS = 6            # 60 ms / 10 ms
POST_TAKEOFF_BUFFER_HOPS = 50  # 500 ms post-takeoff landing window to evaluate smearing
LOG_FILE = "cohort_advanced_fixes_report.txt"

INVERT_S05_IN_PYTHON = True

# --- 2. Unipolar Labeling ---
def construct_unipolar_closing_labels(df):
    y_close = np.zeros(len(df), dtype=np.float64)
    is_m6_takeoff = np.zeros(len(df), dtype=bool)
    is_m6_stroke = np.zeros(len(df), dtype=bool)
    is_m5_open = np.zeros(len(df), dtype=bool)

    stim = df["stimulus"].values
    rep = df["repetition"].values
    is_plat = df["is_hold_plateau"].values.astype(bool)
    v_raw = df["theta_vel_deg_s"].values

    m6_reps = np.unique(rep[stim == 6])
    for r in m6_reps:
        trial_mask = (stim == 6) & (rep == r)
        trial_indices = np.where(trial_mask)[0]
        if len(trial_indices) == 0:
            continue

        plat_in_trial = np.where(trial_mask & is_plat)[0]
        if len(plat_in_trial) > 0:
            first_plat = plat_in_trial[0]
            takeoff_idx = [i for i in trial_indices if i < first_plat]
        else:
            cutoff = trial_indices[0] + int(0.40 * len(trial_indices))
            takeoff_idx = [i for i in trial_indices if i < cutoff]

        if len(takeoff_idx) > 0:
            is_m6_takeoff[takeoff_idx] = True
            y_close[takeoff_idx] = np.abs(v_raw[takeoff_idx])
            
            # Stroke window = Takeoff + 500 ms landing settle
            end_stroke = min(len(df), takeoff_idx[-1] + POST_TAKEOFF_BUFFER_HOPS)
            is_m6_stroke[takeoff_idx[0]:end_stroke] = True

    is_m5_open[stim == 5] = True

    df_labeled = df.copy()
    df_labeled["y_close"] = y_close
    df_labeled["is_m6_takeoff"] = is_m6_takeoff
    df_labeled["is_m6_stroke"] = is_m6_stroke
    df_labeled["is_m5_open"] = is_m5_open
    return df_labeled

def find_contiguous_segments(df, max_dt=0.018, min_len=24):
    t = df["timestamp_s"].values
    if len(t) == 0:
        return []
    dt = np.diff(t)
    break_indices = np.where(dt > max_dt)[0] + 1
    segments = np.split(np.arange(len(df)), break_indices)
    return [seg for seg in segments if len(seg) >= min_len]

# --- 3. Retrospective Context Builder ---
def extract_retrospective_data(df, feat_cols, tight_buffer=False):
    tap_offsets = [0, 2, 4, 6] if tight_buffer else [0, 4, 8, 12]
    max_hist = max(tap_offsets)
    dt_deriv = tap_offsets[1] * DT_SEC  # Elapsed seconds for derivative normalization

    segments = find_contiguous_segments(df, min_len=max_hist + LAG_HOPS + 5)
    seg_records = []

    X_raw_all = df[feat_cols].values
    y_close_all = df["y_close"].values
    theta_raw_all = df["theta_deg"].values
    is_m6_all = df["is_m6_takeoff"].values
    is_stroke_all = df["is_m6_stroke"].values
    is_m5_all = df["is_m5_open"].values

    fds_idx = feat_cols.index("FDS_MAV")
    edc_idx = feat_cols.index("EDC_MAV")

    for seg in segments:
        X_seg = X_raw_all[seg]
        y_seg = y_close_all[seg]
        th_seg = theta_raw_all[seg]
        m6_seg = is_m6_all[seg]
        strk_seg = is_stroke_all[seg]
        m5_seg = is_m5_all[seg]
        L = len(seg)

        X_s, y_s, th_s, m6_s, strk_s, m5_s = [], [], [], [], [], []
        fds_t_s, edc_t_s, delta_fds_s = [], [], []

        start_idx = max(max_hist, LAG_HOPS)
        for i in range(start_idx, L):
            target_idx = i - LAG_HOPS

            taps = [X_seg[i - k] for k in tap_offsets]
            X_s.append(np.hstack(taps))
            y_s.append(y_seg[target_idx])
            th_s.append(th_seg[target_idx])
            m6_s.append(m6_seg[target_idx])
            strk_s.append(strk_seg[target_idx])
            m5_s.append(m5_seg[target_idx])

            fds_curr = X_seg[i, fds_idx]
            fds_past = X_seg[i - tap_offsets[1], fds_idx]
            fds_t_s.append(fds_curr)
            edc_t_s.append(X_seg[i, edc_idx])
            # Normalized recruitment derivative (d/dt MAV)
            delta_fds_s.append((fds_curr - fds_past) / (dt_deriv * 100.0))

        if len(X_s) > 0:
            seg_records.append({
                "X": np.array(X_s),
                "y": np.array(y_s),
                "theta": np.array(th_s),
                "is_m6_takeoff": np.array(m6_s),
                "is_m6_stroke": np.array(strk_s),
                "is_m5_open": np.array(m5_s),
                "fds_t": np.array(fds_t_s),
                "edc_t": np.array(edc_t_s),
                "delta_fds": np.array(delta_fds_s)
            })

    return seg_records

# --- 4. Actuator Post-Processing Pipeline ---
def process_actuation(y_raw, fds_t, edc_t, delta_fds,
                      use_smooth_db=False, use_quench=False, 
                      use_peak_exp=False, use_asym_gate=False,
                      v_db=V_DEADBAND):
    y = np.maximum(0.0, np.copy(y_raw))
    fds_safe = np.maximum(0.0, np.abs(fds_t))
    edc_safe = np.maximum(0.0, np.abs(edc_t))

    # Fix 3: Active Relaxation Quench (Strictly bounded attenuation in [0.0, 1.0])
    if use_quench:
        neg_deriv = np.minimum(0.0, delta_fds)
        norm_deriv = np.clip(neg_deriv / (fds_safe + 1e-4), -10.0, 0.0)
        quench_gain = np.exp(1.5 * norm_deriv)
        y = y * quench_gain

    # Fix 5: Continuous Asymmetric Reciprocal Gate (Bounded in [0.0, 1.0])
    if use_asym_gate:
        recip_ratio = edc_safe / (1.5 * np.maximum(fds_safe, 1e-4))
        recip_gain = np.clip(1.0 - recip_ratio, 0.0, 1.0)
        y = y * recip_gain

    # Fix 2: Smooth Deadband vs. Hard Deadband
    if use_smooth_db:
        mask = y > v_db
        y_cmd = np.zeros_like(y)
        y_cmd[mask] = y[mask] * ((y[mask] - v_db) / y[mask])
    else:
        y_cmd = np.copy(y)
        y_cmd[y_cmd <= v_db] = 0.0

    # Fix 4: Power-Law Dynamic Peak Expansion
    if use_peak_exp:
        mask_fast = y_cmd > 12.0
        v_ref = 140.0
        y_cmd[mask_fast] = v_ref * ((y_cmd[mask_fast] / v_ref) ** 1.18)

    return y_cmd

# --- 5. Timeline Partitioning ---
def partition_timeline_unipolar(df, sub_id, buffer_sec=1.0):
    test_reps = [2, 5] if sub_id == 4 else [2, 6]
    is_test = np.zeros(len(df), dtype=bool)

    for m in [5, 6]:
        for r in test_reps:
            cue_mask = (df["stimulus"] == m) & (df["repetition"] == r)
            if not np.any(cue_mask):
                continue
            t_start = df.loc[cue_mask, "timestamp_s"].min() - buffer_sec
            t_end = df.loc[cue_mask, "timestamp_s"].max() + buffer_sec
            is_test[(df["timestamp_s"] >= t_start) & (df["timestamp_s"] <= t_end)] = True

    is_corrupt = (df["repetition"] == 6).values if sub_id == 4 else np.zeros(len(df), dtype=bool)
    is_plateau = (df["is_hold_plateau"] == 1).values

    # Train excludes full plateaus; Test preserves landing settle to evaluate smearing
    train_mask = ~is_corrupt & ~is_plateau & ~is_test
    test_mask = ~is_corrupt & (df["is_m6_stroke"] | df["is_m5_open"] | ~is_plateau) & is_test

    return train_mask, test_mask

# --- 6. Extended Evaluation Suite ---
def evaluate_pipeline(test_records, predict_fn, gamma,
                      use_smooth_db=False, use_quench=False, 
                      use_peak_exp=False, use_asym_gate=False, dt=DT_SEC):
    m6_y_true, m6_y_cmd = [], []
    m6_th_true, m6_th_recon = [], []
    m5_cmds, rest_cmds = [], []

    excursion_ratios = []
    duration_ratios = []
    peak_ratios = []

    for seg in test_records:
        y_pred = predict_fn(seg["X"]) * gamma
        y_cmd = process_actuation(
            y_pred, seg["fds_t"], seg["edc_t"], seg["delta_fds"],
            use_smooth_db=use_smooth_db, use_quench=use_quench,
            use_peak_exp=use_peak_exp, use_asym_gate=use_asym_gate
        )

        y_seg = seg["y"]
        th_seg = seg["theta"]
        m6_takeoff_mask = seg["is_m6_takeoff"]
        m6_stroke_mask = seg["is_m6_stroke"]
        m5_mask = seg["is_m5_open"]

        # 1. Trajectory Correlation on Active Takeoff
        if np.any(m6_takeoff_mask):
            idx_tk = np.where(m6_takeoff_mask)[0]
            yt_sub = y_seg[idx_tk]
            yp_sub = y_cmd[idx_tk]
            th_sub = th_seg[idx_tk]

            direction = np.sign(th_sub[-1] - th_sub[0]) if len(th_sub) > 1 else 1.0
            if direction == 0.0:
                direction = 1.0
            th_recon_sub = th_sub[0] + direction * np.cumsum(yp_sub * dt)

            m6_y_true.extend(yt_sub)
            m6_y_cmd.extend(yp_sub)
            m6_th_true.extend(th_sub)
            m6_th_recon.extend(th_recon_sub)

            peak_t = np.max(yt_sub)
            peak_p = np.max(yp_sub)
            if peak_t > 10.0:
                peak_ratios.append((peak_p / peak_t) * 100.0)

        # 2. Excursion & Duration Metrics on Full Stroke (Takeoff + Landing Settle)
        if np.any(m6_stroke_mask):
            idx_strk = np.where(m6_stroke_mask)[0]
            yp_strk = y_cmd[idx_strk]
            yt_strk = y_seg[idx_strk]
            th_strk = th_seg[idx_strk]

            delta_th_true = np.abs(np.max(th_strk) - np.min(th_strk))
            delta_th_pred = np.sum(yp_strk * dt)
            if delta_th_true > 5.0:
                excursion_ratios.append((delta_th_pred / delta_th_true) * 100.0)

            dur_true = np.sum(yt_strk > V_DEADBAND) * dt
            dur_pred = np.sum(yp_strk > V_DEADBAND) * dt
            if dur_true > 0.05:
                duration_ratios.append((dur_pred / dur_true) * 100.0)

        if np.any(m5_mask):
            m5_cmds.extend(y_cmd[m5_mask])

        rest_mask = (~m6_stroke_mask) & (~m5_mask) & (y_seg <= 2.0)
        if np.any(rest_mask):
            rest_cmds.extend(y_cmd[rest_mask])

    yt_m6 = np.array(m6_y_true, dtype=np.float64)
    yp_m6 = np.array(m6_y_cmd, dtype=np.float64)
    th_t = np.array(m6_th_true, dtype=np.float64)
    th_p = np.array(m6_th_recon, dtype=np.float64)

    if len(yt_m6) > 10 and np.std(yp_m6) > 0 and np.std(yt_m6) > 0:
        r_vel, _ = stats.pearsonr(yt_m6, yp_m6)
        r2_vel = max(0.0, 1.0 - (np.sum((yt_m6 - yp_m6)**2) / np.sum((yt_m6 - np.mean(yt_m6))**2)))
    else:
        r_vel, r2_vel = 0.0, 0.0

    if len(th_t) > 10 and np.std(th_p) > 0 and np.std(th_t) > 0:
        r_ang, _ = stats.pearsonr(th_t, th_p)
        r2_ang = max(0.0, 1.0 - (np.sum((th_t - th_p)**2) / np.sum((th_t - np.mean(th_t))**2)))
    else:
        r_ang, r2_ang = 0.0, 0.0

    if len(yt_m6) > 20 and np.std(yp_m6) > 0 and np.std(yt_m6) > 0:
        xcorr = signal.correlate(yp_m6 - np.mean(yp_m6), yt_m6 - np.mean(yt_m6), mode="full")
        lags = signal.correlation_lags(len(yp_m6), len(yt_m6), mode="full")
        lag_mask = (lags >= -35) & (lags <= 35)
        recon_lag_ms = lags[lag_mask][np.argmax(xcorr[lag_mask])] * (dt * 1000.0)
    else:
        recon_lag_ms = 0.0

    total_lat_ms = TAU_LAG_MS + recon_lag_ms

    m5_cmds = np.array(m5_cmds)
    m5_fa_pct = (np.sum(m5_cmds > 0.0) / len(m5_cmds) * 100.0) if len(m5_cmds) > 0 else 0.0

    rest_cmds = np.array(rest_cmds)
    rj_pct = (np.sum(rest_cmds > 0.0) / len(rest_cmds) * 100.0) if len(rest_cmds) > 0 else 0.0

    mean_excursion_pct = np.mean(excursion_ratios) if excursion_ratios else 0.0
    mean_duration_pct = np.mean(duration_ratios) if duration_ratios else 0.0
    mean_peak_pct = np.mean(peak_ratios) if peak_ratios else 0.0

    return {
        "r_vel": r_vel, "r2_vel": r2_vel,
        "r_ang": r_ang, "r2_ang": r2_ang,
        "recon_lag": recon_lag_ms, "total_lat": total_lat_ms,
        "excursion_pct": mean_excursion_pct,
        "duration_pct": mean_duration_pct,
        "peak_pct": mean_peak_pct,
        "m5_fa": m5_fa_pct, "rj": rj_pct
    }

def log_print(msg, fh):
    print(msg, end="")
    fh.write(msg)

# --- 7. Main Benchmark Harness ---
def main():
    fh = open(LOG_FILE, "w", encoding="utf-8")

    header = (
        "========================================================================================================================\n"
        "      CONTINUOUS sEMG: ARCHITECTURE SEARCH & INDEPENDENT FIX ABLATION BENCHMARK (tau = 60 ms)                           \n"
        "========================================================================================================================\n"
        f" Cohort      : {len(COHORT)} Subjects | Exclusions: S07 & S11 | S05 Polarity Swapped\n"
        " Focus Goals : \n"
        "   1. Restore Excursion Accuracy (target: 90% - 105% of human degrees)\n"
        "   2. Eliminate Duration Smearing (target: 95% - 110% of human duration)\n"
        "   3. Maintain Extensor Reciprocal Inhibition (M5 False Closing < 5%)\n"
        "========================================================================================================================\n\n"
    )
    log_print(header, fh)

    subject_data = {}
    for sub_id in COHORT:
        csv_path = os.path.join("datasets", "DB2", f"DB2_s{sub_id}", f"S{sub_id}_E1_A1_dynamic_velocity_features.csv")
        if not os.path.exists(csv_path):
            csv_path = f"S{sub_id}_E1_A1_dynamic_velocity_features.csv"
        if not os.path.exists(csv_path):
            continue

        df_raw = pd.read_csv(csv_path)

        if sub_id == 5 and INVERT_S05_IN_PYTHON:
            edc_cols = [c for c in df_raw.columns if c.startswith("EDC_")]
            for e_col in edc_cols:
                f_col = e_col.replace("EDC_", "FDS_")
                if f_col in df_raw.columns:
                    df_raw[e_col], df_raw[f_col] = df_raw[f_col].copy(), df_raw[e_col].copy()
            for ndi_col in ["NDI_MAV", "NDI_RMS", "NDI_WL", "antagonist_diff_ratio"]:
                if ndi_col in df_raw.columns:
                    df_raw[ndi_col] = -df_raw[ndi_col]

        df_labeled = construct_unipolar_closing_labels(df_raw)
        meta_cols = ["timestamp_s", "theta_deg", "theta_vel_deg_s", "stimulus", "repetition", 
                     "is_hold_plateau", "y_close", "is_m6_takeoff", "is_m6_stroke", "is_m5_open"]
        feat_cols = [c for c in df_raw.columns if c not in meta_cols]

        tr_mask, te_mask = partition_timeline_unipolar(df_labeled, sub_id)
        subject_data[sub_id] = {
            "train_df": df_labeled[tr_mask].copy(),
            "test_df": df_labeled[te_mask].copy(),
            "feat_cols": feat_cols
        }

    # ==================================================================================================
    # PART 1: ARCHITECTURE SEARCH (Under Standard Baseline Conditions)
    # ==================================================================================================
    arch_header = (
        "========================================================================================================================\n"
        "                            PART 1: ARCHITECTURE BENCHMARK (STANDARD CONDITIONS)                                        \n"
        "========================================================================================================================\n"
        " Architecture         | Excursion % | Duration %  | Angle R^2   | Angle r     | Latency (ms) | M5 False % | Rest Jitter  \n"
        "------------------------------------------------------------------------------------------------------------------------\n"
    )
    log_print(arch_header, fh)

    arch_names = ["MLP_Standard", "MLP_Compact", "MLP_DeepNarrow", "TwoHeaded_Gated", "Residual_MLP"]
    arch_results = {a: [] for a in arch_names}

    for sub_id, d in subject_data.items():
        tr_rec = extract_retrospective_data(d["train_df"], d["feat_cols"], tight_buffer=False)
        te_rec = extract_retrospective_data(d["test_df"], d["feat_cols"], tight_buffer=False)
        if not tr_rec or not te_rec:
            continue

        X_tr = np.vstack([r["X"] for r in tr_rec])
        y_tr = np.concatenate([r["y"] for r in tr_rec])
        scaler = StandardScaler().fit(X_tr)
        X_tr_z = scaler.transform(X_tr)

        te_rec_sc = []
        for r in te_rec:
            r_c = dict(r)
            r_c["X"] = np.clip(scaler.transform(r["X"]), -20.0, 20.0)
            te_rec_sc.append(r_c)

        active_tr = y_tr > 0.0

        for arch in arch_names:
            if arch == "MLP_Standard":
                m = MLPRegressor(hidden_layer_sizes=(96, 48), activation="relu", alpha=0.15, 
                                 early_stopping=True, n_iter_no_change=10, random_state=42, max_iter=200)
                m.fit(X_tr_z, y_tr)
                fn = lambda X, model=m: np.maximum(0.0, model.predict(X))

            elif arch == "MLP_Compact":
                m = MLPRegressor(hidden_layer_sizes=(32, 16), activation="relu", alpha=0.20, 
                                 early_stopping=True, n_iter_no_change=10, random_state=42, max_iter=200)
                m.fit(X_tr_z, y_tr)
                fn = lambda X, model=m: np.maximum(0.0, model.predict(X))

            elif arch == "MLP_DeepNarrow":
                m = MLPRegressor(hidden_layer_sizes=(48, 24, 12), activation="relu", alpha=0.20, 
                                 early_stopping=True, n_iter_no_change=10, random_state=42, max_iter=200)
                m.fit(X_tr_z, y_tr)
                fn = lambda X, model=m: np.maximum(0.0, model.predict(X))

            elif arch == "TwoHeaded_Gated":
                gate = MLPClassifier(hidden_layer_sizes=(32, 16), activation="relu", alpha=0.15, 
                                     early_stopping=True, n_iter_no_change=10, random_state=42, max_iter=200)
                gate.fit(X_tr_z, (y_tr > 0.0).astype(int))
                reg = MLPRegressor(hidden_layer_sizes=(32, 16), activation="relu", alpha=0.15, 
                                   early_stopping=True, n_iter_no_change=10, random_state=42, max_iter=200)
                if np.sum(active_tr) > 20:
                    reg.fit(X_tr_z[active_tr], y_tr[active_tr])
                else:
                    reg.fit(X_tr_z, y_tr)

                def make_gated_fn(g=gate, r=reg):
                    def _pred(X):
                        p = g.predict_proba(X)[:, 1] if g.classes_.shape[0] > 1 else np.zeros(len(X))
                        mag = np.maximum(0.0, r.predict(X))
                        return p * mag
                    return _pred
                fn = make_gated_fn()

            elif arch == "Residual_MLP":
                ridge = Ridge(alpha=10.0)
                ridge.fit(X_tr_z, y_tr)
                y_ridge_tr = ridge.predict(X_tr_z)
                res_tr = y_tr - y_ridge_tr
                mlp_res = MLPRegressor(hidden_layer_sizes=(32, 16), activation="relu", alpha=0.20, 
                                       early_stopping=True, n_iter_no_change=10, random_state=42, max_iter=200)
                mlp_res.fit(X_tr_z, res_tr)

                def make_res_fn(rd=ridge, mr=mlp_res):
                    def _pred(X):
                        return np.maximum(0.0, rd.predict(X) + mr.predict(X))
                    return _pred
                fn = make_res_fn()

            if np.any(active_tr):
                yp_act = fn(X_tr_z[active_tr])
                denom = np.sum(yp_act ** 2)
                gamma = np.clip(np.sum(y_tr[active_tr] * yp_act) / denom if denom > 1e-6 else 1.0, 0.9, 1.4)
            else:
                gamma = 1.0

            res = evaluate_pipeline(te_rec_sc, fn, gamma)
            arch_results[arch].append(res)

    for arch in arch_names:
        l = arch_results[arch]
        line = (
            f" {arch:<20} |  {np.mean([x['excursion_pct'] for x in l]):5.1f} +/- {np.std([x['excursion_pct'] for x in l]):4.1f}% | "
            f" {np.mean([x['duration_pct'] for x in l]):5.1f} +/- {np.std([x['duration_pct'] for x in l]):4.1f}% | "
            f" {np.mean([x['r2_ang'] for x in l]):5.3f} +/- {np.std([x['r2_ang'] for x in l]):4.3f} | "
            f" {np.mean([x['r_ang'] for x in l]):5.3f} +/- {np.std([x['r_ang'] for x in l]):4.3f} | "
            f" {np.mean([x['total_lat'] for x in l]):5.1f} +/- {np.std([x['total_lat'] for x in l]):4.1f} ms | "
            f" {np.mean([x['m5_fa'] for x in l]):4.2f}% |  {np.mean([x['rj'] for x in l]):4.2f}%\n"
        )
        log_print(line, fh)

    log_print("========================================================================================================================\n\n", fh)

    # ==================================================================================================
    # PART 2: INDEPENDENT FIX ABLATION MATRIX (On MLP_Compact Baseline)
    # ==================================================================================================
    abl_header = (
        "========================================================================================================================\n"
        "                  PART 2: INDEPENDENT FIX ABLATION MATRIX (EVALUATED ON MLP_COMPACT)                                    \n"
        "========================================================================================================================\n"
        " Condition / Fix Mode | Excursion % | Duration %  | Angle R^2   | Peak Speed% | Latency (ms) | M5 False % | Rest Jitter  \n"
        "------------------------------------------------------------------------------------------------------------------------\n"
    )
    log_print(abl_header, fh)

    ablation_modes = [
        ("0. Baseline (No Fixes)",      False, False, False, False, False),
        ("1. + Tight Buffer (60ms)",    True,  False, False, False, False),
        ("2. + Smooth Deadband",        False, True,  False, False, False),
        ("3. + Active Quench",          False, False, True,  False, False),
        ("4. + Peak Expansion (Power)", False, False, False, True,  False),
        ("5. + Asymmetric Gate",        False, False, False, False, True ),
        ("6. ALL FIXES COMBINED",       True,  True,  True,  True,  True )
    ]

    for mode_name, use_tight, use_sdb, use_qn, use_pexp, use_asym in ablation_modes:
        mode_res = []

        for sub_id, d in subject_data.items():
            tr_rec = extract_retrospective_data(d["train_df"], d["feat_cols"], tight_buffer=use_tight)
            te_rec = extract_retrospective_data(d["test_df"], d["feat_cols"], tight_buffer=use_tight)
            if not tr_rec or not te_rec:
                continue

            X_tr = np.vstack([r["X"] for r in tr_rec])
            y_tr = np.concatenate([r["y"] for r in tr_rec])
            scaler = StandardScaler().fit(X_tr)
            X_tr_z = scaler.transform(X_tr)

            te_rec_sc = []
            for r in te_rec:
                r_c = dict(r)
                r_c["X"] = np.clip(scaler.transform(r["X"]), -20.0, 20.0)
                te_rec_sc.append(r_c)

            m = MLPRegressor(hidden_layer_sizes=(32, 16), activation="relu", alpha=0.20, 
                             early_stopping=True, n_iter_no_change=10, random_state=42, max_iter=200)
            m.fit(X_tr_z, y_tr)
            fn = lambda X, model=m: np.maximum(0.0, model.predict(X))

            active_tr = y_tr > 0.0
            if np.any(active_tr):
                yp_act = fn(X_tr_z[active_tr])
                denom = np.sum(yp_act ** 2)
                gamma = np.clip(np.sum(y_tr[active_tr] * yp_act) / denom if denom > 1e-6 else 1.0, 0.9, 1.4)
            else:
                gamma = 1.0

            res = evaluate_pipeline(
                te_rec_sc, fn, gamma,
                use_smooth_db=use_sdb, use_quench=use_qn,
                use_peak_exp=use_pexp, use_asym_gate=use_asym
            )
            mode_res.append(res)

        line = (
            f" {mode_name:<21} |  {np.mean([x['excursion_pct'] for x in mode_res]):5.1f} +/- {np.std([x['excursion_pct'] for x in mode_res]):4.1f}% | "
            f" {np.mean([x['duration_pct'] for x in mode_res]):5.1f} +/- {np.std([x['duration_pct'] for x in mode_res]):4.1f}% | "
            f" {np.mean([x['r2_ang'] for x in mode_res]):5.3f} +/- {np.std([x['r2_ang'] for x in mode_res]):4.3f} | "
            f" {np.mean([x['peak_pct'] for x in mode_res]):5.1f} +/- {np.std([x['peak_pct'] for x in mode_res]):4.1f}% | "
            f" {np.mean([x['total_lat'] for x in mode_res]):5.1f} +/- {np.std([x['total_lat'] for x in mode_res]):4.1f} ms | "
            f" {np.mean([x['m5_fa'] for x in mode_res]):4.2f}% |  {np.mean([x['rj'] for x in mode_res]):4.2f}%\n"
        )
        log_print(line, fh)

    log_print("========================================================================================================================\n", fh)
    print(f"\n[SUCCESS] Benchmark report successfully written to: {LOG_FILE}\n")
    fh.close()

if __name__ == "__main__":
    main()

      CONTINUOUS sEMG: ARCHITECTURE SEARCH & INDEPENDENT FIX ABLATION BENCHMARK (tau = 60 ms)                           
 Cohort      : 10 Subjects | Exclusions: S07 & S11 | S05 Polarity Swapped
 Focus Goals : 
   1. Restore Excursion Accuracy (target: 90% - 105% of human degrees)
   2. Eliminate Duration Smearing (target: 95% - 110% of human duration)
   3. Maintain Extensor Reciprocal Inhibition (M5 False Closing < 5%)

                            PART 1: ARCHITECTURE BENCHMARK (STANDARD CONDITIONS)                                        
 Architecture         | Excursion % | Duration %  | Angle R^2   | Angle r     | Latency (ms) | M5 False % | Rest Jitter  
------------------------------------------------------------------------------------------------------------------------


c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimiza

 MLP_Standard         |   84.6 +/- 49.3% |  146.8 +/- 42.3% |  0.245 +/- 0.252 |  0.799 +/- 0.134 |  132.0 +/- 132.5 ms |  8.47% |  10.28%
 MLP_Compact          |   98.0 +/- 55.9% |  150.8 +/- 43.1% |  0.241 +/- 0.256 |  0.808 +/- 0.118 |  110.0 +/- 149.8 ms |  10.87% |  12.58%
 MLP_DeepNarrow       |   86.6 +/- 54.1% |  139.2 +/- 45.6% |  0.252 +/- 0.279 |  0.806 +/- 0.130 |   66.0 +/- 150.3 ms |  6.25% |  8.02%
 TwoHeaded_Gated      |  106.4 +/- 51.2% |  135.6 +/- 45.3% |  0.268 +/- 0.329 |  0.844 +/- 0.101 |   98.0 +/- 136.2 ms |  7.15% |  11.17%
 Residual_MLP         |  102.5 +/- 29.8% |  150.0 +/- 26.4% |  0.210 +/- 0.278 |  0.804 +/- 0.135 |   95.0 +/- 131.3 ms |  27.10% |  19.77%

                  PART 2: INDEPENDENT FIX ABLATION MATRIX (EVALUATED ON MLP_COMPACT)                                    
 Condition / Fix Mode | Excursion % | Duration %  | Angle R^2   | Peak Speed% | Latency (ms) | M5 False % | Rest Jitter  
------------------------------------------------------------

In [1]:
#!/usr/bin/env python3
"""
====================================================================================================
CONTINUOUS sEMG UNIPOLAR FLIGHT: PURE SPEED MATCHING & ONSET SUITE
State Targets :
  1. Rest Mode       -> Motor Velocity = 0.0 deg/s (Rest Jitter < 5.0%)
  2. Hand Open (M5)  -> Motor Velocity = 0.0 deg/s (M5 False Closures < 5.0%)
  3. Flight (M6)     -> Match Speed & Vigor until Physical Contact (t1 -> t2)
Retrospective : tau = 60 ms (6 hops at 10 ms cadence)
Plateau Logic : 
  - Train: Completely drops hold plateau (stops strictly at t2).
  - Test : Retains exactly LAG_HOPS (60 ms) runway past t2 so target pointer reaches t2, then stops.
Architectures : Bio_Proportional, Linear_Ridge, Random_Forest, Hist_GBDT, MLP_Compact, TwoHeaded_Gated
====================================================================================================
"""

import os
import sys
import time
import numpy as np
import pandas as pd
from scipy import signal, stats
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.neural_network import MLPRegressor, MLPClassifier

# --- 1. Cohort & Protocol Constants ---
COHORT = [1, 2, 3, 4, 5, 6, 8, 9, 10, 12]  # S07 & S11 permanently excluded
V_DEADBAND = 4.0        # deg/s: Actuator deadband floor
V_ONSET_THRESH = 12.0   # deg/s: Threshold to register voluntary takeoff
DT_SEC = 0.010          # 10 ms hop (100 Hz cadence)
TAU_LAG_MS = 60         # 60 ms retrospective target alignment
LAG_HOPS = 6            # 60 ms / 10 ms
POST_TAKEOFF_HOPS = LAG_HOPS  # Exactly 60 ms runway: allows retrospective clock to reach t2, then stops
TIMEOUT_LATENCY_MS = 350.0    # Penalty latency for failed takeoff trigger
LOG_FILE = "cohort_speed_matching_report.txt"

INVERT_S05_IN_PYTHON = True

# --- 2. Unipolar State Labeler ---
def construct_unipolar_closing_labels(df):
    y_close = np.zeros(len(df), dtype=np.float64)
    is_m6_takeoff = np.zeros(len(df), dtype=bool)
    is_m6_stroke = np.zeros(len(df), dtype=bool)
    is_m5_open = np.zeros(len(df), dtype=bool)

    stim = df["stimulus"].values
    rep = df["repetition"].values
    is_plat = df["is_hold_plateau"].values.astype(bool)
    v_raw = df["theta_vel_deg_s"].values

    m6_reps = np.unique(rep[stim == 6])
    for r in m6_reps:
        trial_mask = (stim == 6) & (rep == r)
        trial_indices = np.where(trial_mask)[0]
        if len(trial_indices) == 0:
            continue

        plat_in_trial = np.where(trial_mask & is_plat)[0]
        if len(plat_in_trial) > 0:
            first_plat = plat_in_trial[0]
            takeoff_idx = [i for i in trial_indices if i < first_plat]
        else:
            cutoff = trial_indices[0] + int(0.40 * len(trial_indices))
            takeoff_idx = [i for i in trial_indices if i < cutoff]

        if len(takeoff_idx) > 0:
            is_m6_takeoff[takeoff_idx] = True
            y_close[takeoff_idx] = np.abs(v_raw[takeoff_idx])
            
            # Flight stroke window: Takeoff (t1 -> t2) + exactly 60 ms runway for retrospective pointer
            end_stroke = min(len(df), takeoff_idx[-1] + 1 + POST_TAKEOFF_HOPS)
            is_m6_stroke[takeoff_idx[0]:end_stroke] = True

    # Movement 5 (Hand Open): negative inhibitory data (target = 0.0)
    is_m5_open[stim == 5] = True

    df_labeled = df.copy()
    df_labeled["y_close"] = y_close
    df_labeled["is_m6_takeoff"] = is_m6_takeoff
    df_labeled["is_m6_stroke"] = is_m6_stroke
    df_labeled["is_m5_open"] = is_m5_open
    return df_labeled

def find_contiguous_segments(df, max_dt=0.018, min_len=24):
    t = df["timestamp_s"].values
    if len(t) == 0:
        return []
    dt = np.diff(t)
    break_indices = np.where(dt > max_dt)[0] + 1
    segments = np.split(np.arange(len(df)), break_indices)
    return [seg for seg in segments if len(seg) >= min_len]

# --- 3. Retrospective Context Buffer Builder ---
def extract_speed_data(df, feat_cols):
    # Causal taps: [t, t - 20 ms, t - 40 ms, t - 60 ms] relative to real-time clock
    tap_offsets = [0, 2, 4, 6]
    max_hist = max(tap_offsets)

    segments = find_contiguous_segments(df, min_len=max_hist + LAG_HOPS + 5)
    seg_records = []

    X_raw_all = df[feat_cols].values
    y_close_all = df["y_close"].values
    is_m6_all = df["is_m6_takeoff"].values
    is_stroke_all = df["is_m6_stroke"].values
    is_m5_all = df["is_m5_open"].values

    fds_idx = feat_cols.index("FDS_MAV")
    edc_idx = feat_cols.index("EDC_MAV")

    for seg in segments:
        X_seg = X_raw_all[seg]
        y_seg = y_close_all[seg]
        m6_seg = is_m6_all[seg]
        strk_seg = is_stroke_all[seg]
        m5_seg = is_m5_all[seg]
        L = len(seg)

        X_s, y_s, m6_s, strk_s, m5_s = [], [], [], [], []
        fds_t_s, edc_t_s = [], []

        start_idx = max(max_hist, LAG_HOPS)
        for i in range(start_idx, L):
            target_idx = i - LAG_HOPS  # Target is evaluated at t - 60 ms

            taps = [X_seg[i - k] for k in tap_offsets]
            X_s.append(np.hstack(taps))
            y_s.append(y_seg[target_idx])
            m6_s.append(m6_seg[target_idx])
            strk_s.append(strk_seg[target_idx])
            m5_s.append(m5_seg[target_idx])

            fds_t_s.append(X_seg[i, fds_idx])
            edc_t_s.append(X_seg[i, edc_idx])

        if len(X_s) > 0:
            seg_records.append({
                "X": np.array(X_s),
                "y": np.array(y_s),
                "is_m6_takeoff": np.array(m6_s),
                "is_m6_stroke": np.array(strk_s),
                "is_m5_open": np.array(m5_s),
                "fds_t": np.array(fds_t_s),
                "edc_t": np.array(edc_t_s)
            })

    return seg_records

# --- 4. Timeline Partitioning ---
def partition_timeline(df, sub_id, buffer_sec=1.0):
    test_reps = [2, 5] if sub_id == 4 else [2, 6]
    is_test = np.zeros(len(df), dtype=bool)

    for m in [5, 6]:
        for r in test_reps:
            cue_mask = (df["stimulus"] == m) & (df["repetition"] == r)
            if not np.any(cue_mask):
                continue
            t_start = df.loc[cue_mask, "timestamp_s"].min() - buffer_sec
            t_end = df.loc[cue_mask, "timestamp_s"].max() + buffer_sec
            is_test[(df["timestamp_s"] >= t_start) & (df["timestamp_s"] <= t_end)] = True

    is_corrupt = (df["repetition"] == 6).values if sub_id == 4 else np.zeros(len(df), dtype=bool)
    is_plateau = (df["is_hold_plateau"] == 1).values

    # Train: completely discards plateaus (never trains past t2)
    train_mask = ~is_corrupt & ~is_plateau & ~is_test
    # Test: keeps takeoff + 60 ms runway, M5 hand open, and baseline rest; drops isometric hold
    test_mask = ~is_corrupt & (df["is_m6_stroke"] | df["is_m5_open"] | ~is_plateau) & is_test

    return train_mask, test_mask

# --- 5. Pure Speed Actuator Output ---
def apply_flight_deadband(y_raw, v_db=V_DEADBAND):
    y = np.maximum(0.0, np.copy(y_raw))
    mask = y > v_db
    y_cmd = np.zeros_like(y)
    y_cmd[mask] = y[mask] * ((y[mask] - v_db) / y[mask])
    return y_cmd

# --- 6. Speed-Centric Evaluator ---
def evaluate_speed_pipeline(test_records, predict_fn, gamma=1.0, dt=DT_SEC):
    m6_y_true, m6_y_cmd = [], []
    m5_cmds, rest_cmds = [], []
    
    mean_speed_ratios = []
    peak_speed_ratios = []
    onset_latencies_ms = []

    for seg in test_records:
        y_pred = predict_fn(seg["X"]) * gamma
        y_cmd = apply_flight_deadband(y_pred)

        y_seg = seg["y"]
        m6_tk = seg["is_m6_takeoff"]
        m6_strk = seg["is_m6_stroke"]
        m5_mask = seg["is_m5_open"]

        # Trajectory fidelity during active flight
        if np.any(m6_tk):
            idx_tk = np.where(m6_tk)[0]
            yt_tk = y_seg[idx_tk]
            yp_tk = y_cmd[idx_tk]

            m6_y_true.extend(yt_tk)
            m6_y_cmd.extend(yp_tk)

            peak_t = np.max(yt_tk)
            peak_p = np.max(yp_tk)
            if peak_t > 15.0:
                peak_speed_ratios.append((peak_p / peak_t) * 100.0)

        # Full Stroke: evaluates speed matching and onset latency up to contact
        if np.any(m6_strk):
            idx_strk = np.where(m6_strk)[0]
            yt_strk = y_seg[idx_strk]
            yp_strk = y_cmd[idx_strk]

            active_human = yt_strk > V_DEADBAND
            active_robot = yp_strk > V_DEADBAND

            # Speed Vigor Matching: compares active human flight speed to active robot flight speed
            if np.sum(active_human) > 5 and np.sum(active_robot) > 3:
                mean_t = np.mean(yt_strk[active_human])
                mean_p = np.mean(yp_strk[active_robot])
                mean_speed_ratios.append((mean_p / mean_t) * 100.0)
            elif np.sum(active_human) > 5 and np.sum(active_robot) <= 3:
                mean_speed_ratios.append(0.0)  # Motor stalled

            # Action Onset Latency: delay from human takeoff onset to motor takeoff onset
            onset_t_rel = np.where(yt_strk >= V_ONSET_THRESH)[0]
            onset_p_rel = np.where(yp_strk >= V_ONSET_THRESH)[0]
            if len(onset_t_rel) > 0:
                if len(onset_p_rel) > 0:
                    action_lat = ((onset_p_rel[0] - onset_t_rel[0]) * dt * 1000.0) + TAU_LAG_MS
                    onset_latencies_ms.append(action_lat)
                else:
                    onset_latencies_ms.append(TIMEOUT_LATENCY_MS)

        # Safety Check 1: Hand-Open False Closures
        if np.any(m5_mask):
            m5_cmds.extend(y_cmd[m5_mask])

        # Safety Check 2: Resting Baseline Jitter
        rest_mask = (~m6_strk) & (~m5_mask) & (y_seg <= 2.0)
        if np.any(rest_mask):
            rest_cmds.extend(y_cmd[rest_mask])

    yt_arr = np.array(m6_y_true, dtype=np.float64)
    yp_arr = np.array(m6_y_cmd, dtype=np.float64)

    if len(yt_arr) > 10 and np.std(yp_arr) > 0 and np.std(yt_arr) > 0:
        r_vel, _ = stats.pearsonr(yt_arr, yp_arr)
    else:
        r_vel = 0.0

    m5_cmds = np.array(m5_cmds)
    m5_fa = (np.sum(m5_cmds > 0.0) / len(m5_cmds) * 100.0) if len(m5_cmds) > 0 else 0.0

    rest_cmds = np.array(rest_cmds)
    rj = (np.sum(rest_cmds > 0.0) / len(rest_cmds) * 100.0) if len(rest_cmds) > 0 else 0.0

    return {
        "mean_speed_pct": np.mean(mean_speed_ratios) if mean_speed_ratios else 0.0,
        "peak_speed_pct": np.mean(peak_speed_ratios) if peak_speed_ratios else 0.0,
        "r_vel": r_vel,
        "onset_lat_ms": np.mean(onset_latencies_ms) if onset_latencies_ms else TIMEOUT_LATENCY_MS,
        "m5_fa": m5_fa,
        "rj": rj
    }

def log_print(msg, fh):
    print(msg, end="")
    fh.write(msg)

# --- 7. Main Benchmark Harness ---
def main():
    fh = open(LOG_FILE, "w", encoding="utf-8")

    header = (
        "========================================================================================================================\n"
        "           CONTINUOUS sEMG UNIPOLAR FLIGHT: PURE SPEED MATCHING & ONSET LATENCY SUITE (tau = 60 ms)                    \n"
        "========================================================================================================================\n"
        f" Cohort      : {len(COHORT)} Subjects | Exclusions: S07 & S11 | S05 Polarity Swapped\n"
        " Target Task : Rest -> No Move | Hand Open -> No Move | Flight (t1 -> t2) -> Match Speed\n"
        " Thresholds  : Deadband = 4.0 deg/s | Action Onset = 12.0 deg/s | tau = 60 ms\n"
        " Runway      : Exactly 60 ms post-t2 runway in test set; zero plateau in train set\n"
        "========================================================================================================================\n\n"
    )
    log_print(header, fh)

    # 1. Preload Cohort
    subject_data = {}
    for sub_id in COHORT:
        csv_path = os.path.join("datasets", "DB2", f"DB2_s{sub_id}", f"S{sub_id}_E1_A1_dynamic_velocity_features.csv")
        if not os.path.exists(csv_path):
            csv_path = f"S{sub_id}_E1_A1_dynamic_velocity_features.csv"
        if not os.path.exists(csv_path):
            continue

        df_raw = pd.read_csv(csv_path)

        # Subject 5 Hardware Polarity Swap
        if sub_id == 5 and INVERT_S05_IN_PYTHON:
            edc_cols = [c for c in df_raw.columns if c.startswith("EDC_")]
            for e_col in edc_cols:
                f_col = e_col.replace("EDC_", "FDS_")
                if f_col in df_raw.columns:
                    df_raw[e_col], df_raw[f_col] = df_raw[f_col].copy(), df_raw[e_col].copy()
            for ndi_col in ["NDI_MAV", "NDI_RMS", "NDI_WL"]:
                if ndi_col in df_raw.columns:
                    df_raw[ndi_col] = -df_raw[ndi_col]

        df_labeled = construct_unipolar_closing_labels(df_raw)
        meta_cols = ["timestamp_s", "theta_deg", "theta_vel_deg_s", "stimulus", "repetition", 
                     "is_hold_plateau", "y_close", "is_m6_takeoff", "is_m6_stroke", "is_m5_open"]
        feat_cols = [c for c in df_raw.columns if c not in meta_cols]

        tr_mask, te_mask = partition_timeline(df_labeled, sub_id)
        subject_data[sub_id] = {
            "train_df": df_labeled[tr_mask].copy(),
            "test_df": df_labeled[te_mask].copy(),
            "feat_cols": feat_cols
        }

    # 2. Architectures to Benchmark
    model_names = [
        "Bio_Proportional",
        "Linear_Ridge",
        "Random_Forest",
        "Hist_GBDT",
        "MLP_Compact",
        "TwoHeaded_Gated"
    ]
    
    benchmark_results = {m: [] for m in model_names}
    inference_times_us = {m: [] for m in model_names}

    print(">>> Executing Speed-Matching Benchmark across 10 Subjects...")

    for sub_id, d in subject_data.items():
        tr_rec = extract_speed_data(d["train_df"], d["feat_cols"])
        te_rec = extract_speed_data(d["test_df"], d["feat_cols"])
        if not tr_rec or not te_rec:
            continue

        X_tr = np.vstack([r["X"] for r in tr_rec])
        y_tr = np.concatenate([r["y"] for r in tr_rec])
        scaler = StandardScaler().fit(X_tr)
        X_tr_z = scaler.transform(X_tr)

        te_rec_sc = []
        for r in te_rec:
            r_c = dict(r)
            r_c["X"] = np.clip(scaler.transform(r["X"]), -20.0, 20.0)
            te_rec_sc.append(r_c)

        active_tr = y_tr > 0.0
        fds_idx = d["feat_cols"].index("FDS_MAV")
        edc_idx = d["feat_cols"].index("EDC_MAV")

        # --- MODEL 1: Handcrafted Biomechanical Proportional Baseline ---
        fds_tr = X_tr[:, fds_idx]
        edc_tr = X_tr[:, edc_idx]
        quiet_rest_mask = (y_tr == 0.0) & (edc_tr < np.median(edc_tr))
        fds_rest_floor = np.percentile(fds_tr[quiet_rest_mask], 75) if np.any(quiet_rest_mask) else np.percentile(fds_tr, 20)
        
        diff_active = np.maximum(0.0, fds_tr[active_tr] - edc_tr[active_tr] - fds_rest_floor)
        denom_bio = np.sum(diff_active ** 2)
        k_gain = (np.sum(y_tr[active_tr] * diff_active) / denom_bio) if denom_bio > 1e-6 else 1.0

        def bio_predict(X_z):
            X_orig = scaler.inverse_transform(X_z)
            fds = X_orig[:, fds_idx]
            edc = X_orig[:, edc_idx]
            drive = np.maximum(0.0, fds - edc - fds_rest_floor)
            drive[edc > 1.2 * fds] = 0.0  # Reciprocal extensor inhibition
            return drive * k_gain

        t0 = time.perf_counter()
        _ = bio_predict(te_rec_sc[0]["X"])
        inference_times_us["Bio_Proportional"].append(((time.perf_counter() - t0) / len(te_rec_sc[0]["X"])) * 1e6)
        benchmark_results["Bio_Proportional"].append(evaluate_speed_pipeline(te_rec_sc, bio_predict, 1.0))

        # Dynamic variance calibration gain helper
        def fit_gamma(pred_train):
            if not np.any(active_tr):
                return 1.0
            act_p = pred_train[active_tr]
            denom = np.sum(act_p ** 2)
            if denom < 1e-6:
                return 1.0
            return np.clip(np.sum(y_tr[active_tr] * act_p) / denom, 0.8, 1.5)

        # --- MODEL 2: Linear Ridge Regressor ---
        ridge = Ridge(alpha=20.0)
        ridge.fit(X_tr_z, y_tr)
        ridge_fn = lambda X, m=ridge: np.maximum(0.0, m.predict(X))
        gamma_ridge = fit_gamma(ridge_fn(X_tr_z))

        t0 = time.perf_counter()
        _ = ridge_fn(te_rec_sc[0]["X"])
        inference_times_us["Linear_Ridge"].append(((time.perf_counter() - t0) / len(te_rec_sc[0]["X"])) * 1e6)
        benchmark_results["Linear_Ridge"].append(evaluate_speed_pipeline(te_rec_sc, ridge_fn, gamma_ridge))

        # --- MODEL 3: Random Forest (Bagging Trees) ---
        rf = RandomForestRegressor(n_estimators=35, max_depth=8, n_jobs=-1, random_state=42)
        rf.fit(X_tr_z, y_tr)
        rf_fn = lambda X, m=rf: np.maximum(0.0, m.predict(X))
        gamma_rf = fit_gamma(rf_fn(X_tr_z))

        t0 = time.perf_counter()
        _ = rf_fn(te_rec_sc[0]["X"])
        inference_times_us["Random_Forest"].append(((time.perf_counter() - t0) / len(te_rec_sc[0]["X"])) * 1e6)
        benchmark_results["Random_Forest"].append(evaluate_speed_pipeline(te_rec_sc, rf_fn, gamma_rf))

        # --- MODEL 4: HistGradientBoosting (GBDT) ---
        gbdt = HistGradientBoostingRegressor(max_iter=60, max_depth=6, l2_regularization=0.5, random_state=42)
        gbdt.fit(X_tr_z, y_tr)
        gbdt_fn = lambda X, m=gbdt: np.maximum(0.0, m.predict(X))
        gamma_gbdt = fit_gamma(gbdt_fn(X_tr_z))

        t0 = time.perf_counter()
        _ = gbdt_fn(te_rec_sc[0]["X"])
        inference_times_us["Hist_GBDT"].append(((time.perf_counter() - t0) / len(te_rec_sc[0]["X"])) * 1e6)
        benchmark_results["Hist_GBDT"].append(evaluate_speed_pipeline(te_rec_sc, gbdt_fn, gamma_gbdt))

        # --- MODEL 5: MLP_Compact (32, 16) ---
        mlp = MLPRegressor(hidden_layer_sizes=(32, 16), activation="relu", alpha=0.20,
                           early_stopping=True, n_iter_no_change=10, random_state=42, max_iter=200)
        mlp.fit(X_tr_z, y_tr)
        mlp_fn = lambda X, m=mlp: np.maximum(0.0, m.predict(X))
        gamma_mlp = fit_gamma(mlp_fn(X_tr_z))

        t0 = time.perf_counter()
        _ = mlp_fn(te_rec_sc[0]["X"])
        inference_times_us["MLP_Compact"].append(((time.perf_counter() - t0) / len(te_rec_sc[0]["X"])) * 1e6)
        benchmark_results["MLP_Compact"].append(evaluate_speed_pipeline(te_rec_sc, mlp_fn, gamma_mlp))

        # --- MODEL 6: TwoHeaded_Gated ---
        gate = MLPClassifier(hidden_layer_sizes=(32, 16), activation="relu", alpha=0.15,
                             early_stopping=True, n_iter_no_change=10, random_state=42, max_iter=200)
        gate.fit(X_tr_z, (y_tr > 0.0).astype(int))
        reg = MLPRegressor(hidden_layer_sizes=(32, 16), activation="relu", alpha=0.15,
                           early_stopping=True, n_iter_no_change=10, random_state=42, max_iter=200)
        if np.sum(active_tr) > 20:
            reg.fit(X_tr_z[active_tr], y_tr[active_tr])
        else:
            reg.fit(X_tr_z, y_tr)

        def make_gated(g=gate, r=reg):
            def _pred(X):
                p = g.predict_proba(X)[:, 1] if g.classes_.shape[0] > 1 else np.zeros(len(X))
                mag = np.maximum(0.0, r.predict(X))
                return p * mag
            return _pred
        gated_fn = make_gated()
        gamma_gated = fit_gamma(gated_fn(X_tr_z))

        t0 = time.perf_counter()
        _ = gated_fn(te_rec_sc[0]["X"])
        inference_times_us["TwoHeaded_Gated"].append(((time.perf_counter() - t0) / len(te_rec_sc[0]["X"])) * 1e6)
        benchmark_results["TwoHeaded_Gated"].append(evaluate_speed_pipeline(te_rec_sc, gated_fn, gamma_gated))

    # --- 8. Format Final Output Report ---
    table_header = (
        "========================================================================================================================\n"
        "                  MODEL COMPARISON: PURE SPEED MATCHING & ONSET PERFORMANCE (10 SUBJECTS MEAN +/- STD)                  \n"
        "========================================================================================================================\n"
        " Architecture         | Mean Speed% | Peak Speed% | Vel Corr (r)| Onset Lat (ms) | M5 False % | Rest Jitter | Cost (μs)\n"
        "------------------------------------------------------------------------------------------------------------------------\n"
    )
    log_print(table_header, fh)

    for m in model_names:
        res_list = benchmark_results[m]
        mean_sp = [r["mean_speed_pct"] for r in res_list]
        peak_sp = [r["peak_speed_pct"] for r in res_list]
        r_vel   = [r["r_vel"] for r in res_list]
        lat_ms  = [r["onset_lat_ms"] for r in res_list]
        m5_fa   = [r["m5_fa"] for r in res_list]
        rj      = [r["rj"] for r in res_list]
        us_cost = inference_times_us[m]

        line = (
            f" {m:<20} | {np.mean(mean_sp):5.1f} +/- {np.std(mean_sp):4.1f}% | "
            f"{np.mean(peak_sp):5.1f} +/- {np.std(peak_sp):4.1f}% |  "
            f"{np.mean(r_vel):5.3f} +/- {np.std(r_vel):4.3f} |  "
            f"{np.mean(lat_ms):5.1f} +/- {np.std(lat_ms):4.1f} ms |  "
            f"{np.mean(m5_fa):4.2f}% |   {np.mean(rj):4.2f}% | "
            f"{np.mean(us_cost):6.1f} μs\n"
        )
        log_print(line, fh)

    log_print("========================================================================================================================\n", fh)
    print(f"\n[SUCCESS] Benchmark complete! Full analytical report written to: {LOG_FILE}\n")
    fh.close()

if __name__ == "__main__":
    main()

           CONTINUOUS sEMG UNIPOLAR FLIGHT: PURE SPEED MATCHING & ONSET LATENCY SUITE (tau = 60 ms)                    
 Cohort      : 10 Subjects | Exclusions: S07 & S11 | S05 Polarity Swapped
 Target Task : Rest -> No Move | Hand Open -> No Move | Flight (t1 -> t2) -> Match Speed
 Thresholds  : Deadband = 4.0 deg/s | Action Onset = 12.0 deg/s | tau = 60 ms
 Runway      : Exactly 60 ms post-t2 runway in test set; zero plateau in train set

>>> Executing Speed-Matching Benchmark across 10 Subjects...


c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimiza

                  MODEL COMPARISON: PURE SPEED MATCHING & ONSET PERFORMANCE (10 SUBJECTS MEAN +/- STD)                  
 Architecture         | Mean Speed% | Peak Speed% | Vel Corr (r)| Onset Lat (ms) | M5 False % | Rest Jitter | Cost (μs)
------------------------------------------------------------------------------------------------------------------------
 Bio_Proportional     |  10.1 +/- 21.8% |   5.5 +/- 11.3% |  -0.118 +/- 0.242 |  352.5 +/-  7.5 ms |  6.74% |   1.52% |    1.1 μs
 Linear_Ridge         |  79.0 +/- 33.9% |  58.6 +/- 32.4% |  0.126 +/- 0.394 |   -0.5 +/- 242.9 ms |  42.26% |   34.59% |    0.8 μs
 Random_Forest        |  89.1 +/- 43.2% |  76.4 +/- 32.9% |  0.142 +/- 0.401 |   54.5 +/- 136.8 ms |  27.74% |   19.46% |   44.3 μs
 Hist_GBDT            |  85.0 +/- 42.8% |  78.2 +/- 30.4% |  0.273 +/- 0.390 |   56.0 +/- 83.1 ms |  22.91% |   18.14% |    4.7 μs
 MLP_Compact          | 101.5 +/- 44.0% | 103.0 +/- 41.9% |  0.284 +/- 0.229 |   80.5 +/- 34.2 ms |  16.27% |   1

c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


In [ ]:
#!/usr/bin/env python3
"""
====================================================================================================
CONTINUOUS sEMG UNIPOLAR FLIGHT: ADVANCED NEURAL ARCHITECTURES & HYBRID GATING
Cohort      : Subjects 1 to 12 (Excluding S07 & S11; Inverting S05)
Target Lag  : tau = 60 ms retrospective target alignment
Cadence     : 10 ms hop (100 Hz cadence)
Focus       : Eliminate Rest Jitter (<5%) and M5 False Closures (<5%) while preserving ~100% Speed
Models      :
  1. MLP_Compact_Base       : Baseline single-head regressor (soft smooth deadband)
  2. MLP_Compact_RecipBrake : Single-head MLP + Normalized Reciprocal Brake (0.5 sigma deadband)
  3. TwoHeaded_HardGated    : Decoupled Gate (P >= 0.50) + Pure Active Regressor
  4. TwoHeaded_Hysteresis   : Schmitt Trigger (P_on = 0.55, P_off = 0.35) + Antagonist Abort
  5. TwoHeaded_DeepNarrow   : Deep bottleneck gate (48, 24, 12) + Regressor (32, 16)
  6. TwoHeaded_FullSafe     : Hard Gate (P >= 0.50) + Normalized Antagonist Brake
====================================================================================================
"""

import os
import sys
import time
import numpy as np
import pandas as pd
from scipy import signal, stats
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor, MLPClassifier

# --- 1. Cohort & Protocol Constants ---
COHORT = [1, 2, 3, 4, 5, 6, 8, 9, 10, 12]  # S07 & S11 permanently excluded
V_DEADBAND = 4.0        # deg/s: Actuator deadband floor
V_ONSET_THRESH = 12.0   # deg/s: Velocity threshold to register voluntary takeoff
DT_SEC = 0.010          # 10 ms hop (100 Hz cadence)
TAU_LAG_MS = 60         # 60 ms retrospective target alignment
LAG_HOPS = 6            # 60 ms / 10 ms
POST_TAKEOFF_HOPS = LAG_HOPS  # Exactly 60 ms runway for retrospective pointer to reach t2
TIMEOUT_LATENCY_MS = 350.0    # Penalty latency for failed onset trigger
LOG_FILE = "cohort_advanced_neural_report.txt"

INVERT_S05_IN_PYTHON = True

# --- 2. Unipolar State Labeler ---
def construct_unipolar_closing_labels(df):
    y_close = np.zeros(len(df), dtype=np.float64)
    is_m6_takeoff = np.zeros(len(df), dtype=bool)
    is_m6_stroke = np.zeros(len(df), dtype=bool)
    is_m5_open = np.zeros(len(df), dtype=bool)
    is_genuine_rest = np.zeros(len(df), dtype=bool)

    stim = df["stimulus"].values
    rep = df["repetition"].values
    is_plat = df["is_hold_plateau"].values.astype(bool)
    v_raw = df["theta_vel_deg_s"].values

    # 1. Label M6 Takeoffs & Active Strokes
    m6_reps = np.unique(rep[stim == 6])
    for r in m6_reps:
        trial_mask = (stim == 6) & (rep == r)
        trial_indices = np.where(trial_mask)[0]
        if len(trial_indices) == 0:
            continue

        plat_in_trial = np.where(trial_mask & is_plat)[0]
        if len(plat_in_trial) > 0:
            first_plat = plat_in_trial[0]
            takeoff_idx = [i for i in trial_indices if i < first_plat]
        else:
            cutoff = trial_indices[0] + int(0.40 * len(trial_indices))
            takeoff_idx = [i for i in trial_indices if i < cutoff]

        if len(takeoff_idx) > 0:
            v_mag = np.abs(v_raw[takeoff_idx])
            # Strict deadband suppression: eliminates analog baseline drift during pre-movement pause
            v_mag[v_mag <= V_DEADBAND] = 0.0
            
            is_m6_takeoff[takeoff_idx] = True
            y_close[takeoff_idx] = v_mag
            
            end_stroke = min(len(df), takeoff_idx[-1] + 1 + POST_TAKEOFF_HOPS)
            is_m6_stroke[takeoff_idx[0]:end_stroke] = True

    # 2. Movement 5 (Hand Open): negative inhibitory data (target = 0.0)
    is_m5_open[stim == 5] = True

    # 3. Label Genuine Rest (Exclude active return descents)
    is_genuine_rest[(stim == 0) | ((stim != 5) & (~is_m6_stroke) & (~is_plat) & (np.abs(v_raw) <= 2.0))] = True

    df_labeled = df.copy()
    df_labeled["y_close"] = y_close
    df_labeled["is_m6_takeoff"] = is_m6_takeoff
    df_labeled["is_m6_stroke"] = is_m6_stroke
    df_labeled["is_m5_open"] = is_m5_open
    df_labeled["is_genuine_rest"] = is_genuine_rest
    return df_labeled

def find_contiguous_segments(df, max_dt=0.018, min_len=24):
    t = df["timestamp_s"].values
    if len(t) == 0:
        return []
    dt = np.diff(t)
    break_indices = np.where(dt > max_dt)[0] + 1
    segments = np.split(np.arange(len(df)), break_indices)
    return [seg for seg in segments if len(seg) >= min_len]

# --- 3. Retrospective Context Builder ---
def extract_speed_data(df, feat_cols):
    tap_offsets = [0, 2, 4, 6]  # [t, t-20ms, t-40ms, t-60ms]
    max_hist = max(tap_offsets)

    segments = find_contiguous_segments(df, min_len=max_hist + LAG_HOPS + 5)
    seg_records = []

    X_raw_all = df[feat_cols].values
    y_close_all = df["y_close"].values
    is_m6_all = df["is_m6_takeoff"].values
    is_stroke_all = df["is_m6_stroke"].values
    is_m5_all = df["is_m5_open"].values
    is_rest_all = df["is_genuine_rest"].values

    for seg in segments:
        X_seg = X_raw_all[seg]
        y_seg = y_close_all[seg]
        m6_seg = is_m6_all[seg]
        strk_seg = is_stroke_all[seg]
        m5_seg = is_m5_all[seg]
        rest_seg = is_rest_all[seg]
        L = len(seg)

        X_s, y_s, m6_s, strk_s, m5_s, rest_s = [], [], [], [], [], []

        start_idx = max(max_hist, LAG_HOPS)
        for i in range(start_idx, L):
            target_idx = i - LAG_HOPS

            taps = [X_seg[i - k] for k in tap_offsets]
            X_s.append(np.hstack(taps))
            y_s.append(y_seg[target_idx])
            m6_s.append(m6_seg[target_idx])
            strk_s.append(strk_seg[target_idx])
            m5_s.append(m5_seg[target_idx])
            rest_s.append(rest_seg[target_idx])

        if len(X_s) > 0:
            seg_records.append({
                "X": np.array(X_s),
                "y": np.array(y_s),
                "is_m6_takeoff": np.array(m6_s),
                "is_m6_stroke": np.array(strk_s),
                "is_m5_open": np.array(m5_s),
                "is_genuine_rest": np.array(rest_s)
            })

    return seg_records

# --- 4. Timeline Partitioning ---
def partition_timeline(df, sub_id, buffer_sec=1.0):
    test_reps = [2, 5] if sub_id == 4 else [2, 6]
    is_test = np.zeros(len(df), dtype=bool)

    for m in [5, 6]:
        for r in test_reps:
            cue_mask = (df["stimulus"] == m) & (df["repetition"] == r)
            if not np.any(cue_mask):
                continue
            t_start = df.loc[cue_mask, "timestamp_s"].min() - buffer_sec
            t_end = df.loc[cue_mask, "timestamp_s"].max() + buffer_sec
            is_test[(df["timestamp_s"] >= t_start) & (df["timestamp_s"] <= t_end)] = True

    is_corrupt = (df["repetition"] == 6).values if sub_id == 4 else np.zeros(len(df), dtype=bool)
    is_plateau = (df["is_hold_plateau"] == 1).values

    train_mask = ~is_corrupt & ~is_plateau & ~is_test
    test_mask = ~is_corrupt & (df["is_m6_stroke"] | df["is_m5_open"] | df["is_genuine_rest"]) & is_test

    return train_mask, test_mask

# --- 5. Flight Deadband ---
def apply_flight_deadband(y_raw, v_db=V_DEADBAND):
    y = np.maximum(0.0, np.copy(y_raw))
    mask = y > v_db
    y_cmd = np.zeros_like(y)
    y_cmd[mask] = y[mask] * ((y[mask] - v_db) / y[mask])
    return y_cmd

# --- 6. Speed-Centric Evaluator ---
def evaluate_neural_pipeline(test_records, predict_fn, dt=DT_SEC):
    m6_y_true, m6_y_cmd = [], []
    m5_cmds, rest_cmds = [], []
    
    mean_speed_ratios = []
    peak_speed_ratios = []
    onset_latencies_ms = []

    for seg in test_records:
        y_pred = predict_fn(seg["X"])
        y_cmd = apply_flight_deadband(y_pred)

        y_seg = seg["y"]
        m6_tk = seg["is_m6_takeoff"]
        m6_strk = seg["is_m6_stroke"]
        m5_mask = seg["is_m5_open"]
        rest_mask = seg["is_genuine_rest"]

        if np.any(m6_tk):
            idx_tk = np.where(m6_tk)[0]
            yt_tk = y_seg[idx_tk]
            yp_tk = y_cmd[idx_tk]

            m6_y_true.extend(yt_tk)
            m6_y_cmd.extend(yp_tk)

            peak_t = np.max(yt_tk)
            peak_p = np.max(yp_tk)
            if peak_t > 15.0:
                peak_speed_ratios.append((peak_p / peak_t) * 100.0)

        if np.any(m6_strk):
            idx_strk = np.where(m6_strk)[0]
            yt_strk = y_seg[idx_strk]
            yp_strk = y_cmd[idx_strk]

            active_human = yt_strk > V_DEADBAND
            active_robot = yp_strk > V_DEADBAND

            if np.sum(active_human) > 5 and np.sum(active_robot) > 3:
                mean_t = np.mean(yt_strk[active_human])
                mean_p = np.mean(yp_strk[active_robot])
                mean_speed_ratios.append((mean_p / mean_t) * 100.0)
            elif np.sum(active_human) > 5 and np.sum(active_robot) <= 3:
                mean_speed_ratios.append(0.0)

            onset_t_rel = np.where(yt_strk >= V_ONSET_THRESH)[0]
            onset_p_rel = np.where(yp_strk >= V_ONSET_THRESH)[0]
            if len(onset_t_rel) > 0:
                if len(onset_p_rel) > 0:
                    action_lat = ((onset_p_rel[0] - onset_t_rel[0]) * dt * 1000.0) + TAU_LAG_MS
                    onset_latencies_ms.append(action_lat)
                else:
                    onset_latencies_ms.append(TIMEOUT_LATENCY_MS)

        if np.any(m5_mask):
            m5_cmds.extend(y_cmd[m5_mask])

        if np.any(rest_mask):
            rest_cmds.extend(y_cmd[rest_mask])

    yt_arr = np.array(m6_y_true, dtype=np.float64)
    yp_arr = np.array(m6_y_cmd, dtype=np.float64)

    if len(yt_arr) > 10 and np.std(yp_arr) > 0 and np.std(yt_arr) > 0:
        r_vel, _ = stats.pearsonr(yt_arr, yp_arr)
    else:
        r_vel = 0.0

    m5_cmds = np.array(m5_cmds)
    m5_fa = (np.sum(m5_cmds > 0.0) / len(m5_cmds) * 100.0) if len(m5_cmds) > 0 else 0.0

    rest_cmds = np.array(rest_cmds)
    rj = (np.sum(rest_cmds > 0.0) / len(rest_cmds) * 100.0) if len(rest_cmds) > 0 else 0.0

    return {
        "mean_speed_pct": np.mean(mean_speed_ratios) if mean_speed_ratios else 0.0,
        "peak_speed_pct": np.mean(peak_speed_ratios) if peak_speed_ratios else 0.0,
        "r_vel": r_vel,
        "onset_lat_ms": np.mean(onset_latencies_ms) if onset_latencies_ms else TIMEOUT_LATENCY_MS,
        "m5_fa": m5_fa,
        "rj": rj
    }

def log_print(msg, fh):
    print(msg, end="")
    fh.write(msg)

# --- 7. Main Benchmark Harness ---
def main():
    fh = open(LOG_FILE, "w", encoding="utf-8")

    header = (
        "========================================================================================================================\n"
        "           ADVANCED NEURAL ARCHITECTURES & HYBRID GATING: SPEED MATCHING & ONSET SUITE (tau = 60 ms)                    \n"
        "========================================================================================================================\n"
        f" Cohort      : {len(COHORT)} Subjects | Exclusions: S07 & S11 | S05 Polarity Swapped\n"
        " Focus Goals : Speed Match ~ 100% | Latency < 80 ms | M5 False Closing < 5% | Rest Jitter < 5%\n"
        " Runway      : Exactly 60 ms post-t2 runway in test set; zero plateau in train set\n"
        "========================================================================================================================\n\n"
    )
    log_print(header, fh)

    subject_data = {}
    for sub_id in COHORT:
        csv_path = os.path.join("datasets", "DB2", f"DB2_s{sub_id}", f"S{sub_id}_E1_A1_dynamic_velocity_features.csv")
        if not os.path.exists(csv_path):
            csv_path = f"S{sub_id}_E1_A1_dynamic_velocity_features.csv"
        if not os.path.exists(csv_path):
            continue

        df_raw = pd.read_csv(csv_path)

        if sub_id == 5 and INVERT_S05_IN_PYTHON:
            edc_cols = [c for c in df_raw.columns if c.startswith("EDC_")]
            for e_col in edc_cols:
                f_col = e_col.replace("EDC_", "FDS_")
                if f_col in df_raw.columns:
                    df_raw[e_col], df_raw[f_col] = df_raw[f_col].copy(), df_raw[e_col].copy()
            for ndi_col in ["NDI_MAV", "NDI_RMS", "NDI_WL"]:
                if ndi_col in df_raw.columns:
                    df_raw[ndi_col] = -df_raw[ndi_col]

        df_labeled = construct_unipolar_closing_labels(df_raw)
        meta_cols = ["timestamp_s", "theta_deg", "theta_vel_deg_s", "stimulus", "repetition", 
                     "is_hold_plateau", "y_close", "is_m6_takeoff", "is_m6_stroke", "is_m5_open", "is_genuine_rest"]
        feat_cols = [c for c in df_raw.columns if c not in meta_cols]

        tr_mask, te_mask = partition_timeline(df_labeled, sub_id)
        subject_data[sub_id] = {
            "train_df": df_labeled[tr_mask].copy(),
            "test_df": df_labeled[te_mask].copy(),
            "feat_cols": feat_cols
        }

    model_names = [
        "MLP_Compact_Base",
        "MLP_Compact_RecipBrake",
        "TwoHeaded_HardGated",
        "TwoHeaded_Hysteresis",
        "TwoHeaded_DeepNarrow",
        "TwoHeaded_FullSafe"
    ]
    
    benchmark_results = {m: [] for m in model_names}
    inference_times_us = {m: [] for m in model_names}

    print(">>> Executing Advanced Neural Speed-Matching Benchmark across 10 Subjects...")

    for sub_id, d in subject_data.items():
        tr_rec = extract_speed_data(d["train_df"], d["feat_cols"])
        te_rec = extract_speed_data(d["test_df"], d["feat_cols"])
        if not tr_rec or not te_rec:
            continue

        X_tr = np.vstack([r["X"] for r in tr_rec])
        y_tr = np.concatenate([r["y"] for r in tr_rec])
        scaler = StandardScaler().fit(X_tr)
        X_tr_z = scaler.transform(X_tr)

        te_rec_sc = []
        for r in te_rec:
            r_c = dict(r)
            r_c["X"] = np.clip(scaler.transform(r["X"]), -20.0, 20.0)
            te_rec_sc.append(r_c)

        active_tr = y_tr > 0.0

        fds_tap0_idx = d["feat_cols"].index("FDS_MAV")
        edc_tap0_idx = d["feat_cols"].index("EDC_MAV")

        def fit_gamma(pred_train):
            if not np.any(active_tr):
                return 1.0
            act_p = pred_train[active_tr]
            denom = np.sum(act_p ** 2)
            if denom < 1e-6:
                return 1.0
            return np.clip(np.sum(y_tr[active_tr] * act_p) / denom, 0.8, 1.5)

        # --------------------------------------------------------------------------------------
        # Shared Network Training
        # --------------------------------------------------------------------------------------
        mlp_base = MLPRegressor(hidden_layer_sizes=(32, 16), activation="relu", alpha=0.20,
                                early_stopping=True, n_iter_no_change=10, random_state=42, max_iter=200)
        mlp_base.fit(X_tr_z, y_tr)
        gamma_base = fit_gamma(np.maximum(0.0, mlp_base.predict(X_tr_z)))

        gate_std = MLPClassifier(hidden_layer_sizes=(32, 16), activation="relu", alpha=0.15,
                                 early_stopping=True, n_iter_no_change=10, random_state=42, max_iter=200)
        gate_std.fit(X_tr_z, (y_tr > 0.0).astype(int))

        reg_pure = MLPRegressor(hidden_layer_sizes=(32, 16), activation="relu", alpha=0.15,
                                early_stopping=True, n_iter_no_change=10, random_state=42, max_iter=200)
        if np.sum(active_tr) > 20:
            reg_pure.fit(X_tr_z[active_tr], y_tr[active_tr])
        else:
            reg_pure.fit(X_tr_z, y_tr)
        gamma_reg = fit_gamma(np.maximum(0.0, reg_pure.predict(X_tr_z)))

        gate_dn = MLPClassifier(hidden_layer_sizes=(48, 24, 12), activation="relu", alpha=0.20,
                                early_stopping=True, n_iter_no_change=10, random_state=42, max_iter=200)
        gate_dn.fit(X_tr_z, (y_tr > 0.0).astype(int))

        # --------------------------------------------------------------------------------------
        # CANDIDATE 1: MLP_Compact_Base
        # --------------------------------------------------------------------------------------
        def pred_c1(X, m=mlp_base, g=gamma_base):
            return np.maximum(0.0, m.predict(X)) * g

        t0 = time.perf_counter()
        _ = pred_c1(te_rec_sc[0]["X"])
        inference_times_us["MLP_Compact_Base"].append(((time.perf_counter() - t0) / len(te_rec_sc[0]["X"])) * 1e6)
        benchmark_results["MLP_Compact_Base"].append(evaluate_neural_pipeline(te_rec_sc, pred_c1))

        # --------------------------------------------------------------------------------------
        # CANDIDATE 2: MLP_Compact_RecipBrake (Thresholded at 0.5 sigma excess)
        # --------------------------------------------------------------------------------------
        def pred_c2(X, m=mlp_base, g=gamma_base, e_idx=edc_tap0_idx, f_idx=fds_tap0_idx):
            y_raw = np.maximum(0.0, m.predict(X)) * g
            z_edc = X[:, e_idx]
            z_fds = X[:, f_idx]
            excess_ext = np.maximum(0.0, z_edc - z_fds - 0.5)
            brake = np.clip(1.0 - (excess_ext / 2.0), 0.0, 1.0)
            return y_raw * brake

        t0 = time.perf_counter()
        _ = pred_c2(te_rec_sc[0]["X"])
        inference_times_us["MLP_Compact_RecipBrake"].append(((time.perf_counter() - t0) / len(te_rec_sc[0]["X"])) * 1e6)
        benchmark_results["MLP_Compact_RecipBrake"].append(evaluate_neural_pipeline(te_rec_sc, pred_c2))

        # --------------------------------------------------------------------------------------
        # CANDIDATE 3: TwoHeaded_HardGated (P >= 0.50 Discrete Cut)
        # --------------------------------------------------------------------------------------
        def pred_c3(X, gate=gate_std, reg=reg_pure, g=gamma_reg):
            p = gate.predict_proba(X)[:, 1] if gate.classes_.shape[0] > 1 else np.zeros(len(X))
            mag = np.maximum(0.0, reg.predict(X)) * g
            return np.where(p >= 0.50, mag, 0.0)

        t0 = time.perf_counter()
        _ = pred_c3(te_rec_sc[0]["X"])
        inference_times_us["TwoHeaded_HardGated"].append(((time.perf_counter() - t0) / len(te_rec_sc[0]["X"])) * 1e6)
        benchmark_results["TwoHeaded_HardGated"].append(evaluate_neural_pipeline(te_rec_sc, pred_c3))

        # --------------------------------------------------------------------------------------
        # CANDIDATE 4: TwoHeaded_Hysteresis (Schmitt Trigger + Antagonist Abort)
        # --------------------------------------------------------------------------------------
        def pred_c4(X, gate=gate_std, reg=reg_pure, g=gamma_reg, e_idx=edc_tap0_idx, f_idx=fds_tap0_idx):
            p = gate.predict_proba(X)[:, 1] if gate.classes_.shape[0] > 1 else np.zeros(len(X))
            mag = np.maximum(0.0, reg.predict(X)) * g
            z_edc = X[:, e_idx]
            z_fds = X[:, f_idx]
            antagonist_active = (z_edc > z_fds + 0.8) & (z_edc > 1.0)
            
            out = np.zeros(len(X), dtype=np.float64)
            in_flight = False
            for i in range(len(X)):
                if not in_flight:
                    if p[i] >= 0.55 and not antagonist_active[i]:
                        in_flight = True
                        out[i] = mag[i]
                else:
                    if p[i] < 0.35 or antagonist_active[i]:
                        in_flight = False
                        out[i] = 0.0
                    else:
                        out[i] = mag[i]
            return out

        t0 = time.perf_counter()
        _ = pred_c4(te_rec_sc[0]["X"])
        inference_times_us["TwoHeaded_Hysteresis"].append(((time.perf_counter() - t0) / len(te_rec_sc[0]["X"])) * 1e6)
        benchmark_results["TwoHeaded_Hysteresis"].append(evaluate_neural_pipeline(te_rec_sc, pred_c4))

        # --------------------------------------------------------------------------------------
        # CANDIDATE 5: TwoHeaded_DeepNarrow (Spatial Bottleneck Gate)
        # --------------------------------------------------------------------------------------
        def pred_c5(X, gate=gate_dn, reg=reg_pure, g=gamma_reg):
            p = gate.predict_proba(X)[:, 1] if gate.classes_.shape[0] > 1 else np.zeros(len(X))
            mag = np.maximum(0.0, reg.predict(X)) * g
            return np.where(p >= 0.50, mag, 0.0)

        t0 = time.perf_counter()
        _ = pred_c5(te_rec_sc[0]["X"])
        inference_times_us["TwoHeaded_DeepNarrow"].append(((time.perf_counter() - t0) / len(te_rec_sc[0]["X"])) * 1e6)
        benchmark_results["TwoHeaded_DeepNarrow"].append(evaluate_neural_pipeline(te_rec_sc, pred_c5))

        # --------------------------------------------------------------------------------------
        # CANDIDATE 6: TwoHeaded_FullSafe (Hard Gate + Normalized Antagonist Brake)
        # --------------------------------------------------------------------------------------
        def pred_c6(X, gate=gate_std, reg=reg_pure, g=gamma_reg, e_idx=edc_tap0_idx, f_idx=fds_tap0_idx):
            p = gate.predict_proba(X)[:, 1] if gate.classes_.shape[0] > 1 else np.zeros(len(X))
            mag = np.maximum(0.0, reg.predict(X)) * g
            z_edc = X[:, e_idx]
            z_fds = X[:, f_idx]
            is_valid_flight = (p >= 0.50) & ~((z_edc > z_fds + 0.8) & (z_edc > 1.0))
            return np.where(is_valid_flight, mag, 0.0)

        t0 = time.perf_counter()
        _ = pred_c6(te_rec_sc[0]["X"])
        inference_times_us["TwoHeaded_FullSafe"].append(((time.perf_counter() - t0) / len(te_rec_sc[0]["X"])) * 1e6)
        benchmark_results["TwoHeaded_FullSafe"].append(evaluate_neural_pipeline(te_rec_sc, pred_c6))

    # --- 8. Format Final Output Report ---
    table_header = (
        "========================================================================================================================\n"
        "             ADVANCED NEURAL ARCHITECTURES: SPEED MATCHING & SAFETY METRICS (10 SUBJECTS MEAN +/- STD)                  \n"
        "========================================================================================================================\n"
        " Architecture         | Mean Speed% | Peak Speed% | Vel Corr (r)| Onset Lat (ms) | M5 False % | Rest Jitter | Cost (μs)\n"
        "------------------------------------------------------------------------------------------------------------------------\n"
    )
    log_print(table_header, fh)

    for m in model_names:
        res_list = benchmark_results[m]
        mean_sp = [r["mean_speed_pct"] for r in res_list]
        peak_sp = [r["peak_speed_pct"] for r in res_list]
        r_vel   = [r["r_vel"] for r in res_list]
        lat_ms  = [r["onset_lat_ms"] for r in res_list]
        m5_fa   = [r["m5_fa"] for r in res_list]
        rj      = [r["rj"] for r in res_list]
        us_cost = inference_times_us[m]

        line = (
            f" {m:<20} | {np.mean(mean_sp):5.1f} +/- {np.std(mean_sp):4.1f}% | "
            f"{np.mean(peak_sp):5.1f} +/- {np.std(peak_sp):4.1f}% |  "
            f"{np.mean(r_vel):5.3f} +/- {np.std(r_vel):4.3f} |  "
            f"{np.mean(lat_ms):5.1f} +/- {np.std(lat_ms):4.1f} ms |  "
            f"{np.mean(m5_fa):4.2f}% |   {np.mean(rj):4.2f}% | "
            f"{np.mean(us_cost):6.1f} μs\n"
        )
        log_print(line, fh)

    log_print("========================================================================================================================\n", fh)
    print(f"\n[SUCCESS] Benchmark complete! Report written to: {LOG_FILE}\n")
    fh.close()

if __name__ == "__main__":
    main()

In [2]:
#!/usr/bin/env python3
"""
====================================================================================================
CONTINUOUS sEMG UNIPOLAR FLIGHT: ADVANCED NEURAL ARCHITECTURES & HYBRID GATING
Milestones  : Read directly from S{sub_id}_E1_A1_kinematic_trapezoid_log.csv (t1_takeoff_s, t2_plat_start_s)
Cohort      : Subjects 1 to 12 (Excluding S07 & S11; Inverting S05)
Target Lag  : tau = 60 ms retrospective target alignment
Cadence     : 10 ms hop (100 Hz cadence)
Focus       : Eliminate Rest Jitter (<5%) and M5 False Closures (<5%) while preserving ~100% Speed
Models      :
  1. MLP_Compact_Base       : Baseline single-head regressor (soft smooth deadband)
  2. MLP_Compact_RecipBrake : Single-head MLP + Normalized Reciprocal Brake (0.5 sigma deadband)
  3. TwoHeaded_HardGated    : Decoupled Gate (P >= 0.50) + Pure Active Regressor
  4. TwoHeaded_Hysteresis   : Schmitt Trigger (P_on = 0.55, P_off = 0.35) + Antagonist Abort
  5. TwoHeaded_DeepNarrow   : Deep bottleneck gate (48, 24, 12) + Regressor (32, 16)
  6. TwoHeaded_FullSafe     : Hard Gate (P >= 0.50) + Normalized Antagonist Brake
====================================================================================================
"""

import os
import sys
import time
import numpy as np
import pandas as pd
from scipy import signal, stats
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor, MLPClassifier

# --- 1. Cohort & Protocol Constants ---
COHORT = [1, 2, 3, 4, 5, 6, 8, 9, 10, 12]  # S07 & S11 permanently excluded
V_DEADBAND = 4.0        # deg/s: Actuator deadband floor
V_ONSET_THRESH = 12.0   # deg/s: Velocity threshold to register voluntary takeoff
DT_SEC = 0.010          # 10 ms hop (100 Hz cadence)
TAU_LAG_MS = 60         # 60 ms retrospective target alignment
LAG_HOPS = 6            # 60 ms / 10 ms
POST_TAKEOFF_SEC = 0.060  # Exactly 60 ms runway for retrospective pointer to reach t2
TIMEOUT_LATENCY_MS = 350.0  # Penalty latency for failed onset trigger
LOG_FILE = "cohort_advanced_neural_report.txt"

INVERT_S05_IN_PYTHON = True

# --- 2. Ground-Truth Milestone Unipolar Labeler ---
def construct_unipolar_labels_from_log(df_feat, df_trap):
    y_close = np.zeros(len(df_feat), dtype=np.float64)
    is_m6_takeoff = np.zeros(len(df_feat), dtype=bool)
    is_m6_stroke = np.zeros(len(df_feat), dtype=bool)
    is_m5_open = np.zeros(len(df_feat), dtype=bool)

    t_all = df_feat["timestamp_s"].values
    v_raw = df_feat["theta_vel_deg_s"].values
    stim = df_feat["stimulus"].values

    m6_log = df_trap[df_trap["movement"] == 6]

    for _, row in m6_log.iterrows():
        t1 = row["t1_takeoff_s"]
        t2 = row["t2_plat_start_s"]

        # 1. Active Takeoff Window (t1 -> t2)
        flight_mask = (t_all >= t1) & (t_all < t2)
        flight_idx = np.where(flight_mask)[0]

        if len(flight_idx) > 0:
            is_m6_takeoff[flight_idx] = True
            # Directional flexion speed (ignore any negative extension flutter)
            y_close[flight_idx] = np.maximum(0.0, v_raw[flight_idx])

        # 2. Flight Stroke Window (t1 -> t2 + 60 ms runway)
        stroke_mask = (t_all >= t1) & (t_all < (t2 + POST_TAKEOFF_SEC))
        stroke_idx = np.where(stroke_mask)[0]
        if len(stroke_idx) > 0:
            is_m6_stroke[stroke_idx] = True

    # Movement 5 (Hand Open): negative inhibitory data (target = 0.0)
    is_m5_open[stim == 5] = True

    df_labeled = df_feat.copy()
    df_labeled["y_close"] = y_close
    df_labeled["is_m6_takeoff"] = is_m6_takeoff
    df_labeled["is_m6_stroke"] = is_m6_stroke
    df_labeled["is_m5_open"] = is_m5_open
    return df_labeled

def find_contiguous_segments(df, max_dt=0.018, min_len=24):
    t = df["timestamp_s"].values
    if len(t) == 0:
        return []
    dt = np.diff(t)
    break_indices = np.where(dt > max_dt)[0] + 1
    segments = np.split(np.arange(len(df)), break_indices)
    return [seg for seg in segments if len(seg) >= min_len]

# --- 3. Retrospective Context Builder ---
def extract_speed_data(df, feat_cols):
    tap_offsets = [0, 2, 4, 6]  # [t, t-20ms, t-40ms, t-60ms]
    max_hist = max(tap_offsets)

    segments = find_contiguous_segments(df, min_len=max_hist + LAG_HOPS + 5)
    seg_records = []

    X_raw_all = df[feat_cols].values
    y_close_all = df["y_close"].values
    is_m6_all = df["is_m6_takeoff"].values
    is_stroke_all = df["is_m6_stroke"].values
    is_m5_all = df["is_m5_open"].values

    for seg in segments:
        X_seg = X_raw_all[seg]
        y_seg = y_close_all[seg]
        m6_seg = is_m6_all[seg]
        strk_seg = is_stroke_all[seg]
        m5_seg = is_m5_all[seg]
        L = len(seg)

        X_s, y_s, m6_s, strk_s, m5_s = [], [], [], [], []

        start_idx = max(max_hist, LAG_HOPS)
        for i in range(start_idx, L):
            target_idx = i - LAG_HOPS

            taps = [X_seg[i - k] for k in tap_offsets]
            X_s.append(np.hstack(taps))
            y_s.append(y_seg[target_idx])
            m6_s.append(m6_seg[target_idx])
            strk_s.append(strk_seg[target_idx])
            m5_s.append(m5_seg[target_idx])

        if len(X_s) > 0:
            seg_records.append({
                "X": np.array(X_s),
                "y": np.array(y_s),
                "is_m6_takeoff": np.array(m6_s),
                "is_m6_stroke": np.array(strk_s),
                "is_m5_open": np.array(m5_s)
            })

    return seg_records

# --- 4. Timeline Partitioning ---
def partition_timeline(df, sub_id, buffer_sec=1.0):
    test_reps = [2, 5] if sub_id == 4 else [2, 6]
    is_test = np.zeros(len(df), dtype=bool)

    for m in [5, 6]:
        for r in test_reps:
            cue_mask = (df["stimulus"] == m) & (df["repetition"] == r)
            if not np.any(cue_mask):
                continue
            t_start = df.loc[cue_mask, "timestamp_s"].min() - buffer_sec
            t_end = df.loc[cue_mask, "timestamp_s"].max() + buffer_sec
            is_test[(df["timestamp_s"] >= t_start) & (df["timestamp_s"] <= t_end)] = True

    is_corrupt = (df["repetition"] == 6).values if sub_id == 4 else np.zeros(len(df), dtype=bool)
    is_plateau = (df["is_hold_plateau"] == 1).values

    # Train: completely discards plateaus (never trains past t2)
    train_mask = ~is_corrupt & ~is_plateau & ~is_test
    # Test: keeps takeoff + 60 ms runway, M5 hand open, and full continuous rest
    test_mask = ~is_corrupt & (df["is_m6_stroke"] | df["is_m5_open"] | ~is_plateau) & is_test

    return train_mask, test_mask

# --- 5. Flight Deadband ---
def apply_flight_deadband(y_raw, v_db=V_DEADBAND):
    y = np.maximum(0.0, np.copy(y_raw))
    mask = y > v_db
    y_cmd = np.zeros_like(y)
    y_cmd[mask] = y[mask] * ((y[mask] - v_db) / y[mask])
    return y_cmd

# --- 6. Speed-Centric Evaluator ---
def evaluate_neural_pipeline(test_records, predict_fn, dt=DT_SEC):
    m6_y_true, m6_y_cmd = [], []
    m5_cmds, rest_cmds = [], []

    mean_speed_ratios = []
    peak_speed_ratios = []
    onset_latencies_ms = []

    for seg in test_records:
        y_pred = predict_fn(seg["X"])
        y_cmd = apply_flight_deadband(y_pred)

        y_seg = seg["y"]
        m6_tk = seg["is_m6_takeoff"]
        m6_strk = seg["is_m6_stroke"]
        m5_mask = seg["is_m5_open"]
        rest_mask = (~m6_strk) & (~m5_mask)

        if np.any(m6_tk):
            idx_tk = np.where(m6_tk)[0]
            yt_tk = y_seg[idx_tk]
            yp_tk = y_cmd[idx_tk]

            m6_y_true.extend(yt_tk)
            m6_y_cmd.extend(yp_tk)

            peak_t = np.max(yt_tk)
            peak_p = np.max(yp_tk)
            if peak_t > 15.0:
                peak_speed_ratios.append((peak_p / peak_t) * 100.0)

        if np.any(m6_strk):
            idx_strk = np.where(m6_strk)[0]
            yt_strk = y_seg[idx_strk]
            yp_strk = y_cmd[idx_strk]

            active_human = yt_strk > V_DEADBAND
            active_robot = yp_strk > V_DEADBAND

            if np.sum(active_human) > 5 and np.sum(active_robot) > 3:
                mean_t = np.mean(yt_strk[active_human])
                mean_p = np.mean(yp_strk[active_robot])
                mean_speed_ratios.append((mean_p / mean_t) * 100.0)
            elif np.sum(active_human) > 5 and np.sum(active_robot) <= 3:
                mean_speed_ratios.append(0.0)

            onset_t_rel = np.where(yt_strk >= V_ONSET_THRESH)[0]
            onset_p_rel = np.where(yp_strk >= V_ONSET_THRESH)[0]
            if len(onset_t_rel) > 0:
                if len(onset_p_rel) > 0:
                    action_lat = ((onset_p_rel[0] - onset_t_rel[0]) * dt * 1000.0) + TAU_LAG_MS
                    onset_latencies_ms.append(action_lat)
                else:
                    onset_latencies_ms.append(TIMEOUT_LATENCY_MS)

        if np.any(m5_mask):
            m5_cmds.extend(y_cmd[m5_mask])

        if np.any(rest_mask):
            rest_cmds.extend(y_cmd[rest_mask])

    yt_arr = np.array(m6_y_true, dtype=np.float64)
    yp_arr = np.array(m6_y_cmd, dtype=np.float64)

    if len(yt_arr) > 10 and np.std(yp_arr) > 0 and np.std(yt_arr) > 0:
        r_vel, _ = stats.pearsonr(yt_arr, yp_arr)
    else:
        r_vel = 0.0

    m5_cmds = np.array(m5_cmds)
    m5_fa = (np.sum(m5_cmds > 0.0) / len(m5_cmds) * 100.0) if len(m5_cmds) > 0 else 0.0

    rest_cmds = np.array(rest_cmds)
    rj = (np.sum(rest_cmds > 0.0) / len(rest_cmds) * 100.0) if len(rest_cmds) > 0 else 0.0

    return {
        "mean_speed_pct": np.mean(mean_speed_ratios) if mean_speed_ratios else 0.0,
        "peak_speed_pct": np.mean(peak_speed_ratios) if peak_speed_ratios else 0.0,
        "r_vel": r_vel,
        "onset_lat_ms": np.mean(onset_latencies_ms) if onset_latencies_ms else TIMEOUT_LATENCY_MS,
        "m5_fa": m5_fa,
        "rj": rj
    }

def log_print(msg, fh):
    print(msg, end="")
    fh.write(msg)

# --- 7. Main Benchmark Harness ---
def main():
    fh = open(LOG_FILE, "w", encoding="utf-8")

    header = (
        "========================================================================================================================\n"
        "           ADVANCED NEURAL ARCHITECTURES & HYBRID GATING: SPEED MATCHING & ONSET SUITE (tau = 60 ms)                    \n"
        "========================================================================================================================\n"
        f" Cohort      : {len(COHORT)} Subjects | Exclusions: S07 & S11 | S05 Polarity Swapped\n"
        " Milestones  : Ground-Truth t1_takeoff_s and t2_plat_start_s loaded directly from kinematic_trapezoid_log\n"
        " Focus Goals : Speed Match ~ 100% | Latency < 80 ms | M5 False Closing < 5% | Rest Jitter < 5%\n"
        "========================================================================================================================\n\n"
    )
    log_print(header, fh)

    subject_data = {}
    for sub_id in COHORT:
        feat_path = os.path.join("datasets", "DB2", f"DB2_s{sub_id}", f"S{sub_id}_E1_A1_dynamic_velocity_features.csv")
        trap_path = os.path.join("datasets", "DB2", f"DB2_s{sub_id}", f"S{sub_id}_E1_A1_kinematic_trapezoid_log.csv")

        if not os.path.exists(feat_path):
            feat_path = f"S{sub_id}_E1_A1_dynamic_velocity_features.csv"
        if not os.path.exists(trap_path):
            trap_path = f"S{sub_id}_E1_A1_kinematic_trapezoid_log.csv"

        if not os.path.exists(feat_path) or not os.path.exists(trap_path):
            continue

        df_raw = pd.read_csv(feat_path)
        df_trap = pd.read_csv(trap_path)

        if sub_id == 5 and INVERT_S05_IN_PYTHON:
            edc_cols = [c for c in df_raw.columns if c.startswith("EDC_")]
            for e_col in edc_cols:
                f_col = e_col.replace("EDC_", "FDS_")
                if f_col in df_raw.columns:
                    df_raw[e_col], df_raw[f_col] = df_raw[f_col].copy(), df_raw[e_col].copy()
            for ndi_col in ["NDI_MAV", "NDI_RMS", "NDI_WL"]:
                if ndi_col in df_raw.columns:
                    df_raw[ndi_col] = -df_raw[ndi_col]

        df_labeled = construct_unipolar_labels_from_log(df_raw, df_trap)
        meta_cols = ["timestamp_s", "theta_deg", "theta_vel_deg_s", "stimulus", "repetition", 
                     "is_hold_plateau", "y_close", "is_m6_takeoff", "is_m6_stroke", "is_m5_open"]
        feat_cols = [c for c in df_raw.columns if c not in meta_cols]

        tr_mask, te_mask = partition_timeline(df_labeled, sub_id)
        subject_data[sub_id] = {
            "train_df": df_labeled[tr_mask].copy(),
            "test_df": df_labeled[te_mask].copy(),
            "feat_cols": feat_cols
        }

    model_names = [
        "MLP_Compact_Base",
        "MLP_Compact_RecipBrake",
        "TwoHeaded_HardGated",
        "TwoHeaded_Hysteresis",
        "TwoHeaded_DeepNarrow",
        "TwoHeaded_FullSafe"
    ]

    benchmark_results = {m: [] for m in model_names}
    inference_times_us = {m: [] for m in model_names}

    print(">>> Executing Advanced Neural Speed-Matching Benchmark across 10 Subjects...")

    for sub_id, d in subject_data.items():
        tr_rec = extract_speed_data(d["train_df"], d["feat_cols"])
        te_rec = extract_speed_data(d["test_df"], d["feat_cols"])
        if not tr_rec or not te_rec:
            continue

        X_tr = np.vstack([r["X"] for r in tr_rec])
        y_tr = np.concatenate([r["y"] for r in tr_rec])
        scaler = StandardScaler().fit(X_tr)
        X_tr_z = scaler.transform(X_tr)

        te_rec_sc = []
        for r in te_rec:
            r_c = dict(r)
            r_c["X"] = np.clip(scaler.transform(r["X"]), -20.0, 20.0)
            te_rec_sc.append(r_c)

        active_tr = y_tr > 0.0

        fds_tap0_idx = d["feat_cols"].index("FDS_MAV")
        edc_tap0_idx = d["feat_cols"].index("EDC_MAV")

        def fit_gamma(pred_train):
            if not np.any(active_tr):
                return 1.0
            act_p = pred_train[active_tr]
            denom = np.sum(act_p ** 2)
            if denom < 1e-6:
                return 1.0
            return np.clip(np.sum(y_tr[active_tr] * act_p) / denom, 0.8, 1.5)

        # --------------------------------------------------------------------------------------
        # Shared Network Training
        # --------------------------------------------------------------------------------------
        mlp_base = MLPRegressor(hidden_layer_sizes=(32, 16), activation="relu", alpha=0.20,
                                early_stopping=True, n_iter_no_change=10, random_state=42, max_iter=200)
        mlp_base.fit(X_tr_z, y_tr)
        gamma_base = fit_gamma(np.maximum(0.0, mlp_base.predict(X_tr_z)))

        gate_std = MLPClassifier(hidden_layer_sizes=(32, 16), activation="relu", alpha=0.15,
                                 early_stopping=True, n_iter_no_change=10, random_state=42, max_iter=200)
        gate_std.fit(X_tr_z, (y_tr > 0.0).astype(int))

        reg_pure = MLPRegressor(hidden_layer_sizes=(32, 16), activation="relu", alpha=0.15,
                                early_stopping=True, n_iter_no_change=10, random_state=42, max_iter=200)
        if np.sum(active_tr) > 20:
            reg_pure.fit(X_tr_z[active_tr], y_tr[active_tr])
        else:
            reg_pure.fit(X_tr_z, y_tr)
        gamma_reg = fit_gamma(np.maximum(0.0, reg_pure.predict(X_tr_z)))

        gate_dn = MLPClassifier(hidden_layer_sizes=(48, 24, 12), activation="relu", alpha=0.20,
                                early_stopping=True, n_iter_no_change=10, random_state=42, max_iter=200)
        gate_dn.fit(X_tr_z, (y_tr > 0.0).astype(int))

        # --------------------------------------------------------------------------------------
        # CANDIDATE 1: MLP_Compact_Base
        # --------------------------------------------------------------------------------------
        def pred_c1(X, m=mlp_base, g=gamma_base):
            return np.maximum(0.0, m.predict(X)) * g

        t0 = time.perf_counter()
        _ = pred_c1(te_rec_sc[0]["X"])
        inference_times_us["MLP_Compact_Base"].append(((time.perf_counter() - t0) / len(te_rec_sc[0]["X"])) * 1e6)
        benchmark_results["MLP_Compact_Base"].append(evaluate_neural_pipeline(te_rec_sc, pred_c1))

        # --------------------------------------------------------------------------------------
        # CANDIDATE 2: MLP_Compact_RecipBrake (Thresholded at 0.5 sigma excess)
        # --------------------------------------------------------------------------------------
        def pred_c2(X, m=mlp_base, g=gamma_base, e_idx=edc_tap0_idx, f_idx=fds_tap0_idx):
            y_raw = np.maximum(0.0, m.predict(X)) * g
            z_edc = X[:, e_idx]
            z_fds = X[:, f_idx]
            excess_ext = np.maximum(0.0, z_edc - z_fds - 0.5)
            brake = np.clip(1.0 - (excess_ext / 2.0), 0.0, 1.0)
            return y_raw * brake

        t0 = time.perf_counter()
        _ = pred_c2(te_rec_sc[0]["X"])
        inference_times_us["MLP_Compact_RecipBrake"].append(((time.perf_counter() - t0) / len(te_rec_sc[0]["X"])) * 1e6)
        benchmark_results["MLP_Compact_RecipBrake"].append(evaluate_neural_pipeline(te_rec_sc, pred_c2))

        # --------------------------------------------------------------------------------------
        # CANDIDATE 3: TwoHeaded_HardGated (P >= 0.50 Discrete Cut)
        # --------------------------------------------------------------------------------------
        def pred_c3(X, gate=gate_std, reg=reg_pure, g=gamma_reg):
            p = gate.predict_proba(X)[:, 1] if gate.classes_.shape[0] > 1 else np.zeros(len(X))
            mag = np.maximum(0.0, reg.predict(X)) * g
            return np.where(p >= 0.50, mag, 0.0)

        t0 = time.perf_counter()
        _ = pred_c3(te_rec_sc[0]["X"])
        inference_times_us["TwoHeaded_HardGated"].append(((time.perf_counter() - t0) / len(te_rec_sc[0]["X"])) * 1e6)
        benchmark_results["TwoHeaded_HardGated"].append(evaluate_neural_pipeline(te_rec_sc, pred_c3))

        # --------------------------------------------------------------------------------------
        # CANDIDATE 4: TwoHeaded_Hysteresis (Schmitt Trigger + Antagonist Abort)
        # --------------------------------------------------------------------------------------
        def pred_c4(X, gate=gate_std, reg=reg_pure, g=gamma_reg, e_idx=edc_tap0_idx, f_idx=fds_tap0_idx):
            p = gate.predict_proba(X)[:, 1] if gate.classes_.shape[0] > 1 else np.zeros(len(X))
            mag = np.maximum(0.0, reg.predict(X)) * g
            z_edc = X[:, e_idx]
            z_fds = X[:, f_idx]
            antagonist_active = (z_edc > z_fds + 0.8) & (z_edc > 1.0)

            out = np.zeros(len(X), dtype=np.float64)
            in_flight = False
            for i in range(len(X)):
                if not in_flight:
                    if p[i] >= 0.55 and not antagonist_active[i]:
                        in_flight = True
                        out[i] = mag[i]
                else:
                    if p[i] < 0.35 or antagonist_active[i]:
                        in_flight = False
                        out[i] = 0.0
                    else:
                        out[i] = mag[i]
            return out

        t0 = time.perf_counter()
        _ = pred_c4(te_rec_sc[0]["X"])
        inference_times_us["TwoHeaded_Hysteresis"].append(((time.perf_counter() - t0) / len(te_rec_sc[0]["X"])) * 1e6)
        benchmark_results["TwoHeaded_Hysteresis"].append(evaluate_neural_pipeline(te_rec_sc, pred_c4))

        # --------------------------------------------------------------------------------------
        # CANDIDATE 5: TwoHeaded_DeepNarrow (Spatial Bottleneck Gate)
        # --------------------------------------------------------------------------------------
        def pred_c5(X, gate=gate_dn, reg=reg_pure, g=gamma_reg):
            p = gate.predict_proba(X)[:, 1] if gate.classes_.shape[0] > 1 else np.zeros(len(X))
            mag = np.maximum(0.0, reg.predict(X)) * g
            return np.where(p >= 0.50, mag, 0.0)

        t0 = time.perf_counter()
        _ = pred_c5(te_rec_sc[0]["X"])
        inference_times_us["TwoHeaded_DeepNarrow"].append(((time.perf_counter() - t0) / len(te_rec_sc[0]["X"])) * 1e6)
        benchmark_results["TwoHeaded_DeepNarrow"].append(evaluate_neural_pipeline(te_rec_sc, pred_c5))

        # --------------------------------------------------------------------------------------
        # CANDIDATE 6: TwoHeaded_FullSafe (Hard Gate + Normalized Antagonist Brake)
        # --------------------------------------------------------------------------------------
        def pred_c6(X, gate=gate_std, reg=reg_pure, g=gamma_reg, e_idx=edc_tap0_idx, f_idx=fds_tap0_idx):
            p = gate.predict_proba(X)[:, 1] if gate.classes_.shape[0] > 1 else np.zeros(len(X))
            mag = np.maximum(0.0, reg.predict(X)) * g
            z_edc = X[:, e_idx]
            z_fds = X[:, f_idx]
            is_valid_flight = (p >= 0.50) & ~((z_edc > z_fds + 0.8) & (z_edc > 1.0))
            return np.where(is_valid_flight, mag, 0.0)

        t0 = time.perf_counter()
        _ = pred_c6(te_rec_sc[0]["X"])
        inference_times_us["TwoHeaded_FullSafe"].append(((time.perf_counter() - t0) / len(te_rec_sc[0]["X"])) * 1e6)
        benchmark_results["TwoHeaded_FullSafe"].append(evaluate_neural_pipeline(te_rec_sc, pred_c6))

    # --- 8. Format Final Output Report ---
    table_header = (
        "========================================================================================================================\n"
        "             ADVANCED NEURAL ARCHITECTURES: SPEED MATCHING & SAFETY METRICS (10 SUBJECTS MEAN +/- STD)                  \n"
        "========================================================================================================================\n"
        " Architecture         | Mean Speed% | Peak Speed% | Vel Corr (r)| Onset Lat (ms) | M5 False % | Rest Jitter | Cost (μs)\n"
        "------------------------------------------------------------------------------------------------------------------------\n"
    )
    log_print(table_header, fh)

    for m in model_names:
        res_list = benchmark_results[m]
        mean_sp = [r["mean_speed_pct"] for r in res_list]
        peak_sp = [r["peak_speed_pct"] for r in res_list]
        r_vel   = [r["r_vel"] for r in res_list]
        lat_ms  = [r["onset_lat_ms"] for r in res_list]
        m5_fa   = [r["m5_fa"] for r in res_list]
        rj      = [r["rj"] for r in res_list]
        us_cost = inference_times_us[m]

        line = (
            f" {m:<20} | {np.mean(mean_sp):5.1f} +/- {np.std(mean_sp):4.1f}% | "
            f"{np.mean(peak_sp):5.1f} +/- {np.std(peak_sp):4.1f}% |  "
            f"{np.mean(r_vel):5.3f} +/- {np.std(r_vel):4.3f} |  "
            f"{np.mean(lat_ms):5.1f} +/- {np.std(lat_ms):4.1f} ms |  "
            f"{np.mean(m5_fa):4.2f}% |   {np.mean(rj):4.2f}% | "
            f"{np.mean(us_cost):6.1f} μs\n"
        )
        log_print(line, fh)

    log_print("========================================================================================================================\n", fh)
    print(f"\n[SUCCESS] Benchmark complete! Report written to: {LOG_FILE}\n")
    fh.close()

if __name__ == "__main__":
    main()

           ADVANCED NEURAL ARCHITECTURES & HYBRID GATING: SPEED MATCHING & ONSET SUITE (tau = 60 ms)                    
 Cohort      : 10 Subjects | Exclusions: S07 & S11 | S05 Polarity Swapped
 Milestones  : Ground-Truth t1_takeoff_s and t2_plat_start_s loaded directly from kinematic_trapezoid_log
 Focus Goals : Speed Match ~ 100% | Latency < 80 ms | M5 False Closing < 5% | Rest Jitter < 5%

>>> Executing Advanced Neural Speed-Matching Benchmark across 10 Subjects...


c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimiza

             ADVANCED NEURAL ARCHITECTURES: SPEED MATCHING & SAFETY METRICS (10 SUBJECTS MEAN +/- STD)                  
 Architecture         | Mean Speed% | Peak Speed% | Vel Corr (r)| Onset Lat (ms) | M5 False % | Rest Jitter | Cost (μs)
------------------------------------------------------------------------------------------------------------------------
 MLP_Compact_Base     |  63.2 +/- 22.2% | 109.0 +/- 44.0% |  0.022 +/- 0.241 |  163.5 +/- 125.1 ms |  11.13% |   16.86% |    1.9 μs
 MLP_Compact_RecipBrake |  56.0 +/- 16.7% |  94.8 +/- 26.5% |  -0.006 +/- 0.226 |  171.0 +/- 129.7 ms |  9.20% |   15.75% |    1.8 μs
 TwoHeaded_HardGated  |  92.0 +/- 62.5% |  84.8 +/- 49.9% |  0.007 +/- 0.144 |  343.5 +/- 143.3 ms |  0.59% |   1.41% |    4.7 μs
 TwoHeaded_Hysteresis |  77.6 +/- 64.4% |  73.0 +/- 39.5% |  0.017 +/- 0.126 |  414.5 +/- 130.3 ms |  0.49% |   1.42% |    5.4 μs
 TwoHeaded_DeepNarrow |  96.2 +/- 54.8% |  96.2 +/- 32.6% |  0.018 +/- 0.164 |  265.0 +/- 109.9 ms |  1.52% |   

In [3]:
#!/usr/bin/env python3
"""
====================================================================================================
CONTINUOUS sEMG UNIPOLAR FLIGHT: ADVANCED NEURAL ARCHITECTURES & HYBRID GATING
Milestones  : Read directly from S{sub_id}_E1_A1_kinematic_trapezoid_log.csv (t1_takeoff_s, t2_plat_start_s)
Cohort      : Subjects 1 to 12 (Excluding S07 & S11; Inverting S05)
Target Lag  : tau = 60 ms retrospective target alignment
Cadence     : 10 ms hop (100 Hz cadence)
Focus       : Latency < 80 ms, Speed Match ~ 100%, Rest Jitter < 5%, M5 False Closures < 5%
Diagnostics : Concrete per-subject sample counts, class balance ratios, and kinematic durations logged
====================================================================================================
"""

import os
import sys
import time
import numpy as np
import pandas as pd
from scipy import signal, stats
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor, MLPClassifier

# --- 1. Cohort & Protocol Constants ---
COHORT = [1, 2, 3, 4, 5, 6, 8, 9, 10, 12]  # S07 & S11 permanently excluded
V_DEADBAND = 4.0        # deg/s: Actuator deadband floor
V_ONSET_THRESH = 12.0   # deg/s: Velocity threshold to register voluntary takeoff
DT_SEC = 0.010          # 10 ms hop (100 Hz cadence)
TAU_LAG_MS = 60         # 60 ms retrospective target alignment
LAG_HOPS = 6            # 60 ms / 10 ms
POST_TAKEOFF_SEC = 0.060  # Exactly 60 ms runway for retrospective pointer to reach t2
TIMEOUT_LATENCY_MS = 350.0  # Penalty latency for failed onset trigger
LOG_FILE = "cohort_advanced_neural_report.txt"

INVERT_S05_IN_PYTHON = True

# --- 2. Ground-Truth Milestone Unipolar Labeler ---
def construct_unipolar_labels_from_log(df_feat, df_trap):
    y_close = np.zeros(len(df_feat), dtype=np.float64)
    is_m6_takeoff = np.zeros(len(df_feat), dtype=bool)
    is_m6_stroke = np.zeros(len(df_feat), dtype=bool)
    is_m5_open = np.zeros(len(df_feat), dtype=bool)

    t_all = df_feat["timestamp_s"].values
    v_raw = df_feat["theta_vel_deg_s"].values
    stim = df_feat["stimulus"].values

    m6_log = df_trap[df_trap["movement"] == 6]

    for _, row in m6_log.iterrows():
        t1 = row["t1_takeoff_s"]
        t2 = row["t2_plat_start_s"]

        if t2 <= t1:
            continue

        # 1. Active Takeoff Window (t1 -> t2): Ground-truth physical flight
        flight_mask = (t_all >= t1) & (t_all < t2)
        flight_idx = np.where(flight_mask)[0]

        if len(flight_idx) > 0:
            is_m6_takeoff[flight_idx] = True
            # Directional flexion speed: ignore negative extension twitches
            y_close[flight_idx] = np.maximum(0.0, v_raw[flight_idx])

        # 2. Flight Stroke Window (t1 -> t2 + 60 ms runway)
        stroke_mask = (t_all >= t1) & (t_all < (t2 + POST_TAKEOFF_SEC))
        stroke_idx = np.where(stroke_mask)[0]
        if len(stroke_idx) > 0:
            is_m6_stroke[stroke_idx] = True

    # Movement 5 (Hand Open): negative inhibitory data (target = 0.0)
    is_m5_open[stim == 5] = True

    df_labeled = df_feat.copy()
    df_labeled["y_close"] = y_close
    df_labeled["is_m6_takeoff"] = is_m6_takeoff
    df_labeled["is_m6_stroke"] = is_m6_stroke
    df_labeled["is_m5_open"] = is_m5_open
    return df_labeled

def find_contiguous_segments(df, max_dt=0.018, min_len=24):
    t = df["timestamp_s"].values
    if len(t) == 0:
        return []
    dt = np.diff(t)
    break_indices = np.where(dt > max_dt)[0] + 1
    segments = np.split(np.arange(len(df)), break_indices)
    return [seg for seg in segments if len(seg) >= min_len]

# --- 3. Retrospective Context Builder ---
def extract_speed_data(df, feat_cols):
    tap_offsets = [0, 2, 4, 6]  # [t, t-20ms, t-40ms, t-60ms]
    max_hist = max(tap_offsets)

    segments = find_contiguous_segments(df, min_len=max_hist + LAG_HOPS + 5)
    seg_records = []

    X_raw_all = df[feat_cols].values
    y_close_all = df["y_close"].values
    is_m6_all = df["is_m6_takeoff"].values
    is_stroke_all = df["is_m6_stroke"].values
    is_m5_all = df["is_m5_open"].values

    for seg in segments:
        X_seg = X_raw_all[seg]
        y_seg = y_close_all[seg]
        m6_seg = is_m6_all[seg]
        strk_seg = is_stroke_all[seg]
        m5_seg = is_m5_all[seg]
        L = len(seg)

        X_s, y_s, m6_s, strk_s, m5_s = [], [], [], [], []

        start_idx = max(max_hist, LAG_HOPS)
        for i in range(start_idx, L):
            target_idx = i - LAG_HOPS

            taps = [X_seg[i - k] for k in tap_offsets]
            X_s.append(np.hstack(taps))
            y_s.append(y_seg[target_idx])
            m6_s.append(m6_seg[target_idx])
            strk_seg_val = strk_seg[target_idx]
            strk_s.append(strk_seg_val)
            m5_seg_val = m5_seg[target_idx]
            m5_s.append(m5_seg_val)

        if len(X_s) > 0:
            seg_records.append({
                "X": np.array(X_s),
                "y": np.array(y_s),
                "is_m6_takeoff": np.array(m6_s),
                "is_m6_stroke": np.array(strk_s),
                "is_m5_open": np.array(m5_s)
            })

    return seg_records

# --- 4. Timeline Partitioning ---
def partition_timeline(df, sub_id, buffer_sec=1.0):
    test_reps = [2, 5] if sub_id == 4 else [2, 6]
    is_test = np.zeros(len(df), dtype=bool)

    for m in [5, 6]:
        for r in test_reps:
            cue_mask = (df["stimulus"] == m) & (df["repetition"] == r)
            if not np.any(cue_mask):
                continue
            t_start = df.loc[cue_mask, "timestamp_s"].min() - buffer_sec
            t_end = df.loc[cue_mask, "timestamp_s"].max() + buffer_sec
            is_test[(df["timestamp_s"] >= t_start) & (df["timestamp_s"] <= t_end)] = True

    is_corrupt = (df["repetition"] == 6).values if sub_id == 4 else np.zeros(len(df), dtype=bool)
    is_plateau = (df["is_hold_plateau"] == 1).values

    # Train: completely discards plateaus (never trains past t2)
    train_mask = ~is_corrupt & ~is_plateau & ~is_test
    # Test: keeps takeoff + 60 ms runway, M5 hand open, and full continuous rest
    test_mask = ~is_corrupt & (df["is_m6_stroke"] | df["is_m5_open"] | ~is_plateau) & is_test

    return train_mask, test_mask

# --- 5. Flight Deadband ---
def apply_flight_deadband(y_raw, v_db=V_DEADBAND):
    y = np.maximum(0.0, np.copy(y_raw))
    mask = y > v_db
    y_cmd = np.zeros_like(y)
    y_cmd[mask] = y[mask] * ((y[mask] - v_db) / y[mask])
    return y_cmd

# --- 6. Speed-Centric Evaluator ---
def evaluate_neural_pipeline(test_records, predict_fn, dt=DT_SEC):
    m6_y_true, m6_y_cmd = [], []
    m5_cmds, rest_cmds = [], []

    mean_speed_ratios = []
    peak_speed_ratios = []
    onset_latencies_ms = []

    for seg in test_records:
        y_pred = predict_fn(seg["X"])
        y_cmd = apply_flight_deadband(y_pred)

        y_seg = seg["y"]
        m6_tk = seg["is_m6_takeoff"]
        m6_strk = seg["is_m6_stroke"]
        m5_mask = seg["is_m5_open"]
        rest_mask = (~m6_strk) & (~m5_mask)

        if np.any(m6_tk):
            idx_tk = np.where(m6_tk)[0]
            yt_tk = y_seg[idx_tk]
            yp_tk = y_cmd[idx_tk]

            m6_y_true.extend(yt_tk)
            m6_y_cmd.extend(yp_tk)

            peak_t = np.max(yt_tk)
            peak_p = np.max(yp_tk)
            if peak_t > 15.0:
                peak_speed_ratios.append((peak_p / peak_t) * 100.0)

        if np.any(m6_strk):
            idx_strk = np.where(m6_strk)[0]
            yt_strk = y_seg[idx_strk]
            yp_strk = y_cmd[idx_strk]

            active_human = yt_strk > V_DEADBAND
            active_robot = yp_strk > V_DEADBAND

            if np.sum(active_human) > 5 and np.sum(active_robot) > 3:
                mean_t = np.mean(yt_strk[active_human])
                mean_p = np.mean(yp_strk[active_robot])
                mean_speed_ratios.append((mean_p / mean_t) * 100.0)
            elif np.sum(active_human) > 5 and np.sum(active_robot) <= 3:
                mean_speed_ratios.append(0.0)

            onset_t_rel = np.where(yt_strk >= V_ONSET_THRESH)[0]
            onset_p_rel = np.where(yp_strk >= V_ONSET_THRESH)[0]
            if len(onset_t_rel) > 0:
                if len(onset_p_rel) > 0:
                    action_lat = ((onset_p_rel[0] - onset_t_rel[0]) * dt * 1000.0) + TAU_LAG_MS
                    onset_latencies_ms.append(action_lat)
                else:
                    onset_latencies_ms.append(TIMEOUT_LATENCY_MS)

        if np.any(m5_mask):
            m5_cmds.extend(y_cmd[m5_mask])

        if np.any(rest_mask):
            rest_cmds.extend(y_cmd[rest_mask])

    yt_arr = np.array(m6_y_true, dtype=np.float64)
    yp_arr = np.array(m6_y_cmd, dtype=np.float64)

    if len(yt_arr) > 10 and np.std(yp_arr) > 0 and np.std(yt_arr) > 0:
        r_vel, _ = stats.pearsonr(yt_arr, yp_arr)
    else:
        r_vel = 0.0

    m5_cmds = np.array(m5_cmds)
    m5_fa = (np.sum(m5_cmds > 0.0) / len(m5_cmds) * 100.0) if len(m5_cmds) > 0 else 0.0

    rest_cmds = np.array(rest_cmds)
    rj = (np.sum(rest_cmds > 0.0) / len(rest_cmds) * 100.0) if len(rest_cmds) > 0 else 0.0

    return {
        "mean_speed_pct": np.mean(mean_speed_ratios) if mean_speed_ratios else 0.0,
        "peak_speed_pct": np.mean(peak_speed_ratios) if peak_speed_ratios else 0.0,
        "r_vel": r_vel,
        "onset_lat_ms": np.mean(onset_latencies_ms) if onset_latencies_ms else TIMEOUT_LATENCY_MS,
        "m5_fa": m5_fa,
        "rj": rj
    }

def log_print(msg, fh):
    print(msg, end="")
    fh.write(msg)

# --- 7. Main Benchmark Harness ---
def main():
    fh = open(LOG_FILE, "w", encoding="utf-8")

    header = (
        "========================================================================================================================\n"
        "           ADVANCED NEURAL ARCHITECTURES & HYBRID GATING: SPEED MATCHING & ONSET SUITE (tau = 60 ms)                    \n"
        "========================================================================================================================\n"
        f" Cohort      : {len(COHORT)} Subjects | Exclusions: S07 & S11 | S05 Polarity Swapped\n"
        " Milestones  : Ground-Truth t1_takeoff_s and t2_plat_start_s loaded directly from kinematic_trapezoid_log\n"
        " Optimization: Balanced Gating Prior + Calibrated P Thresholds to Resolve Latency\n"
        "========================================================================================================================\n\n"
    )
    log_print(header, fh)

    # 1. Subject Loading and Validation
    subject_data = {}
    diagnostic_summary = []

    for sub_id in COHORT:
        feat_path = os.path.join("datasets", "DB2", f"DB2_s{sub_id}", f"S{sub_id}_E1_A1_dynamic_velocity_features.csv")
        trap_path = os.path.join("datasets", "DB2", f"DB2_s{sub_id}", f"S{sub_id}_E1_A1_kinematic_trapezoid_log.csv")

        if not os.path.exists(feat_path):
            feat_path = f"S{sub_id}_E1_A1_dynamic_velocity_features.csv"
        if not os.path.exists(trap_path):
            trap_path = f"S{sub_id}_E1_A1_kinematic_trapezoid_log.csv"

        if not os.path.exists(feat_path) or not os.path.exists(trap_path):
            continue

        df_raw = pd.read_csv(feat_path)
        df_trap = pd.read_csv(trap_path)

        if sub_id == 5 and INVERT_S05_IN_PYTHON:
            edc_cols = [c for c in df_raw.columns if c.startswith("EDC_")]
            for e_col in edc_cols:
                f_col = e_col.replace("EDC_", "FDS_")
                if f_col in df_raw.columns:
                    df_raw[e_col], df_raw[f_col] = df_raw[f_col].copy(), df_raw[e_col].copy()
            for ndi_col in ["NDI_MAV", "NDI_RMS", "NDI_WL"]:
                if ndi_col in df_raw.columns:
                    df_raw[ndi_col] = -df_raw[ndi_col]

        df_labeled = construct_unipolar_labels_from_log(df_raw, df_trap)
        meta_cols = ["timestamp_s", "theta_deg", "theta_vel_deg_s", "stimulus", "repetition", 
                     "is_hold_plateau", "y_close", "is_m6_takeoff", "is_m6_stroke", "is_m5_open"]
        feat_cols = [c for c in df_raw.columns if c not in meta_cols]

        tr_mask, te_mask = partition_timeline(df_labeled, sub_id)
        subject_data[sub_id] = {
            "train_df": df_labeled[tr_mask].copy(),
            "test_df": df_labeled[te_mask].copy(),
            "feat_cols": feat_cols,
            "df_trap": df_trap
        }

    model_names = [
        "MLP_Compact_Base",
        "MLP_Compact_RecipBrake",
        "TwoHeaded_HardGated",
        "TwoHeaded_Hysteresis",
        "TwoHeaded_DeepNarrow",
        "TwoHeaded_FullSafe"
    ]

    benchmark_results = {m: [] for m in model_names}
    inference_times_us = {m: [] for m in model_names}

    # Header for diagnostics
    diag_header = (
        "------------------------------------------------------------------------------------------------------------------------\n"
        " DATASET & CLASS DISTRIBUTION DIAGNOSTICS (EMPIRICALLY VERIFIED PER SUBJECT)\n"
        "------------------------------------------------------------------------------------------------------------------------\n"
        " SubID | Feats | M6 Takeoffs (s) | Train Flight | Train Rest/M5 | Actual Ratio | Gating Balanced Set | Test Flight / Rest\n"
        "------------------------------------------------------------------------------------------------------------------------\n"
    )
    log_print(diag_header, fh)

    print(">>> Executing Advanced Neural Speed-Matching Benchmark across Cohort...")

    for sub_id, d in subject_data.items():
        tr_rec = extract_speed_data(d["train_df"], d["feat_cols"])
        te_rec = extract_speed_data(d["test_df"], d["feat_cols"])
        if not tr_rec or not te_rec:
            continue

        X_tr = np.vstack([r["X"] for r in tr_rec])
        y_tr = np.concatenate([r["y"] for r in tr_rec])
        scaler = StandardScaler().fit(X_tr)
        X_tr_z = scaler.transform(X_tr)

        te_rec_sc = []
        for r in te_rec:
            r_c = dict(r)
            r_c["X"] = np.clip(scaler.transform(r["X"]), -20.0, 20.0)
            te_rec_sc.append(r_c)

        active_tr = y_tr > 0.0

        fds_tap0_idx = d["feat_cols"].index("FDS_MAV")
        edc_tap0_idx = d["feat_cols"].index("EDC_MAV")

        def fit_gamma(pred_train):
            if not np.any(active_tr):
                return 1.0
            act_p = pred_train[active_tr]
            denom = np.sum(act_p ** 2)
            if denom < 1e-6:
                return 1.0
            return np.clip(np.sum(y_tr[active_tr] * act_p) / denom, 0.8, 1.5)

        # --------------------------------------------------------------------------------------
        # Shared Network Training
        # --------------------------------------------------------------------------------------
        # 1. Base MLP Regressor
        mlp_base = MLPRegressor(hidden_layer_sizes=(32, 16), activation="relu", alpha=0.20,
                                early_stopping=True, n_iter_no_change=10, random_state=42, max_iter=200)
        mlp_base.fit(X_tr_z, y_tr)
        gamma_base = fit_gamma(np.maximum(0.0, mlp_base.predict(X_tr_z)))

        # 2. Balanced Training for Gating Classifiers
        active_idx = np.where(active_tr)[0]
        rest_idx = np.where(~active_tr)[0]
        np.random.seed(42)
        n_act = len(active_idx)
        if n_act > 0 and len(rest_idx) > n_act:
            sampled_rest = np.random.choice(rest_idx, size=min(len(rest_idx), int(1.5 * n_act)), replace=False)
            gate_train_idx = np.concatenate([active_idx, sampled_rest])
        else:
            gate_train_idx = np.arange(len(y_tr))

        X_gate_tr = X_tr_z[gate_train_idx]
        y_gate_tr = (y_tr[gate_train_idx] > 0.0).astype(int)

        # --- DIAGNOSTIC LOGGING ---
        df_trap_m6 = d["df_trap"][d["df_trap"]["movement"] == 6]
        valid_durations = df_trap_m6["t2_plat_start_s"] - df_trap_m6["t1_takeoff_s"]
        mean_dur_s = valid_durations[valid_durations > 0].mean() if len(valid_durations) > 0 else 0.0

        n_flight_train = int(np.sum(active_tr))
        n_rest_train = int(np.sum(~active_tr))
        empirical_ratio = (n_rest_train / max(1, n_flight_train))

        y_te = np.concatenate([r["y"] for r in te_rec])
        n_flight_test = int(np.sum(y_te > 0.0))
        n_rest_test = int(np.sum(y_te == 0.0))

        diag_line = (
            f" S{sub_id:<4} | {len(d['feat_cols']):<5} | {mean_dur_s:6.3f} s        | "
            f"{n_flight_train:<12} | {n_rest_train:<13} | {empirical_ratio:5.1f}:1      | "
            f"{len(gate_train_idx):<19} | {n_flight_test} / {n_rest_test}\n"
        )
        log_print(diag_line, fh)
        diagnostic_summary.append({
            "flight_train": n_flight_train,
            "rest_train": n_rest_train,
            "ratio": empirical_ratio,
            "duration": mean_dur_s
        })

        # 3. Two-Headed Components (Standard Gating Head)
        gate_std = MLPClassifier(hidden_layer_sizes=(32, 16), activation="relu", alpha=0.15,
                                 early_stopping=True, n_iter_no_change=10, random_state=42, max_iter=200)
        gate_std.fit(X_gate_tr, y_gate_tr)

        # 4. Pure Flight Regressor (Trained strictly on active flight kinematics)
        reg_pure = MLPRegressor(hidden_layer_sizes=(32, 16), activation="relu", alpha=0.15,
                                early_stopping=True, n_iter_no_change=10, random_state=42, max_iter=200)
        if np.sum(active_tr) > 20:
            reg_pure.fit(X_tr_z[active_tr], y_tr[active_tr])
        else:
            reg_pure.fit(X_tr_z, y_tr)
        gamma_reg = fit_gamma(np.maximum(0.0, reg_pure.predict(X_tr_z)))

        # 5. Two-Headed Components (DeepNarrow Bottleneck Gate)
        gate_dn = MLPClassifier(hidden_layer_sizes=(48, 24, 12), activation="relu", alpha=0.20,
                                early_stopping=True, n_iter_no_change=10, random_state=42, max_iter=200)
        gate_dn.fit(X_gate_tr, y_gate_tr)

        # --------------------------------------------------------------------------------------
        # CANDIDATE 1: MLP_Compact_Base
        # --------------------------------------------------------------------------------------
        def pred_c1(X, m=mlp_base, g=gamma_base):
            return np.maximum(0.0, m.predict(X)) * g

        t0 = time.perf_counter()
        _ = pred_c1(te_rec_sc[0]["X"])
        inference_times_us["MLP_Compact_Base"].append(((time.perf_counter() - t0) / len(te_rec_sc[0]["X"])) * 1e6)
        benchmark_results["MLP_Compact_Base"].append(evaluate_neural_pipeline(te_rec_sc, pred_c1))

        # --------------------------------------------------------------------------------------
        # CANDIDATE 2: MLP_Compact_RecipBrake (Thresholded at 0.5 sigma excess)
        # --------------------------------------------------------------------------------------
        def pred_c2(X, m=mlp_base, g=gamma_base, e_idx=edc_tap0_idx, f_idx=fds_tap0_idx):
            y_raw = np.maximum(0.0, m.predict(X)) * g
            z_edc = X[:, e_idx]
            z_fds = X[:, f_idx]
            excess_ext = np.maximum(0.0, z_edc - z_fds - 0.5)
            brake = np.clip(1.0 - (excess_ext / 2.0), 0.0, 1.0)
            return y_raw * brake

        t0 = time.perf_counter()
        _ = pred_c2(te_rec_sc[0]["X"])
        inference_times_us["MLP_Compact_RecipBrake"].append(((time.perf_counter() - t0) / len(te_rec_sc[0]["X"])) * 1e6)
        benchmark_results["MLP_Compact_RecipBrake"].append(evaluate_neural_pipeline(te_rec_sc, pred_c2))

        # --------------------------------------------------------------------------------------
        # CANDIDATE 3: TwoHeaded_HardGated (Balanced Gate, Calibrated P >= 0.35)
        # --------------------------------------------------------------------------------------
        def pred_c3(X, gate=gate_std, reg=reg_pure, g=gamma_reg):
            p = gate.predict_proba(X)[:, 1] if gate.classes_.shape[0] > 1 else np.zeros(len(X))
            mag = np.maximum(0.0, reg.predict(X)) * g
            return np.where(p >= 0.35, mag, 0.0)

        t0 = time.perf_counter()
        _ = pred_c3(te_rec_sc[0]["X"])
        inference_times_us["TwoHeaded_HardGated"].append(((time.perf_counter() - t0) / len(te_rec_sc[0]["X"])) * 1e6)
        benchmark_results["TwoHeaded_HardGated"].append(evaluate_neural_pipeline(te_rec_sc, pred_c3))

        # --------------------------------------------------------------------------------------
        # CANDIDATE 4: TwoHeaded_Hysteresis (P_on = 0.40, P_off = 0.20 + Antagonist Abort)
        # --------------------------------------------------------------------------------------
        def pred_c4(X, gate=gate_std, reg=reg_pure, g=gamma_reg, e_idx=edc_tap0_idx, f_idx=fds_tap0_idx):
            p = gate.predict_proba(X)[:, 1] if gate.classes_.shape[0] > 1 else np.zeros(len(X))
            mag = np.maximum(0.0, reg.predict(X)) * g
            z_edc = X[:, e_idx]
            z_fds = X[:, f_idx]
            antagonist_active = (z_edc > z_fds + 0.8) & (z_edc > 1.0)

            out = np.zeros(len(X), dtype=np.float64)
            in_flight = False
            for i in range(len(X)):
                if not in_flight:
                    if p[i] >= 0.40 and not antagonist_active[i]:
                        in_flight = True
                        out[i] = mag[i]
                else:
                    if p[i] < 0.20 or antagonist_active[i]:
                        in_flight = False
                        out[i] = 0.0
                    else:
                        out[i] = mag[i]
            return out

        t0 = time.perf_counter()
        _ = pred_c4(te_rec_sc[0]["X"])
        inference_times_us["TwoHeaded_Hysteresis"].append(((time.perf_counter() - t0) / len(te_rec_sc[0]["X"])) * 1e6)
        benchmark_results["TwoHeaded_Hysteresis"].append(evaluate_neural_pipeline(te_rec_sc, pred_c4))

        # --------------------------------------------------------------------------------------
        # CANDIDATE 5: TwoHeaded_DeepNarrow (Bottleneck Gate 48->24->12, P >= 0.35)
        # --------------------------------------------------------------------------------------
        def pred_c5(X, gate=gate_dn, reg=reg_pure, g=gamma_reg):
            p = gate.predict_proba(X)[:, 1] if gate.classes_.shape[0] > 1 else np.zeros(len(X))
            mag = np.maximum(0.0, reg.predict(X)) * g
            return np.where(p >= 0.35, mag, 0.0)

        t0 = time.perf_counter()
        _ = pred_c5(te_rec_sc[0]["X"])
        inference_times_us["TwoHeaded_DeepNarrow"].append(((time.perf_counter() - t0) / len(te_rec_sc[0]["X"])) * 1e6)
        benchmark_results["TwoHeaded_DeepNarrow"].append(evaluate_neural_pipeline(te_rec_sc, pred_c5))

        # --------------------------------------------------------------------------------------
        # CANDIDATE 6: TwoHeaded_FullSafe (Balanced Gate P >= 0.35 + Normalized Antagonist Brake)
        # --------------------------------------------------------------------------------------
        def pred_c6(X, gate=gate_std, reg=reg_pure, g=gamma_reg, e_idx=edc_tap0_idx, f_idx=fds_tap0_idx):
            p = gate.predict_proba(X)[:, 1] if gate.classes_.shape[0] > 1 else np.zeros(len(X))
            mag = np.maximum(0.0, reg.predict(X)) * g
            z_edc = X[:, e_idx]
            z_fds = X[:, f_idx]
            is_valid_flight = (p >= 0.35) & ~((z_edc > z_fds + 0.8) & (z_edc > 1.0))
            return np.where(is_valid_flight, mag, 0.0)

        t0 = time.perf_counter()
        _ = pred_c6(te_rec_sc[0]["X"])
        inference_times_us["TwoHeaded_FullSafe"].append(((time.perf_counter() - t0) / len(te_rec_sc[0]["X"])) * 1e6)
        benchmark_results["TwoHeaded_FullSafe"].append(evaluate_neural_pipeline(te_rec_sc, pred_c6))

    # Log Diagnostic Summary Line
    if diagnostic_summary:
        mean_fl = np.mean([s["flight_train"] for s in diagnostic_summary])
        mean_rst = np.mean([s["rest_train"] for s in diagnostic_summary])
        mean_rat = np.mean([s["ratio"] for s in diagnostic_summary])
        mean_d = np.mean([s["duration"] for s in diagnostic_summary])
        diag_footer = (
            "------------------------------------------------------------------------------------------------------------------------\n"
            f" COHORT MEAN | 30    | {mean_d:6.3f} s        | {mean_fl:<12.1f} | {mean_rst:<13.1f} | {mean_rat:5.1f}:1      |\n"
            "------------------------------------------------------------------------------------------------------------------------\n\n"
        )
        log_print(diag_footer, fh)

    # --- 8. Format Final Output Report ---
    table_header = (
        "========================================================================================================================\n"
        "             ADVANCED NEURAL ARCHITECTURES: SPEED MATCHING & SAFETY METRICS (10 SUBJECTS MEAN +/- STD)                  \n"
        "========================================================================================================================\n"
        " Architecture         | Mean Speed% | Peak Speed% | Vel Corr (r)| Onset Lat (ms) | M5 False % | Rest Jitter | Cost (μs)\n"
        "------------------------------------------------------------------------------------------------------------------------\n"
    )
    log_print(table_header, fh)

    for m in model_names:
        res_list = benchmark_results[m]
        mean_sp = [r["mean_speed_pct"] for r in res_list]
        peak_sp = [r["peak_speed_pct"] for r in res_list]
        r_vel   = [r["r_vel"] for r in res_list]
        lat_ms  = [r["onset_lat_ms"] for r in res_list]
        m5_fa   = [r["m5_fa"] for r in res_list]
        rj      = [r["rj"] for r in res_list]
        us_cost = inference_times_us[m]

        line = (
            f" {m:<20} | {np.mean(mean_sp):5.1f} +/- {np.std(mean_sp):4.1f}% | "
            f"{np.mean(peak_sp):5.1f} +/- {np.std(peak_sp):4.1f}% |  "
            f"{np.mean(r_vel):5.3f} +/- {np.std(r_vel):4.3f} |  "
            f"{np.mean(lat_ms):5.1f} +/- {np.std(lat_ms):4.1f} ms |  "
            f"{np.mean(m5_fa):4.2f}% |   {np.mean(rj):4.2f}% | "
            f"{np.mean(us_cost):6.1f} μs\n"
        )
        log_print(line, fh)

    log_print("========================================================================================================================\n", fh)
    print(f"\n[SUCCESS] Benchmark complete! Report written to: {LOG_FILE}\n")
    fh.close()

if __name__ == "__main__":
    main()

           ADVANCED NEURAL ARCHITECTURES & HYBRID GATING: SPEED MATCHING & ONSET SUITE (tau = 60 ms)                    
 Cohort      : 10 Subjects | Exclusions: S07 & S11 | S05 Polarity Swapped
 Milestones  : Ground-Truth t1_takeoff_s and t2_plat_start_s loaded directly from kinematic_trapezoid_log
 Optimization: Balanced Gating Prior + Calibrated P Thresholds to Resolve Latency

------------------------------------------------------------------------------------------------------------------------
 DATASET & CLASS DISTRIBUTION DIAGNOSTICS (EMPIRICALLY VERIFIED PER SUBJECT)
------------------------------------------------------------------------------------------------------------------------
 SubID | Feats | M6 Takeoffs (s) | Train Flight | Train Rest/M5 | Actual Ratio | Gating Balanced Set | Test Flight / Rest
------------------------------------------------------------------------------------------------------------------------
>>> Executing Advanced Neural Speed-Matching Benchmark

c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


 S4    | 30    |  0.698 s        | 221          | 4969          |  22.5:1      | 552                 | 125 / 1970


c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


 S5    | 30    |  0.559 s        | 237          | 7135          |  30.1:1      | 592                 | 98 / 1365
 S6    | 30    |  0.970 s        | 439          | 6645          |  15.1:1      | 1097                | 143 / 1542
 S8    | 30    |  1.093 s        | 470          | 6805          |  14.5:1      | 1175                | 174 / 1208


c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


... (2 outputs left out)

 S10   | 30    |  0.904 s        | 379          | 6609          |  17.4:1      | 947                 | 133 / 1357


c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


 S12   | 30    |  0.564 s        | 218          | 6074          |  27.9:1      | 545                 | 120 / 2956
------------------------------------------------------------------------------------------------------------------------
 COHORT MEAN | 30    |  0.843 s        | 331.2        | 6422.9        |  21.3:1      |
------------------------------------------------------------------------------------------------------------------------

             ADVANCED NEURAL ARCHITECTURES: SPEED MATCHING & SAFETY METRICS (10 SUBJECTS MEAN +/- STD)                  
 Architecture         | Mean Speed% | Peak Speed% | Vel Corr (r)| Onset Lat (ms) | M5 False % | Rest Jitter | Cost (μs)
------------------------------------------------------------------------------------------------------------------------
 MLP_Compact_Base     |  63.2 +/- 22.2% | 109.0 +/- 44.0% |  0.022 +/- 0.241 |  163.5 +/- 125.1 ms |  11.13% |   16.86% |    1.3 μs
 MLP_Compact_RecipBrake |  56.0 +/- 16.7% |  94.8 +/- 26.5% | 

c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


In [4]:
#!/usr/bin/env python3
"""
====================================================================================================
CONTINUOUS sEMG UNIPOLAR FLIGHT: ADVANCED NEURAL ARCHITECTURES & HYBRID GATING
Milestones  : Read directly from S{sub_id}_E1_A1_kinematic_trapezoid_log.csv (t1_takeoff_s, t2_plat_start_s)
Cohort      : Subjects 1 to 12 (Excluding S07 & S11; Inverting S05)
Target Lag  : tau = 60 ms retrospective target alignment
Cadence     : 10 ms hop (100 Hz cadence)
Focus       : Latency < 80 ms, Speed Match ~ 100%, Rest Jitter < 5%, M5 False Closures < 5%
Rebalancing : Manifold SMOTE / Trajectory Interpolation (Preserves 100% of Rest/M5 Negative Distribution)
Diagnostics : Concrete per-subject sample counts, empirical class ratios, and sampler telemetry logged
====================================================================================================
"""

import os
import sys
import time
import numpy as np
import pandas as pd
from scipy import signal, stats
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.neural_network import MLPRegressor, MLPClassifier

# Try importing imbalanced-learn; provide vectorized trajectory interpolation fallback if not installed
try:
    from imblearn.over_sampling import SMOTE
    HAS_IMBLEARN = True
except ImportError:
    HAS_IMBLEARN = False

# --- 1. Cohort & Protocol Constants ---
COHORT = [1, 2, 3, 4, 5, 6, 8, 9, 10, 12]  # S07 & S11 permanently excluded
V_DEADBAND = 4.0        # deg/s: Actuator deadband floor
V_ONSET_THRESH = 12.0   # deg/s: Velocity threshold to register voluntary takeoff
DT_SEC = 0.010          # 10 ms hop (100 Hz cadence)
TAU_LAG_MS = 60         # 60 ms retrospective target alignment
LAG_HOPS = 6            # 60 ms / 10 ms
POST_TAKEOFF_SEC = 0.060  # Exactly 60 ms runway for retrospective pointer to reach t2
TIMEOUT_LATENCY_MS = 350.0  # Penalty latency for failed onset trigger
LOG_FILE = "cohort_advanced_neural_report.txt"

INVERT_S05_IN_PYTHON = True

# --- 2. Ground-Truth Milestone Unipolar Labeler ---
def construct_unipolar_labels_from_log(df_feat, df_trap):
    y_close = np.zeros(len(df_feat), dtype=np.float64)
    is_m6_takeoff = np.zeros(len(df_feat), dtype=bool)
    is_m6_stroke = np.zeros(len(df_feat), dtype=bool)
    is_m5_open = np.zeros(len(df_feat), dtype=bool)

    t_all = df_feat["timestamp_s"].values
    v_raw = df_feat["theta_vel_deg_s"].values
    stim = df_feat["stimulus"].values

    m6_log = df_trap[df_trap["movement"] == 6]

    for _, row in m6_log.iterrows():
        t1 = row["t1_takeoff_s"]
        t2 = row["t2_plat_start_s"]

        if t2 <= t1 or np.isnan(t1) or np.isnan(t2):
            continue

        # 1. Active Takeoff Window (t1 -> t2): Ground-truth physical flight
        flight_mask = (t_all >= t1) & (t_all < t2)
        flight_idx = np.where(flight_mask)[0]

        if len(flight_idx) > 0:
            is_m6_takeoff[flight_idx] = True
            # Directional flexion speed: ignore negative extension twitches
            y_close[flight_idx] = np.maximum(0.0, v_raw[flight_idx])

        # 2. Flight Stroke Window (t1 -> t2 + 60 ms runway)
        stroke_mask = (t_all >= t1) & (t_all < (t2 + POST_TAKEOFF_SEC))
        stroke_idx = np.where(stroke_mask)[0]
        if len(stroke_idx) > 0:
            is_m6_stroke[stroke_idx] = True

    # Movement 5 (Hand Open): negative inhibitory data (target = 0.0)
    is_m5_open[stim == 5] = True

    df_labeled = df_feat.copy()
    df_labeled["y_close"] = y_close
    df_labeled["is_m6_takeoff"] = is_m6_takeoff
    df_labeled["is_m6_stroke"] = is_m6_stroke
    df_labeled["is_m5_open"] = is_m5_open
    return df_labeled

def find_contiguous_segments(df, max_dt=0.018, min_len=24):
    t = df["timestamp_s"].values
    if len(t) == 0:
        return []
    dt = np.diff(t)
    break_indices = np.where(dt > max_dt)[0] + 1
    segments = np.split(np.arange(len(df)), break_indices)
    return [seg for seg in segments if len(seg) >= min_len]

# --- 3. Retrospective Context Builder ---
def extract_speed_data(df, feat_cols):
    tap_offsets = [0, 2, 4, 6]  # [t, t-20ms, t-40ms, t-60ms]
    max_hist = max(tap_offsets)

    segments = find_contiguous_segments(df, min_len=max_hist + LAG_HOPS + 5)
    seg_records = []

    X_raw_all = df[feat_cols].values
    y_close_all = df["y_close"].values
    is_m6_all = df["is_m6_takeoff"].values
    is_stroke_all = df["is_m6_stroke"].values
    is_m5_all = df["is_m5_open"].values

    for seg in segments:
        X_seg = X_raw_all[seg]
        y_seg = y_close_all[seg]
        m6_seg = is_m6_all[seg]
        strk_seg = is_stroke_all[seg]
        m5_seg = is_m5_all[seg]
        L = len(seg)

        X_s, y_s, m6_s, strk_s, m5_s = [], [], [], [], []

        start_idx = max(max_hist, LAG_HOPS)
        for i in range(start_idx, L):
            target_idx = i - LAG_HOPS

            taps = [X_seg[i - k] for k in tap_offsets]
            X_s.append(np.hstack(taps))
            y_s.append(y_seg[target_idx])
            m6_s.append(m6_seg[target_idx])
            strk_s.append(strk_seg[target_idx])
            m5_s.append(m5_seg[target_idx])

        if len(X_s) > 0:
            seg_records.append({
                "X": np.array(X_s),
                "y": np.array(y_s),
                "is_m6_takeoff": np.array(m6_s),
                "is_m6_stroke": np.array(strk_s),
                "is_m5_open": np.array(m5_s)
            })

    return seg_records

# --- 4. Timeline Partitioning ---
def partition_timeline(df, sub_id, buffer_sec=1.0):
    test_reps = [2, 5] if sub_id == 4 else [2, 6]
    is_test = np.zeros(len(df), dtype=bool)

    for m in [5, 6]:
        for r in test_reps:
            cue_mask = (df["stimulus"] == m) & (df["repetition"] == r)
            if not np.any(cue_mask):
                continue
            t_start = df.loc[cue_mask, "timestamp_s"].min() - buffer_sec
            t_end = df.loc[cue_mask, "timestamp_s"].max() + buffer_sec
            is_test[(df["timestamp_s"] >= t_start) & (df["timestamp_s"] <= t_end)] = True

    is_corrupt = (df["repetition"] == 6).values if sub_id == 4 else np.zeros(len(df), dtype=bool)
    is_plateau = (df["is_hold_plateau"] == 1).values

    # Train: completely discards plateaus (never trains past t2)
    train_mask = ~is_corrupt & ~is_plateau & ~is_test
    # Test: keeps takeoff + 60 ms runway, M5 hand open, and full continuous rest
    test_mask = ~is_corrupt & (df["is_m6_stroke"] | df["is_m5_open"] | ~is_plateau) & is_test

    return train_mask, test_mask

# --- 5. Robust Trajectory Oversampler (SMOTE / Manifold Interpolator) ---
def balance_flight_manifold(X_z, y_bin, random_state=42):
    """
    Balances Class 1 (Flight) against Class 0 (Rest/M5) to a 1:1 ratio.
    Retains 100% of genuine resting and M5 samples to prevent rest jitter.
    Synthesizes continuous voluntary flight points along physiological recruitment trajectories.
    """
    n_flight = int(np.sum(y_bin == 1))
    n_rest = int(np.sum(y_bin == 0))

    if n_flight <= 5 or n_rest <= n_flight:
        return X_z, y_bin, "Pass-Through"

    # Strategy A: Use imbalanced-learn SMOTE if available
    if HAS_IMBLEARN:
        k_neighbors = min(5, n_flight - 1)
        smote = SMOTE(sampling_strategy=1.0, k_neighbors=k_neighbors, random_state=random_state)
        X_bal, y_bal = smote.fit_resample(X_z, y_bin)
        return X_bal, y_bal, f"imbalanced-learn SMOTE (k={k_neighbors})"

    # Strategy B: Vectorized Manifold Trajectory Interpolator (Zero-dependency SMOTE)
    np.random.seed(random_state)
    flight_idx = np.where(y_bin == 1)[0]
    rest_idx = np.where(y_bin == 0)[0]
    n_needed = n_rest - n_flight

    k_nn = min(5, len(flight_idx) - 1)
    nn = NearestNeighbors(n_neighbors=k_nn + 1).fit(X_z[flight_idx])
    _, indices = nn.kneighbors(X_z[flight_idx])

    # Sample base points and their nearest neighbors along the voluntary flight manifold
    base_choices = np.random.choice(len(flight_idx), size=n_needed, replace=True)
    neighbor_choices = np.random.randint(1, k_nn + 1, size=n_needed)
    selected_neighbors = indices[base_choices, neighbor_choices]

    alphas = np.random.uniform(0.05, 0.95, size=(n_needed, 1))
    X_base = X_z[flight_idx[base_choices]]
    X_neighbor = X_z[flight_idx[selected_neighbors]]

    # Interpolate along the recruitment vector with mild physiological jitter (sigma = 0.02)
    X_synthetic = X_base + alphas * (X_neighbor - X_base)
    X_synthetic += np.random.normal(0, 0.02, size=X_synthetic.shape)

    X_bal = np.vstack([X_z[rest_idx], X_z[flight_idx], X_synthetic])
    y_bal = np.concatenate([np.zeros(len(rest_idx), dtype=int), np.ones(n_flight + n_needed, dtype=int)])

    shuffle_idx = np.random.permutation(len(y_bal))
    return X_bal[shuffle_idx], y_bal[shuffle_idx], f"Manifold-SMOTE Fallback (k={k_nn})"

# --- 6. Flight Deadband ---
def apply_flight_deadband(y_raw, v_db=V_DEADBAND):
    y = np.maximum(0.0, np.copy(y_raw))
    mask = y > v_db
    y_cmd = np.zeros_like(y)
    y_cmd[mask] = y[mask] * ((y[mask] - v_db) / y[mask])
    return y_cmd

# --- 7. Speed-Centric Evaluator ---
def evaluate_neural_pipeline(test_records, predict_fn, dt=DT_SEC):
    m6_y_true, m6_y_cmd = [], []
    m5_cmds, rest_cmds = [], []

    mean_speed_ratios = []
    peak_speed_ratios = []
    onset_latencies_ms = []

    for seg in test_records:
        y_pred = predict_fn(seg["X"])
        y_cmd = apply_flight_deadband(y_pred)

        y_seg = seg["y"]
        m6_tk = seg["is_m6_takeoff"]
        m6_strk = seg["is_m6_stroke"]
        m5_mask = seg["is_m5_open"]
        rest_mask = (~m6_strk) & (~m5_mask)

        if np.any(m6_tk):
            idx_tk = np.where(m6_tk)[0]
            yt_tk = y_seg[idx_tk]
            yp_tk = y_cmd[idx_tk]

            m6_y_true.extend(yt_tk)
            m6_y_cmd.extend(yp_tk)

            peak_t = np.max(yt_tk)
            peak_p = np.max(yp_tk)
            if peak_t > 15.0:
                peak_speed_ratios.append((peak_p / peak_t) * 100.0)

        if np.any(m6_strk):
            idx_strk = np.where(m6_strk)[0]
            yt_strk = y_seg[idx_strk]
            yp_strk = y_cmd[idx_strk]

            active_human = yt_strk > V_DEADBAND
            active_robot = yp_strk > V_DEADBAND

            if np.sum(active_human) > 5 and np.sum(active_robot) > 3:
                mean_t = np.mean(yt_strk[active_human])
                mean_p = np.mean(yp_strk[active_robot])
                mean_speed_ratios.append((mean_p / mean_t) * 100.0)
            elif np.sum(active_human) > 5 and np.sum(active_robot) <= 3:
                mean_speed_ratios.append(0.0)

            onset_t_rel = np.where(yt_strk >= V_ONSET_THRESH)[0]
            onset_p_rel = np.where(yp_strk >= V_ONSET_THRESH)[0]
            if len(onset_t_rel) > 0:
                if len(onset_p_rel) > 0:
                    action_lat = ((onset_p_rel[0] - onset_t_rel[0]) * dt * 1000.0) + TAU_LAG_MS
                    onset_latencies_ms.append(action_lat)
                else:
                    onset_latencies_ms.append(TIMEOUT_LATENCY_MS)

        if np.any(m5_mask):
            m5_cmds.extend(y_cmd[m5_mask])

        if np.any(rest_mask):
            rest_cmds.extend(y_cmd[rest_mask])

    yt_arr = np.array(m6_y_true, dtype=np.float64)
    yp_arr = np.array(m6_y_cmd, dtype=np.float64)

    if len(yt_arr) > 10 and np.std(yp_arr) > 0 and np.std(yt_arr) > 0:
        r_vel, _ = stats.pearsonr(yt_arr, yp_arr)
    else:
        r_vel = 0.0

    m5_cmds = np.array(m5_cmds)
    m5_fa = (np.sum(m5_cmds > 0.0) / len(m5_cmds) * 100.0) if len(m5_cmds) > 0 else 0.0

    rest_cmds = np.array(rest_cmds)
    rj = (np.sum(rest_cmds > 0.0) / len(rest_cmds) * 100.0) if len(rest_cmds) > 0 else 0.0

    return {
        "mean_speed_pct": np.mean(mean_speed_ratios) if mean_speed_ratios else 0.0,
        "peak_speed_pct": np.mean(peak_speed_ratios) if peak_speed_ratios else 0.0,
        "r_vel": r_vel,
        "onset_lat_ms": np.mean(onset_latencies_ms) if onset_latencies_ms else TIMEOUT_LATENCY_MS,
        "m5_fa": m5_fa,
        "rj": rj
    }

def log_print(msg, fh):
    print(msg, end="")
    fh.write(msg)

# --- 8. Main Benchmark Execution ---
def main():
    fh = open(LOG_FILE, "w", encoding="utf-8")

    header = (
        "========================================================================================================================\n"
        "           ADVANCED NEURAL ARCHITECTURES & HYBRID GATING: SPEED MATCHING & ONSET SUITE (tau = 60 ms)                    \n"
        "========================================================================================================================\n"
        f" Cohort      : {len(COHORT)} Subjects | Exclusions: S07 & S11 | S05 Polarity Swapped\n"
        " Milestones  : Ground-Truth t1_takeoff_s and t2_plat_start_s loaded directly from kinematic_trapezoid_log\n"
        " Rebalancing : 100% of Rest/M5 Retained + Manifold SMOTE Trajectory Synthesis on Voluntary Flight\n"
        " Calibration : Bayesian Prior Balanced to 1:1 | High-Confidence Decision Boundary P >= 0.50\n"
        "========================================================================================================================\n\n"
    )
    log_print(header, fh)

    subject_data = {}
    diagnostic_summary = []

    for sub_id in COHORT:
        feat_path = os.path.join("datasets", "DB2", f"DB2_s{sub_id}", f"S{sub_id}_E1_A1_dynamic_velocity_features.csv")
        trap_path = os.path.join("datasets", "DB2", f"DB2_s{sub_id}", f"S{sub_id}_E1_A1_kinematic_trapezoid_log.csv")

        if not os.path.exists(feat_path):
            feat_path = f"S{sub_id}_E1_A1_dynamic_velocity_features.csv"
        if not os.path.exists(trap_path):
            trap_path = f"S{sub_id}_E1_A1_kinematic_trapezoid_log.csv"

        if not os.path.exists(feat_path) or not os.path.exists(trap_path):
            continue

        df_raw = pd.read_csv(feat_path)
        df_trap = pd.read_csv(trap_path)

        if sub_id == 5 and INVERT_S05_IN_PYTHON:
            edc_cols = [c for c in df_raw.columns if c.startswith("EDC_")]
            for e_col in edc_cols:
                f_col = e_col.replace("EDC_", "FDS_")
                if f_col in df_raw.columns:
                    df_raw[e_col], df_raw[f_col] = df_raw[f_col].copy(), df_raw[e_col].copy()
            for ndi_col in ["NDI_MAV", "NDI_RMS", "NDI_WL"]:
                if ndi_col in df_raw.columns:
                    df_raw[ndi_col] = -df_raw[ndi_col]

        df_labeled = construct_unipolar_labels_from_log(df_raw, df_trap)
        meta_cols = ["timestamp_s", "theta_deg", "theta_vel_deg_s", "stimulus", "repetition", 
                     "is_hold_plateau", "y_close", "is_m6_takeoff", "is_m6_stroke", "is_m5_open"]
        feat_cols = [c for c in df_raw.columns if c not in meta_cols]

        tr_mask, te_mask = partition_timeline(df_labeled, sub_id)
        subject_data[sub_id] = {
            "train_df": df_labeled[tr_mask].copy(),
            "test_df": df_labeled[te_mask].copy(),
            "feat_cols": feat_cols,
            "df_trap": df_trap
        }

    model_names = [
        "MLP_Compact_Base",
        "MLP_Compact_RecipBrake",
        "TwoHeaded_HardGated",
        "TwoHeaded_Hysteresis",
        "TwoHeaded_DeepNarrow",
        "TwoHeaded_FullSafe"
    ]

    benchmark_results = {m: [] for m in model_names}
    inference_times_us = {m: [] for m in model_names}

    diag_header = (
        "------------------------------------------------------------------------------------------------------------------------\n"
        " DATASET & CLASS DISTRIBUTION DIAGNOSTICS (EMPIRICALLY VERIFIED PER SUBJECT)\n"
        "------------------------------------------------------------------------------------------------------------------------\n"
        " SubID | Feats | M6 Takeoffs (s) | Train Flight | Train Rest/M5 | Actual Ratio | Gating Balanced Set | Sampler Engine\n"
        "------------------------------------------------------------------------------------------------------------------------\n"
    )
    log_print(diag_header, fh)

    print(">>> Executing Advanced Neural Speed-Matching Benchmark across Cohort...")

    for sub_id, d in subject_data.items():
        tr_rec = extract_speed_data(d["train_df"], d["feat_cols"])
        te_rec = extract_speed_data(d["test_df"], d["feat_cols"])
        if not tr_rec or not te_rec:
            continue

        X_tr = np.vstack([r["X"] for r in tr_rec])
        y_tr = np.concatenate([r["y"] for r in tr_rec])
        scaler = StandardScaler().fit(X_tr)
        X_tr_z = scaler.transform(X_tr)

        te_rec_sc = []
        for r in te_rec:
            r_c = dict(r)
            r_c["X"] = np.clip(scaler.transform(r["X"]), -20.0, 20.0)
            te_rec_sc.append(r_c)

        active_tr = y_tr > 0.0

        fds_tap0_idx = d["feat_cols"].index("FDS_MAV")
        edc_tap0_idx = d["feat_cols"].index("EDC_MAV")

        def fit_gamma(pred_train):
            if not np.any(active_tr):
                return 1.0
            act_p = pred_train[active_tr]
            denom = np.sum(act_p ** 2)
            if denom < 1e-6:
                return 1.0
            return np.clip(np.sum(y_tr[active_tr] * act_p) / denom, 0.8, 1.5)

        # --------------------------------------------------------------------------------------
        # Shared Network Training
        # --------------------------------------------------------------------------------------
        # 1. Base MLP Regressor
        mlp_base = MLPRegressor(hidden_layer_sizes=(32, 16), activation="relu", alpha=0.20,
                                early_stopping=True, n_iter_no_change=10, random_state=42, max_iter=200)
        mlp_base.fit(X_tr_z, y_tr)
        gamma_base = fit_gamma(np.maximum(0.0, mlp_base.predict(X_tr_z)))

        # 2. Balanced Training for Gating Classifiers via Manifold SMOTE
        y_gate_binary = (y_tr > 0.0).astype(int)
        X_gate_tr, y_gate_tr, sampler_name = balance_flight_manifold(X_tr_z, y_gate_binary)

        # --- DIAGNOSTIC LOGGING ---
        df_trap_m6 = d["df_trap"][d["df_trap"]["movement"] == 6]
        valid_durations = df_trap_m6["t2_plat_start_s"] - df_trap_m6["t1_takeoff_s"]
        mean_dur_s = valid_durations[valid_durations > 0].mean() if len(valid_durations) > 0 else 0.0

        n_flight_train = int(np.sum(active_tr))
        n_rest_train = int(np.sum(~active_tr))
        empirical_ratio = (n_rest_train / max(1, n_flight_train))

        diag_line = (
            f" S{sub_id:<4} | {len(d['feat_cols']):<5} | {mean_dur_s:6.3f} s        | "
            f"{n_flight_train:<12} | {n_rest_train:<13} | {empirical_ratio:5.1f}:1      | "
            f"{len(gate_train_idx_size := y_gate_tr):<19} | {sampler_name}\n"
        )
        log_print(diag_line, fh)
        diagnostic_summary.append({
            "flight_train": n_flight_train,
            "rest_train": n_rest_train,
            "ratio": empirical_ratio,
            "duration": mean_dur_s
        })

        # 3. Two-Headed Components (Standard Gating Head)
        gate_std = MLPClassifier(hidden_layer_sizes=(32, 16), activation="relu", alpha=0.15,
                                 early_stopping=True, n_iter_no_change=10, random_state=42, max_iter=200)
        gate_std.fit(X_gate_tr, y_gate_tr)

        # 4. Pure Flight Regressor (Trained strictly on active flight kinematics)
        reg_pure = MLPRegressor(hidden_layer_sizes=(32, 16), activation="relu", alpha=0.15,
                                early_stopping=True, n_iter_no_change=10, random_state=42, max_iter=200)
        if np.sum(active_tr) > 20:
            reg_pure.fit(X_tr_z[active_tr], y_tr[active_tr])
        else:
            reg_pure.fit(X_tr_z, y_tr)
        gamma_reg = fit_gamma(np.maximum(0.0, reg_pure.predict(X_tr_z)))

        # 5. Two-Headed Components (DeepNarrow Bottleneck Gate)
        gate_dn = MLPClassifier(hidden_layer_sizes=(48, 24, 12), activation="relu", alpha=0.20,
                                early_stopping=True, n_iter_no_change=10, random_state=42, max_iter=200)
        gate_dn.fit(X_gate_tr, y_gate_tr)

        # --------------------------------------------------------------------------------------
        # CANDIDATE 1: MLP_Compact_Base
        # --------------------------------------------------------------------------------------
        def pred_c1(X, m=mlp_base, g=gamma_base):
            return np.maximum(0.0, m.predict(X)) * g

        t0 = time.perf_counter()
        _ = pred_c1(te_rec_sc[0]["X"])
        inference_times_us["MLP_Compact_Base"].append(((time.perf_counter() - t0) / len(te_rec_sc[0]["X"])) * 1e6)
        benchmark_results["MLP_Compact_Base"].append(evaluate_neural_pipeline(te_rec_sc, pred_c1))

        # --------------------------------------------------------------------------------------
        # CANDIDATE 2: MLP_Compact_RecipBrake (Thresholded at 0.5 sigma excess)
        # --------------------------------------------------------------------------------------
        def pred_c2(X, m=mlp_base, g=gamma_base, e_idx=edc_tap0_idx, f_idx=fds_tap0_idx):
            y_raw = np.maximum(0.0, m.predict(X)) * g
            z_edc = X[:, e_idx]
            z_fds = X[:, f_idx]
            excess_ext = np.maximum(0.0, z_edc - z_fds - 0.5)
            brake = np.clip(1.0 - (excess_ext / 2.0), 0.0, 1.0)
            return y_raw * brake

        t0 = time.perf_counter()
        _ = pred_c2(te_rec_sc[0]["X"])
        inference_times_us["MLP_Compact_RecipBrake"].append(((time.perf_counter() - t0) / len(te_rec_sc[0]["X"])) * 1e6)
        benchmark_results["MLP_Compact_RecipBrake"].append(evaluate_neural_pipeline(te_rec_sc, pred_c2))

        # --------------------------------------------------------------------------------------
        # CANDIDATE 3: TwoHeaded_HardGated (SMOTE Gate, Calibrated P >= 0.50 Decision Cut)
        # --------------------------------------------------------------------------------------
        def pred_c3(X, gate=gate_std, reg=reg_pure, g=gamma_reg):
            p = gate.predict_proba(X)[:, 1] if gate.classes_.shape[0] > 1 else np.zeros(len(X))
            mag = np.maximum(0.0, reg.predict(X)) * g
            return np.where(p >= 0.50, mag, 0.0)

        t0 = time.perf_counter()
        _ = pred_c3(te_rec_sc[0]["X"])
        inference_times_us["TwoHeaded_HardGated"].append(((time.perf_counter() - t0) / len(te_rec_sc[0]["X"])) * 1e6)
        benchmark_results["TwoHeaded_HardGated"].append(evaluate_neural_pipeline(te_rec_sc, pred_c3))

        # --------------------------------------------------------------------------------------
        # CANDIDATE 4: TwoHeaded_Hysteresis (P_on = 0.55, P_off = 0.35 + Antagonist Abort)
        # --------------------------------------------------------------------------------------
        def pred_c4(X, gate=gate_std, reg=reg_pure, g=gamma_reg, e_idx=edc_tap0_idx, f_idx=fds_tap0_idx):
            p = gate.predict_proba(X)[:, 1] if gate.classes_.shape[0] > 1 else np.zeros(len(X))
            mag = np.maximum(0.0, reg.predict(X)) * g
            z_edc = X[:, e_idx]
            z_fds = X[:, f_idx]
            antagonist_active = (z_edc > z_fds + 0.8) & (z_edc > 1.0)

            out = np.zeros(len(X), dtype=np.float64)
            in_flight = False
            for i in range(len(X)):
                if not in_flight:
                    if p[i] >= 0.55 and not antagonist_active[i]:
                        in_flight = True
                        out[i] = mag[i]
                else:
                    if p[i] < 0.35 or antagonist_active[i]:
                        in_flight = False
                        out[i] = 0.0
                    else:
                        out[i] = mag[i]
            return out

        t0 = time.perf_counter()
        _ = pred_c4(te_rec_sc[0]["X"])
        inference_times_us["TwoHeaded_Hysteresis"].append(((time.perf_counter() - t0) / len(te_rec_sc[0]["X"])) * 1e6)
        benchmark_results["TwoHeaded_Hysteresis"].append(evaluate_neural_pipeline(te_rec_sc, pred_c4))

        # --------------------------------------------------------------------------------------
        # CANDIDATE 5: TwoHeaded_DeepNarrow (Spatial Bottleneck Gate 48->24->12, P >= 0.50)
        # --------------------------------------------------------------------------------------
        def pred_c5(X, gate=gate_dn, reg=reg_pure, g=gamma_reg):
            p = gate.predict_proba(X)[:, 1] if gate.classes_.shape[0] > 1 else np.zeros(len(X))
            mag = np.maximum(0.0, reg.predict(X)) * g
            return np.where(p >= 0.50, mag, 0.0)

        t0 = time.perf_counter()
        _ = pred_c5(te_rec_sc[0]["X"])
        inference_times_us["TwoHeaded_DeepNarrow"].append(((time.perf_counter() - t0) / len(te_rec_sc[0]["X"])) * 1e6)
        benchmark_results["TwoHeaded_DeepNarrow"].append(evaluate_neural_pipeline(te_rec_sc, pred_c5))

        # --------------------------------------------------------------------------------------
        # CANDIDATE 6: TwoHeaded_FullSafe (Hard Gate P >= 0.50 + Normalized Antagonist Brake)
        # --------------------------------------------------------------------------------------
        def pred_c6(X, gate=gate_std, reg=reg_pure, g=gamma_reg, e_idx=edc_tap0_idx, f_idx=fds_tap0_idx):
            p = gate.predict_proba(X)[:, 1] if gate.classes_.shape[0] > 1 else np.zeros(len(X))
            mag = np.maximum(0.0, reg.predict(X)) * g
            z_edc = X[:, e_idx]
            z_fds = X[:, f_idx]
            is_valid_flight = (p >= 0.50) & ~((z_edc > z_fds + 0.8) & (z_edc > 1.0))
            return np.where(is_valid_flight, mag, 0.0)

        t0 = time.perf_counter()
        _ = pred_c6(te_rec_sc[0]["X"])
        inference_times_us["TwoHeaded_FullSafe"].append(((time.perf_counter() - t0) / len(te_rec_sc[0]["X"])) * 1e6)
        benchmark_results["TwoHeaded_FullSafe"].append(evaluate_neural_pipeline(te_rec_sc, pred_c6))

    # Log Diagnostic Summary Line
    if diagnostic_summary:
        mean_fl = np.mean([s["flight_train"] for s in diagnostic_summary])
        mean_rst = np.mean([s["rest_train"] for s in diagnostic_summary])
        mean_rat = np.mean([s["ratio"] for s in diagnostic_summary])
        mean_d = np.mean([s["duration"] for s in diagnostic_summary])
        diag_footer = (
            "------------------------------------------------------------------------------------------------------------------------\n"
            f" COHORT MEAN | 30    | {mean_d:6.3f} s        | {mean_fl:<12.1f} | {mean_rst:<13.1f} | {mean_rat:5.1f}:1      |\n"
            "------------------------------------------------------------------------------------------------------------------------\n\n"
        )
        log_print(diag_footer, fh)

    # --- 9. Format Final Output Report ---
    table_header = (
        "========================================================================================================================\n"
        "             ADVANCED NEURAL ARCHITECTURES: SPEED MATCHING & SAFETY METRICS (10 SUBJECTS MEAN +/- STD)                  \n"
        "========================================================================================================================\n"
        " Architecture         | Mean Speed% | Peak Speed% | Vel Corr (r)| Onset Lat (ms) | M5 False % | Rest Jitter | Cost (μs)\n"
        "------------------------------------------------------------------------------------------------------------------------\n"
    )
    log_print(table_header, fh)

    for m in model_names:
        res_list = benchmark_results[m]
        mean_sp = [r["mean_speed_pct"] for r in res_list]
        peak_sp = [r["peak_speed_pct"] for r in res_list]
        r_vel   = [r["r_vel"] for r in res_list]
        lat_ms  = [r["onset_lat_ms"] for r in res_list]
        m5_fa   = [r["m5_fa"] for r in res_list]
        rj      = [r["rj"] for r in res_list]
        us_cost = inference_times_us[m]

        line = (
            f" {m:<20} | {np.mean(mean_sp):5.1f} +/- {np.std(mean_sp):4.1f}% | "
            f"{np.mean(peak_sp):5.1f} +/- {np.std(peak_sp):4.1f}% |  "
            f"{np.mean(r_vel):5.3f} +/- {np.std(r_vel):4.3f} |  "
            f"{np.mean(lat_ms):5.1f} +/- {np.std(lat_ms):4.1f} ms |  "
            f"{np.mean(m5_fa):4.2f}% |   {np.mean(rj):4.2f}% | "
            f"{np.mean(us_cost):6.1f} μs\n"
        )
        log_print(line, fh)

    log_print("========================================================================================================================\n", fh)
    print(f"\n[SUCCESS] Benchmark complete! Report written to: {LOG_FILE}\n")
    fh.close()

if __name__ == "__main__":
    main()

           ADVANCED NEURAL ARCHITECTURES & HYBRID GATING: SPEED MATCHING & ONSET SUITE (tau = 60 ms)                    
 Cohort      : 10 Subjects | Exclusions: S07 & S11 | S05 Polarity Swapped
 Milestones  : Ground-Truth t1_takeoff_s and t2_plat_start_s loaded directly from kinematic_trapezoid_log
 Rebalancing : 100% of Rest/M5 Retained + Manifold SMOTE Trajectory Synthesis on Voluntary Flight
 Calibration : Bayesian Prior Balanced to 1:1 | High-Confidence Decision Boundary P >= 0.50

------------------------------------------------------------------------------------------------------------------------
 DATASET & CLASS DISTRIBUTION DIAGNOSTICS (EMPIRICALLY VERIFIED PER SUBJECT)
------------------------------------------------------------------------------------------------------------------------
 SubID | Feats | M6 Takeoffs (s) | Train Flight | Train Rest/M5 | Actual Ratio | Gating Balanced Set | Sampler Engine
-----------------------------------------------------------------------

c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


 S4    | 30    |  0.698 s        | 221          | 4969          |  22.5:1      | 9938                | imbalanced-learn SMOTE (k=5)


c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


 S5    | 30    |  0.559 s        | 237          | 7135          |  30.1:1      | 14270               | imbalanced-learn SMOTE (k=5)
 S6    | 30    |  0.970 s        | 439          | 6645          |  15.1:1      | 13290               | imbalanced-learn SMOTE (k=5)
 S8    | 30    |  1.093 s        | 470          | 6805          |  14.5:1      | 13610               | imbalanced-learn SMOTE (k=5)


c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


... (3 outputs left out)

c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


 S12   | 30    |  0.564 s        | 218          | 6074          |  27.9:1      | 12148               | imbalanced-learn SMOTE (k=5)


c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


------------------------------------------------------------------------------------------------------------------------
 COHORT MEAN | 30    |  0.843 s        | 331.2        | 6422.9        |  21.3:1      |
------------------------------------------------------------------------------------------------------------------------

             ADVANCED NEURAL ARCHITECTURES: SPEED MATCHING & SAFETY METRICS (10 SUBJECTS MEAN +/- STD)                  
 Architecture         | Mean Speed% | Peak Speed% | Vel Corr (r)| Onset Lat (ms) | M5 False % | Rest Jitter | Cost (μs)
------------------------------------------------------------------------------------------------------------------------
 MLP_Compact_Base     |  63.2 +/- 22.2% | 109.0 +/- 44.0% |  0.022 +/- 0.241 |  163.5 +/- 125.1 ms |  11.13% |   16.86% |    1.2 μs
 MLP_Compact_RecipBrake |  56.0 +/- 16.7% |  94.8 +/- 26.5% |  -0.006 +/- 0.226 |  171.0 +/- 129.7 ms |  9.20% |   15.75% |    1.2 μs
 TwoHeaded_HardGated  | 104.3 +/- 46.4% | 

In [5]:
#!/usr/bin/env python3
"""
====================================================================================================
CONTINUOUS sEMG UNIPOLAR FLIGHT: ABLATION BENCHMARK (THRESHOLD TUNING & ZERO-LAG CONTEMPORANEOUS)
Milestones  : Read directly from S{sub_id}_E1_A1_kinematic_trapezoid_log.csv (t1_takeoff_s, t2_plat_start_s)
Cohort      : Subjects 1 to 12 (Excluding S07 & S11; Inverting S05)
Cadence     : 10 ms hop (100 Hz cadence)
Focus       : Latency < 80 ms, Speed Match ~ 100%, Rest Jitter < 5%, M5 False Closures < 5%

Ablation Dimensions (Evaluated Independently):
  1. Threshold Calibration : P >= 0.50 (Standard) vs P >= 0.42 (Early Recruitment Trigger)
  2. Target Alignment Lag  : tau = 60 ms (60 ms history) vs tau = 0 ms (Contemporaneous, 200 ms history)
====================================================================================================
"""

import os
import sys
import time
import numpy as np
import pandas as pd
from scipy import signal, stats
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.neural_network import MLPRegressor, MLPClassifier

try:
    from imblearn.over_sampling import SMOTE
    HAS_IMBLEARN = True
except ImportError:
    HAS_IMBLEARN = False

# --- 1. Cohort & Protocol Constants ---
COHORT = [1, 2, 3, 4, 5, 6, 8, 9, 10, 12]  # S07 & S11 permanently excluded
V_DEADBAND = 4.0        # deg/s: Actuator deadband floor
V_ONSET_THRESH = 12.0   # deg/s: Velocity threshold to register voluntary takeoff
DT_SEC = 0.010          # 10 ms hop (100 Hz cadence)
TIMEOUT_LATENCY_MS = 350.0  # Penalty latency for failed onset trigger
LOG_FILE = "cohort_advanced_neural_report.txt"

INVERT_S05_IN_PYTHON = True

# --- 2. Ground-Truth Milestone Unipolar Labeler ---
def construct_unipolar_labels_from_log(df_feat, df_trap, post_takeoff_sec=0.060):
    y_close = np.zeros(len(df_feat), dtype=np.float64)
    is_m6_takeoff = np.zeros(len(df_feat), dtype=bool)
    is_m6_stroke = np.zeros(len(df_feat), dtype=bool)
    is_m5_open = np.zeros(len(df_feat), dtype=bool)

    t_all = df_feat["timestamp_s"].values
    v_raw = df_feat["theta_vel_deg_s"].values
    stim = df_feat["stimulus"].values

    m6_log = df_trap[df_trap["movement"] == 6]

    for _, row in m6_log.iterrows():
        t1 = row["t1_takeoff_s"]
        t2 = row["t2_plat_start_s"]

        if t2 <= t1 or np.isnan(t1) or np.isnan(t2):
            continue

        # 1. Active Takeoff Window (t1 -> t2): Ground-truth physical flight
        flight_mask = (t_all >= t1) & (t_all < t2)
        flight_idx = np.where(flight_mask)[0]

        if len(flight_idx) > 0:
            is_m6_takeoff[flight_idx] = True
            y_close[flight_idx] = np.maximum(0.0, v_raw[flight_idx])

        # 2. Flight Stroke Window (t1 -> t2 + runway)
        stroke_mask = (t_all >= t1) & (t_all < (t2 + post_takeoff_sec))
        stroke_idx = np.where(stroke_mask)[0]
        if len(stroke_idx) > 0:
            is_m6_stroke[stroke_idx] = True

    is_m5_open[stim == 5] = True

    df_labeled = df_feat.copy()
    df_labeled["y_close"] = y_close
    df_labeled["is_m6_takeoff"] = is_m6_takeoff
    df_labeled["is_m6_stroke"] = is_m6_stroke
    df_labeled["is_m5_open"] = is_m5_open
    return df_labeled

def find_contiguous_segments(df, max_dt=0.018, min_len=24):
    t = df["timestamp_s"].values
    if len(t) == 0:
        return []
    dt = np.diff(t)
    break_indices = np.where(dt > max_dt)[0] + 1
    segments = np.split(np.arange(len(df)), break_indices)
    return [seg for seg in segments if len(seg) >= min_len]

# --- 3. Retrospective Context Builder ---
def extract_speed_data(df, feat_cols, tap_offsets, lag_hops):
    max_hist = max(tap_offsets)
    segments = find_contiguous_segments(df, min_len=max_hist + lag_hops + 5)
    seg_records = []

    X_raw_all = df[feat_cols].values
    y_close_all = df["y_close"].values
    is_m6_all = df["is_m6_takeoff"].values
    is_stroke_all = df["is_m6_stroke"].values
    is_m5_all = df["is_m5_open"].values

    for seg in segments:
        X_seg = X_raw_all[seg]
        y_seg = y_close_all[seg]
        m6_seg = is_m6_all[seg]
        strk_seg = is_stroke_all[seg]
        m5_seg = is_m5_all[seg]
        L = len(seg)

        X_s, y_s, m6_s, strk_s, m5_s = [], [], [], [], []
        start_idx = max(max_hist, lag_hops)

        for i in range(start_idx, L):
            target_idx = i - lag_hops
            taps = [X_seg[i - k] for k in tap_offsets]
            X_s.append(np.hstack(taps))
            y_s.append(y_seg[target_idx])
            m6_s.append(m6_seg[target_idx])
            strk_s.append(strk_seg[target_idx])
            m5_s.append(m5_seg[target_idx])

        if len(X_s) > 0:
            seg_records.append({
                "X": np.array(X_s),
                "y": np.array(y_s),
                "is_m6_takeoff": np.array(m6_s),
                "is_m6_stroke": np.array(strk_s),
                "is_m5_open": np.array(m5_s)
            })

    return seg_records

# --- 4. Timeline Partitioning ---
def partition_timeline(df, sub_id, buffer_sec=1.0):
    test_reps = [2, 5] if sub_id == 4 else [2, 6]
    is_test = np.zeros(len(df), dtype=bool)

    for m in [5, 6]:
        for r in test_reps:
            cue_mask = (df["stimulus"] == m) & (df["repetition"] == r)
            if not np.any(cue_mask):
                continue
            t_start = df.loc[cue_mask, "timestamp_s"].min() - buffer_sec
            t_end = df.loc[cue_mask, "timestamp_s"].max() + buffer_sec
            is_test[(df["timestamp_s"] >= t_start) & (df["timestamp_s"] <= t_end)] = True

    is_corrupt = (df["repetition"] == 6).values if sub_id == 4 else np.zeros(len(df), dtype=bool)
    is_plateau = (df["is_hold_plateau"] == 1).values

    train_mask = ~is_corrupt & ~is_plateau & ~is_test
    test_mask = ~is_corrupt & (df["is_m6_stroke"] | df["is_m5_open"] | ~is_plateau) & is_test
    return train_mask, test_mask

# --- 5. Robust Trajectory Oversampler (SMOTE) ---
def balance_flight_manifold(X_z, y_bin, random_state=42):
    n_flight = int(np.sum(y_bin == 1))
    n_rest = int(np.sum(y_bin == 0))

    if n_flight <= 5 or n_rest <= n_flight:
        return X_z, y_bin, "Pass-Through"

    if HAS_IMBLEARN:
        k_neighbors = min(5, n_flight - 1)
        smote = SMOTE(sampling_strategy=1.0, k_neighbors=k_neighbors, random_state=random_state)
        X_bal, y_bal = smote.fit_resample(X_z, y_bin)
        return X_bal, y_bal, f"imbalanced-learn SMOTE (k={k_neighbors})"

    np.random.seed(random_state)
    flight_idx = np.where(y_bin == 1)[0]
    rest_idx = np.where(y_bin == 0)[0]
    n_needed = n_rest - n_flight

    k_nn = min(5, len(flight_idx) - 1)
    nn = NearestNeighbors(n_neighbors=k_nn + 1).fit(X_z[flight_idx])
    _, indices = nn.kneighbors(X_z[flight_idx])

    base_choices = np.random.choice(len(flight_idx), size=n_needed, replace=True)
    neighbor_choices = np.random.randint(1, k_nn + 1, size=n_needed)
    selected_neighbors = indices[base_choices, neighbor_choices]

    alphas = np.random.uniform(0.05, 0.95, size=(n_needed, 1))
    X_base = X_z[flight_idx[base_choices]]
    X_neighbor = X_z[flight_idx[selected_neighbors]]

    X_synthetic = X_base + alphas * (X_neighbor - X_base)
    X_synthetic += np.random.normal(0, 0.02, size=X_synthetic.shape)

    X_bal = np.vstack([X_z[rest_idx], X_z[flight_idx], X_synthetic])
    y_bal = np.concatenate([np.zeros(len(rest_idx), dtype=int), np.ones(n_flight + n_needed, dtype=int)])
    shuffle_idx = np.random.permutation(len(y_bal))
    return X_bal[shuffle_idx], y_bal[shuffle_idx], f"Manifold-SMOTE Fallback (k={k_nn})"

# --- 6. Flight Deadband ---
def apply_flight_deadband(y_raw, v_db=V_DEADBAND):
    y = np.maximum(0.0, np.copy(y_raw))
    mask = y > v_db
    y_cmd = np.zeros_like(y)
    y_cmd[mask] = y[mask] * ((y[mask] - v_db) / y[mask])
    return y_cmd

# --- 7. Speed-Centric Evaluator ---
def evaluate_neural_pipeline(test_records, predict_fn, tau_lag_ms, dt=DT_SEC):
    m6_y_true, m6_y_cmd = [], []
    m5_cmds, rest_cmds = [], []
    mean_speed_ratios = []
    peak_speed_ratios = []
    onset_latencies_ms = []

    for seg in test_records:
        y_pred = predict_fn(seg["X"])
        y_cmd = apply_flight_deadband(y_pred)

        y_seg = seg["y"]
        m6_tk = seg["is_m6_takeoff"]
        m6_strk = seg["is_m6_stroke"]
        m5_mask = seg["is_m5_open"]
        rest_mask = (~m6_strk) & (~m5_mask)

        if np.any(m6_tk):
            idx_tk = np.where(m6_tk)[0]
            yt_tk = y_seg[idx_tk]
            yp_tk = y_cmd[idx_tk]

            m6_y_true.extend(yt_tk)
            m6_y_cmd.extend(yp_tk)

            peak_t = np.max(yt_tk)
            peak_p = np.max(yp_tk)
            if peak_t > 15.0:
                peak_speed_ratios.append((peak_p / peak_t) * 100.0)

        if np.any(m6_strk):
            idx_strk = np.where(m6_strk)[0]
            yt_strk = y_seg[idx_strk]
            yp_strk = y_cmd[idx_strk]

            active_human = yt_strk > V_DEADBAND
            active_robot = yp_strk > V_DEADBAND

            if np.sum(active_human) > 5 and np.sum(active_robot) > 3:
                mean_t = np.mean(yt_strk[active_human])
                mean_p = np.mean(yp_strk[active_robot])
                mean_speed_ratios.append((mean_p / mean_t) * 100.0)
            elif np.sum(active_human) > 5 and np.sum(active_robot) <= 3:
                mean_speed_ratios.append(0.0)

            onset_t_rel = np.where(yt_strk >= V_ONSET_THRESH)[0]
            onset_p_rel = np.where(yp_strk >= V_ONSET_THRESH)[0]
            if len(onset_t_rel) > 0:
                if len(onset_p_rel) > 0:
                    action_lat = ((onset_p_rel[0] - onset_t_rel[0]) * dt * 1000.0) + tau_lag_ms
                    onset_latencies_ms.append(action_lat)
                else:
                    onset_latencies_ms.append(TIMEOUT_LATENCY_MS)

        if np.any(m5_mask):
            m5_cmds.extend(y_cmd[m5_mask])

        if np.any(rest_mask):
            rest_cmds.extend(y_cmd[rest_mask])

    yt_arr = np.array(m6_y_true, dtype=np.float64)
    yp_arr = np.array(m6_y_cmd, dtype=np.float64)

    if len(yt_arr) > 10 and np.std(yp_arr) > 0 and np.std(yt_arr) > 0:
        r_vel, _ = stats.pearsonr(yt_arr, yp_arr)
    else:
        r_vel = 0.0

    m5_cmds = np.array(m5_cmds)
    m5_fa = (np.sum(m5_cmds > 0.0) / len(m5_cmds) * 100.0) if len(m5_cmds) > 0 else 0.0

    rest_cmds = np.array(rest_cmds)
    rj = (np.sum(rest_cmds > 0.0) / len(rest_cmds) * 100.0) if len(rest_cmds) > 0 else 0.0

    return {
        "mean_speed_pct": np.mean(mean_speed_ratios) if mean_speed_ratios else 0.0,
        "peak_speed_pct": np.mean(peak_speed_ratios) if peak_speed_ratios else 0.0,
        "r_vel": r_vel,
        "onset_lat_ms": np.mean(onset_latencies_ms) if onset_latencies_ms else TIMEOUT_LATENCY_MS,
        "m5_fa": m5_fa,
        "rj": rj
    }

def log_print(msg, fh):
    print(msg, end="")
    fh.write(msg)

# --- 8. Main Benchmark Execution ---
def main():
    fh = open(LOG_FILE, "w", encoding="utf-8")

    header = (
        "========================================================================================================================\n"
        "           ADVANCED NEURAL ARCHITECTURES: THRESHOLD & ZERO-LAG ABLATION SUITE (10 SUBJECTS)                             \n"
        "========================================================================================================================\n"
        f" Cohort      : {len(COHORT)} Subjects | Exclusions: S07 & S11 | S05 Polarity Swapped\n"
        " Milestones  : Ground-Truth t1_takeoff_s and t2_plat_start_s loaded directly from kinematic_trapezoid_log\n"
        " Rebalancing : 100% of Rest/M5 Retained + Manifold SMOTE Trajectory Synthesis on Voluntary Flight\n"
        " Focus Goals : Speed Match ~ 100% | Latency < 80 ms | M5 False Closing < 5% | Rest Jitter < 5%\n"
        "========================================================================================================================\n\n"
    )
    log_print(header, fh)

    # Load Raw Datasets
    raw_subjects = {}
    diagnostic_summary = []

    for sub_id in COHORT:
        feat_path = os.path.join("datasets", "DB2", f"DB2_s{sub_id}", f"S{sub_id}_E1_A1_dynamic_velocity_features.csv")
        trap_path = os.path.join("datasets", "DB2", f"DB2_s{sub_id}", f"S{sub_id}_E1_A1_kinematic_trapezoid_log.csv")

        if not os.path.exists(feat_path):
            feat_path = f"S{sub_id}_E1_A1_dynamic_velocity_features.csv"
        if not os.path.exists(trap_path):
            trap_path = f"S{sub_id}_E1_A1_kinematic_trapezoid_log.csv"

        if not os.path.exists(feat_path) or not os.path.exists(trap_path):
            continue

        df_raw = pd.read_csv(feat_path)
        df_trap = pd.read_csv(trap_path)

        if sub_id == 5 and INVERT_S05_IN_PYTHON:
            edc_cols = [c for c in df_raw.columns if c.startswith("EDC_")]
            for e_col in edc_cols:
                f_col = e_col.replace("EDC_", "FDS_")
                if f_col in df_raw.columns:
                    df_raw[e_col], df_raw[f_col] = df_raw[f_col].copy(), df_raw[e_col].copy()
            for ndi_col in ["NDI_MAV", "NDI_RMS", "NDI_WL"]:
                if ndi_col in df_raw.columns:
                    df_raw[ndi_col] = -df_raw[ndi_col]

        meta_cols = ["timestamp_s", "theta_deg", "theta_vel_deg_s", "stimulus", "repetition", 
                     "is_hold_plateau", "y_close", "is_m6_takeoff", "is_m6_stroke", "is_m5_open"]
        feat_cols = [c for c in df_raw.columns if c not in meta_cols]

        raw_subjects[sub_id] = {
            "df_raw": df_raw,
            "df_trap": df_trap,
            "feat_cols": feat_cols
        }

    # Diagnostic Header
    diag_header = (
        "------------------------------------------------------------------------------------------------------------------------\n"
        " DATASET & CLASS DISTRIBUTION DIAGNOSTICS (EMPIRICALLY VERIFIED PER SUBJECT)\n"
        "------------------------------------------------------------------------------------------------------------------------\n"
        " SubID | Feats | M6 Takeoffs (s) | Train Flight | Train Rest/M5 | Actual Ratio | Gating Balanced Set | Sampler Engine\n"
        "------------------------------------------------------------------------------------------------------------------------\n"
    )
    log_print(diag_header, fh)

    # Ablation Evaluation Registry
    eval_models = [
        # Condition A: Standard Retrospective Target Alignment (tau = 60 ms, 60 ms History Taps)
        "Lag60_Base_P50",
        "Lag60_HardGated_P50",
        "Lag60_FullSafe_P50",
        "Lag60_HardGated_P42",
        "Lag60_FullSafe_P42",

        # Condition B: Contemporaneous Instantaneous Control (tau = 0 ms, 200 ms Deep History Taps)
        "ZeroLag_Base_P50",
        "ZeroLag_HardGated_P50",
        "ZeroLag_FullSafe_P50",
        "ZeroLag_HardGated_P42",
        "ZeroLag_FullSafe_P42"
    ]

    benchmark_results = {m: [] for m in eval_models}
    inference_times_us = {m: [] for m in eval_models}

    print(">>> Executing Independent Threshold & Zero-Lag Ablation across 10 Subjects...")

    for sub_id, d in raw_subjects.items():
        df_raw = d["df_raw"]
        df_trap = d["df_trap"]
        feat_cols = d["feat_cols"]

        fds_tap0_idx = feat_cols.index("FDS_MAV")
        edc_tap0_idx = feat_cols.index("EDC_MAV")

        # ======================================================================================
        # PIPELINE A: tau = 60 ms Retrospective Lag (60 ms Taps: [0, 20, 40, 60 ms])
        # ======================================================================================
        df_labeled_60 = construct_unipolar_labels_from_log(df_raw, df_trap, post_takeoff_sec=0.060)
        tr_mask_60, te_mask_60 = partition_timeline(df_labeled_60, sub_id)

        taps_60 = [0, 2, 4, 6]
        tr_rec_60 = extract_speed_data(df_labeled_60[tr_mask_60], feat_cols, taps_60, lag_hops=6)
        te_rec_60 = extract_speed_data(df_labeled_60[te_mask_60], feat_cols, taps_60, lag_hops=6)

        X_tr_60 = np.vstack([r["X"] for r in tr_rec_60])
        y_tr_60 = np.concatenate([r["y"] for r in tr_rec_60])
        scaler_60 = StandardScaler().fit(X_tr_60)
        X_tr_z_60 = scaler_60.transform(X_tr_60)

        te_rec_sc_60 = []
        for r in te_rec_60:
            r_c = dict(r)
            r_c["X"] = np.clip(scaler_60.transform(r["X"]), -20.0, 20.0)
            te_rec_sc_60.append(r_c)

        active_tr_60 = y_tr_60 > 0.0

        def fit_gamma_60(p_tr):
            if not np.any(active_tr_60):
                return 1.0
            act_p = p_tr[active_tr_60]
            denom = np.sum(act_p ** 2)
            return 1.0 if denom < 1e-6 else np.clip(np.sum(y_tr_60[active_tr_60] * act_p) / denom, 0.8, 1.5)

        # Train Base Regressor
        mlp_base_60 = MLPRegressor(hidden_layer_sizes=(32, 16), activation="relu", alpha=0.20,
                                   early_stopping=True, n_iter_no_change=10, random_state=42, max_iter=200)
        mlp_base_60.fit(X_tr_z_60, y_tr_60)
        gamma_base_60 = fit_gamma_60(np.maximum(0.0, mlp_base_60.predict(X_tr_z_60)))

        # Train Gate (SMOTE)
        y_gate_bin_60 = (y_tr_60 > 0.0).astype(int)
        X_gate_tr_60, y_gate_tr_60, sampler_name = balance_flight_manifold(X_tr_z_60, y_gate_bin_60)
        gate_60 = MLPClassifier(hidden_layer_sizes=(32, 16), activation="relu", alpha=0.15,
                                early_stopping=True, n_iter_no_change=10, random_state=42, max_iter=200)
        gate_60.fit(X_gate_tr_60, y_gate_tr_60)

        # Train Pure Regressor
        reg_pure_60 = MLPRegressor(hidden_layer_sizes=(32, 16), activation="relu", alpha=0.15,
                                   early_stopping=True, n_iter_no_change=10, random_state=42, max_iter=200)
        reg_pure_60.fit(X_tr_z_60[active_tr_60], y_tr_60[active_tr_60])
        gamma_reg_60 = fit_gamma_60(np.maximum(0.0, reg_pure_60.predict(X_tr_z_60)))

        # Log Diagnostics
        df_trap_m6 = df_trap[df_trap["movement"] == 6]
        valid_durations = df_trap_m6["t2_plat_start_s"] - df_trap_m6["t1_takeoff_s"]
        mean_dur_s = valid_durations[valid_durations > 0].mean() if len(valid_durations) > 0 else 0.0
        n_fl = int(np.sum(active_tr_60))
        n_rst = int(np.sum(~active_tr_60))
        diag_line = (
            f" S{sub_id:<4} | {len(feat_cols):<5} | {mean_dur_s:6.3f} s        | "
            f"{n_fl:<12} | {n_rst:<13} | {(n_rst/max(1, n_fl)):5.1f}:1      | "
            f"{len(y_gate_tr_60):<19} | {sampler_name}\n"
        )
        log_print(diag_line, fh)
        diagnostic_summary.append({"flight": n_fl, "rest": n_rst, "ratio": n_rst / max(1, n_fl), "dur": mean_dur_s})

        # Evaluate Condition A Models (tau = 60 ms)
        def pred_lag60_base(X):
            return np.maximum(0.0, mlp_base_60.predict(X)) * gamma_base_60

        def pred_lag60_hg50(X):
            p = gate_60.predict_proba(X)[:, 1] if gate_60.classes_.shape[0] > 1 else np.zeros(len(X))
            return np.where(p >= 0.50, np.maximum(0.0, reg_pure_60.predict(X)) * gamma_reg_60, 0.0)

        def pred_lag60_fs50(X):
            p = gate_60.predict_proba(X)[:, 1] if gate_60.classes_.shape[0] > 1 else np.zeros(len(X))
            mag = np.maximum(0.0, reg_pure_60.predict(X)) * gamma_reg_60
            valid = (p >= 0.50) & ~((X[:, edc_tap0_idx] > X[:, fds_tap0_idx] + 0.8) & (X[:, edc_tap0_idx] > 1.0))
            return np.where(valid, mag, 0.0)

        def pred_lag60_hg42(X):
            p = gate_60.predict_proba(X)[:, 1] if gate_60.classes_.shape[0] > 1 else np.zeros(len(X))
            return np.where(p >= 0.42, np.maximum(0.0, reg_pure_60.predict(X)) * gamma_reg_60, 0.0)

        def pred_lag60_fs42(X):
            p = gate_60.predict_proba(X)[:, 1] if gate_60.classes_.shape[0] > 1 else np.zeros(len(X))
            mag = np.maximum(0.0, reg_pure_60.predict(X)) * gamma_reg_60
            valid = (p >= 0.42) & ~((X[:, edc_tap0_idx] > X[:, fds_tap0_idx] + 0.8) & (X[:, edc_tap0_idx] > 1.0))
            return np.where(valid, mag, 0.0)

        for m_name, fn in [("Lag60_Base_P50", pred_lag60_base),
                           ("Lag60_HardGated_P50", pred_lag60_hg50),
                           ("Lag60_FullSafe_P50", pred_lag60_fs50),
                           ("Lag60_HardGated_P42", pred_lag60_hg42),
                           ("Lag60_FullSafe_P42", pred_lag60_fs42)]:
            t0 = time.perf_counter()
            _ = fn(te_rec_sc_60[0]["X"])
            inference_times_us[m_name].append(((time.perf_counter() - t0) / len(te_rec_sc_60[0]["X"])) * 1e6)
            benchmark_results[m_name].append(evaluate_neural_pipeline(te_rec_sc_60, fn, tau_lag_ms=60))

        # ======================================================================================
        # PIPELINE B: tau = 0 ms Contemporaneous Zero-Lag (200 ms Deep History Taps)
        # Taps: [0, 20, 40, 60, 100, 150, 200 ms] -> 7 taps x 30 feats = 210D input vector
        # ======================================================================================
        df_labeled_0 = construct_unipolar_labels_from_log(df_raw, df_trap, post_takeoff_sec=0.000)
        tr_mask_0, te_mask_0 = partition_timeline(df_labeled_0, sub_id)

        taps_200 = [0, 2, 4, 6, 10, 15, 20]
        tr_rec_0 = extract_speed_data(df_labeled_0[tr_mask_0], feat_cols, taps_200, lag_hops=0)
        te_rec_0 = extract_speed_data(df_labeled_0[te_mask_0], feat_cols, taps_200, lag_hops=0)

        X_tr_0 = np.vstack([r["X"] for r in tr_rec_0])
        y_tr_0 = np.concatenate([r["y"] for r in tr_rec_0])
        scaler_0 = StandardScaler().fit(X_tr_0)
        X_tr_z_0 = scaler_0.transform(X_tr_0)

        te_rec_sc_0 = []
        for r in te_rec_0:
            r_c = dict(r)
            r_c["X"] = np.clip(scaler_0.transform(r["X"]), -20.0, 20.0)
            te_rec_sc_0.append(r_c)

        active_tr_0 = y_tr_0 > 0.0

        def fit_gamma_0(p_tr):
            if not np.any(active_tr_0):
                return 1.0
            act_p = p_tr[active_tr_0]
            denom = np.sum(act_p ** 2)
            return 1.0 if denom < 1e-6 else np.clip(np.sum(y_tr_0[active_tr_0] * act_p) / denom, 0.8, 1.5)

        # Train Contemporaneous Base Regressor
        mlp_base_0 = MLPRegressor(hidden_layer_sizes=(32, 16), activation="relu", alpha=0.20,
                                  early_stopping=True, n_iter_no_change=10, random_state=42, max_iter=200)
        mlp_base_0.fit(X_tr_z_0, y_tr_0)
        gamma_base_0 = fit_gamma_0(np.maximum(0.0, mlp_base_0.predict(X_tr_z_0)))

        # Train Contemporaneous Gate (SMOTE)
        y_gate_bin_0 = (y_tr_0 > 0.0).astype(int)
        X_gate_tr_0, y_gate_tr_0, _ = balance_flight_manifold(X_tr_z_0, y_gate_bin_0)
        gate_0 = MLPClassifier(hidden_layer_sizes=(32, 16), activation="relu", alpha=0.15,
                               early_stopping=True, n_iter_no_change=10, random_state=42, max_iter=200)
        gate_0.fit(X_gate_tr_0, y_gate_tr_0)

        # Train Contemporaneous Pure Regressor
        reg_pure_0 = MLPRegressor(hidden_layer_sizes=(32, 16), activation="relu", alpha=0.15,
                                  early_stopping=True, n_iter_no_change=10, random_state=42, max_iter=200)
        reg_pure_0.fit(X_tr_z_0[active_tr_0], y_tr_0[active_tr_0])
        gamma_reg_0 = fit_gamma_0(np.maximum(0.0, reg_pure_0.predict(X_tr_z_0)))

        # Evaluate Condition B Models (tau = 0 ms)
        def pred_zero_base(X):
            return np.maximum(0.0, mlp_base_0.predict(X)) * gamma_base_0

        def pred_zero_hg50(X):
            p = gate_0.predict_proba(X)[:, 1] if gate_0.classes_.shape[0] > 1 else np.zeros(len(X))
            return np.where(p >= 0.50, np.maximum(0.0, reg_pure_0.predict(X)) * gamma_reg_0, 0.0)

        def pred_zero_fs50(X):
            p = gate_0.predict_proba(X)[:, 1] if gate_0.classes_.shape[0] > 1 else np.zeros(len(X))
            mag = np.maximum(0.0, reg_pure_0.predict(X)) * gamma_reg_0
            valid = (p >= 0.50) & ~((X[:, edc_tap0_idx] > X[:, fds_tap0_idx] + 0.8) & (X[:, edc_tap0_idx] > 1.0))
            return np.where(valid, mag, 0.0)

        def pred_zero_hg42(X):
            p = gate_0.predict_proba(X)[:, 1] if gate_0.classes_.shape[0] > 1 else np.zeros(len(X))
            return np.where(p >= 0.42, np.maximum(0.0, reg_pure_0.predict(X)) * gamma_reg_0, 0.0)

        def pred_zero_fs42(X):
            p = gate_0.predict_proba(X)[:, 1] if gate_0.classes_.shape[0] > 1 else np.zeros(len(X))
            mag = np.maximum(0.0, reg_pure_0.predict(X)) * gamma_reg_0
            valid = (p >= 0.42) & ~((X[:, edc_tap0_idx] > X[:, fds_tap0_idx] + 0.8) & (X[:, edc_tap0_idx] > 1.0))
            return np.where(valid, mag, 0.0)

        for m_name, fn in [("ZeroLag_Base_P50", pred_zero_base),
                           ("ZeroLag_HardGated_P50", pred_zero_hg50),
                           ("ZeroLag_FullSafe_P50", pred_zero_fs50),
                           ("ZeroLag_HardGated_P42", pred_zero_hg42),
                           ("ZeroLag_FullSafe_P42", pred_zero_fs42)]:
            t0 = time.perf_counter()
            _ = fn(te_rec_sc_0[0]["X"])
            inference_times_us[m_name].append(((time.perf_counter() - t0) / len(te_rec_sc_0[0]["X"])) * 1e6)
            benchmark_results[m_name].append(evaluate_neural_pipeline(te_rec_sc_0, fn, tau_lag_ms=0))

    # Diagnostics Footer
    if diagnostic_summary:
        m_fl = np.mean([s["flight"] for s in diagnostic_summary])
        m_rst = np.mean([s["rest"] for s in diagnostic_summary])
        m_rat = np.mean([s["ratio"] for s in diagnostic_summary])
        m_dur = np.mean([s["dur"] for s in diagnostic_summary])
        diag_footer = (
            "------------------------------------------------------------------------------------------------------------------------\n"
            f" COHORT MEAN | 30    | {m_dur:6.3f} s        | {m_fl:<12.1f} | {m_rst:<13.1f} | {m_rat:5.1f}:1      |\n"
            "------------------------------------------------------------------------------------------------------------------------\n\n"
        )
        log_print(diag_footer, fh)

    # Output Report Table
    table_header = (
        "========================================================================================================================\n"
        "           ADVANCED NEURAL ARCHITECTURES: THRESHOLD & ZERO-LAG ABLATION (10 SUBJECTS MEAN +/- STD)                       \n"
        "========================================================================================================================\n"
        " Architecture         | Mean Speed% | Peak Speed% | Vel Corr (r)| Onset Lat (ms) | M5 False % | Rest Jitter | Cost (μs)\n"
        "------------------------------------------------------------------------------------------------------------------------\n"
    )
    log_print(table_header, fh)

    for m in eval_models:
        res_list = benchmark_results[m]
        mean_sp = [r["mean_speed_pct"] for r in res_list]
        peak_sp = [r["peak_speed_pct"] for r in res_list]
        r_vel   = [r["r_vel"] for r in res_list]
        lat_ms  = [r["onset_lat_ms"] for r in res_list]
        m5_fa   = [r["m5_fa"] for r in res_list]
        rj      = [r["rj"] for r in res_list]
        us_cost = inference_times_us[m]

        line = (
            f" {m:<20} | {np.mean(mean_sp):5.1f} +/- {np.std(mean_sp):4.1f}% | "
            f"{np.mean(peak_sp):5.1f} +/- {np.std(peak_sp):4.1f}% |  "
            f"{np.mean(r_vel):5.3f} +/- {np.std(r_vel):4.3f} |  "
            f"{np.mean(lat_ms):5.1f} +/- {np.std(lat_ms):4.1f} ms |  "
            f"{np.mean(m5_fa):4.2f}% |   {np.mean(rj):4.2f}% | "
            f"{np.mean(us_cost):6.1f} μs\n"
        )
        log_print(line, fh)

    log_print("========================================================================================================================\n", fh)
    print(f"\n[SUCCESS] Benchmark complete! Report written to: {LOG_FILE}\n")
    fh.close()

if __name__ == "__main__":
    main()

           ADVANCED NEURAL ARCHITECTURES: THRESHOLD & ZERO-LAG ABLATION SUITE (10 SUBJECTS)                             
 Cohort      : 10 Subjects | Exclusions: S07 & S11 | S05 Polarity Swapped
 Milestones  : Ground-Truth t1_takeoff_s and t2_plat_start_s loaded directly from kinematic_trapezoid_log
 Rebalancing : 100% of Rest/M5 Retained + Manifold SMOTE Trajectory Synthesis on Voluntary Flight
 Focus Goals : Speed Match ~ 100% | Latency < 80 ms | M5 False Closing < 5% | Rest Jitter < 5%

------------------------------------------------------------------------------------------------------------------------
 DATASET & CLASS DISTRIBUTION DIAGNOSTICS (EMPIRICALLY VERIFIED PER SUBJECT)
------------------------------------------------------------------------------------------------------------------------
 SubID | Feats | M6 Takeoffs (s) | Train Flight | Train Rest/M5 | Actual Ratio | Gating Balanced Set | Sampler Engine
--------------------------------------------------------------------

c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


 S2    | 30    |  1.292 s        | 513          | 6656          |  13.0:1      | 13312               | imbalanced-learn SMOTE (k=5)


c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


 S3    | 30    |  0.770 s        | 201          | 6076          |  30.2:1      | 12152               | imbalanced-learn SMOTE (k=5)


c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


... (6 outputs left out)

 S10   | 30    |  0.904 s        | 379          | 6609          |  17.4:1      | 13218               | imbalanced-learn SMOTE (k=5)


c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


 S12   | 30    |  0.564 s        | 218          | 6074          |  27.9:1      | 12148               | imbalanced-learn SMOTE (k=5)
------------------------------------------------------------------------------------------------------------------------
 COHORT MEAN | 30    |  0.843 s        | 331.2        | 6422.9        |  21.3:1      |
------------------------------------------------------------------------------------------------------------------------

           ADVANCED NEURAL ARCHITECTURES: THRESHOLD & ZERO-LAG ABLATION (10 SUBJECTS MEAN +/- STD)                       
 Architecture         | Mean Speed% | Peak Speed% | Vel Corr (r)| Onset Lat (ms) | M5 False % | Rest Jitter | Cost (μs)
------------------------------------------------------------------------------------------------------------------------
 Lag60_Base_P50       |  63.2 +/- 22.2% | 109.0 +/- 44.0% |  0.022 +/- 0.241 |  163.5 +/- 125.1 ms |  11.13% |   16.86% |    1.7 μs
 Lag60_HardGated_P50  | 104.3 +/- 46.4% |  

c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


In [1]:
#!/usr/bin/env python3
"""
====================================================================================================
CONTINUOUS sEMG UNIPOLAR FLIGHT: ENHANCED STM32 EDGE ARCHITECTURES & ABLATION
Milestones  : Read directly from S{sub_id}_E1_A1_kinematic_trapezoid_log.csv (t1_takeoff_s, t2_plat_start_s)
Cohort      : Subjects 1 to 12 (Excluding S07 & S11; Inverting S05)
Cadence     : 10 ms hop (100 Hz cadence)
Target Lag  : tau = 0 ms (Contemporaneous Zero-Lag with 200 ms Causal Lookback)
Threshold   : P >= 0.42 (Early Motor Unit Recruitment Trigger)

Ablation Models:
  1. ZeroLag_FullSafe_P42      : Baseline production reference (from Report 4)
  2. Rich_NeuromechBrake       : Continuous sigmoidal reciprocal inhibition (replaces discrete abort)
  3. Rich_DynamicBaseline      : Drift-referenced online Exponential Moving Average (EMA) noise tracker
  4. Rich_EnsembleTriad        : 3-Model diverse neural ensemble averaging probabilities & velocities
  5. Rich_TemporalFilterBank   : Multi-scale causal recruitment velocity (d/dt) & acceleration (d2/dt2)
  6. Rich_AllIntegrated        : Full combination of all 4 MCU enhancements
====================================================================================================
"""

import os
import sys
import time
import numpy as np
import pandas as pd
from scipy import signal, stats
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.neural_network import MLPRegressor, MLPClassifier

try:
    from imblearn.over_sampling import SMOTE
    HAS_IMBLEARN = True
except ImportError:
    HAS_IMBLEARN = False

# --- 1. Cohort & Protocol Constants ---
COHORT = [1, 2, 3, 4, 5, 6, 8, 9, 10, 12]  # S07 & S11 permanently excluded
V_DEADBAND = 4.0        # deg/s: Actuator deadband floor
V_ONSET_THRESH = 12.0   # deg/s: Velocity threshold to register voluntary takeoff
DT_SEC = 0.010          # 10 ms hop (100 Hz cadence)
TIMEOUT_LATENCY_MS = 350.0  # Penalty latency for failed onset trigger
LOG_FILE = "cohort_advanced_neural_report.txt"

INVERT_S05_IN_PYTHON = True

# --- Fast NumPy MLP Engine for Microsecond MCU Simulation ---
class FastMLP:
    def __init__(self, sklearn_mlp, is_classifier=False):
        self.weights = [w.copy() for w in sklearn_mlp.coefs_]
        self.biases = [b.copy() for b in sklearn_mlp.intercepts_]
        self.is_classifier = is_classifier

    def forward_1d(self, x):
        h = x
        for i in range(len(self.weights) - 1):
            h = np.maximum(0.0, np.dot(h, self.weights[i]) + self.biases[i])
        out = np.dot(h, self.weights[-1]) + self.biases[-1]
        if self.is_classifier:
            return 1.0 / (1.0 + np.exp(-np.clip(out[0], -25.0, 25.0)))
        return out[0]

# --- 2. Ground-Truth Milestone Unipolar Labeler ---
def construct_unipolar_labels_from_log(df_feat, df_trap, post_takeoff_sec=0.000):
    y_close = np.zeros(len(df_feat), dtype=np.float64)
    is_m6_takeoff = np.zeros(len(df_feat), dtype=bool)
    is_m6_stroke = np.zeros(len(df_feat), dtype=bool)
    is_m5_open = np.zeros(len(df_feat), dtype=bool)

    t_all = df_feat["timestamp_s"].values
    v_raw = df_feat["theta_vel_deg_s"].values
    stim = df_feat["stimulus"].values

    m6_log = df_trap[df_trap["movement"] == 6]

    for _, row in m6_log.iterrows():
        t1 = row["t1_takeoff_s"]
        t2 = row["t2_plat_start_s"]

        if t2 <= t1 or np.isnan(t1) or np.isnan(t2):
            continue

        flight_mask = (t_all >= t1) & (t_all < t2)
        flight_idx = np.where(flight_mask)[0]

        if len(flight_idx) > 0:
            is_m6_takeoff[flight_idx] = True
            y_close[flight_idx] = np.maximum(0.0, v_raw[flight_idx])

        stroke_mask = (t_all >= t1) & (t_all < (t2 + post_takeoff_sec + 1e-4))
        stroke_idx = np.where(stroke_mask)[0]
        if len(stroke_idx) > 0:
            is_m6_stroke[stroke_idx] = True

    is_m5_open[stim == 5] = True

    df_labeled = df_feat.copy()
    df_labeled["y_close"] = y_close
    df_labeled["is_m6_takeoff"] = is_m6_takeoff
    df_labeled["is_m6_stroke"] = is_m6_stroke
    df_labeled["is_m5_open"] = is_m5_open
    return df_labeled

def find_contiguous_segments(df, max_dt=0.018, min_len=24):
    t = df["timestamp_s"].values
    if len(t) == 0:
        return []
    dt = np.diff(t)
    break_indices = np.where(dt > max_dt)[0] + 1
    segments = np.split(np.arange(len(df)), break_indices)
    return [seg for seg in segments if len(seg) >= min_len]

# --- 3. Causal Feature Extractor (Standard 7-Tap vs Causal Filterbank) ---
def extract_causal_data(df, feat_cols, include_filterbank=False):
    tap_offsets = [0, 2, 4, 6, 10, 15, 20]  # [0, 20, 40, 60, 100, 150, 200 ms]
    max_hist = max(tap_offsets)

    segments = find_contiguous_segments(df, min_len=max_hist + 5)
    seg_records = []

    X_raw_all = df[feat_cols].values
    y_close_all = df["y_close"].values
    is_m6_all = df["is_m6_takeoff"].values
    is_stroke_all = df["is_m6_stroke"].values
    is_m5_all = df["is_m5_open"].values

    for seg in segments:
        X_seg = X_raw_all[seg]
        y_seg = y_close_all[seg]
        m6_seg = is_m6_all[seg]
        strk_seg = is_stroke_all[seg]
        m5_seg = is_m5_all[seg]
        L = len(seg)

        X_s, y_s, m6_s, strk_s, m5_s = [], [], [], [], []

        for i in range(max_hist, L):
            taps = [X_seg[i - k] for k in tap_offsets]
            feat_vec = np.hstack(taps)

            if include_filterbank:
                # Fast recruitment velocity: d/dt (20 ms differential)
                d_fast = X_seg[i] - X_seg[i - 2]
                # Sustained recruitment velocity: d/dt (60 ms differential)
                d_slow = X_seg[i] - X_seg[i - 6]
                # Recruitment acceleration: d2/dt2
                d2 = X_seg[i] - 2 * X_seg[i - 2] + X_seg[i - 4]
                # Leaky integrator envelope: FIR lowpass
                fir_env = 0.5 * X_seg[i] + 0.3 * X_seg[i - 2] + 0.2 * X_seg[i - 4]

                feat_vec = np.hstack([feat_vec, d_fast, d_slow, d2, fir_env])

            X_s.append(feat_vec)
            y_s.append(y_seg[i])
            m6_s.append(m6_seg[i])
            strk_s.append(strk_seg[i])
            m5_s.append(m5_seg[i])

        if len(X_s) > 0:
            seg_records.append({
                "X": np.array(X_s),
                "y": np.array(y_s),
                "is_m6_takeoff": np.array(m6_s),
                "is_m6_stroke": np.array(strk_s),
                "is_m5_open": np.array(m5_s)
            })

    return seg_records

# --- 4. Timeline Partitioning ---
def partition_timeline(df, sub_id, buffer_sec=1.0):
    test_reps = [2, 5] if sub_id == 4 else [2, 6]
    is_test = np.zeros(len(df), dtype=bool)

    for m in [5, 6]:
        for r in test_reps:
            cue_mask = (df["stimulus"] == m) & (df["repetition"] == r)
            if not np.any(cue_mask):
                continue
            t_start = df.loc[cue_mask, "timestamp_s"].min() - buffer_sec
            t_end = df.loc[cue_mask, "timestamp_s"].max() + buffer_sec
            is_test[(df["timestamp_s"] >= t_start) & (df["timestamp_s"] <= t_end)] = True

    is_corrupt = (df["repetition"] == 6).values if sub_id == 4 else np.zeros(len(df), dtype=bool)
    is_plateau = (df["is_hold_plateau"] == 1).values

    train_mask = ~is_corrupt & ~is_plateau & ~is_test
    test_mask = ~is_corrupt & (df["is_m6_stroke"] | df["is_m5_open"] | ~is_plateau) & is_test
    return train_mask, test_mask

# --- 5. Robust Trajectory Oversampler (SMOTE) ---
def balance_flight_manifold(X_z, y_bin, random_state=42):
    n_flight = int(np.sum(y_bin == 1))
    n_rest = int(np.sum(y_bin == 0))

    if n_flight <= 5 or n_rest <= n_flight:
        return X_z, y_bin, "Pass-Through"

    if HAS_IMBLEARN:
        k_neighbors = min(5, n_flight - 1)
        smote = SMOTE(sampling_strategy=1.0, k_neighbors=k_neighbors, random_state=random_state)
        X_bal, y_bal = smote.fit_resample(X_z, y_bin)
        return X_bal, y_bal, f"imbalanced-learn SMOTE (k={k_neighbors})"

    np.random.seed(random_state)
    flight_idx = np.where(y_bin == 1)[0]
    rest_idx = np.where(y_bin == 0)[0]
    n_needed = n_rest - n_flight

    k_nn = min(5, len(flight_idx) - 1)
    nn = NearestNeighbors(n_neighbors=k_nn + 1).fit(X_z[flight_idx])
    _, indices = nn.kneighbors(X_z[flight_idx])

    base_choices = np.random.choice(len(flight_idx), size=n_needed, replace=True)
    neighbor_choices = np.random.randint(1, k_nn + 1, size=n_needed)
    selected_neighbors = indices[base_choices, neighbor_choices]

    alphas = np.random.uniform(0.05, 0.95, size=(n_needed, 1))
    X_base = X_z[flight_idx[base_choices]]
    X_neighbor = X_z[flight_idx[selected_neighbors]]

    X_synthetic = X_base + alphas * (X_neighbor - X_base)
    X_synthetic += np.random.normal(0, 0.02, size=X_synthetic.shape)

    X_bal = np.vstack([X_z[rest_idx], X_z[flight_idx], X_synthetic])
    y_bal = np.concatenate([np.zeros(len(rest_idx), dtype=int), np.ones(n_flight + n_needed, dtype=int)])
    shuffle_idx = np.random.permutation(len(y_bal))
    return X_bal[shuffle_idx], y_bal[shuffle_idx], f"Manifold-SMOTE Fallback (k={k_nn})"

# --- 6. Flight Deadband ---
def apply_flight_deadband(y_raw, v_db=V_DEADBAND):
    y = np.maximum(0.0, np.copy(y_raw))
    mask = y > v_db
    y_cmd = np.zeros_like(y)
    y_cmd[mask] = y[mask] * ((y[mask] - v_db) / y[mask])
    return y_cmd

# --- 7. Speed-Centric Evaluator ---
def evaluate_neural_pipeline(test_records, predict_fn, dt=DT_SEC):
    m6_y_true, m6_y_cmd = [], []
    m5_cmds, rest_cmds = [], []
    mean_speed_ratios = []
    peak_speed_ratios = []
    onset_latencies_ms = []

    for seg in test_records:
        y_pred = predict_fn(seg["X"])
        y_cmd = apply_flight_deadband(y_pred)

        y_seg = seg["y"]
        m6_tk = seg["is_m6_takeoff"]
        m6_strk = seg["is_m6_stroke"]
        m5_mask = seg["is_m5_open"]
        rest_mask = (~m6_strk) & (~m5_mask)

        if np.any(m6_tk):
            idx_tk = np.where(m6_tk)[0]
            yt_tk = y_seg[idx_tk]
            yp_tk = y_cmd[idx_tk]

            m6_y_true.extend(yt_tk)
            m6_y_cmd.extend(yp_tk)

            peak_t = np.max(yt_tk)
            peak_p = np.max(yp_tk)
            if peak_t > 15.0:
                peak_speed_ratios.append((peak_p / peak_t) * 100.0)

        if np.any(m6_strk):
            idx_strk = np.where(m6_strk)[0]
            yt_strk = y_seg[idx_strk]
            yp_strk = y_cmd[idx_strk]

            active_human = yt_strk > V_DEADBAND
            active_robot = yp_strk > V_DEADBAND

            if np.sum(active_human) > 5 and np.sum(active_robot) > 3:
                mean_t = np.mean(yt_strk[active_human])
                mean_p = np.mean(yp_strk[active_robot])
                mean_speed_ratios.append((mean_p / mean_t) * 100.0)
            elif np.sum(active_human) > 5 and np.sum(active_robot) <= 3:
                mean_speed_ratios.append(0.0)

            onset_t_rel = np.where(yt_strk >= V_ONSET_THRESH)[0]
            onset_p_rel = np.where(yp_strk >= V_ONSET_THRESH)[0]
            if len(onset_t_rel) > 0:
                if len(onset_p_rel) > 0:
                    action_lat = (onset_p_rel[0] - onset_t_rel[0]) * dt * 1000.0
                    onset_latencies_ms.append(action_lat)
                else:
                    onset_latencies_ms.append(TIMEOUT_LATENCY_MS)

        if np.any(m5_mask):
            m5_cmds.extend(y_cmd[m5_mask])

        if np.any(rest_mask):
            rest_cmds.extend(y_cmd[rest_mask])

    yt_arr = np.array(m6_y_true, dtype=np.float64)
    yp_arr = np.array(m6_y_cmd, dtype=np.float64)

    if len(yt_arr) > 10 and np.std(yp_arr) > 0 and np.std(yt_arr) > 0:
        r_vel, _ = stats.pearsonr(yt_arr, yp_arr)
    else:
        r_vel = 0.0

    m5_cmds = np.array(m5_cmds)
    m5_fa = (np.sum(m5_cmds > 0.0) / len(m5_cmds) * 100.0) if len(m5_cmds) > 0 else 0.0

    rest_cmds = np.array(rest_cmds)
    rj = (np.sum(rest_cmds > 0.0) / len(rest_cmds) * 100.0) if len(rest_cmds) > 0 else 0.0

    return {
        "mean_speed_pct": np.mean(mean_speed_ratios) if mean_speed_ratios else 0.0,
        "peak_speed_pct": np.mean(peak_speed_ratios) if peak_speed_ratios else 0.0,
        "r_vel": r_vel,
        "onset_lat_ms": np.mean(onset_latencies_ms) if onset_latencies_ms else TIMEOUT_LATENCY_MS,
        "m5_fa": m5_fa,
        "rj": rj
    }

def log_print(msg, fh):
    print(msg, end="")
    fh.write(msg)

# --- 8. Main Benchmark Execution ---
def main():
    fh = open(LOG_FILE, "w", encoding="utf-8")

    header = (
        "========================================================================================================================\n"
        "           ADVANCED NEURAL ARCHITECTURES: STM32 EDGE ENHANCEMENTS ABLATION SUITE (10 SUBJECTS)                           \n"
        "========================================================================================================================\n"
        f" Cohort      : {len(COHORT)} Subjects | Exclusions: S07 & S11 | S05 Polarity Swapped\n"
        " Milestones  : Ground-Truth t1_takeoff_s and t2_plat_start_s loaded directly from kinematic_trapezoid_log\n"
        " Pipeline    : Contemporaneous Zero-Lag (tau = 0 ms, 200 ms Lookback) | Calibrated Cutoff P >= 0.42\n"
        " Focus Goals : Speed Match ~ 100% | Latency < 80 ms | M5 False Closing < 5% | Rest Jitter < 5%\n"
        "========================================================================================================================\n\n"
    )
    log_print(header, fh)

    raw_subjects = {}
    for sub_id in COHORT:
        feat_path = os.path.join("datasets", "DB2", f"DB2_s{sub_id}", f"S{sub_id}_E1_A1_dynamic_velocity_features.csv")
        trap_path = os.path.join("datasets", "DB2", f"DB2_s{sub_id}", f"S{sub_id}_E1_A1_kinematic_trapezoid_log.csv")

        if not os.path.exists(feat_path):
            feat_path = f"S{sub_id}_E1_A1_dynamic_velocity_features.csv"
        if not os.path.exists(trap_path):
            trap_path = f"S{sub_id}_E1_A1_kinematic_trapezoid_log.csv"

        if not os.path.exists(feat_path) or not os.path.exists(trap_path):
            continue

        df_raw = pd.read_csv(feat_path)
        df_trap = pd.read_csv(trap_path)

        if sub_id == 5 and INVERT_S05_IN_PYTHON:
            edc_cols = [c for c in df_raw.columns if c.startswith("EDC_")]
            for e_col in edc_cols:
                f_col = e_col.replace("EDC_", "FDS_")
                if f_col in df_raw.columns:
                    df_raw[e_col], df_raw[f_col] = df_raw[f_col].copy(), df_raw[e_col].copy()
            for ndi_col in ["NDI_MAV", "NDI_RMS", "NDI_WL"]:
                if ndi_col in df_raw.columns:
                    df_raw[ndi_col] = -df_raw[ndi_col]

        meta_cols = ["timestamp_s", "theta_deg", "theta_vel_deg_s", "stimulus", "repetition", 
                     "is_hold_plateau", "y_close", "is_m6_takeoff", "is_m6_stroke", "is_m5_open"]
        feat_cols = [c for c in df_raw.columns if c not in meta_cols]

        raw_subjects[sub_id] = {
            "df_raw": df_raw,
            "df_trap": df_trap,
            "feat_cols": feat_cols
        }

    eval_models = [
        "ZeroLag_FullSafe_P42",    # Baseline Reference (Report 4)
        "Rich_NeuromechBrake",     # Enhancement 3: Calibrated Continuous Sigmoidal Brake
        "Rich_DynamicBaseline",    # Enhancement 4: Drift-Referenced Streaming EMA Tracker
        "Rich_EnsembleTriad",      # Enhancement 2: 3-Model Diverse Neural Ensemble
        "Rich_TemporalFilterBank", # Enhancement 1: Causal FIR Velocity (d/dt) & Accel (d2/dt2)
        "Rich_AllIntegrated"       # All 4 Enhancements Combined
    ]

    benchmark_results = {m: [] for m in eval_models}
    inference_times_us = {m: [] for m in eval_models}

    print(">>> Running Edge Architecture Benchmarks across 10 Subjects...")

    for sub_id, d in raw_subjects.items():
        df_raw = d["df_raw"]
        df_trap = d["df_trap"]
        feat_cols = d["feat_cols"]

        fds_tap0_idx = feat_cols.index("FDS_MAV")
        edc_tap0_idx = feat_cols.index("EDC_MAV")

        df_labeled = construct_unipolar_labels_from_log(df_raw, df_trap, post_takeoff_sec=0.000)
        tr_mask, te_mask = partition_timeline(df_labeled, sub_id)

        # --------------------------------------------------------------------------------------
        # Data Pipeline 1: Standard 200 ms Lookback (7 Taps, 210D)
        # --------------------------------------------------------------------------------------
        tr_rec_std = extract_causal_data(df_labeled[tr_mask], feat_cols, include_filterbank=False)
        te_rec_std = extract_causal_data(df_labeled[te_mask], feat_cols, include_filterbank=False)

        X_tr_std = np.vstack([r["X"] for r in tr_rec_std])
        y_tr_std = np.concatenate([r["y"] for r in tr_rec_std])
        scaler_std = StandardScaler().fit(X_tr_std)
        X_tr_z_std = scaler_std.transform(X_tr_std)

        te_rec_sc_std = []
        for r in te_rec_std:
            r_c = dict(r)
            r_c["X"] = np.clip(scaler_std.transform(r["X"]), -20.0, 20.0)
            te_rec_sc_std.append(r_c)

        active_tr_std = y_tr_std > 0.0
        # Reference baseline: nominal resting centroid in z-score space
        mu_rest_nominal_std = np.mean(X_tr_z_std[~active_tr_std], axis=0)

        def fit_gamma(act_y, act_p):
            denom = np.sum(act_p ** 2)
            return 1.0 if denom < 1e-6 else np.clip(np.sum(act_y * act_p) / denom, 0.8, 1.5)

        # Train Standard Gate & Regressor (Base Model 1)
        y_gate_bin_std = (y_tr_std > 0.0).astype(int)
        X_gate_tr_std, y_gate_tr_std, _ = balance_flight_manifold(X_tr_z_std, y_gate_bin_std, random_state=42)

        gate_m1 = MLPClassifier(hidden_layer_sizes=(32, 16), activation="relu", alpha=0.15,
                                early_stopping=True, n_iter_no_change=10, random_state=42, max_iter=200)
        gate_m1.fit(X_gate_tr_std, y_gate_tr_std)

        reg_m1 = MLPRegressor(hidden_layer_sizes=(32, 16), activation="relu", alpha=0.15,
                              early_stopping=True, n_iter_no_change=10, random_state=42, max_iter=200)
        reg_m1.fit(X_tr_z_std[active_tr_std], y_tr_std[active_tr_std])
        gamma_m1 = fit_gamma(y_tr_std[active_tr_std], np.maximum(0.0, reg_m1.predict(X_tr_z_std[active_tr_std])))

        fast_gate_m1 = FastMLP(gate_m1, is_classifier=True)
        fast_reg_m1 = FastMLP(reg_m1, is_classifier=False)

        # --------------------------------------------------------------------------------------
        # Model 1: Baseline Reference (ZeroLag_FullSafe_P42)
        # --------------------------------------------------------------------------------------
        def pred_baseline(X):
            p = gate_m1.predict_proba(X)[:, 1] if gate_m1.classes_.shape[0] > 1 else np.zeros(len(X))
            mag = np.maximum(0.0, reg_m1.predict(X)) * gamma_m1
            valid = (p >= 0.42) & ~((X[:, edc_tap0_idx] > X[:, fds_tap0_idx] + 0.8) & (X[:, edc_tap0_idx] > 1.0))
            return np.where(valid, mag, 0.0)

        t0 = time.perf_counter()
        _ = pred_baseline(te_rec_sc_std[0]["X"])
        inference_times_us["ZeroLag_FullSafe_P42"].append(((time.perf_counter() - t0) / len(te_rec_sc_std[0]["X"])) * 1e6)
        benchmark_results["ZeroLag_FullSafe_P42"].append(evaluate_neural_pipeline(te_rec_sc_std, pred_baseline))

        # --------------------------------------------------------------------------------------
        # Model 2: Continuous Neuromechanical Impedance Brake (Enhancement 3)
        # --------------------------------------------------------------------------------------
        def pred_neuromech(X):
            p = gate_m1.predict_proba(X)[:, 1] if gate_m1.classes_.shape[0] > 1 else np.zeros(len(X))
            mag = np.maximum(0.0, reg_m1.predict(X)) * gamma_m1
            z_edc = X[:, edc_tap0_idx]
            z_fds = X[:, fds_tap0_idx]

            diff = z_edc - z_fds
            sig_diff = 1.0 / (1.0 + np.exp(-8.0 * (diff - 0.8)))
            sig_abs = 1.0 / (1.0 + np.exp(-8.0 * (z_edc - 1.0)))
            inhibit_factor = sig_diff * sig_abs
            brake_gain = np.clip(1.0 - 1.25 * inhibit_factor, 0.0, 1.0)

            return np.where(p >= 0.42, mag * brake_gain, 0.0)

        t0 = time.perf_counter()
        _ = pred_neuromech(te_rec_sc_std[0]["X"])
        inference_times_us["Rich_NeuromechBrake"].append(((time.perf_counter() - t0) / len(te_rec_sc_std[0]["X"])) * 1e6)
        benchmark_results["Rich_NeuromechBrake"].append(evaluate_neural_pipeline(te_rec_sc_std, pred_neuromech))

        # --------------------------------------------------------------------------------------
        # Model 3: Drift-Referenced Dynamic Baseline Tracker (Enhancement 4)
        # --------------------------------------------------------------------------------------
        def pred_dynamic_baseline(X):
            out = np.zeros(len(X), dtype=np.float64)
            mu_rest_running = mu_rest_nominal_std.copy()
            alpha_adapt = 0.015

            for i in range(len(X)):
                x_curr = X[i]
                drift = mu_rest_running - mu_rest_nominal_std
                x_adapted = x_curr - drift

                p = fast_gate_m1.forward_1d(x_adapted)
                if p < 0.25:
                    mu_rest_running = (1.0 - alpha_adapt) * mu_rest_running + alpha_adapt * x_curr

                mag = max(0.0, fast_reg_m1.forward_1d(x_adapted)) * gamma_m1
                valid = (p >= 0.42) and not ((x_adapted[edc_tap0_idx] > x_adapted[fds_tap0_idx] + 0.8) and (x_adapted[edc_tap0_idx] > 1.0))
                out[i] = mag if valid else 0.0

            return out

        t0 = time.perf_counter()
        _ = pred_dynamic_baseline(te_rec_sc_std[0]["X"])
        inference_times_us["Rich_DynamicBaseline"].append(((time.perf_counter() - t0) / len(te_rec_sc_std[0]["X"])) * 1e6)
        benchmark_results["Rich_DynamicBaseline"].append(evaluate_neural_pipeline(te_rec_sc_std, pred_dynamic_baseline))

        # --------------------------------------------------------------------------------------
        # Model 4: 3-Model Diverse Neural Ensemble (Enhancement 2)
        # --------------------------------------------------------------------------------------
        gate_m2 = MLPClassifier(hidden_layer_sizes=(48, 24, 12), activation="relu", alpha=0.20,
                                early_stopping=True, n_iter_no_change=10, random_state=101, max_iter=200)
        gate_m2.fit(X_gate_tr_std, y_gate_tr_std)
        reg_m2 = MLPRegressor(hidden_layer_sizes=(48, 24), activation="relu", alpha=0.20,
                              early_stopping=True, n_iter_no_change=10, random_state=101, max_iter=200)
        reg_m2.fit(X_tr_z_std[active_tr_std], y_tr_std[active_tr_std])
        gamma_m2 = fit_gamma(y_tr_std[active_tr_std], np.maximum(0.0, reg_m2.predict(X_tr_z_std[active_tr_std])))

        gate_m3 = MLPClassifier(hidden_layer_sizes=(64, 32), activation="relu", alpha=0.25,
                                early_stopping=True, n_iter_no_change=10, random_state=7, max_iter=200)
        gate_m3.fit(X_gate_tr_std, y_gate_tr_std)
        reg_m3 = MLPRegressor(hidden_layer_sizes=(48, 16), activation="relu", alpha=0.20,
                              early_stopping=True, n_iter_no_change=10, random_state=7, max_iter=200)
        reg_m3.fit(X_tr_z_std[active_tr_std], y_tr_std[active_tr_std])
        gamma_m3 = fit_gamma(y_tr_std[active_tr_std], np.maximum(0.0, reg_m3.predict(X_tr_z_std[active_tr_std])))

        def pred_ensemble(X):
            p1 = gate_m1.predict_proba(X)[:, 1] if gate_m1.classes_.shape[0] > 1 else np.zeros(len(X))
            p2 = gate_m2.predict_proba(X)[:, 1] if gate_m2.classes_.shape[0] > 1 else np.zeros(len(X))
            p3 = gate_m3.predict_proba(X)[:, 1] if gate_m3.classes_.shape[0] > 1 else np.zeros(len(X))
            p_ens = (p1 + p2 + p3) / 3.0

            v1 = np.maximum(0.0, reg_m1.predict(X)) * gamma_m1
            v2 = np.maximum(0.0, reg_m2.predict(X)) * gamma_m2
            v3 = np.maximum(0.0, reg_m3.predict(X)) * gamma_m3
            v_ens = (v1 + v2 + v3) / 3.0

            valid = (p_ens >= 0.42) & ~((X[:, edc_tap0_idx] > X[:, fds_tap0_idx] + 0.8) & (X[:, edc_tap0_idx] > 1.0))
            return np.where(valid, v_ens, 0.0)

        t0 = time.perf_counter()
        _ = pred_ensemble(te_rec_sc_std[0]["X"])
        inference_times_us["Rich_EnsembleTriad"].append(((time.perf_counter() - t0) / len(te_rec_sc_std[0]["X"])) * 1e6)
        benchmark_results["Rich_EnsembleTriad"].append(evaluate_neural_pipeline(te_rec_sc_std, pred_ensemble))

        # --------------------------------------------------------------------------------------
        # Data Pipeline 2: Causal Temporal Filterbank (Enhancement 1: 330D Feature Space)
        # --------------------------------------------------------------------------------------
        tr_rec_fb = extract_causal_data(df_labeled[tr_mask], feat_cols, include_filterbank=True)
        te_rec_fb = extract_causal_data(df_labeled[te_mask], feat_cols, include_filterbank=True)

        X_tr_fb = np.vstack([r["X"] for r in tr_rec_fb])
        y_tr_fb = np.concatenate([r["y"] for r in tr_rec_fb])
        scaler_fb = StandardScaler().fit(X_tr_fb)
        X_tr_z_fb = scaler_fb.transform(X_tr_fb)

        te_rec_sc_fb = []
        for r in te_rec_fb:
            r_c = dict(r)
            r_c["X"] = np.clip(scaler_fb.transform(r["X"]), -20.0, 20.0)
            te_rec_sc_fb.append(r_c)

        active_tr_fb = y_tr_fb > 0.0
        mu_rest_nominal_fb = np.mean(X_tr_z_fb[~active_tr_fb], axis=0)

        y_gate_bin_fb = (y_tr_fb > 0.0).astype(int)
        X_gate_tr_fb, y_gate_tr_fb, _ = balance_flight_manifold(X_tr_z_fb, y_gate_bin_fb, random_state=42)

        gate_fb = MLPClassifier(hidden_layer_sizes=(48, 24), activation="relu", alpha=0.15,
                                early_stopping=True, n_iter_no_change=10, random_state=42, max_iter=200)
        gate_fb.fit(X_gate_tr_fb, y_gate_tr_fb)

        reg_fb = MLPRegressor(hidden_layer_sizes=(48, 24), activation="relu", alpha=0.15,
                              early_stopping=True, n_iter_no_change=10, random_state=42, max_iter=200)
        reg_fb.fit(X_tr_z_fb[active_tr_fb], y_tr_fb[active_tr_fb])
        gamma_fb = fit_gamma(y_tr_fb[active_tr_fb], np.maximum(0.0, reg_fb.predict(X_tr_z_fb[active_tr_fb])))

        fast_gate_fb = FastMLP(gate_fb, is_classifier=True)
        fast_reg_fb = FastMLP(reg_fb, is_classifier=False)

        def pred_temporal_filterbank(X):
            p = gate_fb.predict_proba(X)[:, 1] if gate_fb.classes_.shape[0] > 1 else np.zeros(len(X))
            mag = np.maximum(0.0, reg_fb.predict(X)) * gamma_fb
            valid = (p >= 0.42) & ~((X[:, edc_tap0_idx] > X[:, fds_tap0_idx] + 0.8) & (X[:, edc_tap0_idx] > 1.0))
            return np.where(valid, mag, 0.0)

        t0 = time.perf_counter()
        _ = pred_temporal_filterbank(te_rec_sc_fb[0]["X"])
        inference_times_us["Rich_TemporalFilterBank"].append(((time.perf_counter() - t0) / len(te_rec_sc_fb[0]["X"])) * 1e6)
        benchmark_results["Rich_TemporalFilterBank"].append(evaluate_neural_pipeline(te_rec_sc_fb, pred_temporal_filterbank))

        # --------------------------------------------------------------------------------------
        # Model 6: Full MCU Integrated System (All 4 Combined)
        # --------------------------------------------------------------------------------------
        gate_fb2 = MLPClassifier(hidden_layer_sizes=(64, 32), activation="relu", alpha=0.20,
                                 early_stopping=True, n_iter_no_change=10, random_state=101, max_iter=200)
        gate_fb2.fit(X_gate_tr_fb, y_gate_tr_fb)
        reg_fb2 = MLPRegressor(hidden_layer_sizes=(64, 32), activation="relu", alpha=0.20,
                               early_stopping=True, n_iter_no_change=10, random_state=101, max_iter=200)
        reg_fb2.fit(X_tr_z_fb[active_tr_fb], y_tr_fb[active_tr_fb])
        gamma_fb2 = fit_gamma(y_tr_fb[active_tr_fb], np.maximum(0.0, reg_fb2.predict(X_tr_z_fb[active_tr_fb])))

        fast_gate_fb2 = FastMLP(gate_fb2, is_classifier=True)
        fast_reg_fb2 = FastMLP(reg_fb2, is_classifier=False)

        def pred_all_integrated(X):
            out = np.zeros(len(X), dtype=np.float64)
            mu_rest_running = mu_rest_nominal_fb.copy()
            alpha_adapt = 0.015

            for i in range(len(X)):
                x_curr = X[i]
                drift = mu_rest_running - mu_rest_nominal_fb
                x_adapt = x_curr - drift

                p1 = fast_gate_fb.forward_1d(x_adapt)
                p2 = fast_gate_fb2.forward_1d(x_adapt)
                p_ens = 0.5 * (p1 + p2)

                if p_ens < 0.25:
                    mu_rest_running = (1.0 - alpha_adapt) * mu_rest_running + alpha_adapt * x_curr

                v1 = max(0.0, fast_reg_fb.forward_1d(x_adapt)) * gamma_fb
                v2 = max(0.0, fast_reg_fb2.forward_1d(x_adapt)) * gamma_fb2
                v_ens = 0.5 * (v1 + v2)

                # Continuous neuromechanical brake
                diff = x_adapt[edc_tap0_idx] - x_adapt[fds_tap0_idx]
                z_edc = x_adapt[edc_tap0_idx]
                sig_diff = 1.0 / (1.0 + np.exp(-8.0 * (diff - 0.8)))
                sig_abs = 1.0 / (1.0 + np.exp(-8.0 * (z_edc - 1.0)))
                inhibit_factor = sig_diff * sig_abs
                brake_gain = np.clip(1.0 - 1.25 * inhibit_factor, 0.0, 1.0)

                out[i] = v_ens * brake_gain if p_ens >= 0.42 else 0.0

            return out

        t0 = time.perf_counter()
        _ = pred_all_integrated(te_rec_sc_fb[0]["X"])
        inference_times_us["Rich_AllIntegrated"].append(((time.perf_counter() - t0) / len(te_rec_sc_fb[0]["X"])) * 1e6)
        benchmark_results["Rich_AllIntegrated"].append(evaluate_neural_pipeline(te_rec_sc_fb, pred_all_integrated))

    # Output Report Table
    table_header = (
        "========================================================================================================================\n"
        "           ADVANCED NEURAL ARCHITECTURES: STM32 EDGE ENHANCEMENTS REPORT (10 SUBJECTS MEAN +/- STD)                      \n"
        "========================================================================================================================\n"
        " Architecture         | Mean Speed% | Peak Speed% | Vel Corr (r)| Onset Lat (ms) | M5 False % | Rest Jitter | Cost (μs)\n"
        "------------------------------------------------------------------------------------------------------------------------\n"
    )
    log_print(table_header, fh)

    for m in eval_models:
        res_list = benchmark_results[m]
        mean_sp = [r["mean_speed_pct"] for r in res_list]
        peak_sp = [r["peak_speed_pct"] for r in res_list]
        r_vel   = [r["r_vel"] for r in res_list]
        lat_ms  = [r["onset_lat_ms"] for r in res_list]
        m5_fa   = [r["m5_fa"] for r in res_list]
        rj      = [r["rj"] for r in res_list]
        us_cost = inference_times_us[m]

        line = (
            f" {m:<20} | {np.mean(mean_sp):5.1f} +/- {np.std(mean_sp):4.1f}% | "
            f"{np.mean(peak_sp):5.1f} +/- {np.std(peak_sp):4.1f}% |  "
            f"{np.mean(r_vel):5.3f} +/- {np.std(r_vel):4.3f} |  "
            f"{np.mean(lat_ms):5.1f} +/- {np.std(lat_ms):4.1f} ms |  "
            f"{np.mean(m5_fa):4.2f}% |   {np.mean(rj):4.2f}% | "
            f"{np.mean(us_cost):6.1f} μs\n"
        )
        log_print(line, fh)

    log_print("========================================================================================================================\n", fh)
    print(f"\n[SUCCESS] Benchmark complete! Report written to: {LOG_FILE}\n")
    fh.close()

if __name__ == "__main__":
    main()

           ADVANCED NEURAL ARCHITECTURES: STM32 EDGE ENHANCEMENTS ABLATION SUITE (10 SUBJECTS)                           
 Cohort      : 10 Subjects | Exclusions: S07 & S11 | S05 Polarity Swapped
 Milestones  : Ground-Truth t1_takeoff_s and t2_plat_start_s loaded directly from kinematic_trapezoid_log
 Pipeline    : Contemporaneous Zero-Lag (tau = 0 ms, 200 ms Lookback) | Calibrated Cutoff P >= 0.42
 Focus Goals : Speed Match ~ 100% | Latency < 80 ms | M5 False Closing < 5% | Rest Jitter < 5%

>>> Running Edge Architecture Benchmarks across 10 Subjects...


c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimiza

           ADVANCED NEURAL ARCHITECTURES: STM32 EDGE ENHANCEMENTS REPORT (10 SUBJECTS MEAN +/- STD)                      
 Architecture         | Mean Speed% | Peak Speed% | Vel Corr (r)| Onset Lat (ms) | M5 False % | Rest Jitter | Cost (μs)
------------------------------------------------------------------------------------------------------------------------
 ZeroLag_FullSafe_P42 |  96.1 +/- 36.7% |  91.9 +/- 21.5% |  0.073 +/- 0.270 |   84.5 +/- 74.6 ms |  2.17% |   4.43% |    7.8 μs
 Rich_NeuromechBrake  |  93.1 +/- 33.9% |  90.5 +/- 20.0% |  0.075 +/- 0.270 |   84.0 +/- 73.9 ms |  2.18% |   4.43% |    3.9 μs
 Rich_DynamicBaseline |  65.5 +/- 38.5% |  64.5 +/- 31.0% |  0.077 +/- 0.209 |  140.0 +/- 105.9 ms |  7.90% |   11.78% |   32.7 μs
 Rich_EnsembleTriad   |  93.6 +/- 39.0% |  85.8 +/- 19.2% |  0.046 +/- 0.255 |  105.0 +/- 82.5 ms |  1.84% |   3.57% |   17.5 μs
 Rich_TemporalFilterBank |  90.9 +/- 32.8% |  86.5 +/- 25.8% |  0.058 +/- 0.199 |  100.5 +/- 82.2 ms |  1.88% |   3.53%

In [1]:
#!/usr/bin/env python3
"""
====================================================================================================
CONTINUOUS sEMG UNIPOLAR FLIGHT: SINGLE-CHANNEL (FDS ONLY) EDGE ABLATION BENCHMARK
Milestones  : Read directly from S{sub_id}_E1_A1_kinematic_trapezoid_log.csv (t1_takeoff_s, t2_plat_start_s)
Cohort      : Subjects 1 to 12 (Excluding S07 & S11; Inverting S05)
Cadence     : 10 ms hop (100 Hz cadence)
Target Lag  : tau = 0 ms (Contemporaneous Zero-Lag with 200 ms Causal Lookback)
Channel     : STRICTLY FDS (Single-Site Unipolar Flexor Control; No EDC, No NDI)

Ablation Models (Tested on FDS-Only Feature Space):
  1. FDS_ZeroLag_P50           : Standard decision threshold (P >= 0.50), no antagonist brake
  2. FDS_ZeroLag_P42           : Calibrated early-recruitment threshold (P >= 0.42), no antagonist brake
  3. FDS_DynamicBaseline       : Drift-referenced online Exponential Moving Average (EMA) noise tracker
  4. FDS_EnsembleTriad         : 3-Model diverse neural ensemble averaging on single-channel FDS
  5. FDS_TemporalFilterBank    : Multi-scale causal recruitment velocity (d/dt) & acceleration (d2/dt2)
  6. FDS_AllIntegrated         : Full combination of single-channel MCU enhancements
====================================================================================================
"""

import os
import sys
import time
import numpy as np
import pandas as pd
from scipy import signal, stats
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.neural_network import MLPRegressor, MLPClassifier

try:
    from imblearn.over_sampling import SMOTE
    HAS_IMBLEARN = True
except ImportError:
    HAS_IMBLEARN = False

# --- 1. Cohort & Protocol Constants ---
COHORT = [1, 2, 3, 4, 5, 6, 8, 9, 10, 12]  # S07 & S11 permanently excluded
V_DEADBAND = 4.0        # deg/s: Actuator deadband floor
V_ONSET_THRESH = 12.0   # deg/s: Velocity threshold to register voluntary takeoff
DT_SEC = 0.010          # 10 ms hop (100 Hz cadence)
TIMEOUT_LATENCY_MS = 350.0  # Penalty latency for failed onset trigger
LOG_FILE = "cohort_fds_only_report.txt"

INVERT_S05_IN_PYTHON = True

# --- Fast NumPy MLP Engine for Microsecond MCU Simulation ---
class FastMLP:
    def __init__(self, sklearn_mlp, is_classifier=False):
        self.weights = [w.copy() for w in sklearn_mlp.coefs_]
        self.biases = [b.copy() for b in sklearn_mlp.intercepts_]
        self.is_classifier = is_classifier

    def forward_1d(self, x):
        h = x
        for i in range(len(self.weights) - 1):
            h = np.maximum(0.0, np.dot(h, self.weights[i]) + self.biases[i])
        out = np.dot(h, self.weights[-1]) + self.biases[-1]
        if self.is_classifier:
            return 1.0 / (1.0 + np.exp(-np.clip(out[0], -25.0, 25.0)))
        return out[0]

# --- 2. Ground-Truth Milestone Unipolar Labeler ---
def construct_unipolar_labels_from_log(df_feat, df_trap, post_takeoff_sec=0.000):
    y_close = np.zeros(len(df_feat), dtype=np.float64)
    is_m6_takeoff = np.zeros(len(df_feat), dtype=bool)
    is_m6_stroke = np.zeros(len(df_feat), dtype=bool)
    is_m5_open = np.zeros(len(df_feat), dtype=bool)

    t_all = df_feat["timestamp_s"].values
    v_raw = df_feat["theta_vel_deg_s"].values
    stim = df_feat["stimulus"].values

    m6_log = df_trap[df_trap["movement"] == 6]

    for _, row in m6_log.iterrows():
        t1 = row["t1_takeoff_s"]
        t2 = row["t2_plat_start_s"]

        if t2 <= t1 or np.isnan(t1) or np.isnan(t2):
            continue

        flight_mask = (t_all >= t1) & (t_all < t2)
        flight_idx = np.where(flight_mask)[0]

        if len(flight_idx) > 0:
            is_m6_takeoff[flight_idx] = True
            y_close[flight_idx] = np.maximum(0.0, v_raw[flight_idx])

        stroke_mask = (t_all >= t1) & (t_all < (t2 + post_takeoff_sec + 1e-4))
        stroke_idx = np.where(stroke_mask)[0]
        if len(stroke_idx) > 0:
            is_m6_stroke[stroke_idx] = True

    is_m5_open[stim == 5] = True

    df_labeled = df_feat.copy()
    df_labeled["y_close"] = y_close
    df_labeled["is_m6_takeoff"] = is_m6_takeoff
    df_labeled["is_m6_stroke"] = is_m6_stroke
    df_labeled["is_m5_open"] = is_m5_open
    return df_labeled

def find_contiguous_segments(df, max_dt=0.018, min_len=24):
    t = df["timestamp_s"].values
    if len(t) == 0:
        return []
    dt = np.diff(t)
    break_indices = np.where(dt > max_dt)[0] + 1
    segments = np.split(np.arange(len(df)), break_indices)
    return [seg for seg in segments if len(seg) >= min_len]

# --- 3. Causal Feature Extractor (Standard 7-Tap vs Causal Filterbank) ---
def extract_causal_data(df, feat_cols, include_filterbank=False):
    tap_offsets = [0, 2, 4, 6, 10, 15, 20]  # [0, 20, 40, 60, 100, 150, 200 ms]
    max_hist = max(tap_offsets)

    segments = find_contiguous_segments(df, min_len=max_hist + 5)
    seg_records = []

    X_raw_all = df[feat_cols].values
    y_close_all = df["y_close"].values
    is_m6_all = df["is_m6_takeoff"].values
    is_stroke_all = df["is_m6_stroke"].values
    is_m5_all = df["is_m5_open"].values

    for seg in segments:
        X_seg = X_raw_all[seg]
        y_seg = y_close_all[seg]
        m6_seg = is_m6_all[seg]
        strk_seg = is_stroke_all[seg]
        m5_seg = is_m5_all[seg]
        L = len(seg)

        X_s, y_s, m6_s, strk_s, m5_s = [], [], [], [], []

        for i in range(max_hist, L):
            taps = [X_seg[i - k] for k in tap_offsets]
            feat_vec = np.hstack(taps)

            if include_filterbank:
                # Fast recruitment velocity: d/dt (20 ms differential)
                d_fast = X_seg[i] - X_seg[i - 2]
                # Sustained recruitment velocity: d/dt (60 ms differential)
                d_slow = X_seg[i] - X_seg[i - 6]
                # Recruitment acceleration: d2/dt2
                d2 = X_seg[i] - 2 * X_seg[i - 2] + X_seg[i - 4]
                # Leaky integrator envelope: FIR lowpass
                fir_env = 0.5 * X_seg[i] + 0.3 * X_seg[i - 2] + 0.2 * X_seg[i - 4]

                feat_vec = np.hstack([feat_vec, d_fast, d_slow, d2, fir_env])

            X_s.append(feat_vec)
            y_s.append(y_seg[i])
            m6_s.append(m6_seg[i])
            strk_s.append(strk_seg[i])
            m5_s.append(m5_seg[i])

        if len(X_s) > 0:
            seg_records.append({
                "X": np.array(X_s),
                "y": np.array(y_s),
                "is_m6_takeoff": np.array(m6_s),
                "is_m6_stroke": np.array(strk_s),
                "is_m5_open": np.array(m5_s)
            })

    return seg_records

# --- 4. Timeline Partitioning ---
def partition_timeline(df, sub_id, buffer_sec=1.0):
    test_reps = [2, 5] if sub_id == 4 else [2, 6]
    is_test = np.zeros(len(df), dtype=bool)

    for m in [5, 6]:
        for r in test_reps:
            cue_mask = (df["stimulus"] == m) & (df["repetition"] == r)
            if not np.any(cue_mask):
                continue
            t_start = df.loc[cue_mask, "timestamp_s"].min() - buffer_sec
            t_end = df.loc[cue_mask, "timestamp_s"].max() + buffer_sec
            is_test[(df["timestamp_s"] >= t_start) & (df["timestamp_s"] <= t_end)] = True

    is_corrupt = (df["repetition"] == 6).values if sub_id == 4 else np.zeros(len(df), dtype=bool)
    is_plateau = (df["is_hold_plateau"] == 1).values

    train_mask = ~is_corrupt & ~is_plateau & ~is_test
    test_mask = ~is_corrupt & (df["is_m6_stroke"] | df["is_m5_open"] | ~is_plateau) & is_test
    return train_mask, test_mask

# --- 5. Robust Trajectory Oversampler (SMOTE) ---
def balance_flight_manifold(X_z, y_bin, random_state=42):
    n_flight = int(np.sum(y_bin == 1))
    n_rest = int(np.sum(y_bin == 0))

    if n_flight <= 5 or n_rest <= n_flight:
        return X_z, y_bin, "Pass-Through"

    if HAS_IMBLEARN:
        k_neighbors = min(5, n_flight - 1)
        smote = SMOTE(sampling_strategy=1.0, k_neighbors=k_neighbors, random_state=random_state)
        X_bal, y_bal = smote.fit_resample(X_z, y_bin)
        return X_bal, y_bal, f"imbalanced-learn SMOTE (k={k_neighbors})"

    np.random.seed(random_state)
    flight_idx = np.where(y_bin == 1)[0]
    rest_idx = np.where(y_bin == 0)[0]
    n_needed = n_rest - n_flight

    k_nn = min(5, len(flight_idx) - 1)
    nn = NearestNeighbors(n_neighbors=k_nn + 1).fit(X_z[flight_idx])
    _, indices = nn.kneighbors(X_z[flight_idx])

    base_choices = np.random.choice(len(flight_idx), size=n_needed, replace=True)
    neighbor_choices = np.random.randint(1, k_nn + 1, size=n_needed)
    selected_neighbors = indices[base_choices, neighbor_choices]

    alphas = np.random.uniform(0.05, 0.95, size=(n_needed, 1))
    X_base = X_z[flight_idx[base_choices]]
    X_neighbor = X_z[flight_idx[selected_neighbors]]

    X_synthetic = X_base + alphas * (X_neighbor - X_base)
    X_synthetic += np.random.normal(0, 0.02, size=X_synthetic.shape)

    X_bal = np.vstack([X_z[rest_idx], X_z[flight_idx], X_synthetic])
    y_bal = np.concatenate([np.zeros(len(rest_idx), dtype=int), np.ones(n_flight + n_needed, dtype=int)])
    shuffle_idx = np.random.permutation(len(y_bal))
    return X_bal[shuffle_idx], y_bal[shuffle_idx], f"Manifold-SMOTE Fallback (k={k_nn})"

# --- 6. Flight Deadband ---
def apply_flight_deadband(y_raw, v_db=V_DEADBAND):
    y = np.maximum(0.0, np.copy(y_raw))
    mask = y > v_db
    y_cmd = np.zeros_like(y)
    y_cmd[mask] = y[mask] * ((y[mask] - v_db) / y[mask])
    return y_cmd

# --- 7. Speed-Centric Evaluator ---
def evaluate_neural_pipeline(test_records, predict_fn, dt=DT_SEC):
    m6_y_true, m6_y_cmd = [], []
    m5_cmds, rest_cmds = [], []
    mean_speed_ratios = []
    peak_speed_ratios = []
    onset_latencies_ms = []

    for seg in test_records:
        y_pred = predict_fn(seg["X"])
        y_cmd = apply_flight_deadband(y_pred)

        y_seg = seg["y"]
        m6_tk = seg["is_m6_takeoff"]
        m6_strk = seg["is_m6_stroke"]
        m5_mask = seg["is_m5_open"]
        rest_mask = (~m6_strk) & (~m5_mask)

        if np.any(m6_tk):
            idx_tk = np.where(m6_tk)[0]
            yt_tk = y_seg[idx_tk]
            yp_tk = y_cmd[idx_tk]

            m6_y_true.extend(yt_tk)
            m6_y_cmd.extend(yp_tk)

            peak_t = np.max(yt_tk)
            peak_p = np.max(yp_tk)
            if peak_t > 15.0:
                peak_speed_ratios.append((peak_p / peak_t) * 100.0)

        if np.any(m6_strk):
            idx_strk = np.where(m6_strk)[0]
            yt_strk = y_seg[idx_strk]
            yp_strk = y_cmd[idx_strk]

            active_human = yt_strk > V_DEADBAND
            active_robot = yp_strk > V_DEADBAND

            if np.sum(active_human) > 5 and np.sum(active_robot) > 3:
                mean_t = np.mean(yt_strk[active_human])
                mean_p = np.mean(yp_strk[active_robot])
                mean_speed_ratios.append((mean_p / mean_t) * 100.0)
            elif np.sum(active_human) > 5 and np.sum(active_robot) <= 3:
                mean_speed_ratios.append(0.0)

            onset_t_rel = np.where(yt_strk >= V_ONSET_THRESH)[0]
            onset_p_rel = np.where(yp_strk >= V_ONSET_THRESH)[0]
            if len(onset_t_rel) > 0:
                if len(onset_p_rel) > 0:
                    action_lat = (onset_p_rel[0] - onset_t_rel[0]) * dt * 1000.0
                    onset_latencies_ms.append(action_lat)
                else:
                    onset_latencies_ms.append(TIMEOUT_LATENCY_MS)

        if np.any(m5_mask):
            m5_cmds.extend(y_cmd[m5_mask])

        if np.any(rest_mask):
            rest_cmds.extend(y_cmd[rest_mask])

    yt_arr = np.array(m6_y_true, dtype=np.float64)
    yp_arr = np.array(m6_y_cmd, dtype=np.float64)

    if len(yt_arr) > 10 and np.std(yp_arr) > 0 and np.std(yt_arr) > 0:
        r_vel, _ = stats.pearsonr(yt_arr, yp_arr)
    else:
        r_vel = 0.0

    m5_cmds = np.array(m5_cmds)
    m5_fa = (np.sum(m5_cmds > 0.0) / len(m5_cmds) * 100.0) if len(m5_cmds) > 0 else 0.0

    rest_cmds = np.array(rest_cmds)
    rj = (np.sum(rest_cmds > 0.0) / len(rest_cmds) * 100.0) if len(rest_cmds) > 0 else 0.0

    return {
        "mean_speed_pct": np.mean(mean_speed_ratios) if mean_speed_ratios else 0.0,
        "peak_speed_pct": np.mean(peak_speed_ratios) if peak_speed_ratios else 0.0,
        "r_vel": r_vel,
        "onset_lat_ms": np.mean(onset_latencies_ms) if onset_latencies_ms else TIMEOUT_LATENCY_MS,
        "m5_fa": m5_fa,
        "rj": rj
    }

def log_print(msg, fh):
    print(msg, end="")
    fh.write(msg)

# --- 8. Main Benchmark Execution ---
def main():
    fh = open(LOG_FILE, "w", encoding="utf-8")

    header = (
        "========================================================================================================================\n"
        "           CONTINUOUS sEMG UNIPOLAR FLIGHT: SINGLE-CHANNEL (FDS ONLY) ABLATION SUITE (10 SUBJECTS)                       \n"
        "========================================================================================================================\n"
        f" Cohort      : {len(COHORT)} Subjects | Exclusions: S07 & S11 | S05 Polarity Swapped\n"
        " Channel     : STRICTLY FDS METRICS (No EDC Antagonist Electrode, No NDI Spatial Difference)\n"
        " Milestones  : Ground-Truth t1_takeoff_s and t2_plat_start_s loaded directly from kinematic_trapezoid_log\n"
        " Pipeline    : Contemporaneous Zero-Lag (tau = 0 ms, 200 ms Lookback)\n"
        " Core Audit  : Measures Single-Site Safety Vulnerability (M5 False Closings & Jitter without Antagonist)\n"
        "========================================================================================================================\n\n"
    )
    log_print(header, fh)

    raw_subjects = {}
    for sub_id in COHORT:
        feat_path = os.path.join("datasets", "DB2", f"DB2_s{sub_id}", f"S{sub_id}_E1_A1_dynamic_velocity_features.csv")
        trap_path = os.path.join("datasets", "DB2", f"DB2_s{sub_id}", f"S{sub_id}_E1_A1_kinematic_trapezoid_log.csv")

        if not os.path.exists(feat_path):
            feat_path = f"S{sub_id}_E1_A1_dynamic_velocity_features.csv"
        if not os.path.exists(trap_path):
            trap_path = f"S{sub_id}_E1_A1_kinematic_trapezoid_log.csv"

        if not os.path.exists(feat_path) or not os.path.exists(trap_path):
            continue

        df_raw = pd.read_csv(feat_path)
        df_trap = pd.read_csv(trap_path)

        if sub_id == 5 and INVERT_S05_IN_PYTHON:
            edc_cols = [c for c in df_raw.columns if c.startswith("EDC_")]
            for e_col in edc_cols:
                f_col = e_col.replace("EDC_", "FDS_")
                if f_col in df_raw.columns:
                    df_raw[e_col], df_raw[f_col] = df_raw[f_col].copy(), df_raw[e_col].copy()

        meta_cols = ["timestamp_s", "theta_deg", "theta_vel_deg_s", "stimulus", "repetition", 
                     "is_hold_plateau", "y_close", "is_m6_takeoff", "is_m6_stroke", "is_m5_open"]
        
        # --- STRICT SINGLE-CHANNEL ISOLATION (FDS ONLY) ---
        feat_cols = [c for c in df_raw.columns if c not in meta_cols and c.startswith("FDS_")]

        raw_subjects[sub_id] = {
            "df_raw": df_raw,
            "df_trap": df_trap,
            "feat_cols": feat_cols
        }

    eval_models = [
        "FDS_ZeroLag_P50",         # Standard 1:1 decision boundary (P >= 0.50)
        "FDS_ZeroLag_P42",         # Calibrated cutoff (P >= 0.42) without antagonist brake
        "FDS_DynamicBaseline",     # Drift-referenced streaming EMA noise tracker (P >= 0.42)
        "FDS_EnsembleTriad",       # 3-Model diverse neural ensemble on FDS metrics (P >= 0.42)
        "FDS_TemporalFilterBank",  # Multi-scale causal d/dt and d2/dt2 on FDS (P >= 0.42)
        "FDS_AllIntegrated"        # Combined FilterBank + Ensemble + DynamicBaseline (P >= 0.42)
    ]

    benchmark_results = {m: [] for m in eval_models}
    inference_times_us = {m: [] for m in eval_models}

    diag_header = (
        "------------------------------------------------------------------------------------------------------------------------\n"
        " SINGLE-CHANNEL (FDS ONLY) CLASS DISTRIBUTION & INPUT DIMENSIONS PER SUBJECT\n"
        "------------------------------------------------------------------------------------------------------------------------\n"
        " SubID | FDS Feats | 7-Tap Dim | FilterBank Dim | Train Flight | Train Rest/M5 | Actual Ratio | Gating Set Size\n"
        "------------------------------------------------------------------------------------------------------------------------\n"
    )
    log_print(diag_header, fh)

    print(">>> Running Single-Channel (FDS Only) Benchmarks across 10 Subjects...")

    for sub_id, d in raw_subjects.items():
        df_raw = d["df_raw"]
        df_trap = d["df_trap"]
        feat_cols = d["feat_cols"]

        df_labeled = construct_unipolar_labels_from_log(df_raw, df_trap, post_takeoff_sec=0.000)
        tr_mask, te_mask = partition_timeline(df_labeled, sub_id)

        # --------------------------------------------------------------------------------------
        # Data Pipeline 1: Standard 200 ms Lookback (7 Taps on FDS Only)
        # --------------------------------------------------------------------------------------
        tr_rec_std = extract_causal_data(df_labeled[tr_mask], feat_cols, include_filterbank=False)
        te_rec_std = extract_causal_data(df_labeled[te_mask], feat_cols, include_filterbank=False)

        X_tr_std = np.vstack([r["X"] for r in tr_rec_std])
        y_tr_std = np.concatenate([r["y"] for r in tr_rec_std])
        scaler_std = StandardScaler().fit(X_tr_std)
        X_tr_z_std = scaler_std.transform(X_tr_std)

        te_rec_sc_std = []
        for r in te_rec_std:
            r_c = dict(r)
            r_c["X"] = np.clip(scaler_std.transform(r["X"]), -20.0, 20.0)
            te_rec_sc_std.append(r_c)

        active_tr_std = y_tr_std > 0.0
        mu_rest_nominal_std = np.mean(X_tr_z_std[~active_tr_std], axis=0)

        def fit_gamma(act_y, act_p):
            denom = np.sum(act_p ** 2)
            return 1.0 if denom < 1e-6 else np.clip(np.sum(act_y * act_p) / denom, 0.8, 1.5)

        # Train Standard Gate & Regressor (Base Model 1)
        y_gate_bin_std = (y_tr_std > 0.0).astype(int)
        X_gate_tr_std, y_gate_tr_std, sampler_name = balance_flight_manifold(X_tr_z_std, y_gate_bin_std, random_state=42)

        gate_m1 = MLPClassifier(hidden_layer_sizes=(32, 16), activation="relu", alpha=0.15,
                                early_stopping=True, n_iter_no_change=10, random_state=42, max_iter=200)
        gate_m1.fit(X_gate_tr_std, y_gate_tr_std)

        reg_m1 = MLPRegressor(hidden_layer_sizes=(32, 16), activation="relu", alpha=0.15,
                              early_stopping=True, n_iter_no_change=10, random_state=42, max_iter=200)
        reg_m1.fit(X_tr_z_std[active_tr_std], y_tr_std[active_tr_std])
        gamma_m1 = fit_gamma(y_tr_std[active_tr_std], np.maximum(0.0, reg_m1.predict(X_tr_z_std[active_tr_std])))

        fast_gate_m1 = FastMLP(gate_m1, is_classifier=True)
        fast_reg_m1 = FastMLP(reg_m1, is_classifier=False)

        # --------------------------------------------------------------------------------------
        # Data Pipeline 2: Causal Temporal Filterbank (FDS + d/dt + d2/dt2 + FIR)
        # --------------------------------------------------------------------------------------
        tr_rec_fb = extract_causal_data(df_labeled[tr_mask], feat_cols, include_filterbank=True)
        te_rec_fb = extract_causal_data(df_labeled[te_mask], feat_cols, include_filterbank=True)

        X_tr_fb = np.vstack([r["X"] for r in tr_rec_fb])
        y_tr_fb = np.concatenate([r["y"] for r in tr_rec_fb])
        scaler_fb = StandardScaler().fit(X_tr_fb)
        X_tr_z_fb = scaler_fb.transform(X_tr_fb)

        te_rec_sc_fb = []
        for r in te_rec_fb:
            r_c = dict(r)
            r_c["X"] = np.clip(scaler_fb.transform(r["X"]), -20.0, 20.0)
            te_rec_sc_fb.append(r_c)

        active_tr_fb = y_tr_fb > 0.0
        mu_rest_nominal_fb = np.mean(X_tr_z_fb[~active_tr_fb], axis=0)

        y_gate_bin_fb = (y_tr_fb > 0.0).astype(int)
        X_gate_tr_fb, y_gate_tr_fb, _ = balance_flight_manifold(X_tr_z_fb, y_gate_bin_fb, random_state=42)

        gate_fb = MLPClassifier(hidden_layer_sizes=(48, 24), activation="relu", alpha=0.15,
                                early_stopping=True, n_iter_no_change=10, random_state=42, max_iter=200)
        gate_fb.fit(X_gate_tr_fb, y_gate_tr_fb)

        reg_fb = MLPRegressor(hidden_layer_sizes=(48, 24), activation="relu", alpha=0.15,
                              early_stopping=True, n_iter_no_change=10, random_state=42, max_iter=200)
        reg_fb.fit(X_tr_z_fb[active_tr_fb], y_tr_fb[active_tr_fb])
        gamma_fb = fit_gamma(y_tr_fb[active_tr_fb], np.maximum(0.0, reg_fb.predict(X_tr_z_fb[active_tr_fb])))

        fast_gate_fb = FastMLP(gate_fb, is_classifier=True)
        fast_reg_fb = FastMLP(reg_fb, is_classifier=False)

        # Log Diagnostics
        n_fl = int(np.sum(active_tr_std))
        n_rst = int(np.sum(~active_tr_std))
        ratio = n_rst / max(1, n_fl)
        diag_line = (
            f" S{sub_id:<4} | {len(feat_cols):<9} | {X_tr_std.shape[1]:<9} | {X_tr_fb.shape[1]:<14} | "
            f"{n_fl:<12} | {n_rst:<13} | {ratio:5.1f}:1      | {len(y_gate_tr_std):<16}\n"
        )
        log_print(diag_line, fh)

        # --------------------------------------------------------------------------------------
        # Model 1: FDS_ZeroLag_P50
        # --------------------------------------------------------------------------------------
        def pred_p50(X):
            p = gate_m1.predict_proba(X)[:, 1] if gate_m1.classes_.shape[0] > 1 else np.zeros(len(X))
            mag = np.maximum(0.0, reg_m1.predict(X)) * gamma_m1
            return np.where(p >= 0.50, mag, 0.0)

        t0 = time.perf_counter()
        _ = pred_p50(te_rec_sc_std[0]["X"])
        inference_times_us["FDS_ZeroLag_P50"].append(((time.perf_counter() - t0) / len(te_rec_sc_std[0]["X"])) * 1e6)
        benchmark_results["FDS_ZeroLag_P50"].append(evaluate_neural_pipeline(te_rec_sc_std, pred_p50))

        # --------------------------------------------------------------------------------------
        # Model 2: FDS_ZeroLag_P42 (No Antagonist Brake)
        # --------------------------------------------------------------------------------------
        def pred_p42(X):
            p = gate_m1.predict_proba(X)[:, 1] if gate_m1.classes_.shape[0] > 1 else np.zeros(len(X))
            mag = np.maximum(0.0, reg_m1.predict(X)) * gamma_m1
            return np.where(p >= 0.42, mag, 0.0)

        t0 = time.perf_counter()
        _ = pred_p42(te_rec_sc_std[0]["X"])
        inference_times_us["FDS_ZeroLag_P42"].append(((time.perf_counter() - t0) / len(te_rec_sc_std[0]["X"])) * 1e6)
        benchmark_results["FDS_ZeroLag_P42"].append(evaluate_neural_pipeline(te_rec_sc_std, pred_p42))

        # --------------------------------------------------------------------------------------
        # Model 3: FDS_DynamicBaseline (P >= 0.42)
        # --------------------------------------------------------------------------------------
        def pred_dynamic_baseline(X):
            out = np.zeros(len(X), dtype=np.float64)
            mu_rest_running = mu_rest_nominal_std.copy()
            alpha_adapt = 0.015

            for i in range(len(X)):
                x_curr = X[i]
                drift = mu_rest_running - mu_rest_nominal_std
                x_adapted = x_curr - drift

                p = fast_gate_m1.forward_1d(x_adapted)
                if p < 0.25:
                    mu_rest_running = (1.0 - alpha_adapt) * mu_rest_running + alpha_adapt * x_curr

                mag = max(0.0, fast_reg_m1.forward_1d(x_adapted)) * gamma_m1
                out[i] = mag if p >= 0.42 else 0.0

            return out

        t0 = time.perf_counter()
        _ = pred_dynamic_baseline(te_rec_sc_std[0]["X"])
        inference_times_us["FDS_DynamicBaseline"].append(((time.perf_counter() - t0) / len(te_rec_sc_std[0]["X"])) * 1e6)
        benchmark_results["FDS_DynamicBaseline"].append(evaluate_neural_pipeline(te_rec_sc_std, pred_dynamic_baseline))

        # --------------------------------------------------------------------------------------
        # Model 4: FDS_EnsembleTriad (P >= 0.42)
        # --------------------------------------------------------------------------------------
        gate_m2 = MLPClassifier(hidden_layer_sizes=(48, 24, 12), activation="relu", alpha=0.20,
                                early_stopping=True, n_iter_no_change=10, random_state=101, max_iter=200)
        gate_m2.fit(X_gate_tr_std, y_gate_tr_std)
        reg_m2 = MLPRegressor(hidden_layer_sizes=(48, 24), activation="relu", alpha=0.20,
                              early_stopping=True, n_iter_no_change=10, random_state=101, max_iter=200)
        reg_m2.fit(X_tr_z_std[active_tr_std], y_tr_std[active_tr_std])
        gamma_m2 = fit_gamma(y_tr_std[active_tr_std], np.maximum(0.0, reg_m2.predict(X_tr_z_std[active_tr_std])))

        gate_m3 = MLPClassifier(hidden_layer_sizes=(64, 32), activation="relu", alpha=0.25,
                                early_stopping=True, n_iter_no_change=10, random_state=7, max_iter=200)
        gate_m3.fit(X_gate_tr_std, y_gate_tr_std)
        reg_m3 = MLPRegressor(hidden_layer_sizes=(48, 16), activation="relu", alpha=0.20,
                              early_stopping=True, n_iter_no_change=10, random_state=7, max_iter=200)
        reg_m3.fit(X_tr_z_std[active_tr_std], y_tr_std[active_tr_std])
        gamma_m3 = fit_gamma(y_tr_std[active_tr_std], np.maximum(0.0, reg_m3.predict(X_tr_z_std[active_tr_std])))

        def pred_ensemble(X):
            p1 = gate_m1.predict_proba(X)[:, 1] if gate_m1.classes_.shape[0] > 1 else np.zeros(len(X))
            p2 = gate_m2.predict_proba(X)[:, 1] if gate_m2.classes_.shape[0] > 1 else np.zeros(len(X))
            p3 = gate_m3.predict_proba(X)[:, 1] if gate_m3.classes_.shape[0] > 1 else np.zeros(len(X))
            p_ens = (p1 + p2 + p3) / 3.0

            v1 = np.maximum(0.0, reg_m1.predict(X)) * gamma_m1
            v2 = np.maximum(0.0, reg_m2.predict(X)) * gamma_m2
            v3 = np.maximum(0.0, reg_m3.predict(X)) * gamma_m3
            v_ens = (v1 + v2 + v3) / 3.0

            return np.where(p_ens >= 0.42, v_ens, 0.0)

        t0 = time.perf_counter()
        _ = pred_ensemble(te_rec_sc_std[0]["X"])
        inference_times_us["FDS_EnsembleTriad"].append(((time.perf_counter() - t0) / len(te_rec_sc_std[0]["X"])) * 1e6)
        benchmark_results["FDS_EnsembleTriad"].append(evaluate_neural_pipeline(te_rec_sc_std, pred_ensemble))

        # --------------------------------------------------------------------------------------
        # Model 5: FDS_TemporalFilterBank (P >= 0.42)
        # --------------------------------------------------------------------------------------
        def pred_temporal_filterbank(X):
            p = gate_fb.predict_proba(X)[:, 1] if gate_fb.classes_.shape[0] > 1 else np.zeros(len(X))
            mag = np.maximum(0.0, reg_fb.predict(X)) * gamma_fb
            return np.where(p >= 0.42, mag, 0.0)

        t0 = time.perf_counter()
        _ = pred_temporal_filterbank(te_rec_sc_fb[0]["X"])
        inference_times_us["FDS_TemporalFilterBank"].append(((time.perf_counter() - t0) / len(te_rec_sc_fb[0]["X"])) * 1e6)
        benchmark_results["FDS_TemporalFilterBank"].append(evaluate_neural_pipeline(te_rec_sc_fb, pred_temporal_filterbank))

        # --------------------------------------------------------------------------------------
        # Model 6: FDS_AllIntegrated (FilterBank + Ensemble + DynamicBaseline)
        # --------------------------------------------------------------------------------------
        gate_fb2 = MLPClassifier(hidden_layer_sizes=(64, 32), activation="relu", alpha=0.20,
                                 early_stopping=True, n_iter_no_change=10, random_state=101, max_iter=200)
        gate_fb2.fit(X_gate_tr_fb, y_gate_tr_fb)
        reg_fb2 = MLPRegressor(hidden_layer_sizes=(64, 32), activation="relu", alpha=0.20,
                               early_stopping=True, n_iter_no_change=10, random_state=101, max_iter=200)
        reg_fb2.fit(X_tr_z_fb[active_tr_fb], y_tr_fb[active_tr_fb])
        gamma_fb2 = fit_gamma(y_tr_fb[active_tr_fb], np.maximum(0.0, reg_fb2.predict(X_tr_z_fb[active_tr_fb])))

        fast_gate_fb2 = FastMLP(gate_fb2, is_classifier=True)
        fast_reg_fb2 = FastMLP(reg_fb2, is_classifier=False)

        def pred_all_integrated(X):
            out = np.zeros(len(X), dtype=np.float64)
            mu_rest_running = mu_rest_nominal_fb.copy()
            alpha_adapt = 0.015

            for i in range(len(X)):
                x_curr = X[i]
                drift = mu_rest_running - mu_rest_nominal_fb
                x_adapt = x_curr - drift

                p1 = fast_gate_fb.forward_1d(x_adapt)
                p2 = fast_gate_fb2.forward_1d(x_adapt)
                p_ens = 0.5 * (p1 + p2)

                if p_ens < 0.25:
                    mu_rest_running = (1.0 - alpha_adapt) * mu_rest_running + alpha_adapt * x_curr

                v1 = max(0.0, fast_reg_fb.forward_1d(x_adapt)) * gamma_fb
                v2 = max(0.0, fast_reg_fb2.forward_1d(x_adapt)) * gamma_fb2
                v_ens = 0.5 * (v1 + v2)

                out[i] = v_ens if p_ens >= 0.42 else 0.0

            return out

        t0 = time.perf_counter()
        _ = pred_all_integrated(te_rec_sc_fb[0]["X"])
        inference_times_us["FDS_AllIntegrated"].append(((time.perf_counter() - t0) / len(te_rec_sc_fb[0]["X"])) * 1e6)
        benchmark_results["FDS_AllIntegrated"].append(evaluate_neural_pipeline(te_rec_sc_fb, pred_all_integrated))

    # Output Report Table
    table_header = (
        "\n========================================================================================================================\n"
        "           SINGLE-CHANNEL (FDS ONLY) ARCHITECTURES: BENCHMARK REPORT (10 SUBJECTS MEAN +/- STD)                          \n"
        "========================================================================================================================\n"
        " Architecture             | Mean Speed% | Peak Speed% | Vel Corr (r)| Onset Lat (ms) | M5 False % | Rest Jitter | Cost (μs)\n"
        "------------------------------------------------------------------------------------------------------------------------\n"
    )
    log_print(table_header, fh)

    for m in eval_models:
        res_list = benchmark_results[m]
        mean_sp = [r["mean_speed_pct"] for r in res_list]
        peak_sp = [r["peak_speed_pct"] for r in res_list]
        r_vel   = [r["r_vel"] for r in res_list]
        lat_ms  = [r["onset_lat_ms"] for r in res_list]
        m5_fa   = [r["m5_fa"] for r in res_list]
        rj      = [r["rj"] for r in res_list]
        us_cost = inference_times_us[m]

        line = (
            f" {m:<24} | {np.mean(mean_sp):5.1f} +/- {np.std(mean_sp):4.1f}% | "
            f"{np.mean(peak_sp):5.1f} +/- {np.std(peak_sp):4.1f}% |  "
            f"{np.mean(r_vel):5.3f} +/- {np.std(r_vel):4.3f} |  "
            f"{np.mean(lat_ms):5.1f} +/- {np.std(lat_ms):4.1f} ms |  "
            f"{np.mean(m5_fa):4.2f}% |   {np.mean(rj):4.2f}% | "
            f"{np.mean(us_cost):6.1f} μs\n"
        )
        log_print(line, fh)

    log_print("========================================================================================================================\n", fh)
    print(f"\n[SUCCESS] Benchmark complete! Report written to: {LOG_FILE}\n")
    fh.close()

if __name__ == "__main__":
    main()

           CONTINUOUS sEMG UNIPOLAR FLIGHT: SINGLE-CHANNEL (FDS ONLY) ABLATION SUITE (10 SUBJECTS)                       
 Cohort      : 10 Subjects | Exclusions: S07 & S11 | S05 Polarity Swapped
 Channel     : STRICTLY FDS METRICS (No EDC Antagonist Electrode, No NDI Spatial Difference)
 Milestones  : Ground-Truth t1_takeoff_s and t2_plat_start_s loaded directly from kinematic_trapezoid_log
 Pipeline    : Contemporaneous Zero-Lag (tau = 0 ms, 200 ms Lookback)
 Core Audit  : Measures Single-Site Safety Vulnerability (M5 False Closings & Jitter without Antagonist)

------------------------------------------------------------------------------------------------------------------------
 SINGLE-CHANNEL (FDS ONLY) CLASS DISTRIBUTION & INPUT DIMENSIONS PER SUBJECT
------------------------------------------------------------------------------------------------------------------------
 SubID | FDS Feats | 7-Tap Dim | FilterBank Dim | Train Flight | Train Rest/M5 | Actual Ratio | Gating Set Siz

c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


 S1    | 13        | 91        | 143            | 333          | 6556          |  19.7:1      | 13112           


c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


 S2    | 13        | 91        | 143            | 524          | 6463          |  12.3:1      | 12926           


c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


... (13 outputs left out)

c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimiza

 S12   | 13        | 91        | 143            | 218          | 5892          |  27.0:1      | 11784           


c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(



           SINGLE-CHANNEL (FDS ONLY) ARCHITECTURES: BENCHMARK REPORT (10 SUBJECTS MEAN +/- STD)                          
 Architecture             | Mean Speed% | Peak Speed% | Vel Corr (r)| Onset Lat (ms) | M5 False % | Rest Jitter | Cost (μs)
------------------------------------------------------------------------------------------------------------------------
 FDS_ZeroLag_P50          |  95.3 +/- 32.0% |  88.8 +/- 21.3% |  0.187 +/- 0.191 |  116.5 +/- 73.9 ms |  5.79% |   5.97% |    2.3 μs
 FDS_ZeroLag_P42          |  95.0 +/- 32.1% |  88.8 +/- 21.3% |  0.180 +/- 0.209 |  110.5 +/- 78.1 ms |  6.33% |   6.59% |    1.9 μs
 FDS_DynamicBaseline      |  78.6 +/- 32.4% |  73.9 +/- 22.1% |  0.231 +/- 0.219 |  141.5 +/- 84.9 ms |  11.59% |   8.99% |   26.3 μs
 FDS_EnsembleTriad        |  92.4 +/- 32.3% |  86.3 +/- 16.2% |  0.147 +/- 0.242 |  108.0 +/- 86.7 ms |  5.41% |   5.23% |    7.5 μs
 FDS_TemporalFilterBank   | 100.8 +/- 26.8% |  95.0 +/- 15.4% |  0.138 +/- 0.212 |   69.0 +/- 38.4 